# **Queries File**

In [ ]:
# 1. INSTALLS
%pip install -q elasticsearch requests

# 2. IMPORTS & MOUNT DRIVE
import os, json, requests
from google.colab import drive
from elasticsearch import Elasticsearch

drive.mount('/content/drive')

# 3. CONFIG (Συμπλήρωσε τα στοιχεία σου εδώ)






In [ ]:
!pip install torch==2.8.0 boto3

In [ ]:
!pip install -v -c /content/drive/MyDrive/Project/constraints.txt -r /content/drive/MyDrive/Project/requirements.txt



In [ ]:
!pip install openai elasticsearch tqdm pandas numpy

# **Elser-v1 Pipeline**

## Minimal Rewriter

In [ ]:
# ==============================================================================
# ELSER v1 — MASTER EXPERIMENT CODE (DEEPSEEK-V3.2 via AZURE FOUNDRY)
# VERSION: STATIC 6U/3A - OPTIMIZED XML-STRUCTURAL CONTEXT (DAMO-NLP STYLE)
# MULTI-CORPUS: ClapNQ, FiQA, TopiOCQA, InstructIR
# 🔥 NEW: Saves rewrites to separate JSON file for reuse
# ==============================================================================

import os, json, re, time, threading, requests
from typing import List, Dict, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from google.colab import drive

# Mount Drive
drive.mount('/content/drive', force_remount=True)

# ----------------------------
# 1. SETTINGS & CONFIG
# ----------------------------
BASE_PATH = "/content/drive/MyDrive/Project"
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]

EVAL_SCRIPT = f"{BASE_PATH}/run_retrieval_eval.py"
PRED_DIR = f"{BASE_PATH}/elser_v1_structural_6U3A_predictions"
os.makedirs(PRED_DIR, exist_ok=True)

# 🔥 MULTI-CORPUS CONFIGURATION
CORPORA = [
    {
        "name": "clapnq",
        "queries_file": f"{BASE_PATH}/retrieval/clapnq/clapnq_questions.jsonl",
        "index_name": "clapnq_elser_v1",
        "collection_id": "mt-rag-clapnq-elser-512-100-20240503"
    },
    {
        "name": "fiqa",
        "queries_file": f"{BASE_PATH}/retrieval/fiqa/fiqa_questions.jsonl",
        "index_name": "fiqa_elser_v1",
        "collection_id": "mt-rag-fiqa-beir-elser-512-100-20240501"
    },
    {
        "name": "govt",
        "queries_file": f"{BASE_PATH}/retrieval/govt/govt_questions.jsonl",
        "index_name": "govt_elser_v1",
        "collection_id": "mt-rag-govt-elser-512-100-20240611"
    },
    {
        "name": "cloud",
        "queries_file": f"{BASE_PATH}/retrieval/cloud/cloud_questions.jsonl",
        "index_name": "cloud_elser_v1",
        "collection_id": "mt-rag-ibmcloud-elser-512-100-20240502"
    }
]

# ----------------------------
# 2. COST TRACKER
# ----------------------------
class CostTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt, completion):
        with self.lock:
            self.total_input += prompt
            self.total_output += completion

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        return f"In: {self.total_input} | Out: {self.total_output} | Est. Cost: ${cost:.4f}"

tracker = CostTracker()

def llm_invoke_text(system: str, user: str, max_tokens: int) -> str:
    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-V3.2",
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "max_tokens": max_tokens,
        "temperature": 0.0
    }
    for attempt in range(7):
        try:
            response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
            if response.status_code == 200:
                res_json = response.json()
                usage = res_json.get('usage', {})
                tracker.add_usage(usage.get('prompt_tokens', 0), usage.get('completion_tokens', 0))
                return res_json['choices'][0]['message']['content'].strip()
            elif response.status_code == 429:
                time.sleep(5 * (2 ** attempt))
                continue
        except:
            time.sleep(5)
    return ""

# ----------------------------
# 3. PARSING & STRUCTURAL HISTORY (DAMO-NLP BASED)
# ----------------------------
def _history_to_text(history, user_max=6, assistant_max=3):
    """
    Διαμορφώνει το ιστορικό χρησιμοποιώντας XML tags και διαχωριστικά tokens.
    Αυτή η δομή επιτρέπει στο μοντέλο να επεξεργαστεί καλύτερα τις εξαρτήσεις.
    """
    output = ["<conversation_history>"]
    u_count, a_count = 0, 0

    # Διαβάζουμε το ιστορικό από το παλιότερο προς το νεότερο
    for h in reversed(history):
        if h['role'] == 'user' and u_count < user_max:
            output.append(f"  <turn role='user'>{h['text']}</turn> </s>")
            u_count += 1
        elif h['role'] == 'assistant' and a_count < assistant_max:
            output.append(f"  <turn role='assistant'>{h['text']}</turn> </s>")
            a_count += 1

    output.append("</conversation_history>")
    return "\n".join(reversed(output))

def parse_hybrid_history(text: str):
    user_re = re.compile(r"^\s*\|user\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    assistant_re = re.compile(r"^\s*\|assistant\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    history = []
    for ln in lines:
        u_m, a_m = user_re.match(ln), assistant_re.match(ln)
        if u_m:
            history.append({"role": "user", "text": u_m.group(1).strip()})
        elif a_m:
            history.append({"role": "assistant", "text": a_m.group(1).strip()})
    if not history:
        return text.strip(), []
    return history[-1]["text"], history[:-1]

def load_queries(path: str):
    qmap = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            query, hist = parse_hybrid_history(j.get("text", ""))
            qmap[j["_id"]] = {"query": query, "history": hist}
    return qmap

# ----------------------------
# 4. REWRITER (ORIGINAL PROMPT + STRUCTURAL WRAPPING)
# ----------------------------
class ELSERStructuralRewriter:
    def __init__(self):
        self.name = "elser_minimal_optimized"

    def rewrite(self, query, history):
        if not history:
            return [query]

        hist_text = _history_to_text(history, user_max=6, assistant_max=1)

        # ΤΟ ΑΡΧΙΚΟ ΣΟΥ PROMPT (Αμετάβλητο)
        system = (
            "Given the following conversation, please reword the final utterance from the user into "
            "a single utterance that does not need the history to understand the user's intent. "
            "Output in proper JSON format indicating the 'class' (standalone or non-standalone) "
            "and the 'rewritten version' of the last utterance.\n\n"
            "Rules:\n"
            "- Do not do any unnecessary rephrasing or introduction of new terms or concepts.\n"
            "- Be minimal, staying as close as possible to the shape and meaning of the last user utterance.\n"
            "- If the last user utterance is already clear and standalone, the rewritten version should be THE SAME.\n"
            "- Use information from 'assistant' turns ONLY to resolve pronouns (e.g., 'it', 'he', 'that') or clarify ambiguous references.\n\n"
            "Format: {\"class\": \"...\", \"rewritten version\": \"...\"}"
        )

        # STRUCTURAL WRAPPING
        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 400)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                return [parsed.get("rewritten version", query)]
            return [query]
        except:
            return [query]

# ----------------------------
# 5. MULTI-CORPUS PIPELINE
# ----------------------------
from elasticsearch import Elasticsearch

def run_corpus_experiment(corpus_config: dict, rewriter, es_client):
    """Run experiment for a single corpus"""

    corpus_name = corpus_config["name"]
    queries_file = corpus_config["queries_file"]
    index_name = corpus_config["index_name"]
    collection_id = corpus_config["collection_id"]

    print("\n" + "="*70)
    print(f"🎯 CORPUS: {corpus_name.upper()}")
    print("="*70)

    # Load queries
    data = load_queries(queries_file)
    items = list(data.items())
    print(f"✅ Loaded {len(items)} queries for {corpus_name}")

    # Rewrite queries
    print(f"📝 Rewriting queries with {rewriter.name}...")
    rewrites_map = {}
    with ThreadPoolExecutor(max_workers=5) as ex:
        futs = {ex.submit(rewriter.rewrite, it["query"], it["history"]): qid for qid, it in items}
        for f in tqdm(as_completed(futs), total=len(futs), desc=f"Rewriting {corpus_name}"):
            rewrites_map[futs[f]] = f.result()[0]

    # 🔥 SAVE REWRITES TO JSON FILE
    RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
    rewrites_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}_REWRITES.json")

    rewrites_data = []
    for qid, item in items:
        rewrites_data.append({
            "qid": qid,
            "original_query": item["query"],
            "history": item["history"],
            "rewritten_query": rewrites_map.get(qid, item["query"])
        })

    with open(rewrites_file, "w", encoding="utf-8") as f:
        json.dump(rewrites_data, f, indent=2, ensure_ascii=False)

    print(f"💾 Rewrites saved to: {rewrites_file}")

    # Retrieval
    print(f"🔍 Retrieving from {index_name}...")
    pred_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}.jsonl")

    with open(pred_file, "w", encoding="utf-8") as out:
        for qid, item in tqdm(items, desc=f"Retrieving {corpus_name}"):
            q_final = rewrites_map.get(qid, item["query"])
            try:
                res = es_client.search(index=index_name, body={
                    "query": {
                        "text_expansion": {
                            "text_embedding": {
                                "model_id": ".elser_model_1",
                                "model_text": q_final
                            }
                        }
                    },
                    "size": 100
                })
                contexts = [
                    {"document_id": h["_source"].get("doc_id", h["_id"]), "score": h["_score"]}
                    for h in res["hits"]["hits"]
                ]
            except Exception as e:
                print(f"⚠️ Retrieval error for {qid}: {e}")
                contexts = []

            out.write(json.dumps({
                "task_id": qid,
                "Collection": collection_id,
                "contexts": contexts
            }, ensure_ascii=False) + "\n")

    print(f"✅ Predictions saved to: {pred_file}")

    return pred_file, rewrites_file

def run_evaluation(pred_file: str):
    """Run evaluation for a prediction file"""
    print(f"\n🧪 Evaluating {os.path.basename(pred_file)}...")

    # ❌ OLD (creates double _results):
    # output_eval_file = pred_file.replace(".jsonl", "_results.jsonl")

    # ✅ NEW (correct path):
    output_eval_file = pred_file.replace(".jsonl", ".jsonl")  # Keep same name
    output_eval_base = pred_file.replace(".jsonl", "")  # Remove extension

    cmd = f"python {EVAL_SCRIPT} --input_file {pred_file} --output_file {output_eval_base}_results.jsonl"
    result = os.system(cmd)

    if result != 0:
        print(f"⚠️ Warning: Evaluation failed")
        return None

    print(f"✅ Evaluation completed")
    return output_eval_base + "_results.jsonl"  # Return correct path


def aggregate_results(eval_files: List[str]):
    """Aggregate results from all corpora"""
    all_data = []

    for eval_file in eval_files:
        if eval_file is None:
            continue

        # ✅ CORRECT PATH (single _results):
        agg_csv = eval_file.replace("_results.jsonl", "_results_aggregate.csv")

        if os.path.exists(agg_csv):
            df = pd.read_csv(agg_csv)
            df = df[df["collection"] != "all"]

            corpus_name = os.path.basename(eval_file).split("_")[0]
            df["Corpus"] = corpus_name

            all_data.append(df)
        else:
            print(f"⚠️ Aggregate CSV not found: {agg_csv}")
            # 🔍 DEBUG: List actual files
            eval_dir = os.path.dirname(eval_file)
            eval_base = os.path.basename(eval_file).replace("_results.jsonl", "")
            print(f"   Looking for files matching: {eval_base}*aggregate*")
            import glob
            matches = glob.glob(f"{eval_dir}/{eval_base}*aggregate*")
            if matches:
                print(f"   Found: {matches}")




# ----------------------------
# 6. MAIN EXECUTION
# ----------------------------
def main():
    print("\n" + "="*70)
    print("🚀 ELSER v1 STRUCTURAL REWRITER - MULTI-CORPUS EXPERIMENT")
    print("="*70)
    print(f"Strategy: 6U/3A XML-Structural Context (DAMO-NLP)")
    print(f"Corpora: {', '.join([c['name'] for c in CORPORA])}")
    print(f"Model: DeepSeek-V3.2 via Azure")
    print("="*70)

    # Initialize
    es_client = Elasticsearch(
        cloud_id=CLOUD_ID,
        basic_auth=("elastic", PASSWORD),
        request_timeout=180
    )

    rewriter = ELSERStructuralRewriter()
    eval_files = []
    rewrite_files = []  # 🔥 Track rewrite files

    # Run experiment for each corpus
    for corpus_config in CORPORA:
        try:
            pred_file, rewrites_file = run_corpus_experiment(corpus_config, rewriter, es_client)
            eval_file = run_evaluation(pred_file)
            eval_files.append(eval_file)
            rewrite_files.append(rewrites_file)
        except Exception as e:
            print(f"❌ Error processing {corpus_config['name']}: {e}")
            eval_files.append(None)
            rewrite_files.append(None)

    # Aggregate results
    aggregate_results(eval_files)

    # Cost report
    print("\n" + "="*70)
    print("💰 COST SUMMARY")
    print("="*70)
    print(tracker.get_report())
    print("="*70)

    # 🔥 PRINT REWRITE FILES
    print("\n" + "="*70)
    print("💾 SAVED REWRITE FILES")
    print("="*70)
    for rf in rewrite_files:
        if rf:
            print(f"  ✅ {rf}")
    print("="*70)

if __name__ == "__main__":
    main()


In [ ]:
# ==============================================================================
# ELSER v1 — 4 MINIMAL REWRITER VARIANTS EXPERIMENT (SAFE MODE)
# 🔧 FIXED: Evaluation runs LAST (after all predictions saved)
# ==============================================================================

import os, json, re, time, threading, requests
from typing import List, Dict, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from google.colab import drive
import warnings

# Suppress Elasticsearch deprecation warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Mount Drive
drive.mount('/content/drive', force_remount=True)

# ----------------------------
# 1. SETTINGS & CONFIG
# ----------------------------
BASE_PATH = "/content/drive/MyDrive/Project"
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]

EVAL_SCRIPT = f"{BASE_PATH}/run_retrieval_eval.py"
PRED_DIR = f"{BASE_PATH}/elser_v1_minimal_experiments"
os.makedirs(PRED_DIR, exist_ok=True)

# 🔥 SINGLE CORPUS (ClapNQ only)
CORPORA = [
    {
        "name": "clapnq",
        "queries_file": f"{BASE_PATH}/retrieval/clapnq/clapnq_questions.jsonl",
        "index_name": "clapnq_elser_v1",
        "collection_id": "mt-rag-clapnq-elser-512-100-20240503"
    },
]

# ----------------------------
# 2. COST TRACKER
# ----------------------------
class CostTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt, completion):
        with self.lock:
            self.total_input += prompt
            self.total_output += completion

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        return f"In: {self.total_input} | Out: {self.total_output} | Est. Cost: ${cost:.4f}"

tracker = CostTracker()

def llm_invoke_text(system: str, user: str, max_tokens: int) -> str:
    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-V3.2",
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "max_tokens": max_tokens,
        "temperature": 0.0
    }
    for attempt in range(7):
        try:
            response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
            if response.status_code == 200:
                res_json = response.json()
                usage = res_json.get('usage', {})
                tracker.add_usage(usage.get('prompt_tokens', 0), usage.get('completion_tokens', 0))
                return res_json['choices'][0]['message']['content'].strip()
            elif response.status_code == 429:
                time.sleep(5 * (2 ** attempt))
                continue
        except:
            time.sleep(5)
    return ""

# ----------------------------
# 3. PARSING & HISTORY HELPERS
# ----------------------------
def parse_hybrid_history(text: str):
    user_re = re.compile(r"^\s*\|user\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    assistant_re = re.compile(r"^\s*\|assistant\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    history = []
    for ln in lines:
        u_m, a_m = user_re.match(ln), assistant_re.match(ln)
        if u_m:
            history.append({"role": "user", "text": u_m.group(1).strip()})
        elif a_m:
            history.append({"role": "assistant", "text": a_m.group(1).strip()})
    if not history:
        return text.strip(), []
    return history[-1]["text"], history[:-1]

def load_queries(path: str):
    qmap = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            query, hist = parse_hybrid_history(j.get("text", ""))
            qmap[j["_id"]] = {"query": query, "history": hist}
    return qmap

# ----------------------------
# 4. REWRITER VARIANTS (4 EXPERIMENTS)
# ----------------------------

# 🔥 EXPERIMENT 1: NO XML TAGS/SEPARATORS (Plain text)
class MinimalRewriter_NoXML:
    def __init__(self):
        self.name = "minimal_no_xml"

    def rewrite(self, query, history):
        if not history:
            return [query]

        hist_lines = []
        for h in reversed(history[-6:]):
            if h['role'] == 'user':
                hist_lines.append(f"User: {h['text']}")
            elif h['role'] == 'assistant' and len([x for x in hist_lines if 'Assistant:' in x]) < 3:
                hist_lines.append(f"Assistant: {h['text']}")

        hist_text = "\n".join(reversed(hist_lines))

        system = (
            "Given the following conversation, please reword the final utterance from the user into "
            "a single utterance that does not need the history to understand the user's intent. "
            "Output in proper JSON format indicating the 'class' (standalone or non-standalone) "
            "and the 'rewritten version' of the last utterance.\n\n"
            "Rules:\n"
            "- Do not do any unnecessary rephrasing or introduction of new terms or concepts.\n"
            "- Be minimal, staying as close as possible to the shape and meaning of the last user utterance.\n"
            "- If the last user utterance is already clear and standalone, the rewritten version should be THE SAME.\n"
            "- Use information from 'assistant' turns ONLY to resolve pronouns (e.g., 'it', 'he', 'that') or clarify ambiguous references.\n\n"
            "Format: {\"class\": \"...\", \"rewritten version\": \"...\"}"
        )

        prompt_user = f"{hist_text}\n\nUser: {query}\n\nProvide JSON response:"

        try:
            txt = llm_invoke_text(system, prompt_user, 400)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                return [parsed.get("rewritten version", query)]
            return [query]
        except:
            return [query]

# 🔥 EXPERIMENT 2: MORE ASSISTANT TURNS (All available)
class MinimalRewriter_MoreAssistant:
    def __init__(self):
        self.name = "minimal_more_assistant"

    def _history_to_text(self, history):
        output = ["<conversation_history>"]
        u_count = 0

        for h in reversed(history):
            if h['role'] == 'user' and u_count < 6:
                output.append(f"  <turn role='user'>{h['text']}</turn> </s>")
                u_count += 1
            elif h['role'] == 'assistant':
                output.append(f"  <turn role='assistant'>{h['text']}</turn> </s>")

        output.append("</conversation_history>")
        return "\n".join(reversed(output))

    def rewrite(self, query, history):
        if not history:
            return [query]

        hist_text = self._history_to_text(history)

        system = (
            "Given the following conversation, please reword the final utterance from the user into "
            "a single utterance that does not need the history to understand the user's intent. "
            "Output in proper JSON format indicating the 'class' (standalone or non-standalone) "
            "and the 'rewritten version' of the last utterance.\n\n"
            "Rules:\n"
            "- Do not do any unnecessary rephrasing or introduction of new terms or concepts.\n"
            "- Be minimal, staying as close as possible to the shape and meaning of the last user utterance.\n"
            "- If the last user utterance is already clear and standalone, the rewritten version should be THE SAME.\n"
            "- Use information from 'assistant' turns ONLY to resolve pronouns (e.g., 'it', 'he', 'that') or clarify ambiguous references.\n\n"
            "Format: {\"class\": \"...\", \"rewritten version\": \"...\"}"
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 400)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                return [parsed.get("rewritten version", query)]
            return [query]
        except:
            return [query]

# 🔥 EXPERIMENT 3: FEW-SHOT EXAMPLES
class MinimalRewriter_FewShot:
    def __init__(self):
        self.name = "minimal_few_shot"

    def _history_to_text(self, history):
        output = ["<conversation_history>"]
        u_count, a_count = 0, 0

        for h in reversed(history):
            if h['role'] == 'user' and u_count < 6:
                output.append(f"  <turn role='user'>{h['text']}</turn> </s>")
                u_count += 1
            elif h['role'] == 'assistant' and a_count < 3:
                output.append(f"  <turn role='assistant'>{h['text']}</turn> </s>")
                a_count += 1

        output.append("</conversation_history>")
        return "\n".join(reversed(output))

    def rewrite(self, query, history):
        if not history:
            return [query]

        hist_text = self._history_to_text(history)

        few_shot = """
EXAMPLES OF MINIMAL REWRITING:

Example 1 (Pronoun Resolution):
History:
  User: "What is IBM Cloud?"
  Assistant: "IBM Cloud is a cloud computing platform..."
  User: "What are its main features?"
Rewritten: "What are IBM Cloud's main features?"

Example 2 (Already Standalone - NO CHANGE):
History:
  User: "Tell me about Shakespeare"
  Assistant: "Shakespeare was an English playwright..."
  User: "What historical plays did he write?"
Rewritten: "What historical plays did Shakespeare write?"

Example 3 (Clarification Reference):
History:
  User: "Arizona Cardinals schedule"
  Assistant: "Here's the 2017 schedule..."
  User: "Do they play outside the US?"
Rewritten: "Do the Arizona Cardinals play outside the US?"

Example 4 (Minimal Change):
History:
  User: "Doctor Strange powers"
  Assistant: "His powers come from mystical entities..."
  User: "Which entities?"
Rewritten: "Which mystical entities does Doctor Strange's powers come from?"
"""

        system = (
            "Given the following conversation, please reword the final utterance from the user into "
            "a single utterance that does not need the history to understand the user's intent. "
            "Output in proper JSON format indicating the 'class' (standalone or non-standalone) "
            "and the 'rewritten version' of the last utterance.\n\n"
            "Rules:\n"
            "- Do not do any unnecessary rephrasing or introduction of new terms or concepts.\n"
            "- Be minimal, staying as close as possible to the shape and meaning of the last user utterance.\n"
            "- If the last user utterance is already clear and standalone, the rewritten version should be THE SAME.\n"
            "- Use information from 'assistant' turns ONLY to resolve pronouns (e.g., 'it', 'he', 'that') or clarify ambiguous references.\n\n"
            f"{few_shot}\n"
            "Format: {\"class\": \"...\", \"rewritten version\": \"...\"}"
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 400)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                return [parsed.get("rewritten version", query)]
            return [query]
        except:
            return [query]

# 🔥 EXPERIMENT 4: STRICT STANDALONE (Explicit entity replacement)
class MinimalRewriter_StrictStandalone:
    def __init__(self):
        self.name = "minimal_strict_standalone"

    def _history_to_text(self, history):
        output = ["<conversation_history>"]
        u_count, a_count = 0, 0

        for h in reversed(history):
            if h['role'] == 'user' and u_count < 6:
                output.append(f"  <turn role='user'>{h['text']}</turn> </s>")
                u_count += 1
            elif h['role'] == 'assistant' and a_count < 3:
                output.append(f"  <turn role='assistant'>{h['text']}</turn> </s>")
                a_count += 1

        output.append("</conversation_history>")
        return "\n".join(reversed(output))

    def rewrite(self, query, history):
        if not history:
            return [query]

        hist_text = self._history_to_text(history)

        system = (
            "Given the following conversation, rewrite the final user utterance into a FULLY STANDALONE query "
            "that can be understood WITHOUT any context.\n\n"
            "STRICT RULES:\n"
            "1. ALWAYS replace pronouns (it, he, she, they, this, that) with the EXACT entity from history\n"
            "2. ALWAYS replace vague references ('the company', 'the player', 'the feature') with SPECIFIC names\n"
            "3. If the query asks about 'its features', replace with '[Entity Name] features'\n"
            "4. If the query asks 'when did he...', replace with '[Person Name] when did [Person Name]...'\n"
            "5. DO NOT add extra words - only replace pronouns/references with entities\n"
            "6. Extract entities from BOTH user and assistant turns\n\n"
            "Output JSON format: {\"class\": \"standalone\", \"rewritten version\": \"...\"}\n\n"
            "IMPORTANT: The rewritten version MUST be a complete, self-contained question that someone "
            "with NO knowledge of the conversation could understand."
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"CRITICAL: Replace ALL pronouns and vague references with explicit entities from the conversation history.\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 400)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                return [parsed.get("rewritten version", query)]
            return [query]
        except:
            return [query]

# ----------------------------
# 5. MULTI-CORPUS PIPELINE
# ----------------------------
from elasticsearch import Elasticsearch

def run_corpus_experiment(corpus_config: dict, rewriter, es_client):
    """Run experiment for a single corpus with one rewriter - ONLY PREDICTIONS"""

    corpus_name = corpus_config["name"]
    queries_file = corpus_config["queries_file"]
    index_name = corpus_config["index_name"]
    collection_id = corpus_config["collection_id"]

    print(f"\n  📊 Corpus: {corpus_name}")

    # Load queries
    data = load_queries(queries_file)
    items = list(data.items())

    # Rewrite queries
    rewrites_map = {}
    with ThreadPoolExecutor(max_workers=5) as ex:
        futs = {ex.submit(rewriter.rewrite, it["query"], it["history"]): qid for qid, it in items}
        for f in tqdm(as_completed(futs), total=len(futs), desc=f"  Rewriting {corpus_name}", leave=False):
            rewrites_map[futs[f]] = f.result()[0]

    # Save rewrites
    RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
    rewrites_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}_REWRITES.json")

    rewrites_data = []
    for qid, item in items:
        rewrites_data.append({
            "qid": qid,
            "original_query": item["query"],
            "history": item["history"],
            "rewritten_query": rewrites_map.get(qid, item["query"])
        })

    with open(rewrites_file, "w", encoding="utf-8") as f:
        json.dump(rewrites_data, f, indent=2, ensure_ascii=False)

    # Retrieval
    pred_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}.jsonl")

    with open(pred_file, "w", encoding="utf-8") as out:
        for qid, item in tqdm(items, desc=f"  Retrieving {corpus_name}", leave=False):
            q_final = rewrites_map.get(qid, item["query"])
            try:
                res = es_client.search(index=index_name, body={
                    "query": {
                        "text_expansion": {
                            "text_embedding": {
                                "model_id": ".elser_model_1",
                                "model_text": q_final
                            }
                        }
                    },
                    "size": 100
                })
                contexts = [
                    {"document_id": h["_source"].get("doc_id", h["_id"]), "score": h["_score"]}
                    for h in res["hits"]["hits"]
                ]
            except Exception as e:
                contexts = []

            out.write(json.dumps({
                "task_id": qid,
                "Collection": collection_id,
                "contexts": contexts
            }, ensure_ascii=False) + "\n")

    print(f"  ✅ Saved predictions: {pred_file}")
    print(f"  ✅ Saved rewrites: {rewrites_file}")

    return pred_file, rewrites_file

# 🔥 SEPARATE EVALUATION FUNCTION (runs AFTER all predictions saved)
def run_evaluation(pred_file: str):
    """Run evaluation for a prediction file"""
    print(f"\n  🧪 Evaluating: {os.path.basename(pred_file)}")
    output_eval_base = pred_file.replace(".jsonl", "")
    cmd = f"python {EVAL_SCRIPT} --input_file {pred_file} --output_file {output_eval_base}_results.jsonl"

    try:
        result = os.system(cmd)
        if result == 0:
            print(f"  ✅ Evaluation complete")
            return output_eval_base + "_results.jsonl"
        else:
            print(f"  ⚠️  Evaluation returned code: {result}")
            return None
    except Exception as e:
        print(f"  ❌ Evaluation error: {e}")
        return None

# 🔥 FIXED AGGREGATION FUNCTION
def aggregate_results(eval_files: List[str], rewriter_name: str):
    """Aggregate results from all corpora for one rewriter"""

    metrics_summary = {}

    for eval_file in eval_files:
        if eval_file is None:
            continue

        # Try different possible CSV files
        agg_csv = eval_file.replace("_results.jsonl", "_results_aggregate.csv")

        if not os.path.exists(agg_csv):
            # Try to find any CSV in the same directory
            import glob
            base_dir = os.path.dirname(eval_file)
            base_name = os.path.basename(eval_file).replace("_results.jsonl", "")
            possible_csvs = glob.glob(f"{base_dir}/{base_name}*.csv")
            if possible_csvs:
                agg_csv = possible_csvs[0]
            else:
                print(f"  ⚠️  No CSV found for {eval_file}")
                continue

        try:
            df = pd.read_csv(agg_csv)

            # Extract metrics (adapt based on actual CSV structure)
            for col in df.columns:
                if col.lower() not in ['collection', 'corpus', 'index']:
                    # Get the value
                    if 'collection' in df.columns:
                        # Filter for specific collection (not 'all')
                        values = df[df['collection'] != 'all'][col].values
                    else:
                        values = df[col].values

                    if len(values) > 0 and pd.api.types.is_numeric_dtype(df[col]):
                        metrics_summary[col] = float(values[0]) if len(values) == 1 else float(np.mean(values))

        except Exception as e:
            print(f"  ⚠️  Error reading CSV: {e}")
            continue

    if not metrics_summary:
        print(f"  ⚠️  No metrics found for {rewriter_name}")
        return None

    # Print summary
    print(f"\n  📊 Results for {rewriter_name}:")
    print("  " + "-"*50)
    for metric, value in sorted(metrics_summary.items()):
        print(f"  {metric:<30} {value:>10.4f}")

    return metrics_summary

# ----------------------------
# 6. MAIN EXECUTION (SAFE MODE)
# ----------------------------
def main():
    print("\n" + "="*70)
    print("🧪 MINIMAL REWRITER - 4 VARIANT EXPERIMENTS (SAFE MODE)")
    print("="*70)
    print("Experiment 1: No XML tags/separators (plain text)")
    print("Experiment 2: More assistant turns (all available)")
    print("Experiment 3: Few-shot examples")
    print("Experiment 4: Strict standalone (explicit entities)")
    print("="*70)
    print("🔒 SAFE MODE: Evaluation runs LAST (after all predictions saved)")
    print("="*70)

    # Initialize ES client
    es_client = Elasticsearch(
        cloud_id=CLOUD_ID,
        basic_auth=("elastic", PASSWORD),
        request_timeout=180
    )

    # 🔥 4 REWRITER VARIANTS
    rewriters = [
        MinimalRewriter_NoXML(),
        MinimalRewriter_MoreAssistant(),
        MinimalRewriter_FewShot(),
        MinimalRewriter_StrictStandalone()
    ]

    # 🔥 PHASE 1: GENERATE ALL PREDICTIONS (SAFE)
    print("\n" + "="*70)
    print("📝 PHASE 1: GENERATING PREDICTIONS (4 experiments)")
    print("="*70)

    all_pred_files = {}
    all_rewrite_files = {}

    for i, rewriter in enumerate(rewriters, 1):
        print(f"\n🔬 EXPERIMENT {i}/4: {rewriter.name}")
        print("-"*70)

        pred_files = []
        rewrite_files = []

        for corpus_config in CORPORA:
            try:
                pred_file, rewrites_file = run_corpus_experiment(corpus_config, rewriter, es_client)
                pred_files.append(pred_file)
                rewrite_files.append(rewrites_file)
            except Exception as e:
                print(f"  ❌ Error on {corpus_config['name']}: {e}")
                import traceback
                traceback.print_exc()
                pred_files.append(None)
                rewrite_files.append(None)

        all_pred_files[rewriter.name] = pred_files
        all_rewrite_files[rewriter.name] = rewrite_files

    # 🔥 PHASE 2: RUN ALL EVALUATIONS (SEPARATE)
    print("\n" + "="*70)
    print("🧪 PHASE 2: RUNNING EVALUATIONS")
    print("="*70)
    print("⚠️  If evaluation fails, predictions are already saved!")
    print("="*70)

    all_eval_files = {}

    for rewriter_name, pred_files in all_pred_files.items():
        print(f"\n📊 Evaluating {rewriter_name}...")
        eval_files = []

        for pred_file in pred_files:
            if pred_file:
                eval_file = run_evaluation(pred_file)
                eval_files.append(eval_file)
            else:
                eval_files.append(None)

        all_eval_files[rewriter_name] = eval_files

    # 🔥 PHASE 3: AGGREGATE RESULTS
    print("\n" + "="*70)
    print("📊 PHASE 3: AGGREGATING RESULTS")
    print("="*70)

    all_results = {}

    for rewriter_name, eval_files in all_eval_files.items():
        metrics = aggregate_results(eval_files, rewriter_name)
        all_results[rewriter_name] = metrics

    # Final comparison
    print("\n" + "="*70)
    print("📊 FINAL COMPARISON - ALL EXPERIMENTS")
    print("="*70)

    if all_results and any(all_results.values()):
        # Create comparison table
        all_metrics = set()
        for metrics in all_results.values():
            if metrics:
                all_metrics.update(metrics.keys())

        if all_metrics:
            print(f"\n{'Metric':<30} | " + " | ".join([f"{name:>10}" for name in all_results.keys()]))
            print("-" * (30 + len(all_results) * 14))

            for metric in sorted(all_metrics):
                row = f"{metric:<30} | "
                for name, metrics in all_results.items():
                    value = metrics.get(metric, 0.0) if metrics else 0.0
                    row += f"{value:>10.4f} | "
                print(row)

            # Save comparison
            comp_file = os.path.join(PRED_DIR, "EXPERIMENT_COMPARISON.json")
            with open(comp_file, 'w') as f:
                json.dump(all_results, f, indent=2)
            print(f"\n💾 Comparison saved to: {comp_file}")
        else:
            print("⚠️  No metrics found across experiments")
    else:
        print("⚠️  No evaluation results available")

    # Summary of saved files
    print("\n" + "="*70)
    print("💾 SAVED FILES SUMMARY")
    print("="*70)

    for rewriter_name in all_pred_files.keys():
        print(f"\n{rewriter_name}:")
        pred_files = all_pred_files.get(rewriter_name, [])
        for pf in pred_files:
            if pf:
                print(f"  ✅ Predictions: {pf}")
        rewrite_files = all_rewrite_files.get(rewriter_name, [])
        for rf in rewrite_files:
            if rf:
                print(f"  ✅ Rewrites: {rf}")

    # Cost report
    print("\n" + "="*70)
    print("💰 TOTAL COST (ALL 4 EXPERIMENTS)")
    print("="*70)
    print(tracker.get_report())
    print("="*70)

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import os

csv_files = [
    "/content/drive/MyDrive/Project/elser_v1_minimal_experiments/clapnq_minimal_no_xml_20260114_223618_results_aggregate.csv",









]

all_data = []

for file in csv_files:
    if os.path.exists(file):
        df = pd.read_csv(file)
        df = df[df["collection"] != "all"]
        pipeline_name = file.split("/")[-1].replace("_predictions_results_aggregate.csv", "")
        df["Pipeline"] = pipeline_name
        all_data.append(df)

if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df = final_df.set_index("Pipeline")
    print("\n" + "="*60)
    print("📊 COMPARISON: ELSER vs STELLA")
    print("="*60)
    display(final_df)
else:
    print("⚠️ No results to compare yet")

In [ ]:
import os
import pandas as pd
from IPython.display import display

BASE_PATH = "/content/drive/MyDrive/Project"
PRED_DIR = f"{BASE_PATH}/elser_v1_structural_6U3A_predictions"

# Τα evaluation files που δημιουργήθηκαν
eval_files = [
    f"{PRED_DIR}/clapnq_elser_minimal_optimized_20260112_114149_results.jsonl",
    f"{PRED_DIR}/fiqa_elser_minimal_optimized_20260112_115931_results.jsonl",
    f"{PRED_DIR}/govt_elser_minimal_optimized_20260112_121434_results.jsonl",
    f"{PRED_DIR}/cloud_elser_minimal_optimized_20260112_122934_results.jsonl"
]

all_data = []

for eval_file in eval_files:
    if not os.path.exists(eval_file):
        print(f"⚠️ Eval file not found: {eval_file}")
        continue

    # Get base name without _results.jsonl
    base_name = os.path.basename(eval_file).replace("_results.jsonl", "")

    # Build aggregate CSV path
    agg_csv = os.path.join(PRED_DIR, f"{base_name}_results_aggregate.csv")

    if os.path.exists(agg_csv):
        print(f"✅ Found: {os.path.basename(agg_csv)}")
        df = pd.read_csv(agg_csv)
        df = df[df["collection"] != "all"]

        corpus_name = base_name.split("_")[0]
        df["Corpus"] = corpus_name

        all_data.append(df)
    else:
        print(f"❌ Not found: {os.path.basename(agg_csv)}")

if all_data:
    final_df = pd.concat(all_data, ignore_index=True)

    print("\n" + "="*70)
    print("📊 FINAL RESULTS")
    print("="*70)
    display(final_df)

    print("\n" + "="*70)
    print("📈 SUMMARY BY CORPUS")
    print("="*70)
    summary = final_df.groupby('Corpus')[['nDCG@10', 'Recall@5', 'Recall@100']].mean()
    display(summary)
else:
    print("\n❌ No aggregate files found!")


## Corpus based rewriter

In [ ]:
# ==============================================================================
# ELSER v1 — CORPUS-SPECIFIC REWRITERS EXPERIMENT
# VERSION: Best-per-corpus optimized rewriters
# MULTI-CORPUS: ClapNQ, FiQA, Govt, Cloud (4 CORPORA)
# 🔥 NEW: Saves rewrites to separate JSON file for reuse
# 🔧 FIXED: Correct CSV path parsing for aggregate results
# ==============================================================================

import os, json, re, time, threading, requests
from typing import List, Dict, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from google.colab import drive

# Mount Drive
drive.mount('/content/drive', force_remount=True)

# ----------------------------
# 1. SETTINGS & CONFIG
# ----------------------------
BASE_PATH = "/content/drive/MyDrive/Project"
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]

EVAL_SCRIPT = f"{BASE_PATH}/run_retrieval_eval.py"
PRED_DIR = f"{BASE_PATH}/elser_v1_corpus_specific_predictions"
os.makedirs(PRED_DIR, exist_ok=True)

# ----------------------------
# 2. COST TRACKER
# ----------------------------
class CostTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt, completion):
        with self.lock:
            self.total_input += prompt
            self.total_output += completion

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        return f"In: {self.total_input} | Out: {self.total_output} | Est. Cost: ${cost:.4f}"

tracker = CostTracker()

def llm_invoke_text(system: str, user: str, max_tokens: int) -> str:
    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-V3.2",
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "max_tokens": max_tokens,
        "temperature": 0.0
    }
    for attempt in range(7):
        try:
            response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
            if response.status_code == 200:
                res_json = response.json()
                usage = res_json.get('usage', {})
                tracker.add_usage(usage.get('prompt_tokens', 0), usage.get('completion_tokens', 0))
                return res_json['choices'][0]['message']['content'].strip()
            elif response.status_code == 429:
                time.sleep(5 * (2 ** attempt))
                continue
        except:
            time.sleep(5)
    return ""

# ----------------------------
# 3. HISTORY FORMATTING HELPERS
# ----------------------------
def _history_to_text(history, user_max=6, assistant_max=3):
    """Standard XML-based history formatting (for ClapNQ, Govt, Cloud)"""
    output = ["<conversation_history>"]
    u_count, a_count = 0, 0

    for h in reversed(history):
        if h['role'] == 'user' and u_count < user_max:
            output.append(f"  <turn role='user'>{h['text']}</turn> </s>")
            u_count += 1
        elif h['role'] == 'assistant' and a_count < assistant_max:
            output.append(f"  <turn role='assistant'>{h['text']}</turn> </s>")
            a_count += 1

    output.append("</conversation_history>")
    return "\n".join(reversed(output))

def _history_to_text_optimal(history, u_max=6, a_max=0):
    """Optimized history for FiQA (user-only focus)"""
    output = []
    u_count, a_count = 0, 0

    for h in reversed(history):
        if h['role'] == 'user' and u_count < u_max:
            output.append(f"User: {h['text']}")
            u_count += 1
        elif h['role'] == 'assistant' and a_count < a_max:
            output.append(f"Assistant: {h['text']}")
            a_count += 1

    return "\n".join(reversed(output))

def parse_hybrid_history(text: str):
    user_re = re.compile(r"^\s*\|user\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    assistant_re = re.compile(r"^\s*\|assistant\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    history = []
    for ln in lines:
        u_m, a_m = user_re.match(ln), assistant_re.match(ln)
        if u_m:
            history.append({"role": "user", "text": u_m.group(1).strip()})
        elif a_m:
            history.append({"role": "assistant", "text": a_m.group(1).strip()})
    if not history:
        return text.strip(), []
    return history[-1]["text"], history[:-1]

def load_queries(path: str):
    qmap = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            query, hist = parse_hybrid_history(j.get("text", ""))
            qmap[j["_id"]] = {"query": query, "history": hist}
    return qmap

# ----------------------------
# 4. CORPUS-SPECIFIC REWRITERS
# ----------------------------

class ClapNQCorpusRewriter:
    def __init__(self):
        self.name = "clapnq_wikipedia_optimized2_6U3A"

    def rewrite(self, query, history):
        if not history:
            return [query]

        hist_text = _history_to_text(history, user_max=6, assistant_max=3)

        system = (
            "You are rewriting queries for retrieval from Wikipedia using ELSER (sparse semantic search).\n\n"
            "CORE OBJECTIVE:\n"
            "Transform the user's query into a standalone version that MAXIMIZES OVERLAP with Wikipedia passage text "
            "while preserving the original query's semantic intent.\n\n"
            "CRITICAL WIKIPEDIA + ELSER RETRIEVAL RULES:\n\n"
            "1. ENTITY FORMS - Use MULTIPLE variants when beneficial:\n"
            "   - Include both formal AND common names: 'Apple Inc. Apple' (not just 'Apple Inc.')\n"
            "   - For people: 'Francis Ford Coppola Coppola director'\n"
            "   - For places: 'New York NY New York City'\n"
            "   - For events: 'World War II WW2 Second World War'\n"
            "   ⚠️ ONLY add variants if they ADD retrieval value (don't spam)\n\n"
            "2. PRONOUN RESOLUTION - Be SURGICAL:\n"
            "   - 'he/she/it' → Exact entity name from conversation\n"
            "   - 'they/them' → Plural entity or organization name\n"
            "   - 'this/that' → Specific concept/event from context\n"
            "   - If pronoun refers to TITLE (movie/book): Use exact title, don't expand\n\n"
            "3. TEMPORAL HANDLING - Match Wikipedia's natural language:\n"
            "   ❌ DON'T force 'current' everywhere\n"
            "   ✅ DO use natural temporal phrasing:\n"
            "      - 'Who is the CEO?' → 'Who is CEO of Apple' (Wikipedia says 'serves as CEO', not 'current')\n"
            "      - 'When did X happen?' → Keep as-is\n"
            "      - Only add 'current' if user EXPLICITLY implies present time\n\n"
            "4. FORMALIZATION - MINIMAL ONLY:\n"
            "   ❌ DON'T over-formalize: 'cool facts' → 'notable characteristics' (too academic)\n"
            "   ✅ DO preserve conversational terms that appear in Wikipedia:\n"
            "      - 'facts about X' → 'facts about X' (Wikipedia uses 'facts'!)\n"
            "      - 'why did X happen' → 'why did X happen' OR 'causes of X' (both valid)\n"
            "   Rule: If the term EXISTS in Wikipedia articles, KEEP IT\n\n"
            "5. DISAMBIGUATION - Add ONLY when ambiguous:\n"
            "   - 'Mercury' after planet discussion → 'Mercury planet'\n"
            "   - 'Mercury' with no context → 'Mercury' (let retrieval handle it)\n"
            "   - Format: 'Entity (qualifier)' OR 'Entity qualifier' (both work)\n\n"
            "6. KEYWORD PRESERVATION:\n"
            "   - Preserve question words: who/what/when/where/why/how\n"
            "   - Keep domain-specific terms: director/actor/CEO/president/planet/movie\n"
            "   - Don't replace with synonyms unless they ADD coverage\n\n"
            "7. STRUCTURE:\n"
            "   - Keep original question structure (question → question, keywords → keywords)\n"
            "   - Don't expand 'population of France' into 'What is the population of France?'\n"
            "   - Natural language > Rigid formality\n\n"
            "MINIMAL INTERVENTION:\n"
            "- If query is standalone and has good entity names → Return UNCHANGED\n"
            "- Only rewrite to: resolve pronouns, add entity variants, clarify ambiguity\n"
            "- Goal: Better RETRIEVAL, not prettier English\n\n"
            "OUTPUT FORMAT (strict JSON):\n"
            '{\"class\": \"standalone|non-standalone\", \"rewritten version\": \"query here\"}\n'
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 400)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                return [parsed.get("rewritten version", query)]
            return [query]
        except:
            return [query]

class FiQAOptimalRewriter:
    def __init__(self):
        self.name = "fiqa_optimal_6U0A"

    def rewrite(self, query, history):
        if not history:
            return [query]

        hist_text = _history_to_text_optimal(history, u_max=6, a_max=0)

        system = (
            "You rewrite user queries for a finance Q&A dataset.\n"
            "Rewrite the final user utterance into ONE standalone question.\n"
            "Return STRICT JSON:\n"
            "{\"class\":\"standalone|non-standalone\",\"rewritten version\":\"...\"}\n\n"
            "Rules:\n"
            "- Minimal changes; preserve user's wording.\n"
            "- Resolve references ONLY using the conversation.\n"
            "- Preserve exact amounts, currencies, tickers, percentages, time horizons, product names mentioned.\n"
            "- Do NOT add finance jargon unless it already appears.\n"
            "- Do NOT introduce new entities/assumptions.\n"
            "- If already standalone: output SAME query + class='standalone'."
        )

        prompt_user = f"HISTORY:\n{hist_text}\n\nFINAL QUERY: {query}\n\nJSON:"

        try:
            txt = llm_invoke_text(system, prompt_user, 400)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                return [parsed.get("rewritten version", query).strip()]
            return [query]
        except:
            return [query]

class GovtMinimalRewriter:
    def __init__(self):
        self.name = "govt_corpus_baseline_6U3A"

    def rewrite(self, query, history):
        if not history:
            return [query]

        hist_text = _history_to_text(history, user_max=6, assistant_max=3)

        system = (
            "You rewrite user queries for a government information / forms / benefits dataset.\n"
            "Rewrite the final user utterance into ONE standalone question.\n"
            "Return STRICT JSON:\n"
            "{\"class\":\"standalone|non-standalone\",\"rewritten version\":\"...\"}\n\n"
            "Rules:\n"
            "- Minimal changes; preserve user's wording.\n"
            "- Resolve references ONLY using the conversation.\n"
            "- Preserve exact program names, agency names/acronyms, form IDs, deadlines, eligibility constraints, locations.\n"
            "- Do NOT introduce new agencies, programs, legal terms, requirements.\n"
            "- If already standalone: output SAME query + class='standalone'."
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 400)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                rewritten = parsed.get("rewritten version", query).strip()
                return [rewritten if rewritten else query]
            return [query]
        except Exception as e:
            return [query]

class CloudMinimalRewriter:
    def __init__(self):
        self.name = "cloud_corpus_baseline_6U3A"

    def rewrite(self, query, history):
        if not history:
            return [query]

        hist_text = _history_to_text(history, user_max=6, assistant_max=3)

        system = (
            "You rewrite user queries for a cloud troubleshooting dataset.\n"
            "Rewrite the final user utterance into ONE standalone question.\n"
            "Return STRICT JSON:\n"
            "{\"class\":\"standalone|non-standalone\",\"rewritten version\":\"...\"}\n\n"
            "Rules:\n"
            "- Minimal changes; preserve user's wording.\n"
            "- Resolve references ONLY using the conversation.\n"
            "- Preserve exact error messages/codes, stack trace snippets, CLI commands, flags, config keys, service names.\n"
            "- Include environment details (OS/region/service/resource) ONLY if present.\n"
            "- Do NOT add new tools/providers/services/config keys.\n"
            "- If already standalone: output SAME query + class='standalone'."
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 400)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                rewritten = parsed.get("rewritten version", query).strip()
                return [rewritten if rewritten else query]
            return [query]
        except Exception as e:
            return [query]

# ----------------------------
# 5. CORPUS CONFIGURATION WITH MATCHED REWRITERS
# ----------------------------
CORPORA = [
    {
        "name": "clapnq",
        "queries_file": f"{BASE_PATH}/retrieval/clapnq/clapnq_questions.jsonl",
        "index_name": "clapnq_elser_v1",
        "collection_id": "mt-rag-clapnq-elser-512-100-20240503",
        "rewriter_class": ClapNQCorpusRewriter
    },
    {
        "name": "fiqa",
        "queries_file": f"{BASE_PATH}/retrieval/fiqa/fiqa_questions.jsonl",
        "index_name": "fiqa_elser_v1",
        "collection_id": "mt-rag-fiqa-beir-elser-512-100-20240501",
        "rewriter_class": FiQAOptimalRewriter
    },
    {
        "name": "govt",
        "queries_file": f"{BASE_PATH}/retrieval/govt/govt_questions.jsonl",
        "index_name": "govt_elser_v1",
        "collection_id": "mt-rag-govt-elser-512-100-20240611",
        "rewriter_class": GovtMinimalRewriter
    },
    {
        "name": "cloud",
        "queries_file": f"{BASE_PATH}/retrieval/cloud/cloud_questions.jsonl",
        "index_name": "cloud_elser_v1",
        "collection_id": "mt-rag-ibmcloud-elser-512-100-20240502",
        "rewriter_class": CloudMinimalRewriter
    }
]

# ----------------------------
# 6. MULTI-CORPUS PIPELINE
# ----------------------------
from elasticsearch import Elasticsearch

def run_corpus_experiment(corpus_config: dict, es_client):
    """Run experiment for a single corpus with its specific rewriter"""

    corpus_name = corpus_config["name"]
    queries_file = corpus_config["queries_file"]
    index_name = corpus_config["index_name"]
    collection_id = corpus_config["collection_id"]
    rewriter = corpus_config["rewriter_class"]()  # Instantiate corpus-specific rewriter

    print("\n" + "="*70)
    print(f"🎯 CORPUS: {corpus_name.upper()}")
    print(f"📝 Rewriter: {rewriter.name}")
    print("="*70)

    # Load queries
    data = load_queries(queries_file)
    items = list(data.items())
    print(f"✅ Loaded {len(items)} queries for {corpus_name}")

    # Rewrite queries
    print(f"📝 Rewriting queries with {rewriter.name}...")
    rewrites_map = {}
    with ThreadPoolExecutor(max_workers=5) as ex:
        futs = {ex.submit(rewriter.rewrite, it["query"], it["history"]): qid for qid, it in items}
        for f in tqdm(as_completed(futs), total=len(futs), desc=f"Rewriting {corpus_name}"):
            rewrites_map[futs[f]] = f.result()[0]

    # 🔥 SAVE REWRITES TO JSON FILE
    RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
    rewrites_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}_REWRITES.json")

    rewrites_data = []
    for qid, item in items:
        rewrites_data.append({
            "qid": qid,
            "original_query": item["query"],
            "history": item["history"],
            "rewritten_query": rewrites_map.get(qid, item["query"])
        })

    with open(rewrites_file, "w", encoding="utf-8") as f:
        json.dump(rewrites_data, f, indent=2, ensure_ascii=False)

    print(f"💾 Rewrites saved to: {rewrites_file}")

    # Retrieval
    print(f"🔍 Retrieving from {index_name}...")
    pred_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}.jsonl")

    with open(pred_file, "w", encoding="utf-8") as out:
        for qid, item in tqdm(items, desc=f"Retrieving {corpus_name}"):
            q_final = rewrites_map.get(qid, item["query"])
            try:
                res = es_client.search(index=index_name, body={
                    "query": {
                        "text_expansion": {
                            "text_embedding": {
                                "model_id": ".elser_model_1",
                                "model_text": q_final
                            }
                        }
                    },
                    "size": 100
                })
                contexts = [
                    {"document_id": h["_source"].get("doc_id", h["_id"]), "score": h["_score"]}
                    for h in res["hits"]["hits"]
                ]
            except Exception as e:
                print(f"⚠️ Retrieval error for {qid}: {e}")
                contexts = []

            out.write(json.dumps({
                "task_id": qid,
                "Collection": collection_id,
                "contexts": contexts
            }, ensure_ascii=False) + "\n")

    print(f"✅ Predictions saved to: {pred_file}")

    return pred_file, rewrites_file

def run_evaluation(pred_file: str):
    """Run evaluation for a prediction file"""
    print(f"\n🧪 Evaluating {os.path.basename(pred_file)}...")
    output_eval_file = pred_file.replace(".jsonl", "_results.jsonl")

    cmd = f"python {EVAL_SCRIPT} --input_file {pred_file} --output_file {output_eval_file}"
    result = os.system(cmd)

    if result != 0:
        print(f"⚠️ Warning: Evaluation failed")
        return None

    print(f"✅ Evaluation completed")
    return output_eval_file

def aggregate_results(eval_files: List[str]):
    """Aggregate results from all corpora"""
    all_data = []

    for eval_file in eval_files:
        if eval_file is None:
            continue

        # 🔧 FIXED: Try multiple possible paths
        pred_dir = os.path.dirname(eval_file)
        base_name = os.path.basename(eval_file).replace("_results.jsonl", "")

        possible_paths = [
            os.path.join(pred_dir, f"{base_name}_results_aggregate.csv"),
            os.path.join(pred_dir, f"{base_name}_aggregate.csv"),
            eval_file.replace("_results.jsonl", "_results_aggregate.csv"),
            eval_file.replace(".jsonl", "_aggregate.csv")
        ]

        agg_csv = None
        for path in possible_paths:
            if os.path.exists(path):
                agg_csv = path
                break

        if agg_csv and os.path.exists(agg_csv):
            df = pd.read_csv(agg_csv)
            df = df[df["collection"] != "all"]

            corpus_name = base_name.split("_")[0]
            df["Corpus"] = corpus_name

            all_data.append(df)
            print(f"✅ Loaded results for {corpus_name}")
        else:
            print(f"⚠️ Aggregate CSV not found for: {base_name}")
            # Debug: list available files
            import glob
            matches = glob.glob(os.path.join(pred_dir, f"{base_name}*aggregate*.csv"))
            if matches:
                print(f"   Found files: {[os.path.basename(m) for m in matches]}")

    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)

        print("\n" + "="*70)
        print("📊 FINAL RESULTS - CORPUS-SPECIFIC REWRITERS")
        print("="*70)
        display(final_df)

        combined_file = os.path.join(PRED_DIR, f"corpus_specific_results_{time.strftime('%Y%m%d_%H%M%S')}.csv")
        final_df.to_csv(combined_file, index=False)
        print(f"\n💾 Combined results saved to: {combined_file}")

        print("\n" + "="*70)
        print("📈 SUMMARY BY CORPUS")
        print("="*70)
        summary = final_df.groupby('Corpus')[['nDCG@10', 'Recall@5', 'Recall@100']].mean()
        display(summary)

        return final_df
    else:
        print("⚠️ No results to aggregate")
        return None

# ----------------------------
# 7. MAIN EXECUTION
# ----------------------------
def main():
    print("\n" + "="*70)
    print("🚀 ELSER v1 - CORPUS-SPECIFIC REWRITERS EXPERIMENT")
    print("="*70)
    print("Strategy: Each corpus uses its optimal rewriter")
    print(f"Corpora: {', '.join([c['name'] for c in CORPORA])}")
    print("Rewriters:")
    for c in CORPORA:
        print(f"  • {c['name']}: {c['rewriter_class']().name}")
    print(f"Model: DeepSeek-V3.2 via Azure")
    print("="*70)

    es_client = Elasticsearch(
        cloud_id=CLOUD_ID,
        basic_auth=("elastic", PASSWORD),
        request_timeout=180
    )

    eval_files = []
    rewrite_files = []  # 🔥 Track rewrite files

    for corpus_config in CORPORA:
        try:
            pred_file, rewrites_file = run_corpus_experiment(corpus_config, es_client)
            eval_file = run_evaluation(pred_file)
            eval_files.append(eval_file)
            rewrite_files.append(rewrites_file)
        except Exception as e:
            print(f"❌ Error processing {corpus_config['name']}: {e}")
            import traceback
            traceback.print_exc()
            eval_files.append(None)
            rewrite_files.append(None)

    aggregate_results(eval_files)

    print("\n" + "="*70)
    print("💰 COST SUMMARY")
    print("="*70)
    print(tracker.get_report())
    print("="*70)

    # 🔥 PRINT REWRITE FILES
    print("\n" + "="*70)
    print("💾 SAVED REWRITE FILES")
    print("="*70)
    for rf in rewrite_files:
        if rf:
            print(f"  ✅ {rf}")
    print("="*70)

if __name__ == "__main__":
    main()


## Organizers rewriter

In [ ]:
# ==============================================================================
# ELSER v1 — MASTER EXPERIMENT CODE (DEEPSEEK-V3.2 via AZURE FOUNDRY)
# VERSION: STATIC 6U/3A - OPTIMIZED XML-STRUCTURAL CONTEXT (DAMO-NLP STYLE)
# MULTI-CORPUS: ClapNQ, FiQA, TopiOCQA, InstructIR
# ==============================================================================

import os, json, re, time, threading, requests
from typing import List, Dict, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from google.colab import drive

# Mount Drive
drive.mount('/content/drive', force_remount=True)

# ----------------------------
# 1. SETTINGS & CONFIG
# ----------------------------
BASE_PATH = "/content/drive/MyDrive/Project"
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]

EVAL_SCRIPT = f"{BASE_PATH}/run_retrieval_eval.py"
PRED_DIR = f"{BASE_PATH}/elser_v1_structural_6U3A_predictions"
os.makedirs(PRED_DIR, exist_ok=True)

# 🔥 MULTI-CORPUS CONFIGURATION
CORPORA = [
    {
        "name": "clapnq",
        "queries_file": f"{BASE_PATH}/retrieval/clapnq/clapnq_questions.jsonl",
        "index_name": "clapnq_elser_v1",
        "collection_id": "mt-rag-clapnq-elser-512-100-20240503"
    },
    {
        "name": "fiqa",
        "queries_file": f"{BASE_PATH}/retrieval/fiqa/fiqa_questions.jsonl",
        "index_name": "fiqa_elser_v1",
        "collection_id": "mt-rag-fiqa-beir-elser-512-100-20240501"
    },
    {
        "name": "govt",
        "queries_file": f"{BASE_PATH}/retrieval/govt/govt_questions.jsonl",
        "index_name": "govt_elser_v1",
        "collection_id": "mt-rag-govt-elser-512-100-20240611"
    },
    {
        "name": "cloud",
        "queries_file": f"{BASE_PATH}/retrieval/cloud/cloud_questions.jsonl",
        "index_name": "cloud_elser_v1",
        "collection_id": "mt-rag-ibmcloud-elser-512-100-20240502"
    }
]

# ----------------------------
# 2. COST TRACKER
# ----------------------------
class CostTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt, completion):
        with self.lock:
            self.total_input += prompt
            self.total_output += completion

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        return f"In: {self.total_input} | Out: {self.total_output} | Est. Cost: ${cost:.4f}"

tracker = CostTracker()

def llm_invoke_text(system: str, user: str, max_tokens: int) -> str:
    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-V3.2",
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "max_tokens": max_tokens,
        "temperature": 0.0
    }
    for attempt in range(7):
        try:
            response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
            if response.status_code == 200:
                res_json = response.json()
                usage = res_json.get('usage', {})
                tracker.add_usage(usage.get('prompt_tokens', 0), usage.get('completion_tokens', 0))
                return res_json['choices'][0]['message']['content'].strip()
            elif response.status_code == 429:
                time.sleep(5 * (2 ** attempt))
                continue
        except:
            time.sleep(5)
    return ""

# ----------------------------
# 3. PARSING & STRUCTURAL HISTORY (DAMO-NLP BASED)
# ----------------------------
def _history_to_text(history, user_max=6, assistant_max=3):
    """
    Διαμορφώνει το ιστορικό χρησιμοποιώντας XML tags και διαχωριστικά tokens.
    Αυτή η δομή επιτρέπει στο μοντέλο να επεξεργαστεί καλύτερα τις εξαρτήσεις.
    """
    output = ["<conversation_history>"]
    u_count, a_count = 0, 0

    # Διαβάζουμε το ιστορικό από το παλιότερο προς το νεότερο
    for h in reversed(history):
        if h['role'] == 'user' and u_count < user_max:
            output.append(f"  <turn role='user'>{h['text']}</turn> </s>")
            u_count += 1
        elif h['role'] == 'assistant' and a_count < assistant_max:
            output.append(f"  <turn role='assistant'>{h['text']}</turn> </s>")
            a_count += 1

    output.append("</conversation_history>")
    return "\n".join(reversed(output))

def parse_hybrid_history(text: str):
    user_re = re.compile(r"^\s*\|user\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    assistant_re = re.compile(r"^\s*\|assistant\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    history = []
    for ln in lines:
        u_m, a_m = user_re.match(ln), assistant_re.match(ln)
        if u_m:
            history.append({"role": "user", "text": u_m.group(1).strip()})
        elif a_m:
            history.append({"role": "assistant", "text": a_m.group(1).strip()})
    if not history:
        return text.strip(), []
    return history[-1]["text"], history[:-1]

def load_queries(path: str):
    qmap = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            query, hist = parse_hybrid_history(j.get("text", ""))
            qmap[j["_id"]] = {"query": query, "history": hist}
    return qmap

# ----------------------------
# 4. REWRITER (ORIGINAL PROMPT + STRUCTURAL WRAPPING)
# ----------------------------
class ELSERStructuralRewriter:
    def __init__(self):
        self.name = "elser_organizers"

    def rewrite(self, query, history):
        if not history:
            return [query]

        hist_text = _history_to_text(history, user_max=6, assistant_max=1)

        # ΤΟ ΑΡΧΙΚΟ ΣΟΥ PROMPT (Αμετάβλητο)
        system_prompt = (
            "Given the following conversation, please reword the final utterance from the user "
            "into a single utterance that does not need the history to understand the user's intent. "
            "Output in proper JSON format indicating the \"class\" (standalone or non-standalone) "
            "and the \"reworded version\" of the last utterance. Use this format: "
            "{\"class\": \"type of last utterance\", \"reworded version\": "
            "\"the last utterance rewritten into a standalone question, IF NEEDED\"}.\n\n"
            "In your rewording of the last utterance, do not do any unnecessary rephrasing or "
            "introduction of new terms or concepts that were not mentioned in the prior part of the conversation. "
            "Be minimal, by staying as close as possible to the shape and meaning of the last user utterance. "
            "If the last user utterance is already clear and standalone, the reworded version should be THE SAME "
            "as the last user utterance, and the class should be 'standalone'."
            "Use information from 'assistant' turns ONLY to resolve pronouns (e.g., 'it', 'he', 'that') or clarify ambiguous references.\n\n"
        )

        # STRUCTURAL WRAPPING
        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system_prompt, prompt_user, 400)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                return [parsed.get("reworded version", query)]
            return [query]
        except:
            return [query]

# ----------------------------
# 5. MULTI-CORPUS PIPELINE
# ----------------------------
from elasticsearch import Elasticsearch

def run_corpus_experiment(corpus_config: dict, rewriter, es_client):
    """Run experiment for a single corpus"""

    corpus_name = corpus_config["name"]
    queries_file = corpus_config["queries_file"]
    index_name = corpus_config["index_name"]
    collection_id = corpus_config["collection_id"]

    print("\n" + "="*70)
    print(f"🎯 CORPUS: {corpus_name.upper()}")
    print("="*70)

    # Load queries
    data = load_queries(queries_file)
    items = list(data.items())
    print(f"✅ Loaded {len(items)} queries for {corpus_name}")

    # Rewrite queries
    print(f"📝 Rewriting queries with {rewriter.name}...")
    rewrites_map = {}
    with ThreadPoolExecutor(max_workers=5) as ex:
        futs = {ex.submit(rewriter.rewrite, it["query"], it["history"]): qid for qid, it in items}
        for f in tqdm(as_completed(futs), total=len(futs), desc=f"Rewriting {corpus_name}"):
            rewrites_map[futs[f]] = f.result()[0]

    # Retrieval
    print(f"🔍 Retrieving from {index_name}...")
    RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
    pred_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}.jsonl")

    with open(pred_file, "w", encoding="utf-8") as out:
        for qid, item in tqdm(items, desc=f"Retrieving {corpus_name}"):
            q_final = rewrites_map.get(qid, item["query"])
            try:
                res = es_client.search(index=index_name, body={
                    "query": {
                        "text_expansion": {
                            "text_embedding": {
                                "model_id": ".elser_model_1",
                                "model_text": q_final
                            }
                        }
                    },
                    "size": 100
                })
                contexts = [
                    {"document_id": h["_source"].get("doc_id", h["_id"]), "score": h["_score"]}
                    for h in res["hits"]["hits"]
                ]
            except Exception as e:
                print(f"⚠️ Retrieval error for {qid}: {e}")
                contexts = []

            out.write(json.dumps({
                "task_id": qid,
                "Collection": collection_id,
                "contexts": contexts
            }, ensure_ascii=False) + "\n")

    print(f"✅ Predictions saved to: {pred_file}")

    return pred_file

def run_evaluation(pred_file: str):
    """Run evaluation for a prediction file"""
    print(f"\n🧪 Evaluating {os.path.basename(pred_file)}...")
    output_eval_file = pred_file.replace(".jsonl", "_results.jsonl")

    cmd = f"python {EVAL_SCRIPT} --input_file {pred_file} --output_file {output_eval_file}"
    result = os.system(cmd)

    if result != 0:
        print(f"⚠️ Warning: Evaluation failed")
        return None

    print(f"✅ Evaluation completed")
    return output_eval_file

def aggregate_results(eval_files: List[str]):
    """Aggregate results from all corpora"""
    all_data = []

    for eval_file in eval_files:
        if eval_file is None:
            continue

        # 🔧 FIXED: Correct path to aggregate CSV
        base_name = os.path.basename(eval_file).replace("_results.jsonl", "")
        pred_dir = os.path.dirname(eval_file)
        agg_csv = os.path.join(pred_dir, f"{base_name}_results_aggregate.csv")

        if os.path.exists(agg_csv):
            df = pd.read_csv(agg_csv)
            # Remove 'all' row
            df = df[df["collection"] != "all"]

            # Extract corpus name from file
            corpus_name = base_name.split("_")[0]
            df["Corpus"] = corpus_name

            all_data.append(df)
        else:
            print(f"⚠️ Aggregate CSV not found: {agg_csv}")

    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)

        print("\n" + "="*70)
        print("📊 FINAL RESULTS - ALL CORPORA")
        print("="*70)
        display(final_df)

        # Save combined results
        combined_file = os.path.join(PRED_DIR, f"combined_results_{time.strftime('%Y%m%d_%H%M%S')}.csv")
        final_df.to_csv(combined_file, index=False)
        print(f"\n💾 Combined results saved to: {combined_file}")

        # Summary by corpus
        print("\n" + "="*70)
        print("📈 SUMMARY BY CORPUS")
        print("="*70)
        summary = final_df.groupby('Corpus')[['nDCG@10', 'Recall@5', 'Recall@100']].mean()
        display(summary)

        return final_df
    else:
        print("⚠️ No results to aggregate")
        return None

# ----------------------------
# 6. MAIN EXECUTION
# ----------------------------
def main():
    print("\n" + "="*70)
    print("🚀 ELSER v1 STRUCTURAL REWRITER - MULTI-CORPUS EXPERIMENT")
    print("="*70)
    print(f"Strategy: 6U/3A XML-Structural Context (DAMO-NLP)")
    print(f"Corpora: {', '.join([c['name'] for c in CORPORA])}")
    print(f"Model: DeepSeek-V3.2 via Azure")
    print("="*70)

    # Initialize
    es_client = Elasticsearch(
        cloud_id=CLOUD_ID,
        basic_auth=("elastic", PASSWORD),
        request_timeout=180
    )

    rewriter = ELSERStructuralRewriter()
    eval_files = []

    # Run experiment for each corpus
    for corpus_config in CORPORA:
        try:
            pred_file = run_corpus_experiment(corpus_config, rewriter, es_client)
            eval_file = run_evaluation(pred_file)
            eval_files.append(eval_file)
        except Exception as e:
            print(f"❌ Error processing {corpus_config['name']}: {e}")
            eval_files.append(None)

    # Aggregate results
    aggregate_results(eval_files)

    # Cost report
    print("\n" + "="*70)
    print("💰 COST SUMMARY")
    print("="*70)
    print(tracker.get_report())
    print("="*70)

if __name__ == "__main__":
    main()


## Extra retriever based rewriter

In [ ]:
# ==============================================================================
# ELSER v1 — COT & HYDE REWRITERS EXPERIMENT (DEEPSEEK-V3.2 via AZURE FOUNDRY)
# VERSION: Chain-of-Thought + HyDE Rewriters
# MULTI-CORPUS: ClapNQ, FiQA, Govt, Cloud
# ==============================================================================

import os, json, re, time, threading, requests
from typing import List, Dict, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# ----------------------------
# 1. SETTINGS & CONFIG
# ----------------------------
BASE_PATH = "/content/drive/MyDrive/Project"
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]

EVAL_SCRIPT = f"{BASE_PATH}/run_retrieval_eval.py"
PRED_DIR = f"{BASE_PATH}/elser_v1_cot_hyde_predictions"
os.makedirs(PRED_DIR, exist_ok=True)

CORPORA = [
    {
        "name": "clapnq",
        "queries_file": f"{BASE_PATH}/retrieval/clapnq/clapnq_questions.jsonl",
        "index_name": "clapnq_elser_v1",
        "collection_id": "mt-rag-clapnq-elser-512-100-20240503"
    },
    {
        "name": "fiqa",
        "queries_file": f"{BASE_PATH}/retrieval/fiqa/fiqa_questions.jsonl",
        "index_name": "fiqa_elser_v1",
        "collection_id": "mt-rag-fiqa-beir-elser-512-100-20240501"
    },
    {
        "name": "govt",
        "queries_file": f"{BASE_PATH}/retrieval/govt/govt_questions.jsonl",
        "index_name": "govt_elser_v1",
        "collection_id": "mt-rag-govt-elser-512-100-20240611"
    },
    {
        "name": "cloud",
        "queries_file": f"{BASE_PATH}/retrieval/cloud/cloud_questions.jsonl",
        "index_name": "cloud_elser_v1",
        "collection_id": "mt-rag-ibmcloud-elser-512-100-20240502"
    }
]

# ----------------------------
# 2. COST TRACKER
# ----------------------------
class CostTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt, completion):
        with self.lock:
            self.total_input += prompt
            self.total_output += completion

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        return f"In: {self.total_input} | Out: {self.total_output} | Est. Cost: ${cost:.4f}"

tracker = CostTracker()

def llm_invoke_text(system: str, user: str, max_tokens: int) -> str:
    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-V3.2",
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "max_tokens": max_tokens,
        "temperature": 0.0
    }
    for attempt in range(7):
        try:
            response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
            if response.status_code == 200:
                res_json = response.json()
                usage = res_json.get('usage', {})
                tracker.add_usage(usage.get('prompt_tokens', 0), usage.get('completion_tokens', 0))
                return res_json['choices'][0]['message']['content'].strip()
            elif response.status_code == 429:
                time.sleep(5 * (2 ** attempt))
                continue
        except:
            time.sleep(5)
    return ""

# ----------------------------
# 3. PARSING & STRUCTURAL HISTORY
# ----------------------------
def _history_to_text(history, user_max=6, assistant_max=3):
    output = ["<conversation_history>"]
    u_count, a_count = 0, 0

    for h in reversed(history):
        if h['role'] == 'user' and u_count < user_max:
            output.append(f"  <turn role='user'>{h['text']}</turn> </s>")
            u_count += 1
        elif h['role'] == 'assistant' and a_count < assistant_max:
            output.append(f"  <turn role='assistant'>{h['text']}</turn> </s>")
            a_count += 1

    output.append("</conversation_history>")
    return "\n".join(reversed(output))

def parse_hybrid_history(text: str):
    user_re = re.compile(r"^\s*\|user\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    assistant_re = re.compile(r"^\s*\|assistant\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    history = []
    for ln in lines:
        u_m, a_m = user_re.match(ln), assistant_re.match(ln)
        if u_m:
            history.append({"role": "user", "text": u_m.group(1).strip()})
        elif a_m:
            history.append({"role": "assistant", "text": a_m.group(1).strip()})
    if not history:
        return text.strip(), []
    return history[-1]["text"], history[:-1]

def load_queries(path: str):
    qmap = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            query, hist = parse_hybrid_history(j.get("text", ""))
            qmap[j["_id"]] = {"query": query, "history": hist}
    return qmap

# ----------------------------
# 4. REWRITERS: COT + HYDE
# ----------------------------

class ChainOfThoughtRewriter:
    def __init__(self):
        self.name = "elser_cot_6U3A"

    def rewrite(self, query, history):
        if not history:
            return [query]

        hist_text = _history_to_text(history, user_max=6, assistant_max=3)

        system = (
            "You are an expert query rewriter for information retrieval systems.\n\n"
            "TASK: Rewrite the user's final query into a standalone version for optimal retrieval.\n\n"
            "PROCESS (Chain-of-Thought):\n"
            "1. ANALYZE the conversation history to identify:\n"
            "   - What entities/topics are being discussed\n"
            "   - What pronouns or references need resolution\n"
            "   - What context is needed to understand the final query\n\n"
            "2. REASON about the user's intent:\n"
            "   - Is the query standalone or does it depend on history?\n"
            "   - What information from history is essential?\n"
            "   - What should be preserved vs added?\n\n"
            "3. REWRITE the query:\n"
            "   - Resolve all pronouns (he/she/it/they/this/that)\n"
            "   - Add missing entities/context from history\n"
            "   - Keep original wording when possible\n"
            "   - Make it self-contained and clear\n\n"
            "OUTPUT FORMAT (strict JSON):\n"
            "{\n"
            "  \"reasoning\": \"<your step-by-step analysis>\",\n"
            "  \"class\": \"standalone|non-standalone\",\n"
            "  \"rewritten version\": \"<rewritten query>\"\n"
            "}\n\n"
            "RULES:\n"
            "- Be minimal: don't add unnecessary words\n"
            "- Be precise: use exact entities from history\n"
            "- Be faithful: preserve user's intent and terminology\n"
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE with reasoning):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 600)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                return [parsed.get("rewritten version", query)]
            return [query]
        except:
            return [query]


class HyDERewriter:
    def __init__(self):
        self.name = "elser_hyde_6U3A"

    def rewrite(self, query, history):
        if not history:
            return [query]

        hist_text = _history_to_text(history, user_max=6, assistant_max=3)

        system = (
            "You are generating hypothetical document passages for retrieval optimization.\n\n"
            "TASK: Given a conversation and final user query, generate a HYPOTHETICAL PASSAGE "
            "that would perfectly answer the query. This passage will be used for retrieval.\n\n"
            "STRATEGY (HyDE - Hypothetical Document Embeddings):\n"
            "1. First, understand the query in context of the conversation\n"
            "2. Generate a realistic passage (2-4 sentences) that:\n"
            "   - Directly answers the query\n"
            "   - Uses terminology likely to appear in real documents\n"
            "   - Contains relevant keywords and entities\n"
            "   - Sounds natural and informative\n\n"
            "OUTPUT FORMAT (strict JSON):\n"
            "{\n"
            "  \"standalone_query\": \"<query rewritten to be standalone>\",\n"
            "  \"hypothetical_passage\": \"<2-4 sentence passage that would answer the query>\"\n"
            "}\n\n"
            "EXAMPLE:\n"
            "Query: 'What is its capital?'\n"
            "Context: Discussion about France\n"
            "Output:\n"
            "{\n"
            "  \"standalone_query\": \"What is the capital of France?\",\n"
            "  \"hypothetical_passage\": \"The capital of France is Paris. Paris is located in "
            "the north-central part of the country on the River Seine. It is the most populous "
            "city in France with over 2 million residents in the city proper.\"\n"
            "}\n\n"
            "RULES:\n"
            "- Write as if you're a document that answers the question\n"
            "- Include specific facts, names, dates when relevant\n"
            "- Use vocabulary that would appear in authoritative sources\n"
            "- Keep it concise but informative (2-4 sentences)\n"
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 800)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                standalone = parsed.get("standalone_query", query)
                passage = parsed.get("hypothetical_passage", "")
                combined = f"{standalone} {passage}".strip()
                return [combined]
            return [query]
        except:
            return [query]

# ----------------------------
# 5. MULTI-CORPUS PIPELINE
# ----------------------------
from elasticsearch import Elasticsearch

def run_corpus_experiment(corpus_config: dict, rewriter, es_client):
    corpus_name = corpus_config["name"]
    queries_file = corpus_config["queries_file"]
    index_name = corpus_config["index_name"]
    collection_id = corpus_config["collection_id"]

    print("\n" + "="*70)
    print(f"🎯 CORPUS: {corpus_name.upper()}")
    print(f"📝 Rewriter: {rewriter.name}")
    print("="*70)

    data = load_queries(queries_file)
    items = list(data.items())
    print(f"✅ Loaded {len(items)} queries for {corpus_name}")

    print(f"📝 Rewriting queries with {rewriter.name}...")
    rewrites_map = {}
    with ThreadPoolExecutor(max_workers=5) as ex:
        futs = {ex.submit(rewriter.rewrite, it["query"], it["history"]): qid for qid, it in items}
        for f in tqdm(as_completed(futs), total=len(futs), desc=f"Rewriting {corpus_name}"):
            rewrites_map[futs[f]] = f.result()[0]

    RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
    rewrites_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}_REWRITES.json")

    rewrites_data = []
    for qid, item in items:
        rewrites_data.append({
            "qid": qid,
            "original_query": item["query"],
            "history": item["history"],
            "rewritten_query": rewrites_map.get(qid, item["query"])
        })

    with open(rewrites_file, "w", encoding="utf-8") as f:
        json.dump(rewrites_data, f, indent=2, ensure_ascii=False)

    print(f"💾 Rewrites saved to: {rewrites_file}")

    print(f"🔍 Retrieving from {index_name}...")
    pred_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}.jsonl")

    with open(pred_file, "w", encoding="utf-8") as out:
        for qid, item in tqdm(items, desc=f"Retrieving {corpus_name}"):
            q_final = rewrites_map.get(qid, item["query"])
            try:
                res = es_client.search(index=index_name, body={
                    "query": {
                        "text_expansion": {
                            "text_embedding": {
                                "model_id": ".elser_model_1",
                                "model_text": q_final
                            }
                        }
                    },
                    "size": 100
                })
                contexts = [
                    {"document_id": h["_source"].get("doc_id", h["_id"]), "score": h["_score"]}
                    for h in res["hits"]["hits"]
                ]
            except Exception as e:
                print(f"⚠️ Retrieval error for {qid}: {e}")
                contexts = []

            out.write(json.dumps({
                "task_id": qid,
                "Collection": collection_id,
                "contexts": contexts
            }, ensure_ascii=False) + "\n")

    print(f"✅ Predictions saved to: {pred_file}")
    return pred_file, rewrites_file

def run_evaluation(pred_file: str):
    print(f"\n🧪 Evaluating {os.path.basename(pred_file)}...")
    output_eval_file = pred_file.replace(".jsonl", "_results.jsonl")

    cmd = f"python {EVAL_SCRIPT} --input_file {pred_file} --output_file {output_eval_file}"
    result = os.system(cmd)

    if result != 0:
        print(f"⚠️ Warning: Evaluation failed")
        return None

    print(f"✅ Evaluation completed")
    return output_eval_file

def aggregate_results(eval_files: List[str], rewriter_name: str):
    all_data = []

    for eval_file in eval_files:
        if eval_file is None:
            continue

        base_name = os.path.basename(eval_file).replace("_results.jsonl", "")
        pred_dir = os.path.dirname(eval_file)
        agg_csv = os.path.join(pred_dir, f"{base_name}_results_aggregate.csv")

        if os.path.exists(agg_csv):
            df = pd.read_csv(agg_csv)
            df = df[df["collection"] != "all"]
            corpus_name = base_name.split("_")[0]
            df["Corpus"] = corpus_name
            df["Rewriter"] = rewriter_name
            all_data.append(df)
        else:
            print(f"⚠️ Aggregate CSV not found: {agg_csv}")

    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)

        print("\n" + "="*70)
        print(f"📊 RESULTS - {rewriter_name.upper()}")
        print("="*70)
        display(final_df)

        print("\n" + "="*70)
        print(f"📈 SUMMARY BY CORPUS - {rewriter_name.upper()}")
        print("="*70)
        summary = final_df.groupby('Corpus')[['nDCG@10', 'Recall@5', 'Recall@100']].mean()
        display(summary)

        return final_df
    else:
        print("⚠️ No results to aggregate")
        return None

# ----------------------------
# 6. MAIN EXECUTION
# ----------------------------
def main():
    print("\n" + "="*70)
    print("🚀 ELSER v1 - COT & HYDE REWRITERS EXPERIMENT")
    print("="*70)
    print("Rewriters: Chain-of-Thought + HyDE")
    print(f"Corpora: {', '.join([c['name'] for c in CORPORA])}")
    print("Model: DeepSeek-V3.2 via Azure")
    print("="*70)

    es_client = Elasticsearch(
        cloud_id=CLOUD_ID,
        basic_auth=("elastic", PASSWORD),
        request_timeout=180
    )

    rewriters = [
        ChainOfThoughtRewriter(),
        HyDERewriter()
    ]

    all_results = []

    for rewriter in rewriters:
        print("\n" + "#"*70)
        print(f"### RUNNING REWRITER: {rewriter.name.upper()}")
        print("#"*70)

        eval_files = []
        rewrite_files = []

        for corpus_config in CORPORA:
            try:
                pred_file, rewrites_file = run_corpus_experiment(corpus_config, rewriter, es_client)
                eval_file = run_evaluation(pred_file)
                eval_files.append(eval_file)
                rewrite_files.append(rewrites_file)
            except Exception as e:
                print(f"❌ Error processing {corpus_config['name']}: {e}")
                eval_files.append(None)
                rewrite_files.append(None)

        results_df = aggregate_results(eval_files, rewriter.name)
        if results_df is not None:
            all_results.append(results_df)

        print("\n" + "="*70)
        print(f"💾 SAVED REWRITE FILES - {rewriter.name}")
        print("="*70)
        for rf in rewrite_files:
            if rf:
                print(f"  ✅ {rf}")

    if all_results:
        combined_df = pd.concat(all_results, ignore_index=True)
        combined_file = os.path.join(PRED_DIR, f"cot_hyde_combined_{time.strftime('%Y%m%d_%H%M%S')}.csv")
        combined_df.to_csv(combined_file, index=False)
        print("\n" + "="*70)
        print("📊 COMBINED RESULTS - ALL REWRITERS")
        print("="*70)
        display(combined_df)
        print(f"\n💾 Combined results saved to: {combined_file}")

        print("\n" + "="*70)
        print("📈 COMPARISON BY REWRITER")
        print("="*70)
        comparison = combined_df.groupby('Rewriter')[['nDCG@10', 'Recall@5', 'Recall@100']].mean()
        display(comparison)

    print("\n" + "="*70)
    print("💰 COST SUMMARY")
    print("="*70)
    print(tracker.get_report())
    print("="*70)

if __name__ == "__main__":
    main()


In [ ]:
# ==============================================================================
# ELSER v1 — HYDE REWRITER ONLY (DEEPSEEK-V3.2 via AZURE FOUNDRY)
# VERSION: HyDE Rewriter
# MULTI-CORPUS: ClapNQ, FiQA, Govt, Cloud
# ==============================================================================

import os, json, re, time, threading, requests
from typing import List, Dict, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# ----------------------------
# 1. SETTINGS & CONFIG
# ----------------------------
BASE_PATH = "/content/drive/MyDrive/Project"
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]

EVAL_SCRIPT = f"{BASE_PATH}/run_retrieval_eval.py"
PRED_DIR = f"{BASE_PATH}/elser_v1_cot_hyde_predictions"
os.makedirs(PRED_DIR, exist_ok=True)

CORPORA = [
    {
        "name": "clapnq",
        "queries_file": f"{BASE_PATH}/retrieval/clapnq/clapnq_questions.jsonl",
        "index_name": "clapnq_elser_v1",
        "collection_id": "mt-rag-clapnq-elser-512-100-20240503"
    },
    {
        "name": "fiqa",
        "queries_file": f"{BASE_PATH}/retrieval/fiqa/fiqa_questions.jsonl",
        "index_name": "fiqa_elser_v1",
        "collection_id": "mt-rag-fiqa-beir-elser-512-100-20240501"
    },
    {
        "name": "govt",
        "queries_file": f"{BASE_PATH}/retrieval/govt/govt_questions.jsonl",
        "index_name": "govt_elser_v1",
        "collection_id": "mt-rag-govt-elser-512-100-20240611"
    },
    {
        "name": "cloud",
        "queries_file": f"{BASE_PATH}/retrieval/cloud/cloud_questions.jsonl",
        "index_name": "cloud_elser_v1",
        "collection_id": "mt-rag-ibmcloud-elser-512-100-20240502"
    }
]

# ----------------------------
# 2. COST TRACKER
# ----------------------------
class CostTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt, completion):
        with self.lock:
            self.total_input += prompt
            self.total_output += completion

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        return f"In: {self.total_input} | Out: {self.total_output} | Est. Cost: ${cost:.4f}"

tracker = CostTracker()

def llm_invoke_text(system: str, user: str, max_tokens: int) -> str:
    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-V3.2",
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "max_tokens": max_tokens,
        "temperature": 0.0
    }
    for attempt in range(7):
        try:
            response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
            if response.status_code == 200:
                res_json = response.json()
                usage = res_json.get('usage', {})
                tracker.add_usage(usage.get('prompt_tokens', 0), usage.get('completion_tokens', 0))
                return res_json['choices'][0]['message']['content'].strip()
            elif response.status_code == 429:
                time.sleep(5 * (2 ** attempt))
                continue
        except:
            time.sleep(5)
    return ""

# ----------------------------
# 3. PARSING & STRUCTURAL HISTORY
# ----------------------------
def _history_to_text(history, user_max=6, assistant_max=3):
    output = ["<conversation_history>"]
    u_count, a_count = 0, 0

    for h in reversed(history):
        if h['role'] == 'user' and u_count < user_max:
            output.append(f"  <turn role='user'>{h['text']}</turn> </s>")
            u_count += 1
        elif h['role'] == 'assistant' and a_count < assistant_max:
            output.append(f"  <turn role='assistant'>{h['text']}</turn> </s>")
            a_count += 1

    output.append("</conversation_history>")
    return "\n".join(reversed(output))

def parse_hybrid_history(text: str):
    user_re = re.compile(r"^\s*\|user\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    assistant_re = re.compile(r"^\s*\|assistant\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    history = []
    for ln in lines:
        u_m, a_m = user_re.match(ln), assistant_re.match(ln)
        if u_m:
            history.append({"role": "user", "text": u_m.group(1).strip()})
        elif a_m:
            history.append({"role": "assistant", "text": a_m.group(1).strip()})
    if not history:
        return text.strip(), []
    return history[-1]["text"], history[:-1]

def load_queries(path: str):
    qmap = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            query, hist = parse_hybrid_history(j.get("text", ""))
            qmap[j["_id"]] = {"query": query, "history": hist}
    return qmap

# ----------------------------
# 4. HYDE REWRITER
# ----------------------------
class HyDERewriter:
    def __init__(self):
        self.name = "elser_hyde_6U3A"

    def rewrite(self, query, history):
        if not history:
            return [query]

        hist_text = _history_to_text(history, user_max=6, assistant_max=3)

        system = (
            "You are generating hypothetical document passages for retrieval optimization.\n\n"
            "TASK: Given a conversation and final user query, generate a HYPOTHETICAL PASSAGE "
            "that would perfectly answer the query. This passage will be used for retrieval.\n\n"
            "STRATEGY (HyDE - Hypothetical Document Embeddings):\n"
            "1. First, understand the query in context of the conversation\n"
            "2. Generate a realistic passage (2-4 sentences) that:\n"
            "   - Directly answers the query\n"
            "   - Uses terminology likely to appear in real documents\n"
            "   - Contains relevant keywords and entities\n"
            "   - Sounds natural and informative\n\n"
            "OUTPUT FORMAT (strict JSON):\n"
            "{\n"
            "  \"standalone_query\": \"<query rewritten to be standalone>\",\n"
            "  \"hypothetical_passage\": \"<2-4 sentence passage that would answer the query>\"\n"
            "}\n\n"
            "EXAMPLE:\n"
            "Query: 'What is its capital?'\n"
            "Context: Discussion about France\n"
            "Output:\n"
            "{\n"
            "  \"standalone_query\": \"What is the capital of France?\",\n"
            "  \"hypothetical_passage\": \"The capital of France is Paris. Paris is located in "
            "the north-central part of the country on the River Seine. It is the most populous "
            "city in France with over 2 million residents in the city proper.\"\n"
            "}\n\n"
            "RULES:\n"
            "- Write as if you're a document that answers the question\n"
            "- Include specific facts, names, dates when relevant\n"
            "- Use vocabulary that would appear in authoritative sources\n"
            "- Keep it concise but informative (2-4 sentences)\n"
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 800)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                standalone = parsed.get("standalone_query", query)
                passage = parsed.get("hypothetical_passage", "")
                combined = f"{standalone} {passage}".strip()
                return [combined]
            return [query]
        except:
            return [query]

# ----------------------------
# 5. MULTI-CORPUS PIPELINE
# ----------------------------
from elasticsearch import Elasticsearch

def run_corpus_experiment(corpus_config: dict, rewriter, es_client):
    corpus_name = corpus_config["name"]
    queries_file = corpus_config["queries_file"]
    index_name = corpus_config["index_name"]
    collection_id = corpus_config["collection_id"]

    print("\n" + "="*70)
    print(f"🎯 CORPUS: {corpus_name.upper()}")
    print(f"📝 Rewriter: {rewriter.name}")
    print("="*70)

    data = load_queries(queries_file)
    items = list(data.items())
    print(f"✅ Loaded {len(items)} queries for {corpus_name}")

    print(f"📝 Rewriting queries with {rewriter.name}...")
    rewrites_map = {}
    with ThreadPoolExecutor(max_workers=5) as ex:
        futs = {ex.submit(rewriter.rewrite, it["query"], it["history"]): qid for qid, it in items}
        for f in tqdm(as_completed(futs), total=len(futs), desc=f"Rewriting {corpus_name}"):
            rewrites_map[futs[f]] = f.result()[0]

    RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
    rewrites_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}_REWRITES.json")

    rewrites_data = []
    for qid, item in items:
        rewrites_data.append({
            "qid": qid,
            "original_query": item["query"],
            "history": item["history"],
            "rewritten_query": rewrites_map.get(qid, item["query"])
        })

    with open(rewrites_file, "w", encoding="utf-8") as f:
        json.dump(rewrites_data, f, indent=2, ensure_ascii=False)

    print(f"💾 Rewrites saved to: {rewrites_file}")

    print(f"🔍 Retrieving from {index_name}...")
    pred_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}.jsonl")

    with open(pred_file, "w", encoding="utf-8") as out:
        for qid, item in tqdm(items, desc=f"Retrieving {corpus_name}"):
            q_final = rewrites_map.get(qid, item["query"])
            try:
                res = es_client.search(index=index_name, body={
                    "query": {
                        "text_expansion": {
                            "text_embedding": {
                                "model_id": ".elser_model_1",
                                "model_text": q_final
                            }
                        }
                    },
                    "size": 100
                })
                contexts = [
                    {"document_id": h["_source"].get("doc_id", h["_id"]), "score": h["_score"]}
                    for h in res["hits"]["hits"]
                ]
            except Exception as e:
                print(f"⚠️ Retrieval error for {qid}: {e}")
                contexts = []

            out.write(json.dumps({
                "task_id": qid,
                "Collection": collection_id,
                "contexts": contexts
            }, ensure_ascii=False) + "\n")

    print(f"✅ Predictions saved to: {pred_file}")
    return pred_file, rewrites_file

def run_evaluation(pred_file: str):
    print(f"\n🧪 Evaluating {os.path.basename(pred_file)}...")
    output_eval_file = pred_file.replace(".jsonl", "_results.jsonl")

    cmd = f"python {EVAL_SCRIPT} --input_file {pred_file} --output_file {output_eval_file}"
    result = os.system(cmd)

    if result != 0:
        print(f"⚠️ Warning: Evaluation failed")
        return None

    print(f"✅ Evaluation completed")
    return output_eval_file

def aggregate_results(eval_files: List[str], rewriter_name: str):
    all_data = []

    for eval_file in eval_files:
        if eval_file is None:
            continue

        base_name = os.path.basename(eval_file).replace("_results.jsonl", "")
        pred_dir = os.path.dirname(eval_file)
        agg_csv = os.path.join(pred_dir, f"{base_name}_results_aggregate.csv")

        if os.path.exists(agg_csv):
            df = pd.read_csv(agg_csv)
            df = df[df["collection"] != "all"]
            corpus_name = base_name.split("_")[0]
            df["Corpus"] = corpus_name
            df["Rewriter"] = rewriter_name
            all_data.append(df)
        else:
            print(f"⚠️ Aggregate CSV not found: {agg_csv}")

    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)

        print("\n" + "="*70)
        print(f"📊 RESULTS - {rewriter_name.upper()}")
        print("="*70)
        display(final_df)

        return final_df
    else:
        print("⚠️ No results to aggregate")
        return None

# ----------------------------
# 6. MAIN EXECUTION
# ----------------------------
def main():
    print("\n" + "="*70)
    print("🚀 ELSER v1 - HYDE REWRITER EXPERIMENT")
    print("="*70)
    print("Rewriter: HyDE")
    print(f"Corpora: {', '.join([c['name'] for c in CORPORA])}")
    print("Model: DeepSeek-V3.2 via Azure")
    print("="*70)

    es_client = Elasticsearch(
        cloud_id=CLOUD_ID,
        basic_auth=("elastic", PASSWORD),
        request_timeout=180
    )

    rewriter = HyDERewriter()

    eval_files = []
    rewrite_files = []

    for corpus_config in CORPORA:
        try:
            pred_file, rewrites_file = run_corpus_experiment(corpus_config, rewriter, es_client)
            eval_file = run_evaluation(pred_file)
            eval_files.append(eval_file)
            rewrite_files.append(rewrites_file)
        except Exception as e:
            print(f"❌ Error processing {corpus_config['name']}: {e}")
            import traceback
            traceback.print_exc()
            eval_files.append(None)
            rewrite_files.append(None)

    results_df = aggregate_results(eval_files, rewriter.name)

    print("\n" + "="*70)
    print(f"💾 SAVED REWRITE FILES - {rewriter.name}")
    print("="*70)
    for rf in rewrite_files:
        if rf:
            print(f"  ✅ {rf}")

    print("\n" + "="*70)
    print("💰 COST SUMMARY")
    print("="*70)
    print(tracker.get_report())
    print("="*70)

if __name__ == "__main__":
    main()


In [ ]:
# ==============================================================================
# ELSER v1 — QUERY DECOMPOSITION REWRITER (DEEPSEEK-V3.2 via AZURE FOUNDRY)
# VERSION: Query Decomposition with RRF Fusion
# MULTI-CORPUS: ClapNQ, FiQA, Govt, Cloud
# ==============================================================================

import os, json, re, time, threading, requests
from typing import List, Dict, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# ----------------------------
# 1. SETTINGS & CONFIG
# ----------------------------
BASE_PATH = "/content/drive/MyDrive/Project"
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]

EVAL_SCRIPT = f"{BASE_PATH}/run_retrieval_eval.py"
PRED_DIR = f"{BASE_PATH}/elser_v1_decomposition_predictions"
os.makedirs(PRED_DIR, exist_ok=True)

CORPORA = [
    {
        "name": "clapnq",
        "queries_file": f"{BASE_PATH}/retrieval/clapnq/clapnq_questions.jsonl",
        "index_name": "clapnq_elser_v1",
        "collection_id": "mt-rag-clapnq-elser-512-100-20240503"
    },
    {
        "name": "fiqa",
        "queries_file": f"{BASE_PATH}/retrieval/fiqa/fiqa_questions.jsonl",
        "index_name": "fiqa_elser_v1",
        "collection_id": "mt-rag-fiqa-beir-elser-512-100-20240501"
    },
    {
        "name": "govt",
        "queries_file": f"{BASE_PATH}/retrieval/govt/govt_questions.jsonl",
        "index_name": "govt_elser_v1",
        "collection_id": "mt-rag-govt-elser-512-100-20240611"
    },
    {
        "name": "cloud",
        "queries_file": f"{BASE_PATH}/retrieval/cloud/cloud_questions.jsonl",
        "index_name": "cloud_elser_v1",
        "collection_id": "mt-rag-ibmcloud-elser-512-100-20240502"
    }
]

# ----------------------------
# 2. COST TRACKER
# ----------------------------
class CostTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt, completion):
        with self.lock:
            self.total_input += prompt
            self.total_output += completion

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        return f"In: {self.total_input} | Out: {self.total_output} | Est. Cost: ${cost:.4f}"

tracker = CostTracker()

def llm_invoke_text(system: str, user: str, max_tokens: int) -> str:
    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-V3.2",
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "max_tokens": max_tokens,
        "temperature": 0.0
    }
    for attempt in range(7):
        try:
            response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
            if response.status_code == 200:
                res_json = response.json()
                usage = res_json.get('usage', {})
                tracker.add_usage(usage.get('prompt_tokens', 0), usage.get('completion_tokens', 0))
                return res_json['choices'][0]['message']['content'].strip()
            elif response.status_code == 429:
                time.sleep(5 * (2 ** attempt))
                continue
        except:
            time.sleep(5)
    return ""

# ----------------------------
# 3. PARSING & STRUCTURAL HISTORY
# ----------------------------
def _history_to_text(history, user_max=6, assistant_max=3):
    output = ["<conversation_history>"]
    u_count, a_count = 0, 0

    for h in reversed(history):
        if h['role'] == 'user' and u_count < user_max:
            output.append(f"  <turn role='user'>{h['text']}</turn> </s>")
            u_count += 1
        elif h['role'] == 'assistant' and a_count < assistant_max:
            output.append(f"  <turn role='assistant'>{h['text']}</turn> </s>")
            a_count += 1

    output.append("</conversation_history>")
    return "\n".join(reversed(output))

def parse_hybrid_history(text: str):
    user_re = re.compile(r"^\s*\|user\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    assistant_re = re.compile(r"^\s*\|assistant\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    history = []
    for ln in lines:
        u_m, a_m = user_re.match(ln), assistant_re.match(ln)
        if u_m:
            history.append({"role": "user", "text": u_m.group(1).strip()})
        elif a_m:
            history.append({"role": "assistant", "text": a_m.group(1).strip()})
    if not history:
        return text.strip(), []
    return history[-1]["text"], history[:-1]

def load_queries(path: str):
    qmap = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            query, hist = parse_hybrid_history(j.get("text", ""))
            qmap[j["_id"]] = {"query": query, "history": hist}
    return qmap

# ----------------------------
# 4. RRF FUSION
# ----------------------------
def reciprocal_rank_fusion(results_list: List[List[Dict]], k: int = 60) -> List[Dict]:
    """
    Merge multiple ranked lists using Reciprocal Rank Fusion
    results_list: List of ranked results, each result is [{"document_id": ..., "score": ...}, ...]
    k: RRF constant (default 60)
    """
    doc_scores = {}

    for results in results_list:
        for rank, doc in enumerate(results, start=1):
            doc_id = doc["document_id"]
            rrf_score = 1.0 / (k + rank)
            doc_scores[doc_id] = doc_scores.get(doc_id, 0.0) + rrf_score

    merged = [{"document_id": doc_id, "score": score} for doc_id, score in doc_scores.items()]
    merged.sort(key=lambda x: x["score"], reverse=True)

    return merged

# ----------------------------
# 5. QUERY DECOMPOSITION REWRITER
# ----------------------------
class QueryDecompositionRewriter:
    def __init__(self):
        self.name = "elser_decomposition_6U3A"

    def rewrite(self, query, history):
        if not history:
            return [query]

        hist_text = _history_to_text(history, user_max=6, assistant_max=3)

        system = (
            "You are an expert at decomposing complex queries into simpler sub-queries.\n\n"
            "TASK: Break down the user's final query into 2-4 standalone sub-queries that:\n"
            "1. Each sub-query is a COMPLETE, INDEPENDENT question\n"
            "2. Together, they cover ALL aspects of the original query\n"
            "3. Each can be answered separately\n"
            "4. Use context from conversation to resolve references\n\n"
            "WHEN TO DECOMPOSE:\n"
            "- Comparison queries: 'How do X and Y compare?' → separate queries for X and Y\n"
            "- Multi-aspect queries: 'What are the pros and cons?' → separate for pros and cons\n"
            "- Complex questions with AND/OR: Split into components\n"
            "- Questions about multiple entities: One sub-query per entity\n\n"
            "WHEN NOT TO DECOMPOSE:\n"
            "- Simple factoid questions: 'What is X?' → no decomposition needed\n"
            "- Single-entity queries: 'Tell me about X' → no decomposition\n"
            "- Already atomic questions\n\n"
            "OUTPUT FORMAT (strict JSON):\n"
            "{\n"
            "  \"should_decompose\": true/false,\n"
            "  \"reasoning\": \"<why decompose or not>\",\n"
            "  \"sub_queries\": [\n"
            "    \"<sub-query 1>\",\n"
            "    \"<sub-query 2>\",\n"
            "    \"<sub-query 3 (if needed)>\"\n"
            "  ]\n"
            "}\n\n"
            "If should_decompose=false, return the original query as a single sub-query.\n\n"
            "EXAMPLES:\n\n"
            "Example 1:\n"
            "Query: 'How do their salaries compare?'\n"
            "Context: Discussing CEO of Apple and CEO of Microsoft\n"
            "Output:\n"
            "{\n"
            "  \"should_decompose\": true,\n"
            "  \"reasoning\": \"Comparison query requiring information about two distinct entities\",\n"
            "  \"sub_queries\": [\n"
            "    \"What is the salary of the CEO of Apple?\",\n"
            "    \"What is the salary of the CEO of Microsoft?\"\n"
            "  ]\n"
            "}\n\n"
            "Example 2:\n"
            "Query: 'What are the advantages and disadvantages?'\n"
            "Context: Discussing electric cars\n"
            "Output:\n"
            "{\n"
            "  \"should_decompose\": true,\n"
            "  \"reasoning\": \"Multi-aspect query about pros and cons\",\n"
            "  \"sub_queries\": [\n"
            "    \"What are the advantages of electric cars?\",\n"
            "    \"What are the disadvantages of electric cars?\"\n"
            "  ]\n"
            "}\n\n"
            "Example 3:\n"
            "Query: 'What is the capital?'\n"
            "Context: Discussing France\n"
            "Output:\n"
            "{\n"
            "  \"should_decompose\": false,\n"
            "  \"reasoning\": \"Simple factoid question, no decomposition needed\",\n"
            "  \"sub_queries\": [\n"
            "    \"What is the capital of France?\"\n"
            "  ]\n"
            "}\n\n"
            "RULES:\n"
            "- Each sub-query must be standalone (no pronouns, complete context)\n"
            "- Keep sub-queries concise and focused\n"
            "- Aim for 2-4 sub-queries (not more)\n"
            "- Don't over-decompose simple queries\n"
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_decompose>\nuser: {query}\n</target_query_to_decompose>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 800)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                sub_queries = parsed.get("sub_queries", [query])
                if not sub_queries:
                    sub_queries = [query]
                return sub_queries
            return [query]
        except:
            return [query]

# ----------------------------
# 6. MULTI-CORPUS PIPELINE WITH DECOMPOSITION
# ----------------------------
from elasticsearch import Elasticsearch

def retrieve_with_decomposition(es_client, index_name, sub_queries: List[str], top_k: int = 100) -> List[Dict]:
    """
    Retrieve documents for multiple sub-queries and merge with RRF
    """
    all_results = []

    for sub_q in sub_queries:
        try:
            res = es_client.search(index=index_name, body={
                "query": {
                    "text_expansion": {
                        "text_embedding": {
                            "model_id": ".elser_model_1",
                            "model_text": sub_q
                        }
                    }
                },
                "size": top_k
            })
            results = [
                {"document_id": h["_source"].get("doc_id", h["_id"]), "score": h["_score"]}
                for h in res["hits"]["hits"]
            ]
            all_results.append(results)
        except Exception as e:
            print(f"⚠️ Retrieval error for sub-query '{sub_q}': {e}")
            all_results.append([])

    if not all_results:
        return []

    merged = reciprocal_rank_fusion(all_results, k=60)
    return merged[:top_k]

def run_corpus_experiment(corpus_config: dict, rewriter, es_client):
    corpus_name = corpus_config["name"]
    queries_file = corpus_config["queries_file"]
    index_name = corpus_config["index_name"]
    collection_id = corpus_config["collection_id"]

    print("\n" + "="*70)
    print(f"🎯 CORPUS: {corpus_name.upper()}")
    print(f"📝 Rewriter: {rewriter.name}")
    print("="*70)

    data = load_queries(queries_file)
    items = list(data.items())
    print(f"✅ Loaded {len(items)} queries for {corpus_name}")

    print(f"📝 Decomposing queries with {rewriter.name}...")
    decompositions_map = {}
    with ThreadPoolExecutor(max_workers=5) as ex:
        futs = {ex.submit(rewriter.rewrite, it["query"], it["history"]): qid for qid, it in items}
        for f in tqdm(as_completed(futs), total=len(futs), desc=f"Decomposing {corpus_name}"):
            decompositions_map[futs[f]] = f.result()

    RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
    decomp_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}_DECOMPOSITIONS.json")

    decomp_data = []
    for qid, item in items:
        sub_queries = decompositions_map.get(qid, [item["query"]])
        decomp_data.append({
            "qid": qid,
            "original_query": item["query"],
            "history": item["history"],
            "sub_queries": sub_queries,
            "num_sub_queries": len(sub_queries)
        })

    with open(decomp_file, "w", encoding="utf-8") as f:
        json.dump(decomp_data, f, indent=2, ensure_ascii=False)

    print(f"💾 Decompositions saved to: {decomp_file}")

    avg_subq = sum(len(d["sub_queries"]) for d in decomp_data) / len(decomp_data)
    print(f"📊 Average sub-queries per query: {avg_subq:.2f}")

    print(f"🔍 Retrieving from {index_name} with RRF fusion...")
    pred_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}.jsonl")

    with open(pred_file, "w", encoding="utf-8") as out:
        for qid, item in tqdm(items, desc=f"Retrieving {corpus_name}"):
            sub_queries = decompositions_map.get(qid, [item["query"]])
            contexts = retrieve_with_decomposition(es_client, index_name, sub_queries, top_k=100)

            out.write(json.dumps({
                "task_id": qid,
                "Collection": collection_id,
                "contexts": contexts
            }, ensure_ascii=False) + "\n")

    print(f"✅ Predictions saved to: {pred_file}")
    return pred_file, decomp_file

def run_evaluation(pred_file: str):
    print(f"\n🧪 Evaluating {os.path.basename(pred_file)}...")
    output_eval_file = pred_file.replace(".jsonl", "_results.jsonl")

    cmd = f"python {EVAL_SCRIPT} --input_file {pred_file} --output_file {output_eval_file}"
    result = os.system(cmd)

    if result != 0:
        print(f"⚠️ Warning: Evaluation failed")
        return None

    print(f"✅ Evaluation completed")
    return output_eval_file

def aggregate_results(eval_files: List[str]):
    all_data = []

    for eval_file in eval_files:
        if eval_file is None:
            continue

        base_name = os.path.basename(eval_file).replace("_results.jsonl", "")
        pred_dir = os.path.dirname(eval_file)
        agg_csv = os.path.join(pred_dir, f"{base_name}_results_aggregate.csv")

        if os.path.exists(agg_csv):
            df = pd.read_csv(agg_csv)
            df = df[df["collection"] != "all"]
            corpus_name = base_name.split("_")[0]
            df["Corpus"] = corpus_name
            all_data.append(df)
        else:
            print(f"⚠️ Aggregate CSV not found: {agg_csv}")

    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)

        print("\n" + "="*70)
        print("📊 FINAL RESULTS - QUERY DECOMPOSITION")
        print("="*70)
        display(final_df)

        combined_file = os.path.join(PRED_DIR, f"decomposition_results_{time.strftime('%Y%m%d_%H%M%S')}.csv")
        final_df.to_csv(combined_file, index=False)
        print(f"\n💾 Combined results saved to: {combined_file}")

        print("\n" + "="*70)
        print("📈 SUMMARY BY CORPUS")
        print("="*70)
        summary = final_df.groupby('Corpus')[['nDCG@10', 'Recall@5', 'Recall@100']].mean()
        display(summary)

        return final_df
    else:
        print("⚠️ No results to aggregate")
        return None

# ----------------------------
# 7. MAIN EXECUTION
# ----------------------------
def main():
    print("\n" + "="*70)
    print("🚀 ELSER v1 - QUERY DECOMPOSITION EXPERIMENT")
    print("="*70)
    print("Strategy: Decompose complex queries + RRF fusion")
    print(f"Corpora: {', '.join([c['name'] for c in CORPORA])}")
    print("Model: DeepSeek-V3.2 via Azure")
    print("="*70)

    es_client = Elasticsearch(
        cloud_id=CLOUD_ID,
        basic_auth=("elastic", PASSWORD),
        request_timeout=180
    )

    rewriter = QueryDecompositionRewriter()
    eval_files = []
    decomp_files = []

    for corpus_config in CORPORA:
        try:
            pred_file, decomp_file = run_corpus_experiment(corpus_config, rewriter, es_client)
            eval_file = run_evaluation(pred_file)
            eval_files.append(eval_file)
            decomp_files.append(decomp_file)
        except Exception as e:
            print(f"❌ Error processing {corpus_config['name']}: {e}")
            import traceback
            traceback.print_exc()
            eval_files.append(None)
            decomp_files.append(None)

    aggregate_results(eval_files)

    print("\n" + "="*70)
    print("💰 COST SUMMARY")
    print("="*70)
    print(tracker.get_report())
    print("="*70)

    print("\n" + "="*70)
    print("💾 SAVED DECOMPOSITION FILES")
    print("="*70)
    for df in decomp_files:
        if df:
            print(f"  ✅ {df}")
    print("="*70)

if __name__ == "__main__":
    main()


In [ ]:
# ==============================================================================
# ELSER v1 — MASTER EXPERIMENT CODE (DEEPSEEK-V3.2 via AZURE FOUNDRY)
# VERSION: INTENT-CONTEXTUAL REWRITER (ICR) v1
# STRATEGY: Semantic Intent Expansion & Domain Translation
# MULTI-CORPUS: ClapNQ, FiQA, Govt, Cloud
# ==============================================================================

import os, json, re, time, threading, requests
from typing import List, Dict, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from google.colab import drive
from elasticsearch import Elasticsearch

# Mount Drive
drive.mount('/content/drive', force_remount=True)

# ----------------------------
# 1. SETTINGS & CONFIG
# ----------------------------
BASE_PATH = "/content/drive/MyDrive/Project"
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]

EVAL_SCRIPT = f"{BASE_PATH}/run_retrieval_eval.py"
PRED_DIR = f"{BASE_PATH}/elser_v1_ICR_predictions"
os.makedirs(PRED_DIR, exist_ok=True)

CORPORA = [
    {"name": "clapnq", "queries_file": f"{BASE_PATH}/retrieval/clapnq/clapnq_questions.jsonl", "index_name": "clapnq_elser_v1", "collection_id": "mt-rag-clapnq-elser-512-100-20240503"},
    {"name": "fiqa", "queries_file": f"{BASE_PATH}/retrieval/fiqa/fiqa_questions.jsonl", "index_name": "fiqa_elser_v1", "collection_id": "mt-rag-fiqa-beir-elser-512-100-20240501"},
    {"name": "govt", "queries_file": f"{BASE_PATH}/retrieval/govt/govt_questions.jsonl", "index_name": "govt_elser_v1", "collection_id": "mt-rag-govt-elser-512-100-20240611"},
    {"name": "cloud", "queries_file": f"{BASE_PATH}/retrieval/cloud/cloud_questions.jsonl", "index_name": "cloud_elser_v1", "collection_id": "mt-rag-ibmcloud-elser-512-100-20240502"}
]

# ----------------------------
# 2. LLM & COST TRACKING
# ----------------------------
class CostTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt, completion):
        with self.lock:
            self.total_input += prompt
            self.total_output += completion

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        return f"In: {self.total_input} | Out: {self.total_output} | Est. Cost: ${cost:.4f}"

tracker = CostTracker()

def llm_invoke_text(system: str, user: str, max_tokens: int) -> str:
    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-V3.2",
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "max_tokens": max_tokens,
        "temperature": 0.0
    }
    for attempt in range(7):
        try:
            response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
            if response.status_code == 200:
                res_json = response.json()
                usage = res_json.get('usage', {})
                tracker.add_usage(usage.get('prompt_tokens', 0), usage.get('completion_tokens', 0))
                return res_json['choices'][0]['message']['content'].strip()
            elif response.status_code == 429:
                time.sleep(5 * (2 ** attempt))
                continue
        except:
            time.sleep(5)
    return ""

# ----------------------------
# 3. HISTORY FORMATTING (DAMO-NLP STYLE)
# ----------------------------
def _history_to_text(history, user_max=6, assistant_max=3):
    output = ["<conversation_history>"]
    u_count, a_count = 0, 0
    for h in reversed(history):
        if h['role'] == 'user' and u_count < user_max:
            output.append(f"  <turn role='user'>{h['text']}</turn> </s>")
            u_count += 1
        elif h['role'] == 'assistant' and a_count < assistant_max:
            output.append(f"  <turn role='assistant'>{h['text']}</turn> </s>")
            a_count += 1
    output.append("</conversation_history>")
    return "\n".join(reversed(output))

def parse_hybrid_history(text: str):
    user_re = re.compile(r"^\s*\|user\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    assistant_re = re.compile(r"^\s*\|assistant\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    history = []
    for ln in lines:
        u_m, a_m = user_re.match(ln), assistant_re.match(ln)
        if u_m: history.append({"role": "user", "text": u_m.group(1).strip()})
        elif a_m: history.append({"role": "assistant", "text": a_m.group(1).strip()})
    if not history: return text.strip(), []
    return history[-1]["text"], history[:-1]

def load_queries(path: str):
    qmap = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            query, hist = parse_hybrid_history(j.get("text", ""))
            qmap[j["_id"]] = {"query": query, "history": hist}
    return qmap

# ----------------------------
# 4. THE INTENT-CONTEXTUAL REWRITER (ICR)
# ----------------------------
class ELSERIntentContextualRewriter:
    def __init__(self):
        self.name = "elser_icr_semantic_v1"

    def rewrite(self, query, history):
        hist_text = _history_to_text(history) if history else "<no_history>"

        system = (
            "You are an Expert Search Engineer. Your goal is to rewrite the user's last utterance "
            "into a semantic search query that bridges the gap between casual language and professional documentation.\n\n"
            "Apply these Strategic Transformations:\n"
            "1. SUCCESSION & SEQUENCING: If the query asks for 'next', 'after', or 'rival', focus on identifying the SUCCESSOR or OPPONENT entity. "
            "2. ENTITY ABSTRACTION: For technical topics, include the 'Parent Category' or 'Umbrella Service'.\n"
            "3. COMPARATIVE DECOMPOSITION: Separate comparative entities A vs B into independent semantic fields.\n"
            "4. DOMAIN TRANSLATION: For financial queries, use professional terminology (e.g., 'money' -> 'capital', 'pay' -> 'compensation').\n"
            "5. ANCESTRY MAPPING: Expand 'origin' queries to the historical roots of the broader concept.\n\n"
            "Output ONLY JSON: {\"intent_analysis\": \"...\", \"rewritten_query\": \"...\"}"
        )

        prompt_user = (
            f"STRUCTURAL_CONTEXT_START\n{hist_text}\nSTRUCTURAL_CONTEXT_END\n\n"
            f"<target_utterance>\n{query}\n</target_utterance>\n\n"
            "Identify the Lexical Trap and expand keywords with domain jargon.\n"
            "ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 500)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                return [parsed.get("rewritten_query", query)]
            return [query]
        except: return [query]

# ----------------------------
# 5. PIPELINE EXECUTION
# ----------------------------
def run_corpus_experiment(corpus_config, rewriter, es_client):
    corpus_name = corpus_config["name"]
    data = load_queries(corpus_config["queries_file"])
    items = list(data.items())

    print(f"\n📝 Rewriting {corpus_name}...")
    rewrites_map = {}
    with ThreadPoolExecutor(max_workers=5) as ex:
        futs = {ex.submit(rewriter.rewrite, it["query"], it["history"]): qid for qid, it in items}
        for f in tqdm(as_completed(futs), total=len(futs)):
            rewrites_map[futs[f]] = f.result()[0]

    # Save rewrites
    RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
    rewrites_file = os.path.join(PRED_DIR, f"{corpus_name}_REWRITES_{RUN_TAG}.json")
    with open(rewrites_file, "w", encoding="utf-8") as f:
        json.dump(rewrites_map, f, indent=2, ensure_ascii=False)

    # Retrieval
    print(f"🔍 Retrieving {corpus_name}...")
    pred_file = os.path.join(PRED_DIR, f"{corpus_name}_preds_{RUN_TAG}.jsonl")
    with open(pred_file, "w", encoding="utf-8") as out:
        for qid, it in tqdm(items):
            q_final = rewrites_map.get(qid, it["query"])
            res = es_client.search(index=corpus_config["index_name"], body={
                "query": {"text_expansion": {"text_embedding": {"model_id": ".elser_model_1", "model_text": q_final}}},
                "size": 100
            })
            contexts = [{"document_id": h["_source"].get("doc_id", h["_id"]), "score": h["_score"]} for h in res["hits"]["hits"]]
            out.write(json.dumps({"task_id": qid, "Collection": corpus_config["collection_id"], "contexts": contexts}) + "\n")

    return pred_file

def main():
    es_client = Elasticsearch(cloud_id=CLOUD_ID, basic_auth=("elastic", PASSWORD), request_timeout=180)
    rewriter = ELSERIntentContextualRewriter()

    for corpus_config in CORPORA:
        try:
            pred_file = run_corpus_experiment(corpus_config, rewriter, es_client)
            # Evaluation
            output_eval = pred_file.replace(".jsonl", "_results.jsonl")
            os.system(f"python {EVAL_SCRIPT} --input_file {pred_file} --output_file {output_eval}")
        except Exception as e: print(f"❌ Error in {corpus_config['name']}: {e}")

    print("\n" + "="*50 + "\n" + tracker.get_report() + "\n" + "="*50)

if __name__ == "__main__":
    main()

In [ ]:
# ==============================================================================
# ELSER v1 — MASTER EXPERIMENT (AZURE FOUNDRY chat/completions)
# VERSION: 6U/1A XML Context + ANCHOR/KEYWORD COMBO REWRITER (ELSER-friendly)
# MULTI-CORPUS: ClapNQ, FiQA, Govt, Cloud
# Saves rewrites to JSON for reuse + runs retrieval + runs eval + aggregates
# ==============================================================================

# --- OPTIONAL: installs (Colab) ---
%pip -q install --upgrade elasticsearch requests tqdm pandas numpy

import os, json, re, time, threading, requests, glob
from typing import List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from google.colab import drive
from elasticsearch import Elasticsearch

# Mount Drive
drive.mount("/content/drive", force_remount=True)

# ----------------------------
# 1) SETTINGS & CONFIG
# ----------------------------
BASE_PATH = "/content/drive/MyDrive/Project"

AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]  # <-- ΒΑΛΕ ΤΟ KEY ΣΟΥ ΕΔΩ

CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]  # <-- ΒΑΛΕ ΤΟ PASS ΣΟΥ

EVAL_SCRIPT = f"{BASE_PATH}/run_retrieval_eval.py"
PRED_DIR = f"{BASE_PATH}/elser_v1_anchor_keyword_predictions"
os.makedirs(PRED_DIR, exist_ok=True)

# Για smoke-test βάλε π.χ. 1 ή 10
MAX_QUERIES = None  # None = όλα

# MULTI-CORPUS CONFIGURATION
CORPORA = [
    {
        "name": "clapnq",
        "queries_file": f"{BASE_PATH}/retrieval/clapnq/clapnq_questions.jsonl",
        "index_name": "clapnq_elser_v1",
        "collection_id": "mt-rag-clapnq-elser-512-100-20240503",
    },
    {
        "name": "fiqa",
        "queries_file": f"{BASE_PATH}/retrieval/fiqa/fiqa_questions.jsonl",
        "index_name": "fiqa_elser_v1",
        "collection_id": "mt-rag-fiqa-beir-elser-512-100-20240501",
    },
    {
        "name": "govt",
        "queries_file": f"{BASE_PATH}/retrieval/govt/govt_questions.jsonl",
        "index_name": "govt_elser_v1",
        "collection_id": "mt-rag-govt-elser-512-100-20240611",
    },
    {
        "name": "cloud",
        "queries_file": f"{BASE_PATH}/retrieval/cloud/cloud_questions.jsonl",
        "index_name": "cloud_elser_v1",
        "collection_id": "mt-rag-ibmcloud-elser-512-100-20240502",
    },
]

# ----------------------------
# 2) COST TRACKER
# ----------------------------
class CostTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt_tokens, completion_tokens):
        with self.lock:
            self.total_input += int(prompt_tokens or 0)
            self.total_output += int(completion_tokens or 0)

    def get_report(self):
        # Ενδεικτική κοστολόγηση (βάλε τις δικές σου τιμές αν χρειάζεται)
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        return f"In: {self.total_input} | Out: {self.total_output} | Est. Cost: ${cost:.4f}"

tracker = CostTracker()

def llm_invoke_text(system: str, user: str, max_tokens: int) -> str:
    """Azure DS chat/completions call with retries + token usage tracking."""
    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-V3.2",
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "max_tokens": max_tokens,
        "temperature": 0.0,
    }

    for attempt in range(7):
        try:
            r = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)

            # Retry on rate limit / transient server errors
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(3 * (2 ** attempt))
                continue

            r.raise_for_status()
            res_json = r.json()

            usage = res_json.get("usage", {})
            tracker.add_usage(usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0))

            return res_json["choices"][0]["message"]["content"].strip()

        except Exception:
            time.sleep(2 * (2 ** attempt))

    return ""

# ----------------------------
# 3) PARSING & STRUCTURAL HISTORY (FIXED ORDER)
# ----------------------------
def _history_to_text(history, user_max=6, assistant_max=1):
    """
    XML history formatting (fixed): produces valid tag order + chronological turns.
    Keeps last N user/assistant turns.
    """
    turns = []
    u_count = a_count = 0

    # walk from most recent backwards so we can cap properly
    for h in reversed(history):
        if h["role"] == "user" and u_count < user_max:
            turns.append(f"  <turn role='user'>{h['text']}</turn>")
            u_count += 1
        elif h["role"] == "assistant" and a_count < assistant_max:
            turns.append(f"  <turn role='assistant'>{h['text']}</turn>")
            a_count += 1

    turns.reverse()  # now oldest -> newest
    return "<conversation_history>\n" + "\n".join(turns) + "\n</conversation_history>"

def parse_hybrid_history(text: str):
    user_re = re.compile(r"^\s*\|user\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    assistant_re = re.compile(r"^\s*\|assistant\|\s*:\s*(.*)\s*$", re.IGNORECASE)

    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    history = []

    for ln in lines:
        u_m, a_m = user_re.match(ln), assistant_re.match(ln)
        if u_m:
            history.append({"role": "user", "text": u_m.group(1).strip()})
        elif a_m:
            history.append({"role": "assistant", "text": a_m.group(1).strip()})

    if not history:
        return text.strip(), []

    return history[-1]["text"], history[:-1]

def load_queries(path: str):
    qmap = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            query, hist = parse_hybrid_history(j.get("text", ""))
            qmap[j["_id"]] = {"query": query, "history": hist}
    return qmap

# ----------------------------
# 4) REWRITER: ANCHOR + KEYWORD COMBO (ELSER-friendly)
# ----------------------------
def _norm_term(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[\"'`]", "", s)
    return s

def _dedupe_terms(terms: List[str]) -> List[str]:
    seen = set()
    out = []
    for t in terms:
        if not t:
            continue
        key = _norm_term(t)
        if key and key not in seen:
            seen.add(key)
            out.append(t.strip())
    return out

def _cap_by_words(text: str, max_words: int) -> str:
    w = text.split()
    return text.strip() if len(w) <= max_words else " ".join(w[:max_words]).strip()

class ELSERAnchorKeywordRewriter:
    """
    Same logic as your minimal:
    - if no history -> return original query
    - else call LLM with structured history + strict JSON
    - return 1 string (final query) in a list
    """
    def __init__(self):
        self.name = "elser_anchor_keyword_combo_6U1A"
        self.user_max = 6
        self.assistant_max = 1
        self.max_words = 28
        self.max_anchors = 8
        self.max_keywords = 12

    def rewrite(self, query, history):
        if not history:
            return [query]

        hist_text = _history_to_text(history, user_max=self.user_max, assistant_max=self.assistant_max)

        system = (
            "Given the following conversation, rewrite the final user utterance into ONE standalone query for RETRIEVAL.\n\n"
            "Additionally, extract compact RETRIEVAL TERMS to help ELSER sparse search:\n"
            "- anchors: exact entity names, titles, product/service names, acronyms, IDs, error codes, CLI flags\n"
            "- keywords: short bag-of-terms reflecting core intent\n\n"
            "Rules:\n"
            "- Do NOT invent new entities/facts.\n"
            "- Be minimal; only resolve pronouns/ambiguous references using the conversation.\n"
            "- Preserve numbers/codes/tickers exactly.\n"
            f"- anchors max {self.max_anchors}, keywords max {self.max_keywords}.\n"
            f"- Keep the rewritten query short (aim <= {self.max_words} words).\n\n"
            "Output STRICT JSON ONLY, in this exact schema:\n"
            "{\"class\":\"standalone|non-standalone\",\"rewritten version\":\"...\",\"anchors\":[\"...\"],\"keywords\":[\"...\"]}"
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 450)
            m = re.search(r"\{.*\}", txt, re.DOTALL)
            if not m:
                return [query]

            obj = json.loads(m.group(0))

            base = (obj.get("rewritten version") or query).strip()

            anchors = obj.get("anchors") or []
            keywords = obj.get("keywords") or []

            if not isinstance(anchors, list): anchors = []
            if not isinstance(keywords, list): keywords = []

            anchors = _dedupe_terms([str(x) for x in anchors])[: self.max_anchors]
            keywords = _dedupe_terms([str(x) for x in keywords])[: self.max_keywords]

            combined = " ".join([base] + anchors + keywords).strip()
            combined = _cap_by_words(combined, self.max_words)

            return [combined if combined else query]

        except Exception:
            return [query]

# ----------------------------
# 5) MULTI-CORPUS PIPELINE
# ----------------------------
def run_corpus_experiment(corpus_config: dict, rewriter, es_client):
    corpus_name = corpus_config["name"]
    queries_file = corpus_config["queries_file"]
    index_name = corpus_config["index_name"]
    collection_id = corpus_config["collection_id"]

    print("\n" + "=" * 70)
    print(f"🎯 CORPUS: {corpus_name.upper()}")
    print("=" * 70)

    data = load_queries(queries_file)
    items = list(data.items())

    if MAX_QUERIES is not None:
        items = items[: int(MAX_QUERIES)]

    print(f"✅ Loaded {len(items)} queries for {corpus_name}")

    # Rewrite queries
    print(f"📝 Rewriting queries with {rewriter.name}...")
    rewrites_map = {}

    with ThreadPoolExecutor(max_workers=5) as ex:
        futs = {ex.submit(rewriter.rewrite, it["query"], it["history"]): qid for qid, it in items}
        for f in tqdm(as_completed(futs), total=len(futs), desc=f"Rewriting {corpus_name}"):
            qid = futs[f]
            try:
                rewrites_map[qid] = f.result()[0]
            except Exception:
                rewrites_map[qid] = data[qid]["query"]

    # Save rewrites
    RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
    rewrites_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}_REWRITES.json")

    rewrites_data = []
    for qid, item in items:
        rewrites_data.append({
            "qid": qid,
            "original_query": item["query"],
            "history": item["history"],
            "rewritten_query": rewrites_map.get(qid, item["query"]),
        })

    with open(rewrites_file, "w", encoding="utf-8") as f:
        json.dump(rewrites_data, f, indent=2, ensure_ascii=False)

    print(f"💾 Rewrites saved to: {rewrites_file}")

    # Retrieval
    print(f"🔍 Retrieving from {index_name}...")
    pred_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}.jsonl")

    with open(pred_file, "w", encoding="utf-8") as out:
        for qid, item in tqdm(items, desc=f"Retrieving {corpus_name}"):
            q_final = rewrites_map.get(qid, item["query"])
            try:
                res = es_client.search(
                    index=index_name,
                    body={
                        "query": {
                            "text_expansion": {
                                "text_embedding": {
                                    "model_id": ".elser_model_1",
                                    "model_text": q_final,
                                }
                            }
                        },
                        "size": 100,
                    },
                )
                contexts = [
                    {"document_id": h["_source"].get("doc_id", h["_id"]), "score": h["_score"]}
                    for h in res["hits"]["hits"]
                ]
            except Exception as e:
                print(f"⚠️ Retrieval error for {qid}: {e}")
                contexts = []

            out.write(json.dumps({
                "task_id": qid,
                "Collection": collection_id,   # κράτα όπως το έχεις ήδη
                "contexts": contexts,
            }, ensure_ascii=False) + "\n")

    print(f"✅ Predictions saved to: {pred_file}")
    return pred_file, rewrites_file

def run_evaluation(pred_file: str):
    print(f"\n🧪 Evaluating {os.path.basename(pred_file)}...")

    output_eval_base = pred_file.replace(".jsonl", "")
    cmd = f"python {EVAL_SCRIPT} --input_file {pred_file} --output_file {output_eval_base}_results.jsonl"
    result = os.system(cmd)

    if result != 0:
        print("⚠️ Warning: Evaluation failed")
        return None

    print("✅ Evaluation completed")
    return output_eval_base + "_results.jsonl"

def aggregate_results(eval_files: List[str]):
    all_data = []

    for eval_file in eval_files:
        if eval_file is None:
            continue

        agg_csv = eval_file.replace("_results.jsonl", "_results_aggregate.csv")

        if os.path.exists(agg_csv):
            df = pd.read_csv(agg_csv)
            df = df[df["collection"] != "all"]

            corpus_name = os.path.basename(eval_file).split("_")[0]
            df["Corpus"] = corpus_name
            all_data.append(df)
        else:
            print(f"⚠️ Aggregate CSV not found: {agg_csv}")
            eval_dir = os.path.dirname(eval_file)
            eval_base = os.path.basename(eval_file).replace("_results.jsonl", "")
            matches = glob.glob(f"{eval_dir}/{eval_base}*aggregate*.csv")
            if matches:
                print(f"   Found: {[os.path.basename(m) for m in matches]}")

    if not all_data:
        print("⚠️ No results to aggregate")
        return None

    final_df = pd.concat(all_data, ignore_index=True)

    # display if available
    try:
        from IPython.display import display
        print("\n" + "=" * 70)
        print("📊 FINAL RESULTS - ALL CORPORA")
        print("=" * 70)
        display(final_df)
    except Exception:
        print(final_df.head())

    combined_file = os.path.join(PRED_DIR, f"combined_results_{time.strftime('%Y%m%d_%H%M%S')}.csv")
    final_df.to_csv(combined_file, index=False)
    print(f"\n💾 Combined results saved to: {combined_file}")

    # Summary
    try:
        summary = final_df.groupby("Corpus")[["nDCG@10", "Recall@5", "Recall@100"]].mean()
        print("\n" + "=" * 70)
        print("📈 SUMMARY BY CORPUS")
        print("=" * 70)
        try:
            from IPython.display import display
            display(summary)
        except Exception:
            print(summary)
    except Exception as e:
        print(f"⚠️ Could not compute summary: {e}")

    return final_df

# ----------------------------
# 6) MAIN EXECUTION
# ----------------------------
def main():
    if not AZURE_DS_KEY:
        raise ValueError("Βάλε AZURE_DS_KEY πριν τρέξεις το πείραμα.")
    if not PASSWORD or PASSWORD == "........":
        raise ValueError("Βάλε Elastic PASSWORD πριν τρέξεις το πείραμα.")

    print("\n" + "=" * 70)
    print("🚀 ELSER v1 - ANCHOR/KEYWORD COMBO REWRITER - MULTI-CORPUS EXPERIMENT")
    print("=" * 70)
    print("Strategy: 6U/1A XML context + compact anchors/keywords (ELSER-friendly)")
    print(f"Corpora: {', '.join([c['name'] for c in CORPORA])}")
    print(f"Rewriter: {ELSERAnchorKeywordRewriter().name}")
    if MAX_QUERIES is not None:
        print(f"MAX_QUERIES (smoke test): {MAX_QUERIES}")
    print("=" * 70)

    es_client = Elasticsearch(
        cloud_id=CLOUD_ID,
        basic_auth=("elastic", PASSWORD),
        request_timeout=180,
    )

    rewriter = ELSERAnchorKeywordRewriter()
    eval_files = []
    rewrite_files = []

    for corpus_config in CORPORA:
        try:
            pred_file, rewrites_file = run_corpus_experiment(corpus_config, rewriter, es_client)
            eval_file = run_evaluation(pred_file)
            eval_files.append(eval_file)
            rewrite_files.append(rewrites_file)
        except Exception as e:
            print(f"❌ Error processing {corpus_config['name']}: {e}")
            eval_files.append(None)
            rewrite_files.append(None)

    aggregate_results(eval_files)

    print("\n" + "=" * 70)
    print("💰 COST SUMMARY")
    print("=" * 70)
    print(tracker.get_report())
    print("=" * 70)

    print("\n" + "=" * 70)
    print("💾 SAVED REWRITE FILES")
    print("=" * 70)
    for rf in rewrite_files:
        if rf:
            print(f"  ✅ {rf}")
    print("=" * 70)

if __name__ == "__main__":
    main()


## Question type rewriter

In [ ]:
# ==============================================================================
# ELSER v1 — FIXED TYPE-AWARE REWRITER
# VERSION: Addresses over-Factoid prediction + Multi-turn detection issues
# ==============================================================================


import os, json, re, time, threading, requests, glob
from typing import List, Dict, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd
from google.colab import drive
from elasticsearch import Elasticsearch


drive.mount("/content/drive", force_remount=True)


BASE_PATH = "/content/drive/MyDrive/Project"
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]


CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]


EVAL_SCRIPT = f"{BASE_PATH}/run_retrieval_eval.py"
PRED_DIR = f"{BASE_PATH}/elser_v1_fixed_type_predictions"
os.makedirs(PRED_DIR, exist_ok=True)


MAX_QUERIES = None


CORPORA = [
    {"name": "clapnq", "queries_file": f"{BASE_PATH}/retrieval/clapnq/clapnq_questions.jsonl", "index_name": "clapnq_elser_v1", "collection_id": "mt-rag-clapnq-elser-512-100-20240503"},
    {"name": "fiqa", "queries_file": f"{BASE_PATH}/retrieval/fiqa/fiqa_questions.jsonl", "index_name": "fiqa_elser_v1", "collection_id": "mt-rag-fiqa-beir-elser-512-100-20240501"},
    {"name": "govt", "queries_file": f"{BASE_PATH}/retrieval/govt/govt_questions.jsonl", "index_name": "govt_elser_v1", "collection_id": "mt-rag-govt-elser-512-100-20240611"},
    {"name": "cloud", "queries_file": f"{BASE_PATH}/retrieval/cloud/cloud_questions.jsonl", "index_name": "cloud_elser_v1", "collection_id": "mt-rag-ibmcloud-elser-512-100-20240502"},
]


class CostTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.lock = threading.Lock()


    def add_usage(self, prompt_tokens, completion_tokens):
        with self.lock:
            self.total_input += int(prompt_tokens or 0)
            self.total_output += int(completion_tokens or 0)


    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        return f"In: {self.total_input} | Out: {self.total_output} | Est. Cost: ${cost:.4f}"


tracker = CostTracker()


def llm_invoke_text(system: str, user: str, max_tokens: int = 500) -> str:
    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-V3.2",
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "max_tokens": max_tokens,
        "temperature": 0.0
    }


    for attempt in range(7):
        try:
            r = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(3 * (2 ** attempt))
                continue
            r.raise_for_status()
            res_json = r.json()
            usage = res_json.get("usage", {})
            tracker.add_usage(usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0))
            return res_json["choices"][0]["message"]["content"].strip()
        except:
            time.sleep(2 * (2 ** attempt))
    return ""


def parse_hybrid_history(text: str):
    user_re = re.compile(r"^\s*\|user\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    assistant_re = re.compile(r"^\s*\|assistant\|\s*:\s*(.*)\s*$", re.IGNORECASE)


    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    history = []


    for ln in lines:
        u_m, a_m = user_re.match(ln), assistant_re.match(ln)
        if u_m:
            history.append({"role": "user", "text": u_m.group(1).strip()})
        elif a_m:
            history.append({"role": "assistant", "text": a_m.group(1).strip()})


    if not history:
        return text.strip(), []


    return history[-1]["text"], history[:-1]


def load_queries(path: str):
    qmap = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            query, hist = parse_hybrid_history(j.get("text", ""))
            qmap[j["_id"]] = {"query": query, "history": hist}
    return qmap


# ----------------------------
# FIXED REWRITER
# ----------------------------
class FixedTypeAwareRewriter:
    """
    Fixes:
    1. Over-prediction of Factoid
    2. Multi-turn classification (0% accuracy)
    3. Non-standalone query resolution (complete failures)
    """
    def __init__(self):
        self.name = "fixed_type_aware_v2"


    def _build_system_prompt(self) -> str:
        return """You are an expert Query Rewriter for ELSER v1 sparse retrieval. Your goal: Make non-standalone queries standalone, then classify by type.


═══════════════════════════════════════════════════════════════════════
⚠️ CRITICAL: AVOID OVER-PREDICTING "Factoid"
═══════════════════════════════════════════════════════════════════════
Before choosing Factoid, check if the query is actually:
- **Explanation** (why/how/reason) → NOT Factoid!
- **Summarization** (summary/overview/policy) → NOT Factoid!
- **Keyword** (fragments like "python pandas merge") → NOT Factoid!
- **How-To** (instructions to do something) → NOT Factoid!


Only choose Factoid if it asks for a SINGLE SPECIFIC FACT (name, date, number, yes/no).


═══════════════════════════════════════════════════════════════════════
STEP 1: MULTI-TURN CLASSIFICATION (FIXED)
═══════════════════════════════════════════════════════════════════════
- **first_turn**: No history OR query is 100% standalone (full sentence with no pronouns)
- **follow-up**: Query continues/expands current topic
  * Examples: "What about its features?", "Tell me more", "Any other examples?"
- **clarification**: Query asks to EXPLAIN previous answer
  * Examples: "What do you mean by X?", "Can you clarify that?", "Explain more about..."


🔍 Detection signals:
- Pronouns (it, they, that, this, those) → follow-up or clarification
- Starts with "What about", "How about", "And" → follow-up
- "Explain", "clarify", "what do you mean" → clarification
- Very short (<5 words) with history → follow-up


═══════════════════════════════════════════════════════════════════════
STEP 2: RESOLVE NON-STANDALONE QUERIES (CRITICAL FIX)
═══════════════════════════════════════════════════════════════════════
If follow-up or clarification:
1. Identify the MAIN ENTITY from recent history (last 2-3 turns)
2. Replace pronouns with explicit entity names
3. Add missing context to make query standalone


Examples:
- "any reason to end?" + History: "The Office TV series"
  → "why did The Office TV series end"


- "who was the arch rival to this series?" + History: "Friends TV show"
  → "who was the arch rival TV show to Friends"


- "any awards" + History: "Breaking Bad"
  → "Breaking Bad awards won"


- "When was it found?" + History: "Rosetta Stone"
  → "when was Rosetta Stone found discovered"


═══════════════════════════════════════════════════════════════════════
STEP 3: QUESTION TYPE CLASSIFICATION (10 Types)
═══════════════════════════════════════════════════════════════════════


1. **Factoid** (⚠️ Use SPARINGLY)
   - Asks for ONE specific fact: name, date, quantity, yes/no
   - Examples: "Who is CEO?", "When was X founded?", "How many episodes?"
   - NOT: "Why", "How to", "Tell me about", "Summary"


2. **Explanation**
   - Asks WHY, HOW something works, REASON, CAUSE
   - Keywords: "why", "how does", "reason", "cause", "explain why"
   - Examples: "Why did the stock crash?", "How does photosynthesis work?"


3. **Summarization**
   - Requests summary, overview, policy, general information
   - Keywords: "summarize", "overview", "tell me about", "what is X" (when X is broad topic)
   - Examples: "Summarize the privacy policy", "Tell me about quantum computing"


4. **How-To**
   - Requests INSTRUCTIONS or STEPS to do something
   - Keywords: "how to", "how do I", "steps to", "guide"
   - Examples: "How to reset password?", "How do I install Docker?"


5. **Comparative**
   - Compares 2+ entities
   - Keywords: "vs", "versus", "difference", "compare", "better than"
   - Examples: "iPhone vs Samsung", "Difference between X and Y"


6. **Keyword**
   - Fragmented keywords, NOT a full sentence
   - Examples: "python pandas merge", "react hooks useState", "docker compose"


7. **Composite**
   - Multiple sub-questions in ONE query
   - Keywords: "and", multiple question words
   - Examples: "What is X, how to use it, and what are benefits?"


8. **Troubleshooting**
   - Solving error/problem/bug
   - Keywords: "error", "not working", "fix", "problem", "issue"
   - Examples: "Error 404 when accessing API", "Cannot connect to database"


9. **Non-question**
   - User statement, not asking anything
   - Examples: "I am in California", "The API is slow", "Thank you"


10. **Opinion**
    - Asks for subjective view, preference, recommendation
    - Keywords: "best", "worth it", "recommend", "should I"
    - Examples: "Is Tesla worth buying?", "Best programming language?"


═══════════════════════════════════════════════════════════════════════
STEP 4: OUTPUT FORMAT
═══════════════════════════════════════════════════════════════════════
Return STRICT JSON:
{
  "question_type": "one of 10 types - BE CAREFUL not to over-use Factoid",
  "multi_turn_type": "first_turn | follow-up | clarification",
  "rewritten_query": "STANDALONE query with pronouns resolved and context added",
  "reasoning": "brief explanation: why this type, what context was added"
}


═══════════════════════════════════════════════════════════════════════
CRITICAL RULES:
═══════════════════════════════════════════════════════════════════════
1. ⚠️ If query has "why/how/reason" → Explanation, NOT Factoid
2. ⚠️ If query has "summary/overview/tell me about" → Summarization, NOT Factoid
3. ⚠️ If query has pronouns (it/they/that) → MUST resolve using history
4. ⚠️ If original query = rewritten query AND history exists → YOU FAILED!
5. ⚠️ Multi-turn MUST be "follow-up" or "clarification" if history exists, NOT "first_turn"
"""


    def _build_user_prompt(self, query: str, history: List[Dict]) -> str:
        if not history:
            hist_text = "No conversation history (this is first_turn)."
        else:
            hist_text = "Conversation history (last 5 turns):\n"
            for h in history[-5:]:
                hist_text += f"  {h['role']}: {h['text']}\n"


        return f"""{hist_text}


Current user query: "{query}"


Your task:
1. Classify multi-turn type (first_turn / follow-up / clarification)
2. If follow-up or clarification: RESOLVE pronouns and add context to make standalone
3. Classify question type (avoid over-using Factoid!)
4. Output JSON


⚠️ REMEMBER: If history exists and query has pronouns → rewritten MUST be different!


JSON:"""


    def rewrite(self, query: str, history: List[Dict]) -> Tuple[str, Dict]:
        system = self._build_system_prompt()
        user_prompt = self._build_user_prompt(query, history)


        try:
            response = llm_invoke_text(system, user_prompt, max_tokens=600)


            json_match = re.search(r'\{.*\}', response, re.DOTALL)
            if not json_match:
                return query, {
                    "question_type": "Factoid",
                    "multi_turn_type": "first_turn" if not history else "follow-up",
                    "reasoning": "JSON parse failed",
                    "original_query": query
                }


            obj = json.loads(json_match.group(0))


            rewritten = obj.get("rewritten_query", query).strip()


            # Validation: If history exists and query unchanged → force basic rewrite
            if history and rewritten == query and len(query.split()) < 10:
                # Emergency fallback: Add main entity from last assistant turn
                last_assistant = [h for h in history if h["role"] == "assistant"]
                if last_assistant:
                    # Extract first proper noun from last answer
                    last_text = last_assistant[-1]["text"]
                    words = last_text.split()
                    # Simple heuristic: find capitalized words
                    entities = [w for w in words[:30] if w and w[0].isupper() and len(w) > 3]
                    if entities:
                        rewritten = f"{entities[0]} {query}"


            metadata = {
                "question_type": obj.get("question_type", "Factoid"),
                "multi_turn_type": obj.get("multi_turn_type", "first_turn" if not history else "follow-up"),
                "reasoning": obj.get("reasoning", ""),
                "original_query": query
            }


            return rewritten, metadata


        except Exception as e:
            print(f"⚠️ Rewrite error: {e}")
            return query, {
                "question_type": "error",
                "multi_turn_type": "unknown",
                "reasoning": str(e),
                "original_query": query
            }


# ----------------------------
# PIPELINE (SAME AS BEFORE)
# ----------------------------
def run_corpus_experiment(corpus_config: dict, rewriter, es_client):
    corpus_name = corpus_config["name"]
    queries_file = corpus_config["queries_file"]
    index_name = corpus_config["index_name"]
    collection_id = corpus_config["collection_id"]


    print("\n" + "=" * 70)
    print(f"🎯 CORPUS: {corpus_name.upper()}")
    print("=" * 70)


    data = load_queries(queries_file)
    items = list(data.items())


    if MAX_QUERIES is not None:
        items = items[:int(MAX_QUERIES)]


    print(f"✅ Loaded {len(items)} queries for {corpus_name}")


    print(f"📝 Rewriting queries with {rewriter.name}...")
    rewrites_map = {}


    with ThreadPoolExecutor(max_workers=5) as ex:
        futs = {ex.submit(rewriter.rewrite, it["query"], it["history"]): qid for qid, it in items}
        for f in tqdm(as_completed(futs), total=len(futs), desc=f"Rewriting {corpus_name}"):
            qid = futs[f]
            try:
                rewritten_query, metadata = f.result()
                rewrites_map[qid] = {"query": rewritten_query, "metadata": metadata}
            except Exception as e:
                rewrites_map[qid] = {"query": data[qid]["query"], "metadata": {"question_type": "error", "multi_turn_type": "unknown"}}


    RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
    rewrites_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}_REWRITES.json")


    # ═══════════════════════════════════════════════════════════════════════
    # MODIFIED: Save rewrites with only 4 fields as requested
    # ═══════════════════════════════════════════════════════════════════════
    rewrites_data = []
    for qid, item in items:
        rewrite_info = rewrites_map.get(qid, {})
        rewrites_data.append({
            "qid": qid,
            "original_query": item["query"],
            "history": item["history"],
            "rewritten_query": rewrite_info.get("query", item["query"])
        })


    with open(rewrites_file, "w", encoding="utf-8") as f:
        json.dump(rewrites_data, f, indent=2, ensure_ascii=False)


    print(f"💾 Rewrites saved to: {rewrites_file}")


    print(f"🔍 Retrieving from {index_name}...")
    pred_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}.jsonl")


    with open(pred_file, "w", encoding="utf-8") as out:
        for qid, item in tqdm(items, desc=f"Retrieving {corpus_name}"):
            rewrite_info = rewrites_map.get(qid, {})
            q_final = rewrite_info.get("query", item["query"])
            metadata = rewrite_info.get("metadata", {})


            try:
                res = es_client.search(
                    index=index_name,
                    body={
                        "query": {"text_expansion": {"text_embedding": {"model_id": ".elser_model_1", "model_text": q_final}}},
                        "size": 100,
                    },
                )
                contexts = [{"document_id": h["_source"].get("doc_id", h["_id"]), "score": h["_score"]} for h in res["hits"]["hits"]]
            except Exception as e:
                contexts = []


            out.write(json.dumps({
                "task_id": qid,
                "Collection": collection_id,
                "contexts": contexts,
                "question_type": metadata.get("question_type", "unknown"),
                "multi_turn_type": metadata.get("multi_turn_type", "unknown"),
                "turn_number": len(item["history"]) + 1,
                "original_query": item["query"],
                "rewritten_query": q_final,
            }, ensure_ascii=False) + "\n")


    print(f"✅ Predictions saved to: {pred_file}")
    return pred_file, rewrites_file


def run_evaluation(pred_file: str):
    print(f"\n🧪 Evaluating...")
    output_eval_base = pred_file.replace(".jsonl", "")
    os.system(f"python {EVAL_SCRIPT} --input_file {pred_file} --output_file {output_eval_base}_results.jsonl")
    return output_eval_base + "_results.jsonl"


def main():
    print("🚀 ELSER v1 - FIXED TYPE-AWARE REWRITER")


    es_client = Elasticsearch(cloud_id=CLOUD_ID, basic_auth=("elastic", PASSWORD), request_timeout=180)
    rewriter = FixedTypeAwareRewriter()


    for corpus_config in CORPORA:
        try:
            pred_file, rewrites_file = run_corpus_experiment(corpus_config, rewriter, es_client)
            run_evaluation(pred_file)
        except Exception as e:
            print(f"❌ Error: {e}")


    print("\n💰 Cost:", tracker.get_report())


if __name__ == "__main__":
    main()


In [ ]:
# ==============================================================================
# ELSER v1 — FIXED TYPE-AWARE REWRITER
# VERSION: Addresses over-Factoid prediction + Multi-turn detection issues
# ==============================================================================


import os, json, re, time, threading, requests, glob
from typing import List, Dict, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd
from google.colab import drive
from elasticsearch import Elasticsearch


drive.mount("/content/drive", force_remount=True)


BASE_PATH = "/content/drive/MyDrive/Project"
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]


CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]


EVAL_SCRIPT = f"{BASE_PATH}/run_retrieval_eval.py"
PRED_DIR = f"{BASE_PATH}/elser_v1_fixed_type_predictions"
os.makedirs(PRED_DIR, exist_ok=True)


MAX_QUERIES = None


CORPORA = [

    {"name": "cloud", "queries_file": f"{BASE_PATH}/retrieval/cloud/cloud_questions.jsonl", "index_name": "cloud_elser_v1", "collection_id": "mt-rag-ibmcloud-elser-512-100-20240502"},
]


class CostTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.lock = threading.Lock()


    def add_usage(self, prompt_tokens, completion_tokens):
        with self.lock:
            self.total_input += int(prompt_tokens or 0)
            self.total_output += int(completion_tokens or 0)


    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        return f"In: {self.total_input} | Out: {self.total_output} | Est. Cost: ${cost:.4f}"


tracker = CostTracker()


def llm_invoke_text(system: str, user: str, max_tokens: int = 500) -> str:
    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-V3.2",
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "max_tokens": max_tokens,
        "temperature": 0.0
    }


    for attempt in range(7):
        try:
            r = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(3 * (2 ** attempt))
                continue
            r.raise_for_status()
            res_json = r.json()
            usage = res_json.get("usage", {})
            tracker.add_usage(usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0))
            return res_json["choices"][0]["message"]["content"].strip()
        except:
            time.sleep(2 * (2 ** attempt))
    return ""


def parse_hybrid_history(text: str):
    user_re = re.compile(r"^\s*\|user\|\s*:\s*(.*)\s*$", re.IGNORECASE)
    assistant_re = re.compile(r"^\s*\|assistant\|\s*:\s*(.*)\s*$", re.IGNORECASE)


    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    history = []


    for ln in lines:
        u_m, a_m = user_re.match(ln), assistant_re.match(ln)
        if u_m:
            history.append({"role": "user", "text": u_m.group(1).strip()})
        elif a_m:
            history.append({"role": "assistant", "text": a_m.group(1).strip()})


    if not history:
        return text.strip(), []


    return history[-1]["text"], history[:-1]


def load_queries(path: str):
    qmap = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            query, hist = parse_hybrid_history(j.get("text", ""))
            qmap[j["_id"]] = {"query": query, "history": hist}
    return qmap


# ----------------------------
# FIXED REWRITER
# ----------------------------
class FixedTypeAwareRewriter:
    """
    Fixes:
    1. Over-prediction of Factoid
    2. Multi-turn classification (0% accuracy)
    3. Non-standalone query resolution (complete failures)
    """
    def __init__(self):
        self.name = "fixed_type_aware_v2"


    def _build_system_prompt(self) -> str:
        return """You are an expert Query Rewriter for ELSER v1 sparse retrieval. Your goal: Make non-standalone queries standalone, then classify by type.


═══════════════════════════════════════════════════════════════════════
⚠️ CRITICAL: AVOID OVER-PREDICTING "Factoid"
═══════════════════════════════════════════════════════════════════════
Before choosing Factoid, check if the query is actually:
- **Explanation** (why/how/reason) → NOT Factoid!
- **Summarization** (summary/overview/policy) → NOT Factoid!
- **Keyword** (fragments like "python pandas merge") → NOT Factoid!
- **How-To** (instructions to do something) → NOT Factoid!


Only choose Factoid if it asks for a SINGLE SPECIFIC FACT (name, date, number, yes/no).


═══════════════════════════════════════════════════════════════════════
STEP 1: MULTI-TURN CLASSIFICATION (FIXED)
═══════════════════════════════════════════════════════════════════════
- **first_turn**: No history OR query is 100% standalone (full sentence with no pronouns)
- **follow-up**: Query continues/expands current topic
  * Examples: "What about its features?", "Tell me more", "Any other examples?"
- **clarification**: Query asks to EXPLAIN previous answer
  * Examples: "What do you mean by X?", "Can you clarify that?", "Explain more about..."


🔍 Detection signals:
- Pronouns (it, they, that, this, those) → follow-up or clarification
- Starts with "What about", "How about", "And" → follow-up
- "Explain", "clarify", "what do you mean" → clarification
- Very short (<5 words) with history → follow-up


═══════════════════════════════════════════════════════════════════════
STEP 2: RESOLVE NON-STANDALONE QUERIES (CRITICAL FIX)
═══════════════════════════════════════════════════════════════════════
If follow-up or clarification:
1. Identify the MAIN ENTITY from recent history (last 2-3 turns)
2. Replace pronouns with explicit entity names
3. Add missing context to make query standalone


Examples:
- "any reason to end?" + History: "The Office TV series"
  → "why did The Office TV series end"


- "who was the arch rival to this series?" + History: "Friends TV show"
  → "who was the arch rival TV show to Friends"


- "any awards" + History: "Breaking Bad"
  → "Breaking Bad awards won"


- "When was it found?" + History: "Rosetta Stone"
  → "when was Rosetta Stone found discovered"


═══════════════════════════════════════════════════════════════════════
STEP 3: QUESTION TYPE CLASSIFICATION (10 Types)
═══════════════════════════════════════════════════════════════════════


1. **Factoid** (⚠️ Use SPARINGLY)
   - Asks for ONE specific fact: name, date, quantity, yes/no
   - Examples: "Who is CEO?", "When was X founded?", "How many episodes?"
   - NOT: "Why", "How to", "Tell me about", "Summary"


2. **Explanation**
   - Asks WHY, HOW something works, REASON, CAUSE
   - Keywords: "why", "how does", "reason", "cause", "explain why"
   - Examples: "Why did the stock crash?", "How does photosynthesis work?"


3. **Summarization**
   - Requests summary, overview, policy, general information
   - Keywords: "summarize", "overview", "tell me about", "what is X" (when X is broad topic)
   - Examples: "Summarize the privacy policy", "Tell me about quantum computing"


4. **How-To**
   - Requests INSTRUCTIONS or STEPS to do something
   - Keywords: "how to", "how do I", "steps to", "guide"
   - Examples: "How to reset password?", "How do I install Docker?"


5. **Comparative**
   - Compares 2+ entities
   - Keywords: "vs", "versus", "difference", "compare", "better than"
   - Examples: "iPhone vs Samsung", "Difference between X and Y"


6. **Keyword**
   - Fragmented keywords, NOT a full sentence
   - Examples: "python pandas merge", "react hooks useState", "docker compose"


7. **Composite**
   - Multiple sub-questions in ONE query
   - Keywords: "and", multiple question words
   - Examples: "What is X, how to use it, and what are benefits?"


8. **Troubleshooting**
   - Solving error/problem/bug
   - Keywords: "error", "not working", "fix", "problem", "issue"
   - Examples: "Error 404 when accessing API", "Cannot connect to database"


9. **Non-question**
   - User statement, not asking anything
   - Examples: "I am in California", "The API is slow", "Thank you"


10. **Opinion**
    - Asks for subjective view, preference, recommendation
    - Keywords: "best", "worth it", "recommend", "should I"
    - Examples: "Is Tesla worth buying?", "Best programming language?"


═══════════════════════════════════════════════════════════════════════
STEP 4: OUTPUT FORMAT
═══════════════════════════════════════════════════════════════════════
Return STRICT JSON:
{
  "question_type": "one of 10 types - BE CAREFUL not to over-use Factoid",
  "multi_turn_type": "first_turn | follow-up | clarification",
  "rewritten_query": "STANDALONE query with pronouns resolved and context added",
  "reasoning": "brief explanation: why this type, what context was added"
}


═══════════════════════════════════════════════════════════════════════
CRITICAL RULES:
═══════════════════════════════════════════════════════════════════════
1. ⚠️ If query has "why/how/reason" → Explanation, NOT Factoid
2. ⚠️ If query has "summary/overview/tell me about" → Summarization, NOT Factoid
3. ⚠️ If query has pronouns (it/they/that) → MUST resolve using history
4. ⚠️ If original query = rewritten query AND history exists → YOU FAILED!
5. ⚠️ Multi-turn MUST be "follow-up" or "clarification" if history exists, NOT "first_turn"
"""


    def _build_user_prompt(self, query: str, history: List[Dict]) -> str:
        if not history:
            hist_text = "No conversation history (this is first_turn)."
        else:
            hist_text = "Conversation history (last 5 turns):\n"
            for h in history[-5:]:
                hist_text += f"  {h['role']}: {h['text']}\n"


        return f"""{hist_text}


Current user query: "{query}"


Your task:
1. Classify multi-turn type (first_turn / follow-up / clarification)
2. If follow-up or clarification: RESOLVE pronouns and add context to make standalone
3. Classify question type (avoid over-using Factoid!)
4. Output JSON


⚠️ REMEMBER: If history exists and query has pronouns → rewritten MUST be different!


JSON:"""


    def rewrite(self, query: str, history: List[Dict]) -> Tuple[str, Dict]:
        system = self._build_system_prompt()
        user_prompt = self._build_user_prompt(query, history)


        try:
            response = llm_invoke_text(system, user_prompt, max_tokens=600)


            json_match = re.search(r'\{.*\}', response, re.DOTALL)
            if not json_match:
                return query, {
                    "question_type": "Factoid",
                    "multi_turn_type": "first_turn" if not history else "follow-up",
                    "reasoning": "JSON parse failed",
                    "original_query": query
                }


            obj = json.loads(json_match.group(0))


            rewritten = obj.get("rewritten_query", query).strip()


            # Validation: If history exists and query unchanged → force basic rewrite
            if history and rewritten == query and len(query.split()) < 10:
                # Emergency fallback: Add main entity from last assistant turn
                last_assistant = [h for h in history if h["role"] == "assistant"]
                if last_assistant:
                    # Extract first proper noun from last answer
                    last_text = last_assistant[-1]["text"]
                    words = last_text.split()
                    # Simple heuristic: find capitalized words
                    entities = [w for w in words[:30] if w and w[0].isupper() and len(w) > 3]
                    if entities:
                        rewritten = f"{entities[0]} {query}"


            metadata = {
                "question_type": obj.get("question_type", "Factoid"),
                "multi_turn_type": obj.get("multi_turn_type", "first_turn" if not history else "follow-up"),
                "reasoning": obj.get("reasoning", ""),
                "original_query": query
            }


            return rewritten, metadata


        except Exception as e:
            print(f"⚠️ Rewrite error: {e}")
            return query, {
                "question_type": "error",
                "multi_turn_type": "unknown",
                "reasoning": str(e),
                "original_query": query
            }


# ----------------------------
# PIPELINE (SAME AS BEFORE)
# ----------------------------
def run_corpus_experiment(corpus_config: dict, rewriter, es_client):
    corpus_name = corpus_config["name"]
    queries_file = corpus_config["queries_file"]
    index_name = corpus_config["index_name"]
    collection_id = corpus_config["collection_id"]


    print("\n" + "=" * 70)
    print(f"🎯 CORPUS: {corpus_name.upper()}")
    print("=" * 70)


    data = load_queries(queries_file)
    items = list(data.items())


    if MAX_QUERIES is not None:
        items = items[:int(MAX_QUERIES)]


    print(f"✅ Loaded {len(items)} queries for {corpus_name}")


    print(f"📝 Rewriting queries with {rewriter.name}...")
    rewrites_map = {}


    with ThreadPoolExecutor(max_workers=5) as ex:
        futs = {ex.submit(rewriter.rewrite, it["query"], it["history"]): qid for qid, it in items}
        for f in tqdm(as_completed(futs), total=len(futs), desc=f"Rewriting {corpus_name}"):
            qid = futs[f]
            try:
                rewritten_query, metadata = f.result()
                rewrites_map[qid] = {"query": rewritten_query, "metadata": metadata}
            except Exception as e:
                rewrites_map[qid] = {"query": data[qid]["query"], "metadata": {"question_type": "error", "multi_turn_type": "unknown"}}


    RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
    rewrites_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}_REWRITES.json")


    # ═══════════════════════════════════════════════════════════════════════
    # MODIFIED: Save rewrites with only 4 fields as requested
    # ═══════════════════════════════════════════════════════════════════════
    rewrites_data = []
    for qid, item in items:
        rewrite_info = rewrites_map.get(qid, {})
        rewrites_data.append({
            "qid": qid,
            "original_query": item["query"],
            "history": item["history"],
            "rewritten_query": rewrite_info.get("query", item["query"])
        })


    with open(rewrites_file, "w", encoding="utf-8") as f:
        json.dump(rewrites_data, f, indent=2, ensure_ascii=False)


    print(f"💾 Rewrites saved to: {rewrites_file}")


    print(f"🔍 Retrieving from {index_name}...")
    pred_file = os.path.join(PRED_DIR, f"{corpus_name}_{rewriter.name}_{RUN_TAG}.jsonl")


    with open(pred_file, "w", encoding="utf-8") as out:
        for qid, item in tqdm(items, desc=f"Retrieving {corpus_name}"):
            rewrite_info = rewrites_map.get(qid, {})
            q_final = rewrite_info.get("query", item["query"])
            metadata = rewrite_info.get("metadata", {})


            try:
                res = es_client.search(
                    index=index_name,
                    body={
                        "query": {"text_expansion": {"text_embedding": {"model_id": ".elser_model_1", "model_text": q_final}}},
                        "size": 100,
                    },
                )
                contexts = [{"document_id": h["_source"].get("doc_id", h["_id"]), "score": h["_score"]} for h in res["hits"]["hits"]]
            except Exception as e:
                contexts = []


            out.write(json.dumps({
                "task_id": qid,
                "Collection": collection_id,
                "contexts": contexts,
                "question_type": metadata.get("question_type", "unknown"),
                "multi_turn_type": metadata.get("multi_turn_type", "unknown"),
                "turn_number": len(item["history"]) + 1,
                "original_query": item["query"],
                "rewritten_query": q_final,
            }, ensure_ascii=False) + "\n")


    print(f"✅ Predictions saved to: {pred_file}")
    return pred_file, rewrites_file


def run_evaluation(pred_file: str):
    print(f"\n🧪 Evaluating...")
    output_eval_base = pred_file.replace(".jsonl", "")
    os.system(f"python {EVAL_SCRIPT} --input_file {pred_file} --output_file {output_eval_base}_results.jsonl")
    return output_eval_base + "_results.jsonl"


def main():
    print("🚀 ELSER v1 - FIXED TYPE-AWARE REWRITER")


    es_client = Elasticsearch(cloud_id=CLOUD_ID, basic_auth=("elastic", PASSWORD), request_timeout=180)
    rewriter = FixedTypeAwareRewriter()


    for corpus_config in CORPORA:
        try:
            pred_file, rewrites_file = run_corpus_experiment(corpus_config, rewriter, es_client)
            run_evaluation(pred_file)
        except Exception as e:
            print(f"❌ Error: {e}")


    print("\n💰 Cost:", tracker.get_report())


if __name__ == "__main__":
    main()


In [ ]:
# ================================
# CELL 1: IMPORTS & INSTALLATION
# ================================

!pip install -q cohere faiss-cpu numpy pandas boto3

import os
import json
import time
import numpy as np
import pandas as pd
import faiss
import cohere
from pathlib import Path
from typing import Dict, List, Optional
from collections import defaultdict

# ================================
# CELL 2: DATA SETUP + HELPERS (UPDATED FOR COHERE)
# ================================

# ---------- Paths ----------
DATA_PATH = "/content/drive/MyDrive/Project"
OUTPUT_PATH = "/content/drive/MyDrive/Project/predictions"
REWRITES_BASE_PATH = "/content/drive/MyDrive/Project"
os.makedirs(OUTPUT_PATH, exist_ok=True)

EVAL_SCRIPT_PATH = os.path.join(DATA_PATH, "run_retrieval_eval.py")

# ---------- Cohere Configuration ----------
# 🔥 CHANGED: Direct Cohere API key (get from dashboard.cohere.com)
COHERE_API_KEY = os.environ["API_KEY"]  # Replace with your actual key

# ---------- Helpers φόρτωσης δεδομένων ----------

def load_corpus_jsonl(path: str) -> Dict[str, Dict[str, str]]:
    """Load corpus from JSONL file"""
    corpus = {}
    if not os.path.exists(path):
        print(f"⚠️ Corpus file not found: {path}")
        return corpus
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            did = j.get("_id") or j.get("id") or j.get("document_id")
            if did:
                corpus[did] = {
                    "title": j.get("title", ""),
                    "text": j.get("text", ""),
                }
    print(f"✅ Loaded {len(corpus)} documents from {os.path.basename(path)}")
    return corpus

def load_queries_jsonl(path: str) -> Dict[str, Dict[str, str]]:
    """Load queries from JSONL file"""
    q = {}
    if not os.path.exists(path):
        print(f"⚠️ Queries file not found: {path}")
        return q
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            qid = j.get("_id") or j.get("id") or j.get("query_id") or j.get("task_id")
            text = j.get("text") or j.get("query") or ""
            if qid:
                q[qid] = {
                    "query": text,
                    "task_id": qid
                }
    print(f"✅ Loaded {len(q)} queries from {os.path.basename(path)}")
    return q

def load_history_from_questions(questions_file: str) -> Dict[str, List[Dict[str, str]]]:
    """Ανακατασκευάζει ιστορικό συνομιλίας από το questions.jsonl του MTRAG"""
    if not os.path.exists(questions_file):
        print(f"⚠️ History questions file not found: {questions_file}")
        return {}

    conversations = defaultdict(list)
    histories = {}

    with open(questions_file, "r", encoding="utf-8") as f:
        for line in f:
            try:
                j = json.loads(line)
            except Exception:
                continue
            qid = j.get("_id") or j.get("id") or j.get("query_id") or j.get("task_id")
            text = j.get("text") or j.get("query") or ""
            if not qid:
                continue

            if "<::>" in qid:
                conv_id, turn_str = qid.rsplit("<::>", 1)
                try:
                    turn_num = int(turn_str)
                except:
                    turn_num = 0

                role = "user"
                clean_text = text
                if text.startswith("|user|:"):
                    clean_text = text.replace("|user|:", "").strip()
                    role = "user"
                elif text.startswith("|assistant|:"):
                    clean_text = text.replace("|assistant|:", "").strip()
                    role = "assistant"

                conversations[conv_id].append({
                    "turn_num": turn_num,
                    "role": role,
                    "text": clean_text,
                    "query_id": qid
                })

    for conv_id, turns in conversations.items():
        turns_sorted = sorted(turns, key=lambda x: x["turn_num"])
        for i, turn in enumerate(turns_sorted):
            qid = turn["query_id"]
            history = [
                {"role": t["role"], "text": t["text"]}
                for t in turns_sorted[:i]
            ]
            histories[qid] = history

    print(f"✅ Loaded history for {len(histories)} queries from {os.path.basename(questions_file)}")
    return histories

def load_task_metadata(path: str) -> pd.DataFrame:
    """Load metadata about first/later turns and standalone/non-standalone queries"""
    if not os.path.exists(path):
        return pd.DataFrame()

    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))

    df = pd.DataFrame(records)
    print(f"✅ Loaded metadata for {len(df)} tasks from {os.path.basename(path)}")
    return df

def load_rewrites_from_json(file_path: str) -> Dict[str, List[str]]:
    """Load pre-generated rewrites from JSON file"""
    if not os.path.exists(file_path):
        print(f"⚠️ Rewrites file not found: {file_path}")
        return {}

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    rewrites_dict = {}

    # Handle list format
    if isinstance(data, list):
        for entry in data:
            task_id = (entry.get("task_id") or
                      entry.get("qid") or
                      entry.get("id") or
                      entry.get("query_id"))

            if not task_id:
                continue

            single_rewrite = (entry.get("rewritten_query") or
                            entry.get("rewrite") or
                            entry.get("query"))

            multi_rewrites = (entry.get("rewrites") or
                            entry.get("decompositions") or
                            entry.get("queries"))

            if multi_rewrites and isinstance(multi_rewrites, list):
                rewrites_dict[task_id] = multi_rewrites
            elif single_rewrite and isinstance(single_rewrite, str):
                rewrites_dict[task_id] = [single_rewrite]
            else:
                rewrites_dict[task_id] = []

    # Handle dict format
    elif isinstance(data, dict):
        for task_id, content in data.items():
            if isinstance(content, list):
                rewrites_dict[task_id] = content
            elif isinstance(content, dict):
                rewrites = content.get("rewrites", [])
                decompositions = content.get("decompositions", [])
                rewrites_dict[task_id] = rewrites if rewrites else decompositions
            elif isinstance(content, str):
                rewrites_dict[task_id] = [content]

    print(f"✅ Loaded rewrites for {len(rewrites_dict)} tasks from {os.path.basename(file_path)}")

    if rewrites_dict:
        sample_id = list(rewrites_dict.keys())[0]
        sample_rewrites = rewrites_dict[sample_id]
        print(f"   📋 Sample ID: {sample_id[:40]}...")
        print(f"   📋 Rewrites count: {len(sample_rewrites)}")
        if sample_rewrites:
            print(f"   📋 First rewrite: {sample_rewrites[0][:100]}...")

    return rewrites_dict

# ---------- Pre-generated Rewrites File Map ----------
FILE_MAP = {
    "clapnq": {
        "Type": f"{REWRITES_BASE_PATH}/elser_v1_fixed_type_predictions/clapnq_fixed_type_aware_v2_20260117_223328_REWRITES.json"

    },
    "govt":{
        "Type": f"{REWRITES_BASE_PATH}/elser_v1_fixed_type_predictions/govt_fixed_type_aware_v2_20260117_230421_REWRITES.json"
    },
    "cloud":{
        "Type": f"{REWRITES_BASE_PATH}/elser_v1_fixed_type_predictions/cloud_fixed_type_aware_v2_20260117_234154_REWRITES.json"
    },
    "fiqa":{
        "Type":f"{REWRITES_BASE_PATH}/elser_v1_fixed_type_predictions/fiqa_fixed_type_aware_v2_20260117_224418_REWRITES.json"
    }

}

# ---------- Config για ΟΛΑ τα corpora ----------
CORPORA_CONFIG = [
    {
        "name": "clapnq",
        "corpus_file": os.path.join(DATA_PATH, "corpora", "clapnq.jsonl"),
        "queries_file": os.path.join(DATA_PATH, "retrieval", "clapnq", "clapnq_lastturn.jsonl"),
        "history_file": os.path.join(DATA_PATH, "retrieval", "clapnq", "clapnq_questions.jsonl"),
        "emb_file": os.path.join(DATA_PATH, "embeddings_clapnq_cohere_v4_crossregion.npy"),
        "metadata_file": os.path.join(DATA_PATH, "retrieval", "clapnq", "dev_metadata.jsonl"),
        "collection_name": "mt-rag-clapnq-elser-512-100-20240503",
    },
    {
        "name": "govt",
        "corpus_file": os.path.join(DATA_PATH, "corpora", "govt.jsonl"),
        "queries_file": os.path.join(DATA_PATH, "retrieval", "govt", "govt_lastturn.jsonl"),
        "history_file": os.path.join(DATA_PATH, "retrieval", "govt", "govt_questions.jsonl"),
        "emb_file": os.path.join(DATA_PATH, "embeddings_govt_cohere_v4_crossregion.npy"),
        "metadata_file": os.path.join(DATA_PATH, "retrieval", "govt", "dev_metadata.jsonl"),
        "collection_name": "mt-rag-govt-elser-512-100-20240611",
    },
    {
        "name": "cloud",
        "corpus_file": os.path.join(DATA_PATH, "corpora", "cloud.jsonl"),
        "queries_file": os.path.join(DATA_PATH, "retrieval", "cloud", "cloud_lastturn.jsonl"),
        "history_file": os.path.join(DATA_PATH, "retrieval", "cloud", "cloud_questions.jsonl"),
        "emb_file": os.path.join(DATA_PATH, "embeddings_cloud_cohere_v4_crossregion.npy"),
        "metadata_file": os.path.join(DATA_PATH, "retrieval", "cloud", "dev_metadata.jsonl"),
        "collection_name": "mt-rag-ibmcloud-elser-512-100-20240502",
    },
    {
        "name": "fiqa",
        "corpus_file": os.path.join(DATA_PATH, "corpora", "fiqa.jsonl"),
        "queries_file": os.path.join(DATA_PATH, "retrieval", "fiqa", "fiqa_lastturn.jsonl"),
        "history_file": os.path.join(DATA_PATH, "retrieval", "fiqa", "fiqa_questions.jsonl"),
        "emb_file": os.path.join(DATA_PATH, "embeddings_fiqa_cohere_v4_crossregion.npy"),
        "metadata_file": os.path.join(DATA_PATH, "retrieval", "fiqa", "dev_metadata.jsonl"),
        "collection_name": "mt-rag-fiqa-beir-elser-512-100-20240501",
    },
]

print("\n" + "="*80)
print("CONFIGURATION SUMMARY - DIRECT COHERE API")
print("="*80)
print(f"Data Path: {DATA_PATH}")
print(f"Output Path: {OUTPUT_PATH}")
print(f"Cohere API: Direct (dashboard.cohere.com)")
print(f"Corpora: {[c['name'] for c in CORPORA_CONFIG]}")
print("Mode: Single query retrieval (rewritten query preferred)")
print("="*80 + "\n")

# ================================
# CELL 3: COHERE EMBEDDER + RETRIEVER V4
# ================================

class CohereEmbedderV4:
    """Embedder using direct Cohere API with embed-v4"""
    def __init__(self, api_key: str, model_id="embed-english-v3.0"):
        self.model_id = model_id
        # 🔥 CHANGED: Direct Cohere client without base_url
        self.client = cohere.Client(api_key=api_key)
        print(f"✅ Cohere Embedder initialized with model: {model_id}")

    def _sanitize_text(self, text: str) -> str:
        """Clean and sanitize input text"""
        if not text or not isinstance(text, str):
            return "empty document"
        text = text.replace('\x00', '').replace('\r\n', ' ').replace('\n', ' ').replace('\r', ' ')
        text = ''.join(ch for ch in text if ch.isprintable() or ch.isspace())
        text = ' '.join(text.split())
        return text if text.strip() else "empty document"

    def embed_query(self, query: str) -> np.ndarray:
        """Embed a single query using Cohere API"""
        query = self._sanitize_text(query)[:1500]
        max_retries = 3

        for attempt in range(max_retries):
            try:
                response = self.client.embed(
                    texts=[query],
                    model=self.model_id,
                    input_type="search_query",
                    embedding_types=["float"]
                )

                embedding = response.embeddings.float[0]
                return np.array(embedding, dtype=np.float32)

            except Exception as e:
                if attempt == max_retries - 1:
                    print(f"❌ Query embedding failed after {max_retries} attempts: {e}")
                    return np.zeros(1024, dtype=np.float32)
                time.sleep(0.5 * (attempt + 1))

    def embed_documents(self, texts: List[str], batch_size: int = 96) -> np.ndarray:
        """Embed multiple documents in batches"""
        all_embeddings = []

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            batch_clean = [self._sanitize_text(t)[:2048] for t in batch]

            try:
                response = self.client.embed(
                    texts=batch_clean,
                    model=self.model_id,
                    input_type="search_document",
                    embedding_types=["float"]
                )

                batch_embs = [np.array(emb, dtype=np.float32) for emb in response.embeddings.float]
                all_embeddings.extend(batch_embs)
                print(f"✅ Embedded batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}")

                time.sleep(0.2)

            except Exception as e:
                print(f"❌ Batch {i//batch_size + 1} failed: {e}")
                dim = 1024 if not all_embeddings else len(all_embeddings[0])
                all_embeddings.extend([np.zeros(dim, dtype=np.float32)] * len(batch))

        return np.array(all_embeddings, dtype=np.float32)

class CohereRetrieverV4:
    """Retriever with Cohere embeddings and FAISS index"""
    def __init__(self, docs: List[str], doc_ids: List[str], persist_file: str,
                 embedder: CohereEmbedderV4):
        self.docs = docs
        self.doc_ids = doc_ids
        self.embedder = embedder
        self.index = None
        self.embs = None

        if Path(persist_file).exists():
            print(f"📂 Loading embeddings from {persist_file}")
            self.embs = np.load(persist_file).astype(np.float32)
            print(f"✅ Loaded {len(self.embs)} embeddings (shape: {self.embs.shape})")
        else:
            print(f"⚠️ Embeddings file {persist_file} not found. Creating new embeddings...")
            self.embs = embedder.embed_documents(docs)
            np.save(persist_file, self.embs)
            print(f"✅ Saved new embeddings to {persist_file}")

        dim = self.embs.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        faiss.normalize_L2(self.embs)
        self.index.add(self.embs)
        print(f"✅ FAISS index built (dim={dim}, docs={len(self.docs)})")

    def retrieve(self, query: str, history: Optional[List[Dict]] = None, top_k: int = 100) -> List[Dict]:
        """Retrieve top-k documents for a query"""
        if self.index is None or self.embs is None:
            return []

        q_emb = self.embedder.embed_query(query)
        if np.all(q_emb == 0):
            return []

        q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-8)
        q_emb = q_emb.astype(np.float32).reshape(1, -1)

        D, I = self.index.search(q_emb, top_k)

        results = []
        for j, idx in enumerate(I[0]):
            if 0 <= idx < len(self.doc_ids):
                results.append({
                    "id": self.doc_ids[idx],
                    "text": self.docs[idx],
                    "score": float(D[0][j]),
                })
        return results

# ================================
# CELL 4: EVALUATION HELPERS
# ================================

def load_query_results(results_file: str) -> pd.DataFrame:
    """Load per-query results from evaluation output"""
    if not os.path.exists(results_file):
        return pd.DataFrame()

    records = []
    with open(results_file, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))

    return pd.DataFrame(records)

def summarize_group(df: pd.DataFrame, group_name: str):
    """Summarize metrics for a group of queries"""
    if df.empty:
        print(f"⚠️ {group_name}: No data")
        return

    metrics = ["recall@1", "recall@3", "recall@5", "ndcg@1", "ndcg@3", "ndcg@5"]
    available_metrics = [m for m in metrics if m in df.columns]

    if not available_metrics:
        print(f"⚠️ {group_name}: No metrics available")
        return

    print(f"\n{'='*60}")
    print(f"{group_name} (n={len(df)})")
    print('='*60)

    summary = df[available_metrics].mean()
    for metric, value in summary.items():
        print(f"{metric:15s}: {value:.4f}")

# ================================
# CELL 5: MAIN EXPERIMENT LOOP
# ================================

# 🔥 CHANGED: Initialize with direct Cohere API
cohere_embedder = CohereEmbedderV4(
    api_key=COHERE_API_KEY,
    model_id="embed-v4.0"  # or "embed-v-4-0" if using v4
)

EXPERIMENT_STRATEGIES = [
    "Type"

]

for cfg in CORPORA_CONFIG:
    corpus_name = cfg["name"]
    corpus_file = cfg["corpus_file"]
    queries_file = cfg["queries_file"]
    history_file = cfg["history_file"]
    emb_file = cfg["emb_file"]
    metadata_file = cfg["metadata_file"]
    collection_name = cfg["collection_name"]

    print("\n" + "#" * 80)
    print(f"### CORPUS: {corpus_name.upper()}")
    print("#" * 80)

    print(f"=== LOADING {corpus_name.upper()} DATA ===")
    corpus = load_corpus_jsonl(corpus_file)
    queries = load_queries_jsonl(queries_file)
    histories = load_history_from_questions(history_file)

    if not corpus or not queries:
        print(f"❌ Skipping {corpus_name} - missing corpus or queries")
        continue

    docs = [v["text"] for v in corpus.values()]
    doc_ids = list(corpus.keys())

    meta_df = load_task_metadata(metadata_file) if metadata_file else pd.DataFrame()
    if not meta_df.empty:
        print(f"✅ Metadata loaded for {corpus_name}: shape={meta_df.shape}")

    print(f"\n=== BUILDING RETRIEVER FOR {corpus_name.upper()} ===")
    # 🔥 CHANGED: Use CohereRetrieverV4 instead of AzureCohereRetrieverV4
    retriever = CohereRetrieverV4(
        docs=docs,
        doc_ids=doc_ids,
        persist_file=emb_file,
        embedder=cohere_embedder
    )

    if retriever.index is None:
        print(f"❌ Skipping {corpus_name} - retriever initialization failed")
        continue

    for strategy in EXPERIMENT_STRATEGIES:
        print(f"\n{'='*70}")
        print(f"{corpus_name.upper()} – STRATEGY: {strategy}")
        print('='*70)

        if corpus_name not in FILE_MAP or strategy not in FILE_MAP[corpus_name]:
            print(f"⚠️ Strategy {strategy} not available for {corpus_name}, skipping")
            continue

        rewrites_file = FILE_MAP[corpus_name][strategy]
        rewrites_dict = load_rewrites_from_json(rewrites_file)

        if not rewrites_dict:
            print(f"⚠️ No rewrites loaded for {corpus_name}/{strategy}, skipping")
            continue

        output_file = os.path.join(OUTPUT_PATH, f"{corpus_name}_cohere_{strategy}_predictions.jsonl")
        results_file = output_file.replace(".jsonl", "_results.jsonl")

        print(f"▶ Running retrieval for {corpus_name}/{strategy}...")

        predictions = []
        rewrites_used = 0
        originals_used = 0

        for task_id, query_data in queries.items():
            original_query = query_data["query"]
            history = histories.get(task_id, [])

            task_rewrites = rewrites_dict.get(task_id, [])

            if task_rewrites and len(task_rewrites) > 0:
                query_to_use = task_rewrites[0]
                rewrites_used += 1
            else:
                query_to_use = original_query
                originals_used += 1

            results = retriever.retrieve(query_to_use, history, top_k=100)

            pred_entry = {
                "task_id": task_id,
                "Collection": collection_name,
                "contexts": [
                    {
                        "document_id": doc["id"],
                        "score": doc["score"]
                    }
                    for doc in results[:100]
                ]
            }
            predictions.append(pred_entry)

        with open(output_file, 'w', encoding='utf-8') as f:
            for pred in predictions:
                f.write(json.dumps(pred) + '\n')

        print(f"✅ Saved {len(predictions)} predictions to {output_file}")
        print(f"   📊 Rewrites used: {rewrites_used}/{len(queries)} ({100*rewrites_used/len(queries):.1f}%)")
        print(f"   📊 Originals used: {originals_used}/{len(queries)} ({100*originals_used/len(queries):.1f}%)")

        if os.path.exists(EVAL_SCRIPT_PATH):
            print(f"▶ Running evaluation for {corpus_name}/{strategy}...")
            cmd = f"python3 {EVAL_SCRIPT_PATH} --input_file {output_file} --output_file {results_file}"
            result = os.system(cmd)
            if result != 0:
                print(f"⚠️ Evaluation failed for {corpus_name}/{strategy}")
            else:
                print(f"✅ Evaluation completed for {corpus_name}/{strategy}")

        csv_file = output_file.replace(".jsonl", "_results_aggregate.csv")
        if os.path.exists(csv_file):
            df = pd.read_csv(csv_file)
            df = df[df["collection"] != "all"]
            df["Pipeline"] = f"{corpus_name}_Cohere_{strategy}"
            print("\n" + "="*60)
            print(f"AGGREGATE RESULTS: {corpus_name}_Cohere_{strategy}")
            print("="*60)
            print(df.to_string(index=False))

        if os.path.exists(results_file) and not meta_df.empty:
            qdf = load_query_results(results_file)
            if not qdf.empty:
                merged_df = qdf.merge(meta_df, on="task_id", how="left")

                if "is_first_turn" in merged_df.columns:
                    summarize_group(
                        merged_df[merged_df["is_first_turn"] == True],
                        f"{corpus_name}/{strategy}: FIRST TURNS"
                    )
                    summarize_group(
                        merged_df[merged_df["is_first_turn"] == False],
                        f"{corpus_name}/{strategy}: LATER TURNS"
                    )

                if "is_standalone" in merged_df.columns:
                    summarize_group(
                        merged_df[merged_df["is_standalone"] == True],
                        f"{corpus_name}/{strategy}: STANDALONE"
                    )
                    summarize_group(
                        merged_df[merged_df["is_standalone"] == False],
                        f"{corpus_name}/{strategy}: NON-STANDALONE"
                    )

print("\n" + "="*80)
print("🎉 ALL EXPERIMENTS COMPLETED FOR ALL CORPORA!")
print("="*80)


## Reranking per rewriter

In [ ]:
import os, json, time, requests, subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import Dict, List, Tuple

# ==============================================================================
# 1. SETTINGS & CONFIGURATION
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
BASE_DIR = "/content/drive/MyDrive/Project"

EVAL_SCRIPT_PATH = os.path.join(BASE_PATH, "run_retrieval_eval.py")

# Azure Cohere Credentials
AZURE_COHERE_ENDPOINT = os.environ["ENDPOINT"]
AZURE_COHERE_KEY = os.environ["_KEY"]

RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = os.path.join(BASE_PATH, "cohere_v4_rerank_results", RUN_TAG)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Τα αρχεία εισόδου σου (ClapNQ)
INPUT_FILES = {
    "clapnq": {
       "COT": f"{BASE_PATH}/elser_v1_cot_hyde_predictions/clapnq_elser_cot_6U3A_20260114_140449.jsonl",
        "Hyde": f"{BASE_PATH}/elser_v1_cot_hyde_predictions/clapnq_elser_hyde_6U3A_20260114_161648.jsonl",
        "Decomp": f"{BASE_PATH}/elser_v1_decomposition_predictions/clapnq_elser_decomposition_6U3A_20260114_152005.jsonl",
    },

}

CORPUS_FILE = os.path.join(BASE_PATH, "corpora/clapnq.jsonl")
QUERIES_FILE = os.path.join(BASE_PATH, "retrieval/clapnq/clapnq_lastturn.jsonl")

TOP_K_RERANK = 100
COHERE_WEIGHTS = [0.1, 0.3, 0.5, 0.7, 0.9] # Βάρη για το Grid Search

# ==============================================================================
# 2. HELPER FUNCTIONS (IO & PROCESSING)
# ==============================================================================

def load_jsonl(path: str) -> List[dict]:
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip(): data.append(json.loads(line))
    return data

def load_corpus(path: str) -> Dict[str, str]:
    corpus = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            did = str(j.get("_id") or j.get("id"))
            # Συνδυάζουμε τίτλο και κείμενο για καλύτερο reranking
            text = (j.get("title", "") + " " + (j.get("text") or j.get("contents") or "")).strip()
            corpus[did] = text[:2000] # Περιορισμός tokens για οικονομία
    return corpus

def load_queries(path: str) -> Dict[str, str]:
    queries = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            qid = str(j.get("_id") or j.get("id") or j.get("task_id"))
            queries[qid] = j.get("text") or j.get("query")
    return queries

def azure_cohere_rerank(query: str, docs_text: List[str]) -> List[float]:
    """Κλήση του Cohere v4 Reranker μέσω Azure Endpoint."""
    if not query or not docs_text: return [0.0] * len(docs_text)
    headers = {"Content-Type": "application/json", "api-key": AZURE_COHERE_KEY}
    payload = {
        "model": "Cohere-rerank-v4.0-pro",
        "query": query,
        "documents": docs_text,
        "top_n": len(docs_text)
    }
    try:
        response = requests.post(AZURE_COHERE_ENDPOINT, headers=headers, json=payload, timeout=60)
        if response.status_code == 200:
            results = response.json()["results"]
            scores = [0.0] * len(docs_text)
            for r in results:
                scores[r["index"]] = float(r["relevance_score"])
            return scores
        else:
            print(f"⚠️ API Error: {response.status_code} - {response.text}")
    except Exception as e:
        print(f"❌ Connection Error: {e}")
    return [0.0] * len(docs_text)

def minmax_norm(scores: List[float]) -> np.ndarray:
    """Κανονικοποίηση scores στο εύρος [0, 1]."""
    arr = np.array(scores, dtype=np.float32)
    if arr.max() == arr.min(): return np.zeros_like(arr)
    return (arr - arr.min()) / (arr.max() - arr.min())

def run_eval(input_jsonl: str) -> Tuple[float, float]:
    """Εκτελεί το eval script και επιστρέφει Recall@5 και nDCG@5."""
    result_file = input_jsonl.replace(".jsonl", "_results.jsonl")
    csv_file = result_file.replace(".jsonl", "_aggregate.csv")

    cmd = f"python3 {EVAL_SCRIPT_PATH} --input_file {input_jsonl} --output_file {result_file}"
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        if "collection" in df.columns: df = df[df["collection"] != "all"]
        row = df.iloc[0]
        # Μετατροπή σε float αν είναι string list (π.χ. '[0.1, 0.2, 0.5]')
        def parse_metric(val):
            if isinstance(val, str) and "[" in val:
                import ast
                return ast.literal_eval(val)[2] # Index 2 is usually @5
            return val
        return parse_metric(row.get("Recall")), parse_metric(row.get("nDCG"))
    return 0.0, 0.0

# ==============================================================================
# 3. MAIN RERANKING & FUSION PIPELINE
# ==============================================================================



print("📖 Φόρτωση δεδομένων (Corpus & Queries)...")
corpus_map = load_corpus(CORPUS_FILE)
query_map = load_queries(QUERIES_FILE)

performance_summary = []

for corpus_name, files in INPUT_FILES.items():
    for strategy_name, file_path in files.items():
        if not os.path.exists(file_path):
            print(f"⚠️ Το αρχείο {file_path} δεν βρέθηκε. Παράλειψη...")
            continue

        print(f"\n🚀 Επεξεργασία στρατηγικής: {strategy_name}")
        preds = load_jsonl(file_path)

        # --- BHMA 1: Υπολογισμός Cohere Scores (Μία φορά ανά ερώτηση) ---
        cache = []
        for p in tqdm(preds, desc="Azure Cohere v4 Rerank"):
            qid = str(p["task_id"])
            q_txt = query_map.get(qid, "")

            top_ctxs = p["contexts"][:TOP_K_RERANK]
            rest_ctxs = p["contexts"][TOP_K_RERANK:]

            # Λήψη κειμένων από το corpus
            doc_texts = [corpus_map.get(str(c["document_id"]), "N/A") for c in top_ctxs]

            # Κλήση Azure Cohere
            coh_scores = azure_cohere_rerank(q_txt, doc_texts)
            orig_scores = [float(c["score"]) for c in top_ctxs]

            cache.append({
                "task_id": qid,
                "Collection": p.get("Collection"),
                "top_ctxs": top_ctxs,
                "rest": rest_ctxs,
                "orig_scores": orig_scores,
                "coh_scores": coh_scores
            })

        # --- BHMA 2: Αξιολόγηση Cohere-Only (cw=1.0) ---
        cohere_only_file = os.path.join(OUTPUT_DIR, f"{strategy_name}_COHERE_ONLY.jsonl")
        with open(cohere_only_file, "w", encoding="utf-8") as f:
            for item in cache:
                new_ctxs = []
                for i, ctx in enumerate(item["top_ctxs"]):
                    nc = ctx.copy()
                    nc["score"] = float(item["coh_scores"][i])
                    new_ctxs.append(nc)
                new_ctxs.sort(key=lambda x: x["score"], reverse=True)
                f.write(json.dumps({"task_id": item["task_id"], "Collection": item["Collection"], "contexts": new_ctxs + item["rest"]}) + "\n")

        r5, n5 = run_eval(cohere_only_file)
        performance_summary.append({"Strategy": strategy_name, "Method": "Cohere-Only", "Recall@5": r5, "nDCG@5": n5})

        # --- BHMA 3: Grid Search Hybrid Fusion ---
        for cw in COHERE_WEIGHTS:
            ow = 1.0 - cw
            hybrid_file = os.path.join(OUTPUT_DIR, f"{strategy_name}_HYBRID_cw{cw}.jsonl")

            with open(hybrid_file, "w", encoding="utf-8") as f:
                for item in cache:
                    n_orig = minmax_norm(item["orig_scores"])
                    n_coh = minmax_norm(item["coh_scores"])
                    # Τύπος Υβριδικού Σκορ
                    combined = (ow * n_orig) + (cw * n_coh)

                    new_ctxs = []
                    for i, ctx in enumerate(item["top_ctxs"]):
                        nc = ctx.copy()
                        nc["score"] = float(combined[i])
                        new_ctxs.append(nc)
                    new_ctxs.sort(key=lambda x: x["score"], reverse=True)
                    f.write(json.dumps({"task_id": item["task_id"], "Collection": item["Collection"], "contexts": new_ctxs + item["rest"]}) + "\n")

            r5, n5 = run_eval(hybrid_file)
            performance_summary.append({"Strategy": strategy_name, "Method": f"Hybrid (cw={cw})", "Recall@5": r5, "nDCG@5": n5})

# ==============================================================================
# 4. ΕΜΦΑΝΙΣΗ ΤΕΛΙΚΩΝ ΑΠΟΤΕΛΕΣΜΑΤΩΝ
# ==============================================================================
df_final = pd.DataFrame(performance_summary)
print("\n" + "="*60)
print("🏆 ΤΕΛΙΚΗ ΑΞΙΟΛΟΓΗΣΗ RERANKING & HYBRID FUSION")
print("="*60)
display(df_final.sort_values(["Strategy", "nDCG@5"], ascending=[True, False]))

# Αποθήκευση αναφοράς
df_final.to_csv(os.path.join(OUTPUT_DIR, "reranking_final_report.csv"), index=False)

In [ ]:
import os, json, time, requests, subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import Dict, List, Tuple

# ==============================================================================
# 1. SETTINGS & CONFIGURATION
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
BASE_DIR = "/content/drive/MyDrive/Project"

EVAL_SCRIPT_PATH = os.path.join(BASE_PATH, "run_retrieval_eval.py")

# Azure Cohere Credentials
AZURE_COHERE_ENDPOINT = os.environ["ENDPOINT"]
AZURE_COHERE_KEY = os.environ["_KEY"]

RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = os.path.join(BASE_PATH, "cohere_v4_rerank_results", RUN_TAG)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Multi-Corpus Configuration (FiQA, Govt, Cloud)
INPUT_FILES = {
    "fiqa": {
        "Anchor_Keyword": f"{BASE_DIR}/elser_v1_anchor_keyword_predictions/fiqa_elser_anchor_keyword_combo_6U1A_20260112_201042.jsonl",
        "Corpus_Specific": f"{BASE_DIR}/elser_v1_corpus_specific_predictions/fiqa_fiqa_optimal_6U0A_20260112_130952.jsonl",
        "Structural_Minimal": f"{BASE_DIR}/elser_v1_structural_6U3A_predictions/fiqa_elser_minimal_optimized_20260112_115931.jsonl",
        "COT": f"{BASE_DIR}/elser_v1_cot_hyde_predictions/fiqa_elser_cot_6U3A_20260114_142219.jsonl",
        "Hyde": f"{BASE_DIR}/elser_v1_cot_hyde_predictions/fiqa_elser_hyde_6U3A_20260114_162827.jsonl",
        "Decomp": f"{BASE_DIR}/elser_v1_decomposition_predictions/fiqa_elser_decomposition_6U3A_20260114_153356.jsonl"
    },
    "govt": {
        "Anchor_Keyword": f"{BASE_DIR}/elser_v1_anchor_keyword_predictions/govt_elser_anchor_keyword_combo_6U1A_20260112_202122.jsonl",
        "Corpus_Specific": f"{BASE_DIR}/elser_v1_corpus_specific_predictions/govt_govt_corpus_baseline_6U3A_20260112_132109.jsonl",
        "Structural_Minimal": f"{BASE_DIR}/elser_v1_structural_6U3A_predictions/govt_elser_minimal_optimized_20260112_121434.jsonl",
        "COT": f"{BASE_DIR}/elser_v1_cot_hyde_predictions/govt_elser_cot_6U3A_20260114_144153.jsonl",
        "Hyde": f"{BASE_DIR}/elser_v1_cot_hyde_predictions/govt_elser_hyde_6U3A_20260114_164138.jsonl",
        "Decomp": f"{BASE_DIR}/elser_v1_decomposition_predictions/govt_elser_decomposition_6U3A_20260114_154710.jsonl"
    },
    "cloud": {
        "Anchor_Keyword": f"{BASE_DIR}/elser_v1_anchor_keyword_predictions/cloud_elser_anchor_keyword_combo_6U1A_20260112_203200.jsonl",
        "Corpus_Specific": f"{BASE_DIR}/elser_v1_corpus_specific_predictions/cloud_cloud_corpus_baseline_6U3A_20260112_133206.jsonl",
        "Structural_Minimal": f"{BASE_DIR}/elser_v1_structural_6U3A_predictions/cloud_elser_minimal_optimized_20260112_122934.jsonl",
        "COT": f"{BASE_DIR}/elser_v1_cot_hyde_predictions/cloud_elser_cot_6U3A_20260114_145758.jsonl",
        "Hyde": f"{BASE_DIR}/elser_v1_cot_hyde_predictions/cloud_elser_hyde_6U3A_20260114_165333.jsonl",
        "Decomp": f"{BASE_DIR}/elser_v1_decomposition_predictions/cloud_elser_decomposition_6U3A_20260114_160014.jsonl"
    }
}

# Corpus-specific files
CORPUS_CONFIG = {
    "fiqa": {
        "corpus_file": f"{BASE_PATH}/corpora/fiqa.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/fiqa/fiqa_lastturn.jsonl"
    },
    "govt": {
        "corpus_file": f"{BASE_PATH}/corpora/govt.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/govt/govt_lastturn.jsonl"
    },
    "cloud": {
        "corpus_file": f"{BASE_PATH}/corpora/cloud.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/cloud/cloud_lastturn.jsonl"
    }
}

TOP_K_RERANK = 100
COHERE_WEIGHTS = [0.1, 0.3, 0.5, 0.7, 0.9]  # Grid search weights

# ==============================================================================
# 2. HELPER FUNCTIONS (IO & PROCESSING)
# ==============================================================================

def load_jsonl(path: str) -> List[dict]:
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

def load_corpus(path: str) -> Dict[str, str]:
    """Load corpus documents with combined title + text."""
    corpus = {}
    if not os.path.exists(path):
        print(f"⚠️  Corpus file not found: {path}")
        return corpus

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            did = str(j.get("_id") or j.get("id"))
            # Combine title and text for better reranking
            text = (j.get("title", "") + " " + (j.get("text") or j.get("contents") or "")).strip()
            corpus[did] = text[:2000]  # Token limit for cost efficiency
    return corpus

def load_queries(path: str) -> Dict[str, str]:
    """Load query texts by ID."""
    queries = {}
    if not os.path.exists(path):
        print(f"⚠️  Queries file not found: {path}")
        return queries

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            qid = str(j.get("_id") or j.get("id") or j.get("task_id"))
            queries[qid] = j.get("text") or j.get("query")
    return queries

def azure_cohere_rerank(query: str, docs_text: List[str]) -> List[float]:
    """Call Cohere v4 Reranker via Azure Endpoint."""
    if not query or not docs_text:
        return [0.0] * len(docs_text)

    headers = {"Content-Type": "application/json", "api-key": AZURE_COHERE_KEY}
    payload = {
        "model": "Cohere-rerank-v4.0-pro",
        "query": query,
        "documents": docs_text,
        "top_n": len(docs_text)
    }

    try:
        response = requests.post(AZURE_COHERE_ENDPOINT, headers=headers, json=payload, timeout=60)
        if response.status_code == 200:
            results = response.json()["results"]
            scores = [0.0] * len(docs_text)
            for r in results:
                scores[r["index"]] = float(r["relevance_score"])
            return scores
        else:
            print(f"⚠️  API Error: {response.status_code} - {response.text[:200]}")
    except Exception as e:
        print(f"❌ Connection Error: {e}")

    return [0.0] * len(docs_text)

def minmax_norm(scores: List[float]) -> np.ndarray:
    """Normalize scores to [0, 1] range."""
    arr = np.array(scores, dtype=np.float32)
    if arr.max() == arr.min():
        return np.zeros_like(arr)
    return (arr - arr.min()) / (arr.max() - arr.min())

def run_eval(input_jsonl: str) -> Tuple[float, float]:
    """Run evaluation script and return Recall@5 and nDCG@5."""
    result_file = input_jsonl.replace(".jsonl", "_results.jsonl")
    csv_file = result_file.replace(".jsonl", "_aggregate.csv")

    cmd = f"python3 {EVAL_SCRIPT_PATH} --input_file {input_jsonl} --output_file {result_file}"
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        if "collection" in df.columns:
            df = df[df["collection"] != "all"]
        if df.empty:
            return 0.0, 0.0
        row = df.iloc[0]

        # Parse metric (handle string list format)
        def parse_metric(val):
            if isinstance(val, str) and "[" in val:
                import ast
                return ast.literal_eval(val)[2]  # Index 2 is usually @5
            return val

        return parse_metric(row.get("Recall", 0.0)), parse_metric(row.get("nDCG", 0.0))

    return 0.0, 0.0

# ==============================================================================
# 3. MAIN RERANKING & FUSION PIPELINE
# ==============================================================================

print("="*80)
print("🚀 COHERE V4 RERANKING - MULTI-CORPUS (FiQA, Govt, Cloud)")
print("="*80)
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Run Tag: {RUN_TAG}\n")

performance_summary = []

for corpus_name, files in INPUT_FILES.items():
    print(f"\n{'='*80}")
    print(f"📚 CORPUS: {corpus_name.upper()}")
    print(f"{'='*80}")

    # Load corpus-specific data
    corpus_cfg = CORPUS_CONFIG[corpus_name]
    corpus_map = load_corpus(corpus_cfg["corpus_file"])
    query_map = load_queries(corpus_cfg["queries_file"])

    if not corpus_map or not query_map:
        print(f"⚠️  Skipping {corpus_name} - missing corpus or queries\n")
        continue

    print(f"✅ Loaded {len(corpus_map)} docs and {len(query_map)} queries")

    for strategy_name, file_path in files.items():
        if not os.path.exists(file_path):
            print(f"⚠️  File not found: {file_path}")
            continue

        print(f"\n🎯 Processing: {strategy_name}")
        preds = load_jsonl(file_path)

        # --- STEP 1: Compute Cohere Scores (once per query) ---
        cache = []
        for p in tqdm(preds, desc=f"{corpus_name}/{strategy_name} Rerank", leave=False):
            qid = str(p["task_id"])
            q_txt = query_map.get(qid, "")

            top_ctxs = p["contexts"][:TOP_K_RERANK]
            rest_ctxs = p["contexts"][TOP_K_RERANK:]

            # Get document texts from corpus
            doc_texts = [corpus_map.get(str(c["document_id"]), "N/A") for c in top_ctxs]

            # Call Azure Cohere
            coh_scores = azure_cohere_rerank(q_txt, doc_texts)
            orig_scores = [float(c["score"]) for c in top_ctxs]

            cache.append({
                "task_id": qid,
                "Collection": p.get("Collection"),
                "top_ctxs": top_ctxs,
                "rest": rest_ctxs,
                "orig_scores": orig_scores,
                "coh_scores": coh_scores
            })

        # --- STEP 2: Evaluate Cohere-Only (cw=1.0) ---
        cohere_only_file = os.path.join(OUTPUT_DIR, f"{corpus_name}_{strategy_name}_COHERE_ONLY.jsonl")
        with open(cohere_only_file, "w", encoding="utf-8") as f:
            for item in cache:
                new_ctxs = []
                for i, ctx in enumerate(item["top_ctxs"]):
                    nc = ctx.copy()
                    nc["score"] = float(item["coh_scores"][i])
                    new_ctxs.append(nc)
                new_ctxs.sort(key=lambda x: x["score"], reverse=True)
                f.write(json.dumps({
                    "task_id": item["task_id"],
                    "Collection": item["Collection"],
                    "contexts": new_ctxs + item["rest"]
                }) + "\n")

        r5, n5 = run_eval(cohere_only_file)
        performance_summary.append({
            "Corpus": corpus_name,
            "Strategy": strategy_name,
            "Method": "Cohere-Only",
            "Recall@5": r5,
            "nDCG@5": n5
        })
        print(f"  ✅ Cohere-Only: Recall@5={r5:.4f}, nDCG@5={n5:.4f}")

        # --- STEP 3: Grid Search Hybrid Fusion ---
        best_recall = r5
        best_ndcg = n5
        best_weight = 1.0

        for cw in COHERE_WEIGHTS:
            ow = 1.0 - cw
            hybrid_file = os.path.join(OUTPUT_DIR, f"{corpus_name}_{strategy_name}_HYBRID_cw{cw}.jsonl")

            with open(hybrid_file, "w", encoding="utf-8") as f:
                for item in cache:
                    n_orig = minmax_norm(item["orig_scores"])
                    n_coh = minmax_norm(item["coh_scores"])
                    # Hybrid scoring formula
                    combined = (ow * n_orig) + (cw * n_coh)

                    new_ctxs = []
                    for i, ctx in enumerate(item["top_ctxs"]):
                        nc = ctx.copy()
                        nc["score"] = float(combined[i])
                        new_ctxs.append(nc)
                    new_ctxs.sort(key=lambda x: x["score"], reverse=True)
                    f.write(json.dumps({
                        "task_id": item["task_id"],
                        "Collection": item["Collection"],
                        "contexts": new_ctxs + item["rest"]
                    }) + "\n")

            r5, n5 = run_eval(hybrid_file)
            performance_summary.append({
                "Corpus": corpus_name,
                "Strategy": strategy_name,
                "Method": f"Hybrid (cw={cw})",
                "Recall@5": r5,
                "nDCG@5": n5
            })

            if n5 > best_ndcg:
                best_ndcg = n5
                best_recall = r5
                best_weight = cw

        print(f"  🏆 Best Hybrid: cw={best_weight}, Recall@5={best_recall:.4f}, nDCG@5={best_ndcg:.4f}")

# ==============================================================================
# 4. FINAL RESULTS DISPLAY
# ==============================================================================
df_final = pd.DataFrame(performance_summary)

print("\n" + "="*80)
print("🏆 FINAL RERANKING & HYBRID FUSION RESULTS")
print("="*80 + "\n")

# Display sorted by corpus, strategy, and nDCG
display(df_final.sort_values(["Corpus", "Strategy", "nDCG@5"], ascending=[True, True, False]))

# Save detailed report
report_file = os.path.join(OUTPUT_DIR, "reranking_final_report.csv")
df_final.to_csv(report_file, index=False)
print(f"\n💾 Saved detailed report: {report_file}")

# Summary statistics by corpus
print("\n" + "="*80)
print("📊 SUMMARY BY CORPUS")
print("="*80 + "\n")
summary_by_corpus = df_final.groupby("Corpus")[["Recall@5", "nDCG@5"]].agg(["mean", "max"])
display(summary_by_corpus)

# Best method per corpus
print("\n" + "="*80)
print("🥇 BEST METHOD PER CORPUS")
print("="*80 + "\n")
for corpus in df_final["Corpus"].unique():
    corpus_df = df_final[df_final["Corpus"] == corpus]
    best_row = corpus_df.loc[corpus_df["nDCG@5"].idxmax()]
    print(f"{corpus.upper()}:")
    print(f"  Strategy: {best_row['Strategy']}")
    print(f"  Method: {best_row['Method']}")
    print(f"  Recall@5: {best_row['Recall@5']:.4f}")
    print(f"  nDCG@5: {best_row['nDCG@5']:.4f}\n")

print("="*80)
print("✅ RERANKING COMPLETE!")
print(f"📁 All results saved to: {OUTPUT_DIR}")
print("="*80)

## Retrieval results and reranked results

In [ ]:
import json
import csv
import os
import pandas as pd
import numpy as np
from collections import defaultdict

# ==============================================================================
# 1. ΡΥΘΜΙΣΕΙΣ (ΟΠΩΣ ΠΡΙΝ)
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
BASE_DIR = "/content/drive/MyDrive/Project"
FILE_MAP = {
    "clapnq": {
        "Anchor_Keyword": f"{BASE_PATH}/elser_v1_anchor_keyword_predictions/clapnq_elser_anchor_keyword_combo_6U1A_20260112_200002.jsonl",
        "Corpus_Specific": f"{BASE_PATH}/elser_v1_corpus_specific_predictions/clapnq_clapnq_wikipedia_optimized2_6U3A_20260112_125554.jsonl",
        "Structural_Minimal": f"{BASE_PATH}/elser_v1_structural_6U3A_predictions/clapnq_elser_minimal_optimized_20260112_114149.jsonl",
        "COT": f"{BASE_PATH}/elser_v1_cot_hyde_predictions/clapnq_elser_cot_6U3A_20260114_140449.jsonl",
        "Hyde": f"{BASE_PATH}/elser_v1_cot_hyde_predictions/clapnq_elser_hyde_6U3A_20260114_161648.jsonl",
        "Hyde prf": f"{BASE_PATH}/elser_v1_context_hyde_predictions/clapnq_context_hyde_top3_20260207_165315.jsonl",
        "Decomp": f"{BASE_PATH}/elser_v1_decomposition_predictions/clapnq_elser_decomposition_6U3A_20260114_152005.jsonl",
        "ICR_Semantic": f"{BASE_PATH}/elser_v1_ICR_predictions/clapnq_preds_20260113_144416.jsonl",
        "Type": f"{BASE_PATH}/elser_v1_type_optimized_predictions/clapnq_type_optimized_mtrag_v1_20260116_191916.jsonl"

    },
    "fiqa": {
        "Anchor_Keyword": f"{BASE_DIR}/elser_v1_anchor_keyword_predictions/fiqa_elser_anchor_keyword_combo_6U1A_20260112_201042.jsonl",
        "Organizers": f"{BASE_DIR}/elser_v1_structural_6U3A_predictions/fiqa_elser_organizers_20260112_140752.jsonl",
        "Corpus_Specific": f"{BASE_DIR}/elser_v1_corpus_specific_predictions/fiqa_fiqa_optimal_6U0A_20260112_130952.jsonl",
        "Structural_Minimal": f"{BASE_DIR}/elser_v1_structural_6U3A_predictions/fiqa_elser_minimal_optimized_20260112_115931.jsonl",
        "COT": f"{BASE_DIR}/elser_v1_cot_hyde_predictions/fiqa_elser_cot_6U3A_20260114_142219.jsonl",
        "Hyde": f"{BASE_DIR}/elser_v1_cot_hyde_predictions/fiqa_elser_hyde_6U3A_20260114_162827.jsonl",
        "Hyde prf": f"{BASE_PATH}/elser_v1_context_hyde_predictions/fiqa_context_hyde_top3_20260207_170331.jsonl",
        "Decomp": f"{BASE_DIR}/elser_v1_decomposition_predictions/fiqa_elser_decomposition_6U3A_20260114_153356.jsonl",
        "ICR_Semantic": f"{BASE_PATH}/elser_v1_ICR_predictions/fiqa_preds_20260113_145706.jsonl",
    },
    "govt": {
        "Anchor_Keyword": f"{BASE_DIR}/elser_v1_anchor_keyword_predictions/govt_elser_anchor_keyword_combo_6U1A_20260112_202122.jsonl",
        "Organizers": f"{BASE_DIR}/elser_v1_structural_6U3A_predictions/govt_elser_organizers_20260112_142144.jsonl",
        "Corpus_Specific": f"{BASE_DIR}/elser_v1_corpus_specific_predictions/govt_govt_corpus_baseline_6U3A_20260112_132109.jsonl",
        "Structural_Minimal": f"{BASE_DIR}/elser_v1_structural_6U3A_predictions/govt_elser_minimal_optimized_20260112_121434.jsonl",
        "COT": f"{BASE_DIR}/elser_v1_cot_hyde_predictions/govt_elser_cot_6U3A_20260114_144153.jsonl",
        "Hyde": f"{BASE_DIR}/elser_v1_cot_hyde_predictions/govt_elser_hyde_6U3A_20260114_164138.jsonl",
        "Hyde prf": f"{BASE_PATH}/elser_v1_context_hyde_predictions/govt_context_hyde_top3_20260207_171350.jsonl",
        "Decomp": f"{BASE_DIR}/elser_v1_decomposition_predictions/govt_elser_decomposition_6U3A_20260114_154710.jsonl"
    },
    "cloud": {
        "Anchor_Keyword": f"{BASE_DIR}/elser_v1_anchor_keyword_predictions/cloud_elser_anchor_keyword_combo_6U1A_20260112_203200.jsonl",
        "Organizers": f"{BASE_DIR}/elser_v1_structural_6U3A_predictions/cloud_elser_organizers_20260112_143650.jsonl",
        "Corpus_Specific": f"{BASE_DIR}/elser_v1_corpus_specific_predictions/cloud_cloud_corpus_baseline_6U3A_20260112_133206.jsonl",
        "Corpus prf":"/content/drive/MyDrive/Project/elser_v1_context_corpus_specific_predictions/cloud_cloud_context_corpus_specific_top3_20260208_160532.jsonl",
        "Structural_Minimal": f"{BASE_DIR}/elser_v1_structural_6U3A_predictions/cloud_elser_minimal_optimized_20260112_122934.jsonl",
        "COT": f"{BASE_DIR}/elser_v1_cot_hyde_predictions/cloud_elser_cot_6U3A_20260114_145758.jsonl",
        "Hyde": f"{BASE_DIR}/elser_v1_cot_hyde_predictions/cloud_elser_hyde_6U3A_20260114_165333.jsonl",
        "Hyde prf": f"{BASE_PATH}/elser_v1_context_hyde_predictions/cloud_context_hyde_top3_20260207_172324.jsonl",
        "Decomp": f"{BASE_DIR}/elser_v1_decomposition_predictions/cloud_elser_decomposition_6U3A_20260114_160014.jsonl"
    }
}

REWRITES_MAP = {
    "clapnq": f"{BASE_PATH}/elser_v1_structural_6U3A_predictions/clapnq_elser_minimal_optimized_20260112_114149_REWRITES.json",
    "fiqa": f"{BASE_PATH}/elser_v1_structural_6U3A_predictions/fiqa_elser_minimal_optimized_20260112_115931_REWRITES.json",
    "govt": f"{BASE_PATH}/elser_v1_structural_6U3A_predictions/govt_elser_minimal_optimized_20260112_121434_REWRITES.json",
    "cloud": f"{BASE_PATH}/elser_v1_structural_6U3A_predictions/cloud_elser_minimal_optimized_20260112_122934_REWRITES.json"
}

K_VALUES = [5, 10, 20, 50, 100]

# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================
def load_qrels(corpus):
    path = os.path.join(BASE_PATH, "retrieval", corpus, "dev.tsv")
    qrels = {}
    with open(path, 'r', encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        next(reader)
        for row in reader:
            qid, docid, score = row[0], row[1], int(row[2])
            if score > 0:
                if qid not in qrels: qrels[qid] = set()
                qrels[qid].add(docid)
    return qrels

def calculate_recall(qrels, results_dict, k_values):
    scores = {k: 0.0 for k in k_values}
    num_queries = len(results_dict)
    if num_queries == 0: return {f"Recall@{k}": 0.0 for k in k_values}
    for qid, retrieved in results_dict.items():
        gold = qrels.get(qid, set())
        if not gold: continue
        for k in k_values:
            top_k = set(retrieved[:k])
            scores[k] += len(top_k.intersection(gold)) / len(gold)
    return {f"Recall@{k}": round(scores[k] / num_queries, 5) for k in k_values}

# ==============================================================================
# 3. ΜΑΖΙΚΗ ΑΝΑΛΥΣΗ ΜΕ ΥΠΟΛΟΓΙΣΜΟ RANKS
# ==============================================================================
all_rows = []
study_cases = []

for corpus in ["clapnq", "fiqa", "govt", "cloud"]:
    print(f"Processing {corpus.upper()}...")
    qrels = load_qrels(corpus)

    # Φόρτωση Rewrites και Corpus Texts
    rewrites = {}
    if os.path.exists(REWRITES_MAP[corpus]):
        with open(REWRITES_MAP[corpus], 'r') as f:
            for item in json.load(f): rewrites[item['qid']] = item['rewritten_query']

    texts = {}
    with open(os.path.join(BASE_PATH, "corpora", f"{corpus}.jsonl"), 'r') as f:
        for line in f:
            j = json.loads(line)
            texts[j['_id']] = j['text']

    strat_finds_ranks = {} # strat -> {(qid, did): rank}
    strat_metrics = {}
    rank1_mistakes = {}

    for strat_name, fpath in FILE_MAP[corpus].items():
        if not os.path.exists(fpath): continue

        res_list = {}
        found_with_ranks = {} # (qid, did) -> rank
        with open(fpath, 'r') as f:
            for line in f:
                item = json.loads(line)
                qid = item["task_id"]
                if qid not in qrels: continue

                retrieved = [c["document_id"] for c in item.get("contexts", [])]
                res_list[qid] = retrieved

                gold_ids = qrels[qid]
                # Καταγραφή θέσης (rank) για κάθε gold doc που βρέθηκε στο Top-100
                for rank, did in enumerate(retrieved[:100], 1):
                    if did in gold_ids:
                        found_with_ranks[(qid, did)] = rank

                if strat_name == "Structural_Minimal" and retrieved:
                    rank1_mistakes[qid] = texts.get(retrieved[0], "N/A")

        strat_finds_ranks[strat_name] = found_with_ranks
        strat_metrics[strat_name] = calculate_recall(qrels, res_list, K_VALUES)

    # Unique Analysis & Mean Ranks
    all_found_pairs = set().union(*(d.keys() for d in strat_finds_ranks.values()))

    for name, found_ranks in strat_finds_ranks.items():
        # Εύρεση Unique Finds για τη συγκεκριμένη στρατηγική
        others_found = set().union(*(d.keys() for k, d in strat_finds_ranks.items() if k != name))
        unique_pairs = set(found_ranks.keys()) - others_found

        # Υπολογισμός Mean Ranks
        all_ranks = list(found_ranks.values())
        unique_ranks = [found_ranks[p] for p in unique_pairs]

        mean_rank = round(np.mean(all_ranks), 2) if all_ranks else 0
        mean_unique_rank = round(np.mean(unique_ranks), 2) if unique_ranks else 0

        row = {
            "Corpus": corpus,
            "Strategy": name,
            "UniqueFinds": len(unique_pairs),
            "MeanRank": mean_rank,
            "MeanUniqueRank": mean_unique_rank
        }
        row.update(strat_metrics[name])
        all_rows.append(row)

# Export & Display
df_final = pd.DataFrame(all_rows)
cols = ["Corpus", "Strategy"] + [f"Recall@{k}" for k in K_VALUES] + ["UniqueFinds", "MeanRank", "MeanUniqueRank"]
df_final = df_final[cols]

display(df_final)

df_final.to_csv(f"{BASE_PATH}/detailed_retrieval_comparison.csv", index=False)
print(f"\n✅ Η ανάλυση ολοκληρώθηκε! Τα αποτελέσματα αποθηκεύτηκαν στο detailed_retrieval_comparison.csv")

In [ ]:
import json
import csv
import os
import pandas as pd
import numpy as np
from collections import defaultdict

# ==============================================================================
# 1. ΡΥΘΜΙΣΕΙΣ
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
BASE_DIR = "/content/drive/MyDrive/Project"

FILE_MAP = {
    "clapnq": {
        "Anchor_Keyword": f"{BASE_PATH}/cohere_v4_rerank_results/20260113_155329/Anchor_Keyword_HYBRID_cw0.5.jsonl",
        "Corpus_Specific": f"{BASE_PATH}/cohere_v4_rerank_results/20260113_155329/Corpus_Specific_HYBRID_cw0.5.jsonl",
        "Structural_Minimal": f"{BASE_PATH}/cohere_v4_rerank_results/20260113_155329/Structural_Minimal_HYBRID_cw0.5.jsonl",
        "COT": f"{BASE_PATH}/cohere_v4_rerank_results/20260115_105920/COT_HYBRID_cw0.5.jsonl",
        "Hyde": f"{BASE_PATH}/cohere_v4_rerank_results/20260115_105920/Hyde_HYBRID_cw0.5.jsonl",
        "Decomp": f"{BASE_PATH}/cohere_v4_rerank_results/20260115_105920/Decomp_HYBRID_cw0.5.jsonl",
    },
    "fiqa": {
        "Anchor_Keyword": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/fiqa_Anchor_Keyword_HYBRID_cw0.5.jsonl",
        "Corpus_Specific": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/fiqa_Corpus_Specific_HYBRID_cw0.5.jsonl",
        "Structural_Minimal": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/fiqa_Structural_Minimal_HYBRID_cw0.5.jsonl",
        "COT": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/fiqa_COT_HYBRID_cw0.5.jsonl",
        "Hyde": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/fiqa_Hyde_HYBRID_cw0.5.jsonl",
        "Decomp": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/fiqa_Decomp_HYBRID_cw0.5.jsonl"
    },
    "govt": {
        "Anchor_Keyword": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/govt_Anchor_Keyword_HYBRID_cw0.5.jsonl",
        "Corpus_Specific": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/govt_Corpus_Specific_HYBRID_cw0.5.jsonl",
        "Structural_Minimal": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/govt_Structural_Minimal_HYBRID_cw0.5.jsonl",
        "COT": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/govt_COT_HYBRID_cw0.5.jsonl",
        "Hyde": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/govt_Hyde_HYBRID_cw0.5.jsonl",
        "Decomp": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/govt_Decomp_HYBRID_cw0.5.jsonl"
    },
    "cloud": {
        "Anchor_Keyword": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/cloud_Anchor_Keyword_HYBRID_cw0.5.jsonl",
        "Corpus_Specific": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/cloud_Corpus_Specific_HYBRID_cw0.5.jsonl",
        "Structural_Minimal": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/cloud_Structural_Minimal_HYBRID_cw0.5.jsonl",
        "COT": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/cloud_COT_HYBRID_cw0.5.jsonl",
        "Hyde": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/cloud_Hyde_HYBRID_cw0.5.jsonl",
        "Decomp": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/cloud_Decomp_HYBRID_cw0.5.jsonl"
    }
}

K_VALUES = [5, 10, 20, 50, 100]

# ==============================================================================
# 2. HELPER FUNCTIONS με DEBUG
# ==============================================================================
def load_qrels(corpus):
    path = os.path.join(BASE_PATH, "retrieval", corpus, "dev.tsv")
    qrels = {}

    if not os.path.exists(path):
        print(f"⚠️  QRELS NOT FOUND: {path}")
        return qrels

    with open(path, 'r', encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        header = next(reader)
        print(f"  📄 QRELS Header: {header}")

        for row in reader:
            qid, docid, score = row[0], row[1], int(row[2])
            if score > 0:
                if qid not in qrels:
                    qrels[qid] = set()
                qrels[qid].add(docid)

    print(f"  ✅ Loaded {len(qrels)} queries with gold docs")
    if qrels:
        sample_qid = list(qrels.keys())[0]
        print(f"  📌 Sample qrel ID: '{sample_qid}' -> {len(qrels[sample_qid])} gold docs")

    return qrels


def calculate_recall(qrels, results_dict, k_values, corpus_name="", strat_name=""):
    scores = {k: 0.0 for k in k_values}
    num_queries = len(results_dict)

    if num_queries == 0:
        print(f"    ⚠️  [{corpus_name}/{strat_name}] No queries in results_dict!")
        return {f"Recall@{k}": 0.0 for k in k_values}

    matched_queries = 0
    for qid, retrieved in results_dict.items():
        gold = qrels.get(qid, set())
        if not gold:
            continue

        matched_queries += 1
        for k in k_values:
            top_k = set(retrieved[:k])
            scores[k] += len(top_k.intersection(gold)) / len(gold)

    print(f"    📊 [{corpus_name}/{strat_name}] Matched {matched_queries}/{num_queries} queries with qrels")

    return {f"Recall@{k}": round(scores[k] / num_queries, 5) for k in k_values}


# ==============================================================================
# 3. ΜΑΖΙΚΗ ΑΝΑΛΥΣΗ ΜΕ DEBUG
# ==============================================================================
all_rows = []

for corpus in ["clapnq", "fiqa", "govt", "cloud"]:
    print(f"\n{'='*70}")
    print(f"🔍 Processing {corpus.upper()}")
    print(f"{'='*70}")

    qrels = load_qrels(corpus)

    if not qrels:
        print(f"⚠️  Skipping {corpus} - no qrels loaded\n")
        continue

    strat_finds_ranks = {}
    strat_metrics = {}

    for strat_name, fpath in FILE_MAP[corpus].items():
        print(f"\n  📂 Strategy: {strat_name}")
        print(f"     Path: {fpath}")

        if not os.path.exists(fpath):
            print(f"     ❌ File not found!")
            continue

        res_list = {}
        found_with_ranks = {}

        with open(fpath, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            print(f"     ✅ Loaded {len(lines)} lines")

            # Debug first line
            if lines:
                first_item = json.loads(lines[0])
                print(f"     📌 Sample task_id: '{first_item.get('task_id')}'")
                print(f"     📌 Sample Collection: '{first_item.get('Collection')}'")
                contexts_count = len(first_item.get('contexts', []))
                print(f"     📌 Sample contexts: {contexts_count} docs")

            for line in lines:
                item = json.loads(line)
                qid = str(item["task_id"])  # 🔥 Ensure string conversion

                if qid not in qrels:
                    continue

                retrieved = [str(c["document_id"]) for c in item.get("contexts", [])]  # 🔥 Convert to string
                res_list[qid] = retrieved

                gold_ids = qrels[qid]
                for rank, did in enumerate(retrieved[:100], 1):
                    if did in gold_ids:
                        found_with_ranks[(qid, did)] = rank

        print(f"     📊 Matched {len(res_list)} queries with qrels")

        if len(res_list) == 0:
            print(f"     ⚠️  NO MATCHES! Checking ID mismatch...")
            # Debug: Compare first 3 task_ids with first 3 qrel IDs
            with open(fpath, 'r') as f:
                sample_tasks = [json.loads(line)["task_id"] for line in f.readlines()[:3]]
            sample_qrels = list(qrels.keys())[:3]
            print(f"     🔍 Sample task_ids from file: {sample_tasks}")
            print(f"     🔍 Sample qrel IDs: {sample_qrels}")

        strat_finds_ranks[strat_name] = found_with_ranks
        strat_metrics[strat_name] = calculate_recall(qrels, res_list, K_VALUES, corpus, strat_name)

    # Unique Analysis & Mean Ranks
    all_found_pairs = set().union(*(d.keys() for d in strat_finds_ranks.values()))

    for name, found_ranks in strat_finds_ranks.items():
        others_found = set().union(*(d.keys() for k, d in strat_finds_ranks.items() if k != name))
        unique_pairs = set(found_ranks.keys()) - others_found

        all_ranks = list(found_ranks.values())
        unique_ranks = [found_ranks[p] for p in unique_pairs]

        mean_rank = round(np.mean(all_ranks), 2) if all_ranks else 0
        mean_unique_rank = round(np.mean(unique_ranks), 2) if unique_ranks else 0

        row = {
            "Corpus": corpus,
            "Strategy": name,
            "UniqueFinds": len(unique_pairs),
            "MeanRank": mean_rank,
            "MeanUniqueRank": mean_unique_rank
        }
        row.update(strat_metrics[name])
        all_rows.append(row)

# Export & Display
print(f"\n{'='*70}")
print("📊 FINAL RESULTS")
print(f"{'='*70}\n")

df_final = pd.DataFrame(all_rows)
cols = ["Corpus", "Strategy"] + [f"Recall@{k}" for k in K_VALUES] + ["UniqueFinds", "MeanRank", "MeanUniqueRank"]
df_final = df_final[cols]

display(df_final)

output_path = f"{BASE_PATH}/detailed_retrieval_comparison_DEBUG.csv"
df_final.to_csv(output_path, index=False)
print(f"\n✅ Η ανάλυση ολοκληρώθηκε! Αποθηκεύτηκε στο: {output_path}")


In [ ]:
import os, json, time, requests, subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import Dict, List, Tuple


# ==============================================================================
# 1. CONFIGURATION - BATCH PROCESSING ΓΙΑ 3 CORPORA
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"

# Ορισμός όλων των corpora και των αρχείων τους
CORPUS_CONFIGS = {
    "clapnq": {
        "input_file": f"{BASE_PATH}/elser_v1_fixed_type_predictions/clapnq_fixed_type_aware_v2_20260117_223328.jsonl",
        "corpus_file": f"{BASE_PATH}/corpora/clapnq.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/clapnq/clapnq_lastturn.jsonl",
        "rewrites_file": f"{BASE_PATH}/elser_v1_fixed_type_predictions/clapnq_fixed_type_aware_v2_20260117_223328_REWRITES.json"
    },
    "fiqa": {
        "input_file": f"{BASE_PATH}/elser_v1_fixed_type_predictions/fiqa_fixed_type_aware_v2_20260117_224418.jsonl",
        "corpus_file": f"{BASE_PATH}/corpora/fiqa.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/fiqa/fiqa_lastturn.jsonl",
        "rewrites_file": f"{BASE_PATH}/elser_v1_fixed_type_predictions/fiqa_fixed_type_aware_v2_20260117_224418_REWRITES.json"
    },
    "govt": {
        "input_file": f"{BASE_PATH}/elser_v1_fixed_type_predictions/govt_fixed_type_aware_v2_20260117_230421.jsonl",
        "corpus_file": f"{BASE_PATH}/corpora/govt.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/govt/govt_lastturn.jsonl",
        "rewrites_file": f"{BASE_PATH}/elser_v1_fixed_type_predictions/govt_fixed_type_aware_v2_20260117_230421_REWRITES.json"
    }
}

# Output directory
RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_BASE = os.path.join(BASE_PATH, "cohere_v4_fixed_type_rerank", RUN_TAG)
os.makedirs(OUTPUT_BASE, exist_ok=True)

# Azure Cohere Credentials
AZURE_COHERE_ENDPOINT = os.environ["ENDPOINT"]
AZURE_COHERE_KEY = os.environ["_KEY"]

EVAL_SCRIPT_PATH = os.path.join(BASE_PATH, "run_retrieval_eval.py")
TOP_K_RERANK = 100
COHERE_WEIGHTS = [0.1, 0.3, 0.5, 0.7, 0.9]


# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================

def load_jsonl(path: str) -> List[dict]:
    data = []
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        return data
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data


def load_json(path: str) -> Dict:
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_corpus(path: str) -> Dict[str, str]:
    corpus = {}
    if not os.path.exists(path): return corpus
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            did = str(j.get("_id") or j.get("id"))
            text = (j.get("title", "") + " " + (j.get("text") or j.get("contents") or "")).strip()
            corpus[did] = text[:2000]
    return corpus


def load_queries(path: str) -> Dict[str, str]:
    queries = {}
    if not os.path.exists(path): return queries
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            qid = str(j.get("_id") or j.get("id") or j.get("task_id"))
            queries[qid] = j.get("text") or j.get("query")
    return queries


def load_rewrites(path: str) -> Dict[str, str]:
    rewrites_map = {}

    if not os.path.exists(path):
        print(f"⚠️ Rewrites file not found: {path}")
        return rewrites_map

    try:
        rewrites_data = load_json(path)

        if isinstance(rewrites_data, list):
            for item in rewrites_data:
                qid = item.get("qid") or item.get("task_id")
                rewrite = item.get("rewritten_query") or item.get("rewrite")
                if qid and rewrite:
                    rewrites_map[str(qid)] = rewrite
        elif isinstance(rewrites_data, dict):
            rewrites_map = {str(k): str(v) for k, v in rewrites_data.items()}

        print(f"✅ Loaded {len(rewrites_map)} rewrites")

    except Exception as e:
        print(f"❌ Error loading rewrites: {e}")

    return rewrites_map


def azure_cohere_rerank(query: str, docs_text: List[str]) -> List[float]:
    if not query or not docs_text: return [0.0] * len(docs_text)
    headers = {"Content-Type": "application/json", "api-key": AZURE_COHERE_KEY}
    payload = {
        "model": "Cohere-rerank-v4.0-pro",
        "query": query,
        "documents": docs_text,
        "top_n": len(docs_text)
    }
    try:
        response = requests.post(AZURE_COHERE_ENDPOINT, headers=headers, json=payload, timeout=60)
        if response.status_code == 200:
            results = response.json()["results"]
            scores = [0.0] * len(docs_text)
            for r in results:
                scores[r["index"]] = float(r["relevance_score"])
            return scores
        elif response.status_code == 429:
            time.sleep(2)  # Rate limit - wait and retry once
            return azure_cohere_rerank(query, docs_text)
    except Exception as e:
        print(f"❌ Cohere API Error: {e}")
    return [0.0] * len(docs_text)


def minmax_norm(scores: List[float]) -> np.ndarray:
    arr = np.array(scores, dtype=np.float32)
    if arr.max() == arr.min(): return np.zeros_like(arr)
    return (arr - arr.min()) / (arr.max() - arr.min())


def run_eval(input_jsonl: str) -> Tuple[float, float]:
    result_file = input_jsonl.replace(".jsonl", "_results.jsonl")
    csv_file = result_file.replace(".jsonl", "_aggregate.csv")
    cmd = f"python3 {EVAL_SCRIPT_PATH} --input_file {input_jsonl} --output_file {result_file}"
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        if "collection" in df.columns:
            df = df[df["collection"] != "all"]
        if df.empty:
            return 0.0, 0.0
        row = df.iloc[0]
        def parse_metric(val):
            if isinstance(val, str) and "[" in val:
                import ast
                return ast.literal_eval(val)[2]
            return val
        return parse_metric(row.get("Recall", 0.0)), parse_metric(row.get("nDCG", 0.0))
    return 0.0, 0.0


# ==============================================================================
# 3. PROCESS SINGLE CORPUS
# ==============================================================================

def process_corpus(corpus_name: str, config: Dict) -> pd.DataFrame:
    """Process a single corpus and return results DataFrame"""

    print("\n" + "="*80)
    print(f"🎯 PROCESSING: {corpus_name.upper()}")
    print("="*80)

    # Create corpus-specific output directory
    output_dir = os.path.join(OUTPUT_BASE, corpus_name)
    os.makedirs(output_dir, exist_ok=True)

    # Load data
    print("📥 Loading data...")
    corpus_map = load_corpus(config["corpus_file"])
    original_queries = load_queries(config["queries_file"])
    rewrites_map = load_rewrites(config["rewrites_file"])
    preds = load_jsonl(config["input_file"])

    if not preds:
        print(f"❌ Error: No predictions found for {corpus_name}")
        return pd.DataFrame()

    print(f"✅ Loaded {len(corpus_map)} corpus documents")
    print(f"✅ Loaded {len(original_queries)} original queries")
    print(f"✅ Loaded {len(rewrites_map)} rewritten queries")
    print(f"✅ Loaded {len(preds)} predictions\n")

    # Statistics
    queries_with_rewrites = 0
    queries_with_fallback = 0

    performance_summary = []
    cache = []

    # Step 1: Run Reranking
    print("🎯 Running Cohere reranking...")
    for p in tqdm(preds, desc=f"Reranking {corpus_name}", ncols=100):
        qid = str(p["task_id"])

        # Try rewritten query first
        if qid in rewrites_map:
            q_txt = rewrites_map[qid]
            queries_with_rewrites += 1
        else:
            q_txt = original_queries.get(qid, "")
            queries_with_fallback += 1

        top_ctxs = p["contexts"][:TOP_K_RERANK]
        rest_ctxs = p["contexts"][TOP_K_RERANK:]

        doc_texts = [corpus_map.get(str(c["document_id"]), "N/A") for c in top_ctxs]
        coh_scores = azure_cohere_rerank(q_txt, doc_texts)
        orig_scores = [float(c["score"]) for c in top_ctxs]

        cache.append({
            "task_id": qid,
            "Collection": p.get("Collection"),
            "top_ctxs": top_ctxs,
            "rest": rest_ctxs,
            "orig_scores": orig_scores,
            "coh_scores": coh_scores,
            "query_used": q_txt,
            "used_rewrite": qid in rewrites_map
        })

    print(f"\n📊 Query source statistics:")
    print(f"   • Queries with rewrites: {queries_with_rewrites} ({queries_with_rewrites/len(preds)*100:.1f}%)")
    print(f"   • Queries using fallback: {queries_with_fallback} ({queries_with_fallback/len(preds)*100:.1f}%)")

    # Step 2: Cohere-Only
    print("\n📊 Evaluating: Cohere-Only...")
    cohere_file = os.path.join(output_dir, "COHERE_ONLY.jsonl")
    with open(cohere_file, "w") as f:
        for item in cache:
            new_ctxs = []
            for i, ctx in enumerate(item["top_ctxs"]):
                nc = ctx.copy()
                nc["score"] = float(item["coh_scores"][i])
                new_ctxs.append(nc)
            new_ctxs.sort(key=lambda x: x["score"], reverse=True)
            f.write(json.dumps({
                "task_id": item["task_id"],
                "Collection": item["Collection"],
                "contexts": new_ctxs + item["rest"]
            }) + "\n")

    r5, n5 = run_eval(cohere_file)
    performance_summary.append({
        "Corpus": corpus_name,
        "Method": "Cohere-Only",
        "Weight": "1.0",
        "Recall@5": r5,
        "nDCG@5": n5
    })
    print(f"   Recall@5: {r5:.4f} | nDCG@5: {n5:.4f}")

    # Step 3: Hybrid Fusion
    for cw in COHERE_WEIGHTS:
        ow = 1.0 - cw
        print(f"\n📊 Evaluating: Hybrid (cw={cw})...")
        hybrid_file = os.path.join(output_dir, f"HYBRID_cw{cw}.jsonl")

        with open(hybrid_file, "w") as f:
            for item in cache:
                n_orig = minmax_norm(item["orig_scores"])
                n_coh = minmax_norm(item["coh_scores"])
                combined = (ow * n_orig) + (cw * n_coh)

                new_ctxs = []
                for i, ctx in enumerate(item["top_ctxs"]):
                    nc = ctx.copy()
                    nc["score"] = float(combined[i])
                    new_ctxs.append(nc)
                new_ctxs.sort(key=lambda x: x["score"], reverse=True)

                f.write(json.dumps({
                    "task_id": item["task_id"],
                    "Collection": item["Collection"],
                    "contexts": new_ctxs + item["rest"]
                }) + "\n")

        r5_h, n5_h = run_eval(hybrid_file)
        performance_summary.append({
            "Corpus": corpus_name,
            "Method": "Hybrid",
            "Weight": f"cw={cw}",
            "Recall@5": r5_h,
            "nDCG@5": n5_h
        })
        print(f"   Recall@5: {r5_h:.4f} | nDCG@5: {n5_h:.4f}")

    # Step 4: Original-Only
    print(f"\n📊 Evaluating: Original-Only...")
    orig_file = os.path.join(output_dir, "ORIGINAL_ONLY.jsonl")
    with open(orig_file, "w") as f:
        for item in cache:
            new_ctxs = []
            for i, ctx in enumerate(item["top_ctxs"]):
                nc = ctx.copy()
                nc["score"] = float(item["orig_scores"][i])
                new_ctxs.append(nc)
            new_ctxs.sort(key=lambda x: x["score"], reverse=True)
            f.write(json.dumps({
                "task_id": item["task_id"],
                "Collection": item["Collection"],
                "contexts": new_ctxs + item["rest"]
            }) + "\n")

    r5_orig, n5_orig = run_eval(orig_file)
    performance_summary.append({
        "Corpus": corpus_name,
        "Method": "Original-Only",
        "Weight": "0.0",
        "Recall@5": r5_orig,
        "nDCG@5": n5_orig
    })
    print(f"   Recall@5: {r5_orig:.4f} | nDCG@5: {n5_orig:.4f}")

    # Save corpus-specific report
    df_corpus = pd.DataFrame(performance_summary)
    df_corpus = df_corpus.sort_values("nDCG@5", ascending=False)

    report_path = os.path.join(output_dir, f"{corpus_name}_report.csv")
    df_corpus.to_csv(report_path, index=False)

    print(f"\n✅ {corpus_name} report saved: {report_path}")

    return df_corpus


# ==============================================================================
# 4. MAIN - BATCH PROCESSING
# ==============================================================================

def main():
    print("="*80)
    print("🔥 COHERE V4 BATCH RERANKING - FIXED TYPE AWARE")
    print("="*80)
    print(f"\n📂 Processing {len(CORPUS_CONFIGS)} corpora: {list(CORPUS_CONFIGS.keys())}")
    print(f"💾 Output base: {OUTPUT_BASE}\n")

    all_results = []

    # Process each corpus
    for corpus_name, config in CORPUS_CONFIGS.items():
        df_corpus = process_corpus(corpus_name, config)
        if not df_corpus.empty:
            all_results.append(df_corpus)

    # Aggregate results
    if all_results:
        print("\n" + "="*80)
        print("🏆 FINAL AGGREGATE RESULTS")
        print("="*80)

        df_all = pd.concat(all_results, ignore_index=True)

        # Group by method and weight
        df_summary = df_all.groupby(["Method", "Weight"]).agg({
            "Recall@5": "mean",
            "nDCG@5": "mean"
        }).reset_index()
        df_summary = df_summary.sort_values("nDCG@5", ascending=False)

        print("\nAGGREGATE PERFORMANCE (Average across all corpora):")
        print(df_summary.to_string(index=False))

        # Save aggregate report
        aggregate_report = os.path.join(OUTPUT_BASE, "AGGREGATE_REPORT.csv")
        df_summary.to_csv(aggregate_report, index=False)

        # Save detailed report
        detailed_report = os.path.join(OUTPUT_BASE, "DETAILED_REPORT.csv")
        df_all.to_csv(detailed_report, index=False)

        print(f"\n✅ Aggregate report: {aggregate_report}")
        print(f"✅ Detailed report: {detailed_report}")

        # Best configuration
        best = df_summary.iloc[0]
        print(f"\n🎯 BEST OVERALL CONFIGURATION:")
        print(f"   Method: {best['Method']}")
        print(f"   Weight: {best['Weight']}")
        print(f"   Avg Recall@5: {best['Recall@5']:.4f}")
        print(f"   Avg nDCG@5: {best['nDCG@5']:.4f}")

    print(f"\n📁 All results saved to: {OUTPUT_BASE}")


if __name__ == "__main__":
    main()

## RRF results

In [ ]:
import json
import os
from collections import defaultdict
import pandas as pd

# ==============================================================================
# RRF SETTINGS
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
BASE_DIR = "/content/drive/MyDrive/Project"
OUTPUT_DIR = f"{BASE_PATH}/rrf_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

K_RRF = 20  # RRF constant

# Manual weights per strategy (adjust as needed)
STRATEGY_WEIGHTS = {
    "Anchor_Keyword": 0.0,
    "Corpus_Specific": 1.0,
    "Structural_Minimal": 2.0,
    "COT": 0.0,
    "Hyde": 0.4,
    "Decomp": 0.0
}

FILE_MAP = {
    "clapnq": {
        "Anchor_Keyword": f"{BASE_PATH}/cohere_v4_rerank_results/20260113_155329/Anchor_Keyword_HYBRID_cw0.5.jsonl",
        "Corpus_Specific": f"{BASE_PATH}/cohere_v4_rerank_results/20260113_155329/Corpus_Specific_HYBRID_cw0.5.jsonl",
        "Structural_Minimal": f"{BASE_PATH}/cohere_v4_rerank_results/20260113_155329/Structural_Minimal_HYBRID_cw0.5.jsonl",
        "COT": f"{BASE_PATH}/cohere_v4_rerank_results/20260115_105920/COT_HYBRID_cw0.5.jsonl",
        "Hyde": f"{BASE_PATH}/cohere_v4_rerank_results/20260115_105920/Hyde_HYBRID_cw0.5.jsonl",
        "Decomp": f"{BASE_PATH}/cohere_v4_rerank_results/20260115_105920/Decomp_HYBRID_cw0.5.jsonl",
    },
    "fiqa": {
        "Anchor_Keyword": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/fiqa_Anchor_Keyword_HYBRID_cw0.5.jsonl",
        "Corpus_Specific": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/fiqa_Corpus_Specific_HYBRID_cw0.5.jsonl",
        "Structural_Minimal": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/fiqa_Structural_Minimal_HYBRID_cw0.5.jsonl",
        "COT": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/fiqa_COT_HYBRID_cw0.5.jsonl",
        "Hyde": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/fiqa_Hyde_HYBRID_cw0.5.jsonl",
        "Decomp": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/fiqa_Decomp_HYBRID_cw0.5.jsonl"
    },
    "govt": {
        "Anchor_Keyword": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/govt_Anchor_Keyword_HYBRID_cw0.5.jsonl",
        "Corpus_Specific": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/govt_Corpus_Specific_HYBRID_cw0.5.jsonl",
        "Structural_Minimal": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/govt_Structural_Minimal_HYBRID_cw0.5.jsonl",
        "COT": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/govt_COT_HYBRID_cw0.5.jsonl",
        "Hyde": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/govt_Hyde_HYBRID_cw0.5.jsonl",
        "Decomp": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/govt_Decomp_HYBRID_cw0.5.jsonl"
    },
    "cloud": {
        "Anchor_Keyword": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/cloud_Anchor_Keyword_HYBRID_cw0.5.jsonl",
        "Corpus_Specific": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/cloud_Corpus_Specific_HYBRID_cw0.5.jsonl",
        "Structural_Minimal": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/cloud_Structural_Minimal_HYBRID_cw0.5.jsonl",
        "COT": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/cloud_COT_HYBRID_cw0.5.jsonl",
        "Hyde": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/cloud_Hyde_HYBRID_cw0.5.jsonl",
        "Decomp": f"{BASE_DIR}/cohere_v4_rerank_results/20260115_011450/cloud_Decomp_HYBRID_cw0.5.jsonl"
    }
}


# ==============================================================================
# RRF FUSION FUNCTION
# ==============================================================================
def rrf_fusion(strategy_results, k=20, weights=None):
    if weights is None:
        weights = {s: 1.0 for s in strategy_results.keys()}

    all_qids = set()
    for strat_res in strategy_results.values():
        all_qids.update(strat_res.keys())

    fused = {}

    for qid in all_qids:
        doc_scores = defaultdict(float)

        for strat_name, qid_results in strategy_results.items():
            if qid not in qid_results:
                continue

            weight = weights.get(strat_name, 1.0)
            ranked_docs = qid_results[qid]

            for rank, doc_id in enumerate(ranked_docs, start=1):
                rrf_score = weight / (k + rank)
                doc_scores[doc_id] += rrf_score

        sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
        fused[qid] = sorted_docs

    return fused


# ==============================================================================
# LOAD STRATEGY RESULTS
# ==============================================================================
def load_strategy_results(corpus):
    strategy_results = {}
    query_metadata = {}

    for strat_name, fpath in FILE_MAP[corpus].items():
        if not os.path.exists(fpath):
            print(f"  ⚠️  File not found: {fpath}")
            continue

        strat_res = {}

        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                item = json.loads(line)
                qid = str(item["task_id"])

                if qid not in query_metadata:
                    query_metadata[qid] = {
                        "query": item.get("query", ""),
                        "Collection": item.get("Collection", corpus)
                    }

                contexts = item.get("contexts", [])
                ranked_docs = [str(c["document_id"]) for c in contexts]
                strat_res[qid] = ranked_docs

        strategy_results[strat_name] = strat_res
        print(f"  ✅ Loaded {strat_name}: {len(strat_res)} queries")

    return strategy_results, query_metadata


# ==============================================================================
# SAVE RRF RESULTS
# ==============================================================================
def save_rrf_results(corpus, fused_results, query_metadata, output_dir):
    output_path = os.path.join(output_dir, f"elserv1_{corpus}_rrf.jsonl")

    with open(output_path, 'w', encoding='utf-8') as f:
        for qid, ranked_docs in fused_results.items():
            metadata = query_metadata.get(qid, {})

            contexts = []
            for doc_id, rrf_score in ranked_docs:
                contexts.append({
                    "document_id": doc_id,
                    "score": round(rrf_score, 6)
                })

            output_item = {
                "task_id": qid,
                "Collection": metadata.get("Collection", corpus),
                "contexts": contexts
            }

            f.write(json.dumps(output_item, ensure_ascii=False) + "\n")

    print(f"  💾 Saved: {output_path} ({len(fused_results)} queries)")
    return output_path


# ==============================================================================
# MAIN EXECUTION - RRF FUSION
# ==============================================================================
print("=" * 70)
print("🔀 RECIPROCAL RANK FUSION (RRF)")
print("=" * 70)
print(f"\nRRF constant k = {K_RRF}")
print(f"Strategy weights: {STRATEGY_WEIGHTS}\n")

saved_files = []

for corpus in ["clapnq", "fiqa", "govt", "cloud"]:
    print(f"\n{'='*70}")
    print(f"📊 Processing: {corpus.upper()}")
    print(f"{'='*70}")

    strategy_results, query_metadata = load_strategy_results(corpus)

    if not strategy_results:
        print(f"  ⚠️  No results loaded for {corpus}, skipping...")
        continue

    print(f"\n  🔀 Fusing {len(strategy_results)} strategies...")
    fused_results = rrf_fusion(strategy_results, k=K_RRF, weights=STRATEGY_WEIGHTS)
    print(f"  ✅ Fused {len(fused_results)} queries")

    output_path = save_rrf_results(corpus, fused_results, query_metadata, OUTPUT_DIR)
    saved_files.append(output_path)

print(f"\n{'='*70}")
print("✅ RRF FUSION COMPLETE!")
print(f"{'='*70}")
print(f"\nSaved {len(saved_files)} files to: {OUTPUT_DIR}")
for fpath in saved_files:
    print(f"  📄 {os.path.basename(fpath)}")


# ==============================================================================
# EVALUATION OF RRF RESULTS
# ==============================================================================
print(f"\n\n{'='*70}")
print("📈 RUNNING EVALUATION ON RRF RESULTS")
print(f"{'='*70}\n")

evaluation_script = "/content/drive/MyDrive/Project/run_retrieval_eval.py"

if not os.path.exists(evaluation_script):
    print(f"⚠️ Evaluation script not found: {evaluation_script}")
    print("Skipping evaluation...")
else:
    for corpus in ["clapnq", "fiqa", "govt", "cloud"]:
        input_path = os.path.join(OUTPUT_DIR, f"elserv1_{corpus}_rrf.jsonl")
        output_path = os.path.join(OUTPUT_DIR, f"elserv1_{corpus}_rrf_results.jsonl")

        if not os.path.exists(input_path):
            print(f"⚠️ Skipping {corpus} - file not found: {input_path}")
            continue

        print(f"\n{'='*50}")
        print(f"Running evaluation for {corpus} ...")
        print(f"Input: {input_path}")
        print(f"Output: {output_path}")
        print(f"{'='*50}")

        cmd = f"python {evaluation_script} --input_file {input_path} --output_file {output_path}"
        result = os.system(cmd)

        if result != 0:
            print(f"⚠️ Warning: Evaluation failed for {corpus}")
        else:
            print(f"✅ Evaluation completed for {corpus}")


# ==============================================================================
# AGGREGATE RESULTS COMPARISON
# ==============================================================================
print(f"\n\n{'='*70}")
print("📊 AGGREGATING EVALUATION RESULTS")
print(f"{'='*70}\n")

csv_files = [
    f"{OUTPUT_DIR}/elserv1_clapnq_rrf_results_aggregate.csv",
    f"{OUTPUT_DIR}/elserv1_fiqa_rrf_results_aggregate.csv",
    f"{OUTPUT_DIR}/elserv1_govt_rrf_results_aggregate.csv",
    f"{OUTPUT_DIR}/elserv1_cloud_rrf_results_aggregate.csv"
]

all_data = []

for file in csv_files:
    if os.path.exists(file):
        df = pd.read_csv(file)
        df = df[df["collection"] != "all"]  # Remove aggregate row if present
        corpus_name = file.split("/")[-1].replace("elserv1_", "").replace("_rrf_results_aggregate.csv", "")
        df["Corpus"] = corpus_name
        all_data.append(df)
        print(f"✅ Loaded: {corpus_name}")
    else:
        print(f"⚠️ Not found: {file}")

if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df = final_df.set_index("Corpus")
    print("\n" + "="*70)
    print("📊 RRF EVALUATION RESULTS - ALL CORPORA")
    print("="*70)
    display(final_df)

    # Save combined results
    combined_output = f"{OUTPUT_DIR}/rrf_all_corpora_results.csv"
    final_df.to_csv(combined_output)
    print(f"\n💾 Combined results saved to: {combined_output}")
else:
    print("\n⚠️ No evaluation results found")

print("\n" + "="*70)
print("✅ ALL PROCESSING COMPLETE!")
print("="*70)


## Fusion of rewrites per corpora

# **Elser-v2 Pipeline**

In [ ]:
# ================================
# COMPLETE ELSER v1 SYSTEM - MAXIMUM SPEED (FIXED)
# ================================
!pip install -q elasticsearch==8.11.0 tqdm pandas numpy

import os
import json
import time
import base64
from typing import List, Dict, Optional
from elasticsearch import Elasticsearch, helpers
from tqdm import tqdm
from pathlib import Path
import elasticsearch
import concurrent.futures

print(f"✅ Elasticsearch client version: {elasticsearch.__version__}")

# ================================
# 1. SETUP & CONNECTION
# ================================

print("\n" + "="*70)
print("🚀 ELSER v1 COMPLETE PIPELINE - MAXIMUM SPEED")
print("="*70)

# Mount Google Drive
print("\n💾 Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Connect to Elasticsearch
print("\n🔌 Connecting to Elasticsearch...")
CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]

es = Elasticsearch(
    cloud_id=CLOUD_ID,
    basic_auth=("elastic", PASSWORD),
    request_timeout=180,
    max_retries=5,
    retry_on_timeout=True,
    verify_certs=True
)

if es.ping():
    info = es.info()
    print(f"✅ Connected to Elasticsearch!")
    print(f"   Cluster: {info['cluster_name']}")
    print(f"   Version: {info['version']['number']}")
else:
    raise ConnectionError("❌ Cannot connect to Elasticsearch!")

# ================================
# 2. DEPLOY ELSER v1 - MAXIMUM THROUGHPUT
# ================================

print("\n" + "="*70)
print("🔥 DEPLOYING ELSER v2 MODEL - MAX PERFORMANCE")
print("="*70)

MODEL_ID = ".elser_model_2_linux-x86_64"

# Check if model exists
print("\n1️⃣ Checking ELSER v1 model...")
try:
    model_info = es.ml.get_trained_models(model_id=MODEL_ID)
    print("✅ ELSER v1 model already exists")
except:
    # Download model
    try:
        es.ml.put_trained_model(
            model_id=MODEL_ID,
            input={"field_names": ["text_field"]}
        )
        print("✅ ELSER v1 model download initiated")
    except Exception as e:
        print(f"⚠️ Download error: {e}")

# Wait for full download
print("\n2️⃣ Waiting for model to be fully downloaded...")
max_wait = 600
elapsed = 0

while elapsed < max_wait:
    try:
        model_info = es.ml.get_trained_models(
            model_id=MODEL_ID,
            include="definition_status"
        )
        if model_info.get('trained_model_configs'):
            config = model_info['trained_model_configs'][0]
            if config.get('fully_defined', False):
                print(f"\n✅ ELSER v2 model fully downloaded!")
                break
    except:
        pass

    print(f"\r⏱️ Downloading... {elapsed}s", end='')
    time.sleep(10)
    elapsed += 10

# Stop existing deployment and redeploy with more allocations
print("\n3️⃣ Optimizing deployment for MAX throughput...")

try:
    # Stop existing deployment
    es.ml.stop_trained_model_deployment(model_id=MODEL_ID, force=True)
    print("   Stopped existing deployment")
    time.sleep(5)
except:
    pass

try:
    # Start with maximum allocations
    es.ml.start_trained_model_deployment(
        model_id=MODEL_ID,
        wait_for="started",
        number_of_allocations=4,
        threads_per_allocation=1
    )
    print("✅ ELSER v2 deployed with 4 allocations!")
except Exception as e:
    if "already started" in str(e).lower():
        print("✅ ELSER v2 deployment running")
    else:
        print(f"⚠️ Using existing deployment: {e}")

# Wait for deployment
print("\n4️⃣ Waiting for ELSER v2 to be ready...")
max_wait = 300
elapsed = 0

while elapsed < max_wait:
    try:
        stats = es.ml.get_trained_models_stats(model_id=MODEL_ID)
        if stats.get('trained_model_stats'):
            model_stat = stats['trained_model_stats'][0]
            if 'deployment_stats' in model_stat:
                state = model_stat['deployment_stats'].get('state')
                alloc = model_stat['deployment_stats'].get('allocation_status', {})
                current = alloc.get('allocation_count', 0)

                if state == 'started':
                    print(f"\n✅ ELSER v1 ready with {current} allocations!")
                    break

                print(f"\r⏱️ {elapsed}s | State: {state} | Alloc: {current}", end='')
    except:
        pass

    time.sleep(5)
    elapsed += 5

# ================================
# 3. LOAD DATA
# ================================

print("\n" + "="*70)
print("📂 LOADING DATASETS")
print("="*70)

DATA_PATH = "/content/drive/MyDrive/Project"

def load_corpus_jsonl(path: str) -> Dict[str, Dict[str, str]]:
    """Load corpus from JSONL file"""
    corpus = {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    j = json.loads(line)
                    did = j.get("_id") or j.get("id") or j.get("document_id")
                    if did:
                        corpus[did] = {"title": j.get("title", ""), "text": j.get("text", "")}
                except:
                    continue
    except FileNotFoundError:
        return {}
    return corpus

# Load datasets
datasets_info = {
    "clapnq": os.path.join(DATA_PATH, "corpora", "clapnq.jsonl"),
    "cloud": os.path.join(DATA_PATH, "corpora", "cloud.jsonl"),
    "fiqa": os.path.join(DATA_PATH, "corpora", "fiqa.jsonl"),
    "govt": os.path.join(DATA_PATH, "corpora", "govt.jsonl"),
}

datasets = {}
for name, path in datasets_info.items():
    print(f"\n📖 Loading {name}...")
    if not os.path.exists(path):
        print(f"   ⚠️ Skipping - file not found")
        continue

    corpus = load_corpus_jsonl(path)
    if corpus:
        docs = [v["text"] for v in corpus.values()]
        doc_ids = list(corpus.keys())
        datasets[name] = (docs, doc_ids)
        print(f"   ✓ {len(docs):,} documents loaded")

if not datasets:
    raise FileNotFoundError("❌ No datasets loaded!")

# ================================
# 4. UTILITY FUNCTIONS
# ================================

def check_existing_embeddings(backup_dir: str, index_name: str) -> bool:
    """Check if embeddings backup exists"""
    embeddings_file = os.path.join(backup_dir, f"{index_name}_embeddings.json")
    config_file = os.path.join(backup_dir, "configs", f"{index_name}_metadata.json")

    if os.path.exists(embeddings_file) and os.path.exists(config_file):
        try:
            with open(config_file, 'r') as f:
                metadata = json.load(f)
            file_size = os.path.getsize(embeddings_file) / 1e6
            print(f"\n📂 Found backup! ({metadata.get('doc_count', 0):,} docs, {file_size:.1f} MB)")
            return True
        except:
            return False
    return False

def export_embeddings_to_drive(es_client, index_name: str, backup_dir: str):
    """Export embeddings to Google Drive"""
    print(f"\n💾 Creating backup...")

    os.makedirs(backup_dir, exist_ok=True)
    os.makedirs(os.path.join(backup_dir, "configs"), exist_ok=True)

    embeddings_file = os.path.join(backup_dir, f"{index_name}_embeddings.json")
    all_docs = []

    response = es_client.search(
        index=index_name,
        scroll='5m',
        size=1000,
        body={"_source": ["doc_id", "text", "text_embedding"], "query": {"match_all": {}}}
    )

    scroll_id = response['_scroll_id']
    total = response['hits']['total']['value']

    for hit in response['hits']['hits']:
        all_docs.append({
            "doc_id": hit['_source']['doc_id'],
            "text": hit['_source']['text'][:200],
            "embedding": hit['_source']['text_embedding']
        })

    while len(response['hits']['hits']) > 0:
        response = es_client.scroll(scroll_id=scroll_id, scroll='5m')
        for hit in response['hits']['hits']:
            all_docs.append({
                "doc_id": hit['_source']['doc_id'],
                "text": hit['_source']['text'][:200],
                "embedding": hit['_source']['text_embedding']
            })
        print(f"   Progress: {len(all_docs):,}/{total:,}", end='\r')

    es_client.clear_scroll(scroll_id=scroll_id)

    with open(embeddings_file, 'w') as f:
        json.dump(all_docs, f, ensure_ascii=False, indent=2)

    # Save config
    mapping = es_client.indices.get_mapping(index=index_name)
    with open(os.path.join(backup_dir, "configs", f"{index_name}_mapping.json"), 'w') as f:
        json.dump(mapping[index_name]['mappings'], f, indent=2)

    settings = es_client.indices.get_settings(index=index_name)
    clean_settings = {"index": {"number_of_shards": 1, "number_of_replicas": 0}}
    with open(os.path.join(backup_dir, "configs", f"{index_name}_settings.json"), 'w') as f:
        json.dump(clean_settings, f, indent=2)

    with open(os.path.join(backup_dir, "configs", f"{index_name}_metadata.json"), 'w') as f:
        json.dump({
            "index_name": index_name,
            "doc_count": len(all_docs),
            "elser_model": MODEL_ID,
            "export_date": time.strftime("%Y-%m-%d %H:%M:%S")
        }, f, indent=2)

    print(f"\n✅ Backup saved ({len(all_docs):,} docs)")

def restore_from_backup(es_client, index_name: str, backup_dir: str):
    """Restore index from backup"""
    print(f"\n🔄 Restoring from backup...")

    embeddings_file = os.path.join(backup_dir, f"{index_name}_embeddings.json")
    config_dir = os.path.join(backup_dir, "configs")

    with open(os.path.join(config_dir, f"{index_name}_mapping.json")) as f:
        mapping = json.load(f)
    with open(os.path.join(config_dir, f"{index_name}_settings.json")) as f:
        settings = json.load(f)

    if es_client.indices.exists(index=index_name):
        es_client.indices.delete(index=index_name)

    es_client.indices.create(index=index_name, body={"settings": settings, "mappings": mapping})

    with open(embeddings_file, 'r') as f:
        all_docs = json.load(f)

    def generate_actions():
        for doc in all_docs:
            yield {
                "_index": index_name,
                "_id": doc["doc_id"],
                "_source": {
                    "doc_id": doc["doc_id"],
                    "text": doc["text"],
                    "text_embedding": doc["embedding"]
                }
            }

    success = 0
    for ok, _ in helpers.streaming_bulk(es_client, generate_actions(), chunk_size=500):
        if ok:
            success += 1

    es_client.indices.refresh(index=index_name)
    print(f"✅ Restored {success:,} documents")

def create_and_index_elser_v1(
    es_client,
    index_name: str,
    docs: list,
    doc_ids: list,
    checkpoint_dir: str,
    backup_dir: str,
    checkpoint_frequency: float = 0.05
):
    """Create and index with ELSER v1 - MAXIMUM SPEED"""
    print(f"\n{'='*60}")
    print(f"⚡ INDEXING: {index_name}")
    print(f"   Documents: {len(docs):,}")
    print(f"{'='*60}")

    os.makedirs(checkpoint_dir, exist_ok=True)
    os.makedirs(backup_dir, exist_ok=True)
    os.makedirs(os.path.join(backup_dir, "configs"), exist_ok=True)

    # Check backup
    if check_existing_embeddings(backup_dir, index_name):
        if es_client.indices.exists(index=index_name):
            count = es_client.count(index=index_name)['count']
            if count >= len(docs) * 0.95:
                print(f"✅ Already complete!")
                return
            else:
                restore_from_backup(es_client, index_name, backup_dir)
                return
        else:
            restore_from_backup(es_client, index_name, backup_dir)
            return

    # Check checkpoint
    checkpoint_file = os.path.join(checkpoint_dir, f"{index_name}_progress.json")
    processed_count = 0
    start_time = time.time()

    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r") as f:
            progress = json.load(f)
        processed_count = progress.get("processed_count", 0)
        print(f"\n📂 Resuming from {processed_count:,}/{len(docs):,}")

    # Create index
    if not es_client.indices.exists(index=index_name):
        print(f"\n📋 Creating index with MAX SPEED settings...")

        pipeline_id = f"{index_name}_elser_v2_pipeline"
        es_client.ingest.put_pipeline(
            id=pipeline_id,
            body={
                "description": f"ELSER v2 for {index_name}",
                "processors": [{
                    "inference": {
                        "model_id": MODEL_ID,
                        "input_output": [{"input_field": "text", "output_field": "text_embedding"}]
                    }
                }]
            }
        )

        es_client.indices.create(
            index=index_name,
            body={
                "settings": {
                    "number_of_shards": 1,
                    "number_of_replicas": 0,
                    "index": {
                        "default_pipeline": pipeline_id,
                        "refresh_interval": "-1",
                        "translog.durability": "async"
                    }
                },
                "mappings": {
                    "properties": {
                        "doc_id": {"type": "keyword"},
                        "text": {"type": "text"},
                        "text_embedding": {"type": "sparse_vector"}
                    }
                }
            }
        )
        print(f"✅ Index created (refresh disabled for speed)")

    # MAXIMUM SPEED indexing
    chunk_size = max(1000, int(len(docs) * checkpoint_frequency))
    total_success = 0
    total_errors = 0

    print(f"\n⚡ MAXIMUM SPEED INDEXING")
    print(f"   Chunk size: {chunk_size:,} docs")
    print(f"   Bulk size: 200 docs per request\n")

    for chunk_start in range(processed_count, len(docs), chunk_size):
        chunk_end = min(chunk_start + chunk_size, len(docs))
        chunk_docs = docs[chunk_start:chunk_end]
        chunk_ids = doc_ids[chunk_start:chunk_end]

        chunk_num = (chunk_start // chunk_size) + 1
        total_chunks = (len(docs) + chunk_size - 1) // chunk_size

        print(f"📦 Chunk {chunk_num}/{total_chunks}: {chunk_start:,} → {chunk_end:,}")
        chunk_start_time = time.time()

        def generate_actions():
            for doc_id, text in zip(chunk_ids, chunk_docs):
                yield {
                    "_index": index_name,
                    "_id": doc_id,
                    "_source": {"doc_id": doc_id, "text": text}
                }

        success = 0
        errors = 0

        # OPTIMIZED: Large chunk_size, no thread_count parameter
        for ok, response in tqdm(
            helpers.streaming_bulk(
                es_client,
                generate_actions(),
                chunk_size=200,
                raise_on_error=False,
                max_retries=2,
                request_timeout=300
            ),
            total=len(chunk_docs),
            desc=f"Chunk {chunk_num}",
            ncols=80
        ):
            if ok:
                success += 1
            else:
                errors += 1

        total_success += success
        total_errors += errors

        # Performance metrics
        chunk_elapsed = time.time() - chunk_start_time
        chunk_rate = len(chunk_docs) / chunk_elapsed if chunk_elapsed > 0 else 0

        total_elapsed = time.time() - start_time
        overall_rate = (chunk_end - processed_count) / total_elapsed if total_elapsed > 0 else 0

        remaining = len(docs) - chunk_end
        eta_seconds = remaining / overall_rate if overall_rate > 0 else 0
        eta_hours = eta_seconds / 3600
        eta_minutes = (eta_seconds % 3600) / 60

        # Checkpoint
        with open(checkpoint_file, "w") as f:
            json.dump({
                "processed_count": chunk_end,
                "success_count": total_success,
                "error_count": total_errors,
                "timestamp": time.time()
            }, f)

        print(f"✅ {success:,} success | {errors:,} errors")
        print(f"⚡ {chunk_rate:.1f} docs/sec (chunk) | {overall_rate:.1f} docs/sec (avg)")
        print(f"⏱️  ETA: {int(eta_hours)}h {int(eta_minutes)}m\n")

    # Finalize
    print(f"🔄 Finalizing...")
    es_client.indices.put_settings(
        index=index_name,
        body={"index": {"refresh_interval": "1s", "translog.durability": "request"}}
    )
    es_client.indices.refresh(index=index_name)

    final_count = es_client.count(index=index_name)['count']
    total_time = time.time() - start_time
    avg_rate = final_count / total_time if total_time > 0 else 0

    print(f"\n{'='*60}")
    print(f"✅ COMPLETE!")
    print(f"   Indexed: {final_count:,}/{len(docs):,}")
    print(f"   Time: {int(total_time/3600)}h {int((total_time%3600)/60)}m")
    print(f"   Speed: {avg_rate:.1f} docs/sec")
    print(f"{'='*60}")

    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)

    export_embeddings_to_drive(es_client, index_name, backup_dir)

# ================================
# 5. PROCESS ALL DATASETS (SMART BACKUP/INDEXING)
# ================================

print("\n" + "="*70)
print("🚀 PROCESSING DATASETS (BACKUP-FIRST STRATEGY)")
print("="*70)

CHECKPOINT_DIR = "/content/drive/MyDrive/Project/elser_checkpoints"
BACKUP_DIR = "/content/drive/MyDrive/Project/elser_backups"

# Λίστα με όλα τα datasets που θέλουμε να ελέγξουμε
target_datasets = ["clapnq", "cloud", "fiqa", "govt"]

for name in target_datasets:
    index_name = f"{name}_elser"
    print(f"\n{'='*70}")
    print(f"📊 DATASET: {name.upper()}")
    print(f"{'='*70}")

    # 1. Έλεγχος αν υπάρχει ήδη Backup στο Drive
    backup_exists = check_existing_embeddings(BACKUP_DIR, index_name)

    if backup_exists:
        print(f"📦 Backup found for {name}. Checking Elasticsearch index...")
        # Αν υπάρχει backup, ελέγχουμε αν ο index λείπει ή είναι ημιτελής
        index_ready = False
        if es.indices.exists(index=index_name):
            count = es.count(index=index_name)['count']
            if count > 0:
                print(f"✅ Index {index_name} already exists with {count:,} docs.")
                index_ready = True

        if not index_ready:
            print(f"🔄 Index missing or empty. Restoring {name} from Drive backup...")
            try:
                restore_from_backup(es, index_name, BACKUP_DIR)
                print(f"✅ {name.upper()} restored successfully from backup.")
            except Exception as e:
                print(f"❌ Failed to restore {name}: {e}")
        continue # Πάμε στο επόμενο dataset αφού τελειώσαμε με το backup

    # 2. Αν ΔΕΝ υπάρχει Backup, προχωράμε σε κανονικό Indexing (αν υπάρχει το αρχείο)
    print(f"🔍 No backup found for {name}. Looking for source JSONL file...")

    # Φόρτωση των δεδομένων μόνο αν χρειάζεται indexing
    path = datasets_info.get(name)
    if path and os.path.exists(path):
        corpus = load_corpus_jsonl(path)
        if corpus:
            docs = [v["text"] for v in corpus.values()]
            doc_ids = list(corpus.keys())

            try:
                create_and_index_elser_v1(
                    es_client=es,
                    index_name=index_name,
                    docs=docs,
                    doc_ids=doc_ids,
                    checkpoint_dir=CHECKPOINT_DIR,
                    backup_dir=BACKUP_DIR,
                    checkpoint_frequency=0.05
                )
                print(f"✅ {name.upper()} indexed and backed up.")
            except Exception as e:
                print(f"❌ Error indexing {name}: {e}")
        else:
            print(f"⚠️ Source file for {name} is empty or invalid.")
    else:
        print(f"❌ Source file not found at {path} and no backup exists. Skipping.")

print(f"\n{'='*70}")
print("🎉 ALL TARGET DATASETS PROCESSED!")
print("="*70)

# ================================
# 6. VERIFICATION
# ================================

print("\n🔍 VERIFICATION")
for name in datasets.keys():
    index_name = f"{name}_elser"
    if es.indices.exists(index=index_name):
        count = es.count(index=index_name)['count']
        expected = len(datasets[name][0])
        pct = (count/expected*100) if expected > 0 else 0
        status = "✅" if pct >= 95 else "⚠️"
        print(f"{status} {index_name}: {count:,}/{expected:,} ({pct:.1f}%)")
    else:
        print(f"❌ {index_name}: NOT FOUND")

print("\n✅ DONE!")


In [ ]:
# ==============================================================================
# ELSER v2 — PRE-COMPUTED REWRITES RETRIEVAL (FIXED)
# ==============================================================================

import os, json, re, time
from typing import List, Dict, Optional
from tqdm.auto import tqdm
import pandas as pd
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# ----------------------------
# 1. SETTINGS & CONFIG
# ----------------------------
BASE_PATH = "/content/drive/MyDrive/Project"
REWRITES_BASE_PATH = BASE_PATH

CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]

EVAL_SCRIPT = f"{BASE_PATH}/run_retrieval_eval.py"
PRED_DIR = f"{BASE_PATH}/elser_v2_precomputed_rewrites_predictions"
os.makedirs(PRED_DIR, exist_ok=True)

ELSER_V2_MODEL_ID = ".elser_model_2_linux-x86_64"

# ----------------------------
# 2. PRE-COMPUTED REWRITES FILE MAP
# ----------------------------
FILE_MAP = {
    "clapnq": {
        "Anchor_Keyword": f"{REWRITES_BASE_PATH}/elser_v1_anchor_keyword_predictions/clapnq_elser_anchor_keyword_combo_6U1A_20260112_200002_REWRITES.json",
        "Organizers": f"{REWRITES_BASE_PATH}/elser_v1_structural_6U3A_predictions/clapnq_elser_organizers_20260112_135408_REWRITES.json",
        "Corpus_Specific": f"{REWRITES_BASE_PATH}/elser_v1_corpus_specific_predictions/clapnq_clapnq_wikipedia_optimized2_6U3A_20260112_125554_REWRITES.json",
        "Structural_Minimal": f"{REWRITES_BASE_PATH}/elser_v1_structural_6U3A_predictions/clapnq_elser_minimal_optimized_20260112_114149_REWRITES.json",
        "COT": f"{REWRITES_BASE_PATH}/elser_v1_cot_hyde_predictions/clapnq_elser_cot_6U3A_20260114_140449_REWRITES.json",
        "Hyde": f"{REWRITES_BASE_PATH}/elser_v1_cot_hyde_predictions/clapnq_elser_hyde_6U3A_20260114_161648_REWRITES.json",
        "Decomp": f"{REWRITES_BASE_PATH}/elser_v1_decomposition_predictions/clapnq_elser_decomposition_6U3A_20260114_152005_DECOMPOSITIONS.json",
        "Type": f"{REWRITES_BASE_PATH}/elser_v1_fixed_type_predictions/clapnq_fixed_type_aware_v2_20260117_223328_REWRITES.json"
    },
    "fiqa": {
        "Anchor_Keyword": f"{REWRITES_BASE_PATH}/elser_v1_anchor_keyword_predictions/fiqa_elser_anchor_keyword_combo_6U1A_20260112_201042_REWRITES.json",
        "Organizers": f"{REWRITES_BASE_PATH}/elser_v1_structural_6U3A_predictions/fiqa_elser_organizers_20260112_140752_REWRITES.json",
        "Corpus_Specific": f"{REWRITES_BASE_PATH}/elser_v1_corpus_specific_predictions/fiqa_fiqa_optimal_6U0A_20260112_130952_REWRITES.json",
        "Structural_Minimal": f"{REWRITES_BASE_PATH}/elser_v1_structural_6U3A_predictions/fiqa_elser_minimal_optimized_20260112_115931_REWRITES.json",
        "COT": f"{REWRITES_BASE_PATH}/elser_v1_cot_hyde_predictions/fiqa_elser_cot_6U3A_20260114_142219_REWRITES.json",
        "Hyde": f"{REWRITES_BASE_PATH}/elser_v1_cot_hyde_predictions/fiqa_elser_hyde_6U3A_20260114_162827_REWRITES.json",
        "Decomp": f"{REWRITES_BASE_PATH}/elser_v1_decomposition_predictions/fiqa_elser_decomposition_6U3A_20260114_153356_DECOMPOSITIONS.json",
        "Type": f"{REWRITES_BASE_PATH}/elser_v1_fixed_type_predictions/fiqa_fixed_type_aware_v2_20260117_224418_REWRITES.json"
    },
    "govt": {
        "Anchor_Keyword": f"{REWRITES_BASE_PATH}/elser_v1_anchor_keyword_predictions/govt_elser_anchor_keyword_combo_6U1A_20260112_202122_REWRITES.json",
        "Organizers": f"{REWRITES_BASE_PATH}/elser_v1_structural_6U3A_predictions/govt_elser_organizers_20260112_142144_REWRITES.json",
        "Corpus_Specific": f"{REWRITES_BASE_PATH}/elser_v1_corpus_specific_predictions/govt_govt_corpus_baseline_6U3A_20260112_132109_REWRITES.json",
        "Structural_Minimal": f"{REWRITES_BASE_PATH}/elser_v1_structural_6U3A_predictions/govt_elser_minimal_optimized_20260112_121434_REWRITES.json",
        "COT": f"{REWRITES_BASE_PATH}/elser_v1_cot_hyde_predictions/govt_elser_cot_6U3A_20260114_144153_REWRITES.json",
        "Hyde": f"{REWRITES_BASE_PATH}/elser_v1_cot_hyde_predictions/govt_elser_hyde_6U3A_20260114_164138_REWRITES.json",
        "Decomp": f"{REWRITES_BASE_PATH}/elser_v1_decomposition_predictions/govt_elser_decomposition_6U3A_20260114_154710_DECOMPOSITIONS.json",
        "Type": f"{REWRITES_BASE_PATH}/elser_v1_fixed_type_predictions/govt_fixed_type_aware_v2_20260117_230421_REWRITES.json"
    },
    "cloud": {
        "Anchor_Keyword": f"{REWRITES_BASE_PATH}/elser_v1_anchor_keyword_predictions/cloud_elser_anchor_keyword_combo_6U1A_20260112_203200_REWRITES.json",
        "Organizers": f"{REWRITES_BASE_PATH}/elser_v1_structural_6U3A_predictions/cloud_elser_organizers_20260112_143650_REWRITES.json",
        "Corpus_Specific": f"{REWRITES_BASE_PATH}/elser_v1_corpus_specific_predictions/cloud_cloud_corpus_baseline_6U3A_20260112_133206_REWRITES.json",
        "Structural_Minimal": f"{REWRITES_BASE_PATH}/elser_v1_structural_6U3A_predictions/cloud_elser_minimal_optimized_20260112_122934_REWRITES.json",
        "COT": f"{REWRITES_BASE_PATH}/elser_v1_cot_hyde_predictions/cloud_elser_cot_6U3A_20260114_145758_REWRITES.json",
        "Hyde": f"{REWRITES_BASE_PATH}/elser_v1_cot_hyde_predictions/cloud_elser_hyde_6U3A_20260114_165333_REWRITES.json",
        "Decomp": f"{REWRITES_BASE_PATH}/elser_v1_decomposition_predictions/cloud_elser_decomposition_6U3A_20260114_160014_DECOMPOSITIONS.json",
        "Type": f"{REWRITES_BASE_PATH}/elser_v1_fixed_type_predictions/cloud_fixed_type_aware_v2_20260117_234154_REWRITES.json"
    }
}

CORPORA = [
    {
        "name": "clapnq",
        "index_name": "clapnq_elser",
        "collection_id": "mt-rag-clapnq-elser-512-100-20240503"
    },
    {
        "name": "fiqa",
        "index_name": "fiqa_elser",
        "collection_id": "mt-rag-fiqa-beir-elser-512-100-20240501"
    },
    {
        "name": "govt",
        "index_name": "govt_elser",
        "collection_id": "mt-rag-govt-elser-512-100-20240611"
    },
    {
        "name": "cloud",
        "index_name": "cloud_elser",
        "collection_id": "mt-rag-ibmcloud-elser-512-100-20240502"
    }
]

# ----------------------------
# 3. LOAD REWRITES (FIXED VERSION FROM COHERE CODE)
# ----------------------------
def load_rewrites_from_json(file_path: str) -> Dict[str, str]:
    """Load pre-generated rewrites from JSON file - FIXED VERSION"""
    if not os.path.exists(file_path):
        print(f"⚠️ Rewrites file not found: {file_path}")
        return {}

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    rewrites_dict = {}

    # Handle list format (your case)
    if isinstance(data, list):
        for entry in data:
            # 🔥 FIX: Try multiple possible keys for task_id
            task_id = (entry.get("task_id") or
                      entry.get("qid") or
                      entry.get("id") or
                      entry.get("query_id"))

            if not task_id:
                continue

            # 🔥 FIX: Try multiple possible keys for rewrites
            # Single rewrite (string)
            single_rewrite = (entry.get("rewritten_query") or
                            entry.get("rewrite") or
                            entry.get("query"))

            # Multiple rewrites (list) - TAKE FIRST ONE FOR ELSER
            multi_rewrites = (entry.get("rewrites") or
                            entry.get("decompositions") or
                            entry.get("queries"))

            # Build rewrites dict - USE FIRST REWRITE AS STRING
            if multi_rewrites and isinstance(multi_rewrites, list) and len(multi_rewrites) > 0:
                rewrites_dict[str(task_id)] = str(multi_rewrites[0])
            elif single_rewrite and isinstance(single_rewrite, str):
                rewrites_dict[str(task_id)] = str(single_rewrite)
            else:
                # Skip if no valid rewrite
                continue

    # Handle dict format
    elif isinstance(data, dict):
        for task_id, content in data.items():
            if isinstance(content, list) and len(content) > 0:
                rewrites_dict[str(task_id)] = str(content[0])
            elif isinstance(content, dict):
                rewrites = content.get("rewrites", [])
                decompositions = content.get("decompositions", [])
                if rewrites and len(rewrites) > 0:
                    rewrites_dict[str(task_id)] = str(rewrites[0])
                elif decompositions and len(decompositions) > 0:
                    rewrites_dict[str(task_id)] = str(decompositions[0])
            elif isinstance(content, str):
                rewrites_dict[str(task_id)] = str(content)

    print(f"✅ Loaded rewrites for {len(rewrites_dict)} tasks from {os.path.basename(file_path)}")

    # 🔍 DEBUG: Show sample
    if rewrites_dict:
        sample_id = list(rewrites_dict.keys())[0]
        sample_rewrite = rewrites_dict[sample_id]
        print(f"   📋 Sample ID: {sample_id[:40]}...")
        print(f"   📋 Sample rewrite: {sample_rewrite[:100]}...")

    return rewrites_dict

# ----------------------------
# 4. RETRIEVAL PIPELINE
# ----------------------------
from elasticsearch import Elasticsearch

def run_retrieval_experiment(
    corpus_name: str,
    index_name: str,
    collection_id: str,
    rewrite_type: str,
    rewrites_file: str,
    es_client
):
    """Run retrieval experiment with pre-computed rewrites"""

    print("\n" + "="*70)
    print(f"🎯 CORPUS: {corpus_name.upper()}")
    print(f"📝 Rewrite Type: {rewrite_type}")
    print(f"🔧 Index: {index_name}")
    print(f"📂 Rewrites: {os.path.basename(rewrites_file)}")
    print("="*70)

    # Load rewrites
    rewrites_map = load_rewrites_from_json(rewrites_file)

    if not rewrites_map:
        print(f"❌ No rewrites loaded for {corpus_name}/{rewrite_type}")
        return None

    # Prepare output
    RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
    pred_file = os.path.join(
        PRED_DIR,
        f"{corpus_name}_elser_v2_{rewrite_type}_{RUN_TAG}.jsonl"
    )

    print(f"🔍 Retrieving from {index_name} using ELSER v2...")

    success_count = 0
    error_count = 0

    with open(pred_file, "w", encoding="utf-8") as out:
        for task_id, rewritten_query in tqdm(rewrites_map.items(), desc=f"Retrieving"):
            try:
                res = es_client.search(index=index_name, body={
                    "query": {
                        "text_expansion": {
                            "text_embedding": {
                                "model_id": ELSER_V2_MODEL_ID,
                                "model_text": rewritten_query
                            }
                        }
                    },
                    "size": 100
                })

                contexts = [
                    {"document_id": h["_source"].get("doc_id", h["_id"]), "score": h["_score"]}
                    for h in res["hits"]["hits"]
                ]

                out.write(json.dumps({
                    "task_id": task_id,
                    "Collection": collection_id,
                    "contexts": contexts,
                    "metadata": {
                        "rewrite_type": rewrite_type,
                        "rewrite_source": os.path.basename(rewrites_file),
                        "elser_model": ELSER_V2_MODEL_ID
                    }
                }, ensure_ascii=False) + "\n")

                success_count += 1

            except Exception as e:
                error_count += 1
                if error_count <= 3:  # Show first 3 errors only
                    print(f"⚠️ Retrieval error for {task_id}: {e}")

    print(f"✅ Retrieval complete: {success_count} success, {error_count} errors")
    print(f"💾 Saved to: {pred_file}")

    return pred_file

def run_evaluation(pred_file: str):
    """Run evaluation on predictions"""
    print(f"\n🧪 Evaluating {os.path.basename(pred_file)}...")
    output_eval_file = pred_file.replace(".jsonl", "_results.jsonl")

    cmd = f"python {EVAL_SCRIPT} --input_file {pred_file} --output_file {output_eval_file} > /dev/null 2>&1"
    result = os.system(cmd)

    if result != 0:
        print(f"⚠️ Warning: Evaluation failed")
        return None

    print(f"✅ Evaluation completed")
    return output_eval_file

def load_evaluation_results(eval_file: str, corpus_name: str, rewrite_type: str):
    """Load evaluation results from aggregate CSV"""
    if eval_file is None:
        return None

    base_name = os.path.basename(eval_file).replace("_results.jsonl", "")
    pred_dir = os.path.dirname(eval_file)
    agg_csv = os.path.join(pred_dir, f"{base_name}_results_aggregate.csv")

    if not os.path.exists(agg_csv):
        print(f"⚠️ Aggregate CSV not found: {agg_csv}")
        return None

    try:
        df = pd.read_csv(agg_csv)
        df = df[df["collection"] != "all"]
        df["Corpus"] = corpus_name
        df["Rewrite_Type"] = rewrite_type
        return df
    except Exception as e:
        print(f"❌ Error loading results: {e}")
        return None

# ----------------------------
# 5. MAIN EXECUTION
# ----------------------------
def main():
    print("\n" + "="*70)
    print("🚀 ELSER v2 - PRE-COMPUTED REWRITES RETRIEVAL (FIXED)")
    print("="*70)
    print(f"Corpora: {', '.join([c['name'] for c in CORPORA])}")
    print(f"ELSER Model: {ELSER_V2_MODEL_ID}")
    print(f"Rewrite Types: {len(FILE_MAP['clapnq'])} per corpus")
    print("="*70)

    # Connect to Elasticsearch
    es_client = Elasticsearch(
        cloud_id=CLOUD_ID,
        basic_auth=("elastic", PASSWORD),
        request_timeout=180
    )

    # Verify ELSER v2 deployment
    try:
        stats = es_client.ml.get_trained_models_stats(model_id=ELSER_V2_MODEL_ID)
        if stats.get('trained_model_stats'):
            model_stat = stats['trained_model_stats'][0]
            if 'deployment_stats' in model_stat:
                state = model_stat['deployment_stats'].get('state')
                print(f"✅ ELSER v2 deployment status: {state}\n")
    except Exception as e:
        print(f"⚠️ Warning: Could not verify ELSER v2 deployment: {e}\n")

    # Run experiments
    all_results = []

    for corpus_config in CORPORA:
        corpus_name = corpus_config["name"]
        index_name = corpus_config["index_name"]
        collection_id = corpus_config["collection_id"]

        if corpus_name not in FILE_MAP:
            print(f"⚠️ No rewrite files configured for {corpus_name}")
            continue

        for rewrite_type, rewrites_file in FILE_MAP[corpus_name].items():
            try:
                # Run retrieval
                pred_file = run_retrieval_experiment(
                    corpus_name=corpus_name,
                    index_name=index_name,
                    collection_id=collection_id,
                    rewrite_type=rewrite_type,
                    rewrites_file=rewrites_file,
                    es_client=es_client
                )

                if pred_file is None:
                    continue

                # Run evaluation
                eval_file = run_evaluation(pred_file)

                # Load results
                results_df = load_evaluation_results(eval_file, corpus_name, rewrite_type)
                if results_df is not None:
                    all_results.append(results_df)

            except Exception as e:
                print(f"❌ Error processing {corpus_name}/{rewrite_type}: {e}")
                import traceback
                traceback.print_exc()

    # Aggregate all results
    if all_results:
        final_df = pd.concat(all_results, ignore_index=True)

        print("\n" + "="*70)
        print("📊 AGGREGATE RESULTS - ALL REWRITE TYPES")
        print("="*70)

        # Summary by rewrite type
        summary = final_df[["Corpus", "Rewrite_Type", "Recall", "nDCG"]].copy()

        display(summary)

        # Save aggregate results
        summary_file = os.path.join(PRED_DIR, f"elser_v2_all_rewrites_summary_{time.strftime('%Y%m%d_%H%M%S')}.csv")
        summary.to_csv(summary_file, index=False)
        print(f"\n💾 Summary saved to: {summary_file}")
    else:
        print("\n⚠️ No results to aggregate")

    print("\n" + "="*70)
    print("✅ ALL EXPERIMENTS COMPLETE!")
    print(f"📂 Results saved to: {PRED_DIR}")
    print("="*70)

if __name__ == "__main__":
    main()


# **Cohere-v4 Pipeline**

In [ ]:
# ================================
# CELL 1: IMPORTS & INSTALLATION
# ================================

!pip install -q cohere faiss-cpu numpy pandas boto3

import os
import json
import time
import numpy as np
import pandas as pd
import faiss
import cohere
from pathlib import Path
from typing import Dict, List, Optional
from collections import defaultdict

# ================================
# CELL 2: AWS & DATA SETUP + HELPERS (FIXED)
# ================================

# ---------- Paths ----------
DATA_PATH = "/content/drive/MyDrive/Project"
OUTPUT_PATH = "/content/drive/MyDrive/Project/predictions"
REWRITES_BASE_PATH = "/content/drive/MyDrive/Project"
os.makedirs(OUTPUT_PATH, exist_ok=True)

EVAL_SCRIPT_PATH = os.path.join(DATA_PATH, "run_retrieval_eval.py")

# ---------- Azure Cohere Configuration ----------
AZURE_COHERE_ENDPOINT = os.environ["ENDPOINT"]
AZURE_COHERE_KEY = os.environ["_KEY"]

# ---------- Helpers φόρτωσης δεδομένων ----------

def load_corpus_jsonl(path: str) -> Dict[str, Dict[str, str]]:
    """Load corpus from JSONL file"""
    corpus = {}
    if not os.path.exists(path):
        print(f"⚠️ Corpus file not found: {path}")
        return corpus
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            did = j.get("_id") or j.get("id") or j.get("document_id")
            if did:
                corpus[did] = {
                    "title": j.get("title", ""),
                    "text": j.get("text", ""),
                }
    print(f"✅ Loaded {len(corpus)} documents from {os.path.basename(path)}")
    return corpus

def load_queries_jsonl(path: str) -> Dict[str, Dict[str, str]]:
    """Load queries from JSONL file"""
    q = {}
    if not os.path.exists(path):
        print(f"⚠️ Queries file not found: {path}")
        return q
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            qid = j.get("_id") or j.get("id") or j.get("query_id") or j.get("task_id")
            text = j.get("text") or j.get("query") or ""
            if qid:
                q[qid] = {
                    "query": text,
                    "task_id": qid
                }
    print(f"✅ Loaded {len(q)} queries from {os.path.basename(path)}")
    return q

def load_history_from_questions(questions_file: str) -> Dict[str, List[Dict[str, str]]]:
    """Ανακατασκευάζει ιστορικό συνομιλίας από το questions.jsonl του MTRAG"""
    if not os.path.exists(questions_file):
        print(f"⚠️ History questions file not found: {questions_file}")
        return {}

    conversations = defaultdict(list)
    histories = {}

    with open(questions_file, "r", encoding="utf-8") as f:
        for line in f:
            try:
                j = json.loads(line)
            except Exception:
                continue
            qid = j.get("_id") or j.get("id") or j.get("query_id") or j.get("task_id")
            text = j.get("text") or j.get("query") or ""
            if not qid:
                continue

            if "<::>" in qid:
                conv_id, turn_str = qid.rsplit("<::>", 1)
                try:
                    turn_num = int(turn_str)
                except:
                    turn_num = 0

                role = "user"
                clean_text = text
                if text.startswith("|user|:"):
                    clean_text = text.replace("|user|:", "").strip()
                    role = "user"
                elif text.startswith("|assistant|:"):
                    clean_text = text.replace("|assistant|:", "").strip()
                    role = "assistant"

                conversations[conv_id].append({
                    "turn_num": turn_num,
                    "role": role,
                    "text": clean_text,
                    "query_id": qid
                })

    for conv_id, turns in conversations.items():
        turns_sorted = sorted(turns, key=lambda x: x["turn_num"])
        for i, turn in enumerate(turns_sorted):
            qid = turn["query_id"]
            history = [
                {"role": t["role"], "text": t["text"]}
                for t in turns_sorted[:i]
            ]
            histories[qid] = history

    print(f"✅ Loaded history for {len(histories)} queries from {os.path.basename(questions_file)}")
    return histories

def load_task_metadata(path: str) -> pd.DataFrame:
    """Load metadata about first/later turns and standalone/non-standalone queries"""
    if not os.path.exists(path):
        return pd.DataFrame()

    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))

    df = pd.DataFrame(records)
    print(f"✅ Loaded metadata for {len(df)} tasks from {os.path.basename(path)}")
    return df

def load_rewrites_from_json(file_path: str) -> Dict[str, List[str]]:
    """Load pre-generated rewrites from JSON file - FIXED VERSION"""
    if not os.path.exists(file_path):
        print(f"⚠️ Rewrites file not found: {file_path}")
        return {}

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    rewrites_dict = {}

    # Handle list format (your case)
    if isinstance(data, list):
        for entry in data:
            # 🔥 FIX: Try multiple possible keys for task_id
            task_id = (entry.get("task_id") or
                      entry.get("qid") or
                      entry.get("id") or
                      entry.get("query_id"))

            if not task_id:
                continue

            # 🔥 FIX: Try multiple possible keys for rewrites
            # Single rewrite (string)
            single_rewrite = (entry.get("rewritten_query") or
                            entry.get("rewrite") or
                            entry.get("query"))

            # Multiple rewrites (list)
            multi_rewrites = (entry.get("rewrites") or
                            entry.get("decompositions") or
                            entry.get("queries"))

            # Build rewrites list
            if multi_rewrites and isinstance(multi_rewrites, list):
                rewrites_dict[task_id] = multi_rewrites
            elif single_rewrite and isinstance(single_rewrite, str):
                rewrites_dict[task_id] = [single_rewrite]  # Wrap in list
            else:
                rewrites_dict[task_id] = []

    # Handle dict format
    elif isinstance(data, dict):
        for task_id, content in data.items():
            if isinstance(content, list):
                rewrites_dict[task_id] = content
            elif isinstance(content, dict):
                rewrites = content.get("rewrites", [])
                decompositions = content.get("decompositions", [])
                rewrites_dict[task_id] = rewrites if rewrites else decompositions
            elif isinstance(content, str):
                rewrites_dict[task_id] = [content]

    print(f"✅ Loaded rewrites for {len(rewrites_dict)} tasks from {os.path.basename(file_path)}")

    # 🔍 DEBUG: Show sample
    if rewrites_dict:
        sample_id = list(rewrites_dict.keys())[0]
        sample_rewrites = rewrites_dict[sample_id]
        print(f"   📋 Sample ID: {sample_id[:40]}...")
        print(f"   📋 Rewrites count: {len(sample_rewrites)}")
        if sample_rewrites:
            print(f"   📋 First rewrite: {sample_rewrites[0][:100]}...")

    return rewrites_dict

# ---------- Pre-generated Rewrites File Map ----------
FILE_MAP = {
    "clapnq": {
        "Anchor_Keyword": f"{REWRITES_BASE_PATH}/elser_v1_anchor_keyword_predictions/clapnq_elser_anchor_keyword_combo_6U1A_20260112_200002_REWRITES.json",
        "Organizers": f"{REWRITES_BASE_PATH}/elser_v1_structural_6U3A_predictions/clapnq_elser_organizers_20260112_135408_REWRITES.json",
        "Corpus_Specific": f"{REWRITES_BASE_PATH}/elser_v1_corpus_specific_predictions/clapnq_clapnq_wikipedia_optimized2_6U3A_20260112_125554_REWRITES.json",
        "Structural_Minimal": f"{REWRITES_BASE_PATH}/elser_v1_structural_6U3A_predictions/clapnq_elser_minimal_optimized_20260112_114149_REWRITES.json",
        "COT": f"{REWRITES_BASE_PATH}/elser_v1_cot_hyde_predictions/clapnq_elser_cot_6U3A_20260114_140449_REWRITES.json",
        "Hyde": f"{REWRITES_BASE_PATH}/elser_v1_cot_hyde_predictions/clapnq_elser_hyde_6U3A_20260114_161648_REWRITES.json",
        "Decomp": f"{REWRITES_BASE_PATH}/elser_v1_decomposition_predictions/clapnq_elser_decomposition_6U3A_20260114_152005_DECOMPOSITIONS.json"
    },
    "fiqa": {
        "Anchor_Keyword": f"{REWRITES_BASE_PATH}/elser_v1_anchor_keyword_predictions/fiqa_elser_anchor_keyword_combo_6U1A_20260112_201042_REWRITES.json",
        "Organizers": f"{REWRITES_BASE_PATH}/elser_v1_structural_6U3A_predictions/fiqa_elser_organizers_20260112_140752_REWRITES.json",
        "Corpus_Specific": f"{REWRITES_BASE_PATH}/elser_v1_corpus_specific_predictions/fiqa_fiqa_optimal_6U0A_20260112_130952_REWRITES.json",
        "Structural_Minimal": f"{REWRITES_BASE_PATH}/elser_v1_structural_6U3A_predictions/fiqa_elser_minimal_optimized_20260112_115931_REWRITES.json",
        "COT": f"{REWRITES_BASE_PATH}/elser_v1_cot_hyde_predictions/fiqa_elser_cot_6U3A_20260114_142219_REWRITES.json",
        "Hyde": f"{REWRITES_BASE_PATH}/elser_v1_cot_hyde_predictions/fiqa_elser_hyde_6U3A_20260114_162827_REWRITES.json",
        "Decomp": f"{REWRITES_BASE_PATH}/elser_v1_decomposition_predictions/fiqa_elser_decomposition_6U3A_20260114_153356_DECOMPOSITIONS.json"
    },
    "govt": {
        "Anchor_Keyword": f"{REWRITES_BASE_PATH}/elser_v1_anchor_keyword_predictions/govt_elser_anchor_keyword_combo_6U1A_20260112_202122_REWRITES.json",
        "Organizers": f"{REWRITES_BASE_PATH}/elser_v1_structural_6U3A_predictions/govt_elser_organizers_20260112_142144_REWRITES.json",
        "Corpus_Specific": f"{REWRITES_BASE_PATH}/elser_v1_corpus_specific_predictions/govt_govt_corpus_baseline_6U3A_20260112_132109_REWRITES.json",
        "Structural_Minimal": f"{REWRITES_BASE_PATH}/elser_v1_structural_6U3A_predictions/govt_elser_minimal_optimized_20260112_121434_REWRITES.json",
        "COT": f"{REWRITES_BASE_PATH}/elser_v1_cot_hyde_predictions/govt_elser_cot_6U3A_20260114_144153_REWRITES.json",
        "Hyde": f"{REWRITES_BASE_PATH}/elser_v1_cot_hyde_predictions/govt_elser_hyde_6U3A_20260114_164138_REWRITES.json",
        "Decomp": f"{REWRITES_BASE_PATH}/elser_v1_decomposition_predictions/govt_elser_decomposition_6U3A_20260114_154710_DECOMPOSITIONS.json"
    },
    "cloud": {
        "Anchor_Keyword": f"{REWRITES_BASE_PATH}/elser_v1_anchor_keyword_predictions/cloud_elser_anchor_keyword_combo_6U1A_20260112_203200_REWRITES.json",
        "Organizers": f"{REWRITES_BASE_PATH}/elser_v1_structural_6U3A_predictions/cloud_elser_organizers_20260112_143650_REWRITES.json",
        "Corpus_Specific": f"{REWRITES_BASE_PATH}/elser_v1_corpus_specific_predictions/cloud_cloud_corpus_baseline_6U3A_20260112_133206_REWRITES.json",
        "Structural_Minimal": f"{REWRITES_BASE_PATH}/elser_v1_structural_6U3A_predictions/cloud_elser_minimal_optimized_20260112_122934_REWRITES.json",
        "COT": f"{REWRITES_BASE_PATH}/elser_v1_cot_hyde_predictions/cloud_elser_cot_6U3A_20260114_145758_REWRITES.json",
        "Hyde": f"{REWRITES_BASE_PATH}/elser_v1_cot_hyde_predictions/cloud_elser_hyde_6U3A_20260114_165333_REWRITES.json",
        "Decomp": f"{REWRITES_BASE_PATH}/elser_v1_decomposition_predictions/cloud_elser_decomposition_6U3A_20260114_160014_DECOMPOSITIONS.json"
    }
}

# ---------- Config για ΟΛΑ τα corpora ----------
CORPORA_CONFIG = [
    {
        "name": "clapnq",
        "corpus_file": os.path.join(DATA_PATH, "corpora", "clapnq.jsonl"),
        "queries_file": os.path.join(DATA_PATH, "retrieval", "clapnq", "clapnq_lastturn.jsonl"),
        "history_file": os.path.join(DATA_PATH, "retrieval", "clapnq", "clapnq_questions.jsonl"),
        "emb_file": os.path.join(DATA_PATH, "embeddings_clapnq_cohere_v4_crossregion.npy"),
        "metadata_file": os.path.join(DATA_PATH, "retrieval", "clapnq", "dev_metadata.jsonl"),
        "collection_name": "mt-rag-clapnq-elser-512-100-20240503",
    },
    {
        "name": "govt",
        "corpus_file": os.path.join(DATA_PATH, "corpora", "govt.jsonl"),
        "queries_file": os.path.join(DATA_PATH, "retrieval", "govt", "govt_lastturn.jsonl"),
        "history_file": os.path.join(DATA_PATH, "retrieval", "govt", "govt_questions.jsonl"),
        "emb_file": os.path.join(DATA_PATH, "embeddings_govt_cohere_v4_crossregion.npy"),
        "metadata_file": os.path.join(DATA_PATH, "retrieval", "govt", "dev_metadata.jsonl"),
        "collection_name": "mt-rag-govt-elser-512-100-20240611",
    },
    {
        "name": "cloud",
        "corpus_file": os.path.join(DATA_PATH, "corpora", "cloud.jsonl"),
        "queries_file": os.path.join(DATA_PATH, "retrieval", "cloud", "cloud_lastturn.jsonl"),
        "history_file": os.path.join(DATA_PATH, "retrieval", "cloud", "cloud_questions.jsonl"),
        "emb_file": os.path.join(DATA_PATH, "embeddings_cloud_cohere_v4_crossregion.npy"),
        "metadata_file": os.path.join(DATA_PATH, "retrieval", "cloud", "dev_metadata.jsonl"),
        "collection_name": "mt-rag-ibmcloud-elser-512-100-20240502",
    },
    {
        "name": "fiqa",
        "corpus_file": os.path.join(DATA_PATH, "corpora", "fiqa.jsonl"),
        "queries_file": os.path.join(DATA_PATH, "retrieval", "fiqa", "fiqa_lastturn.jsonl"),
        "history_file": os.path.join(DATA_PATH, "retrieval", "fiqa", "fiqa_questions.jsonl"),
        "emb_file": os.path.join(DATA_PATH, "embeddings_fiqa_cohere_v4_crossregion.npy"),
        "metadata_file": os.path.join(DATA_PATH, "retrieval", "fiqa", "dev_metadata.jsonl"),
        "collection_name": "mt-rag-fiqa-beir-elser-512-100-20240501",
    },
]

print("\n" + "="*80)
print("CONFIGURATION SUMMARY - FIXED REWRITE LOADING")
print("="*80)
print(f"Data Path: {DATA_PATH}")
print(f"Output Path: {OUTPUT_PATH}")
print(f"Azure Cohere Endpoint: {AZURE_COHERE_ENDPOINT}")
print(f"Corpora: {[c['name'] for c in CORPORA_CONFIG]}")
print("Mode: Single query retrieval (rewritten query preferred)")
print("="*80 + "\n")

# ================================
# CELL 3: AZURE COHERE EMBEDDER + RETRIEVER V4
# ================================

class AzureCohereEmbedderV4:
    """Embedder using Azure-hosted Cohere embed-v4"""
    def __init__(self, azure_endpoint: str, azure_api_key: str, model_id="embed-english-v3.0"):
        self.model_id = model_id
        self.client = cohere.Client(
            api_key=azure_api_key,
            base_url=azure_endpoint
        )
        print(f"✅ Azure Cohere Embedder initialized with model: {model_id}")

    def _sanitize_text(self, text: str) -> str:
        """Clean and sanitize input text"""
        if not text or not isinstance(text, str):
            return "empty document"
        text = text.replace('\x00', '').replace('\r\n', ' ').replace('\n', ' ').replace('\r', ' ')
        text = ''.join(ch for ch in text if ch.isprintable() or ch.isspace())
        text = ' '.join(text.split())
        return text if text.strip() else "empty document"

    def embed_query(self, query: str) -> np.ndarray:
        """Embed a single query using Azure Cohere API"""
        query = self._sanitize_text(query)[:1500]
        max_retries = 3

        for attempt in range(max_retries):
            try:
                response = self.client.embed(
                    texts=[query],
                    model=self.model_id,
                    input_type="search_query",
                    embedding_types=["float"]
                )

                embedding = response.embeddings.float[0]
                return np.array(embedding, dtype=np.float32)

            except Exception as e:
                if attempt == max_retries - 1:
                    print(f"❌ Query embedding failed after {max_retries} attempts: {e}")
                    return np.zeros(1024, dtype=np.float32)
                time.sleep(0.5 * (attempt + 1))

    def embed_documents(self, texts: List[str], batch_size: int = 96) -> np.ndarray:
        """Embed multiple documents in batches"""
        all_embeddings = []

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            batch_clean = [self._sanitize_text(t)[:2048] for t in batch]

            try:
                response = self.client.embed(
                    texts=batch_clean,
                    model=self.model_id,
                    input_type="search_document",
                    embedding_types=["float"]
                )

                batch_embs = [np.array(emb, dtype=np.float32) for emb in response.embeddings.float]
                all_embeddings.extend(batch_embs)
                print(f"✅ Embedded batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}")

                time.sleep(0.2)

            except Exception as e:
                print(f"❌ Batch {i//batch_size + 1} failed: {e}")
                dim = 1024 if not all_embeddings else len(all_embeddings[0])
                all_embeddings.extend([np.zeros(dim, dtype=np.float32)] * len(batch))

        return np.array(all_embeddings, dtype=np.float32)

class AzureCohereRetrieverV4:
    """Retriever with Azure Cohere embeddings and FAISS index"""
    def __init__(self, docs: List[str], doc_ids: List[str], persist_file: str,
                 embedder: AzureCohereEmbedderV4):
        self.docs = docs
        self.doc_ids = doc_ids
        self.embedder = embedder
        self.index = None
        self.embs = None

        if Path(persist_file).exists():
            print(f"📂 Loading embeddings from {persist_file}")
            self.embs = np.load(persist_file).astype(np.float32)
            print(f"✅ Loaded {len(self.embs)} embeddings (shape: {self.embs.shape})")
        else:
            print(f"⚠️ Embeddings file {persist_file} not found. Creating new embeddings...")
            self.embs = embedder.embed_documents(docs)
            np.save(persist_file, self.embs)
            print(f"✅ Saved new embeddings to {persist_file}")

        dim = self.embs.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        faiss.normalize_L2(self.embs)
        self.index.add(self.embs)
        print(f"✅ FAISS index built (dim={dim}, docs={len(self.docs)})")

    def retrieve(self, query: str, history: Optional[List[Dict]] = None, top_k: int = 100) -> List[Dict]:
        """Retrieve top-k documents for a query"""
        if self.index is None or self.embs is None:
            return []

        q_emb = self.embedder.embed_query(query)
        if np.all(q_emb == 0):
            return []

        q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-8)
        q_emb = q_emb.astype(np.float32).reshape(1, -1)

        D, I = self.index.search(q_emb, top_k)

        results = []
        for j, idx in enumerate(I[0]):
            if 0 <= idx < len(self.doc_ids):
                results.append({
                    "id": self.doc_ids[idx],
                    "text": self.docs[idx],
                    "score": float(D[0][j]),
                })
        return results

# ================================
# CELL 4: EVALUATION HELPERS
# ================================

def load_query_results(results_file: str) -> pd.DataFrame:
    """Load per-query results from evaluation output"""
    if not os.path.exists(results_file):
        return pd.DataFrame()

    records = []
    with open(results_file, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))

    return pd.DataFrame(records)

def summarize_group(df: pd.DataFrame, group_name: str):
    """Summarize metrics for a group of queries"""
    if df.empty:
        print(f"⚠️ {group_name}: No data")
        return

    metrics = ["recall@1", "recall@3", "recall@5", "ndcg@1", "ndcg@3", "ndcg@5"]
    available_metrics = [m for m in metrics if m in df.columns]

    if not available_metrics:
        print(f"⚠️ {group_name}: No metrics available")
        return

    print(f"\n{'='*60}")
    print(f"{group_name} (n={len(df)})")
    print('='*60)

    summary = df[available_metrics].mean()
    for metric, value in summary.items():
        print(f"{metric:15s}: {value:.4f}")

# ================================
# CELL 5: MAIN EXPERIMENT LOOP (FIXED)
# ================================

azure_embedder = AzureCohereEmbedderV4(
    azure_endpoint=AZURE_COHERE_ENDPOINT,
    azure_api_key=AZURE_COHERE_KEY,
    model_id="embed-v-4-0"
)

EXPERIMENT_STRATEGIES = [
    "Anchor_Keyword",
    "Organizers",
    "Corpus_Specific",
    "Structural_Minimal",
    "COT",
    "Hyde",
    "Decomp"
]

for cfg in CORPORA_CONFIG:
    corpus_name = cfg["name"]
    corpus_file = cfg["corpus_file"]
    queries_file = cfg["queries_file"]
    history_file = cfg["history_file"]
    emb_file = cfg["emb_file"]
    metadata_file = cfg["metadata_file"]
    collection_name = cfg["collection_name"]

    print("\n" + "#" * 80)
    print(f"### CORPUS: {corpus_name.upper()}")
    print("#" * 80)

    print(f"=== LOADING {corpus_name.upper()} DATA ===")
    corpus = load_corpus_jsonl(corpus_file)
    queries = load_queries_jsonl(queries_file)
    histories = load_history_from_questions(history_file)

    if not corpus or not queries:
        print(f"❌ Skipping {corpus_name} - missing corpus or queries")
        continue

    docs = [v["text"] for v in corpus.values()]
    doc_ids = list(corpus.keys())

    meta_df = load_task_metadata(metadata_file) if metadata_file else pd.DataFrame()
    if not meta_df.empty:
        print(f"✅ Metadata loaded for {corpus_name}: shape={meta_df.shape}")

    print(f"\n=== BUILDING RETRIEVER FOR {corpus_name.upper()} ===")
    retriever = AzureCohereRetrieverV4(
        docs=docs,
        doc_ids=doc_ids,
        persist_file=emb_file,
        embedder=azure_embedder
    )

    if retriever.index is None:
        print(f"❌ Skipping {corpus_name} - retriever initialization failed")
        continue

    for strategy in EXPERIMENT_STRATEGIES:
        print(f"\n{'='*70}")
        print(f"{corpus_name.upper()} – STRATEGY: {strategy}")
        print('='*70)

        if corpus_name not in FILE_MAP or strategy not in FILE_MAP[corpus_name]:
            print(f"⚠️ Strategy {strategy} not available for {corpus_name}, skipping")
            continue

        rewrites_file = FILE_MAP[corpus_name][strategy]
        rewrites_dict = load_rewrites_from_json(rewrites_file)

        if not rewrites_dict:
            print(f"⚠️ No rewrites loaded for {corpus_name}/{strategy}, skipping")
            continue

        output_file = os.path.join(OUTPUT_PATH, f"{corpus_name}_azure_cohere_{strategy}_predictions.jsonl")
        results_file = output_file.replace(".jsonl", "_results.jsonl")

        print(f"▶ Running retrieval for {corpus_name}/{strategy}...")

        predictions = []
        rewrites_used = 0
        originals_used = 0

        for task_id, query_data in queries.items():
            original_query = query_data["query"]
            history = histories.get(task_id, [])

            task_rewrites = rewrites_dict.get(task_id, [])

            # 🔥 Use rewrite if available, otherwise original
            if task_rewrites and len(task_rewrites) > 0:
                query_to_use = task_rewrites[0]
                rewrites_used += 1
            else:
                query_to_use = original_query
                originals_used += 1

            results = retriever.retrieve(query_to_use, history, top_k=100)

            pred_entry = {
                "task_id": task_id,
                "Collection": collection_name,
                "contexts": [
                    {
                        "document_id": doc["id"],
                        "score": doc["score"]
                    }
                    for doc in results[:100]
                ]
            }
            predictions.append(pred_entry)

        with open(output_file, 'w', encoding='utf-8') as f:
            for pred in predictions:
                f.write(json.dumps(pred) + '\n')

        print(f"✅ Saved {len(predictions)} predictions to {output_file}")
        print(f"   📊 Rewrites used: {rewrites_used}/{len(queries)} ({100*rewrites_used/len(queries):.1f}%)")
        print(f"   📊 Originals used: {originals_used}/{len(queries)} ({100*originals_used/len(queries):.1f}%)")

        if os.path.exists(EVAL_SCRIPT_PATH):
            print(f"▶ Running evaluation for {corpus_name}/{strategy}...")
            cmd = f"python3 {EVAL_SCRIPT_PATH} --input_file {output_file} --output_file {results_file}"
            result = os.system(cmd)
            if result != 0:
                print(f"⚠️ Evaluation failed for {corpus_name}/{strategy}")
            else:
                print(f"✅ Evaluation completed for {corpus_name}/{strategy}")

        csv_file = output_file.replace(".jsonl", "_results_aggregate.csv")
        if os.path.exists(csv_file):
            df = pd.read_csv(csv_file)
            df = df[df["collection"] != "all"]
            df["Pipeline"] = f"{corpus_name}_Azure_Cohere_{strategy}"
            print("\n" + "="*60)
            print(f"AGGREGATE RESULTS: {corpus_name}_Azure_Cohere_{strategy}")
            print("="*60)
            print(df.to_string(index=False))

        if os.path.exists(results_file) and not meta_df.empty:
            qdf = load_query_results(results_file)
            if not qdf.empty:
                merged_df = qdf.merge(meta_df, on="task_id", how="left")

                if "is_first_turn" in merged_df.columns:
                    summarize_group(
                        merged_df[merged_df["is_first_turn"] == True],
                        f"{corpus_name}/{strategy}: FIRST TURNS"
                    )
                    summarize_group(
                        merged_df[merged_df["is_first_turn"] == False],
                        f"{corpus_name}/{strategy}: LATER TURNS"
                    )

                if "is_standalone" in merged_df.columns:
                    summarize_group(
                        merged_df[merged_df["is_standalone"] == True],
                        f"{corpus_name}/{strategy}: STANDALONE"
                    )
                    summarize_group(
                        merged_df[merged_df["is_standalone"] == False],
                        f"{corpus_name}/{strategy}: NON-STANDALONE"
                    )

print("\n" + "="*80)
print("🎉 ALL EXPERIMENTS COMPLETED FOR ALL CORPORA!")
print("="*80)


In [ ]:
# ================================
# CELL 1: IMPORTS & INSTALLATION
# ================================

!pip install -q cohere faiss-cpu numpy pandas boto3

import os
import json
import time
import numpy as np
import pandas as pd
import faiss
import cohere
from pathlib import Path
from typing import Dict, List, Optional
from collections import defaultdict

# ================================
# CELL 2: DATA SETUP + HELPERS (UPDATED FOR COHERE)
# ================================

# ---------- Paths ----------
DATA_PATH = "/content/drive/MyDrive/Project"
OUTPUT_PATH = "/content/drive/MyDrive/Project/predictions"
REWRITES_BASE_PATH = "/content/drive/MyDrive/Project"
os.makedirs(OUTPUT_PATH, exist_ok=True)

EVAL_SCRIPT_PATH = os.path.join(DATA_PATH, "run_retrieval_eval.py")

# ---------- Cohere Configuration ----------
# 🔥 CHANGED: Direct Cohere API key (get from dashboard.cohere.com)
COHERE_API_KEY = os.environ["API_KEY"]  # Replace with your actual key

# ---------- Helpers φόρτωσης δεδομένων ----------

def load_corpus_jsonl(path: str) -> Dict[str, Dict[str, str]]:
    """Load corpus from JSONL file"""
    corpus = {}
    if not os.path.exists(path):
        print(f"⚠️ Corpus file not found: {path}")
        return corpus
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            did = j.get("_id") or j.get("id") or j.get("document_id")
            if did:
                corpus[did] = {
                    "title": j.get("title", ""),
                    "text": j.get("text", ""),
                }
    print(f"✅ Loaded {len(corpus)} documents from {os.path.basename(path)}")
    return corpus

def load_queries_jsonl(path: str) -> Dict[str, Dict[str, str]]:
    """Load queries from JSONL file"""
    q = {}
    if not os.path.exists(path):
        print(f"⚠️ Queries file not found: {path}")
        return q
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            qid = j.get("_id") or j.get("id") or j.get("query_id") or j.get("task_id")
            text = j.get("text") or j.get("query") or ""
            if qid:
                q[qid] = {
                    "query": text,
                    "task_id": qid
                }
    print(f"✅ Loaded {len(q)} queries from {os.path.basename(path)}")
    return q

def load_history_from_questions(questions_file: str) -> Dict[str, List[Dict[str, str]]]:
    """Ανακατασκευάζει ιστορικό συνομιλίας από το questions.jsonl του MTRAG"""
    if not os.path.exists(questions_file):
        print(f"⚠️ History questions file not found: {questions_file}")
        return {}

    conversations = defaultdict(list)
    histories = {}

    with open(questions_file, "r", encoding="utf-8") as f:
        for line in f:
            try:
                j = json.loads(line)
            except Exception:
                continue
            qid = j.get("_id") or j.get("id") or j.get("query_id") or j.get("task_id")
            text = j.get("text") or j.get("query") or ""
            if not qid:
                continue

            if "<::>" in qid:
                conv_id, turn_str = qid.rsplit("<::>", 1)
                try:
                    turn_num = int(turn_str)
                except:
                    turn_num = 0

                role = "user"
                clean_text = text
                if text.startswith("|user|:"):
                    clean_text = text.replace("|user|:", "").strip()
                    role = "user"
                elif text.startswith("|assistant|:"):
                    clean_text = text.replace("|assistant|:", "").strip()
                    role = "assistant"

                conversations[conv_id].append({
                    "turn_num": turn_num,
                    "role": role,
                    "text": clean_text,
                    "query_id": qid
                })

    for conv_id, turns in conversations.items():
        turns_sorted = sorted(turns, key=lambda x: x["turn_num"])
        for i, turn in enumerate(turns_sorted):
            qid = turn["query_id"]
            history = [
                {"role": t["role"], "text": t["text"]}
                for t in turns_sorted[:i]
            ]
            histories[qid] = history

    print(f"✅ Loaded history for {len(histories)} queries from {os.path.basename(questions_file)}")
    return histories

def load_task_metadata(path: str) -> pd.DataFrame:
    """Load metadata about first/later turns and standalone/non-standalone queries"""
    if not os.path.exists(path):
        return pd.DataFrame()

    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))

    df = pd.DataFrame(records)
    print(f"✅ Loaded metadata for {len(df)} tasks from {os.path.basename(path)}")
    return df

def load_rewrites_from_json(file_path: str) -> Dict[str, List[str]]:
    """Load pre-generated rewrites from JSON file"""
    if not os.path.exists(file_path):
        print(f"⚠️ Rewrites file not found: {file_path}")
        return {}

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    rewrites_dict = {}

    # Handle list format
    if isinstance(data, list):
        for entry in data:
            task_id = (entry.get("task_id") or
                      entry.get("qid") or
                      entry.get("id") or
                      entry.get("query_id"))

            if not task_id:
                continue

            single_rewrite = (entry.get("rewritten_query") or
                            entry.get("rewrite") or
                            entry.get("query"))

            multi_rewrites = (entry.get("rewrites") or
                            entry.get("decompositions") or
                            entry.get("queries"))

            if multi_rewrites and isinstance(multi_rewrites, list):
                rewrites_dict[task_id] = multi_rewrites
            elif single_rewrite and isinstance(single_rewrite, str):
                rewrites_dict[task_id] = [single_rewrite]
            else:
                rewrites_dict[task_id] = []

    # Handle dict format
    elif isinstance(data, dict):
        for task_id, content in data.items():
            if isinstance(content, list):
                rewrites_dict[task_id] = content
            elif isinstance(content, dict):
                rewrites = content.get("rewrites", [])
                decompositions = content.get("decompositions", [])
                rewrites_dict[task_id] = rewrites if rewrites else decompositions
            elif isinstance(content, str):
                rewrites_dict[task_id] = [content]

    print(f"✅ Loaded rewrites for {len(rewrites_dict)} tasks from {os.path.basename(file_path)}")

    if rewrites_dict:
        sample_id = list(rewrites_dict.keys())[0]
        sample_rewrites = rewrites_dict[sample_id]
        print(f"   📋 Sample ID: {sample_id[:40]}...")
        print(f"   📋 Rewrites count: {len(sample_rewrites)}")
        if sample_rewrites:
            print(f"   📋 First rewrite: {sample_rewrites[0][:100]}...")

    return rewrites_dict

# ---------- Pre-generated Rewrites File Map ----------
FILE_MAP = {
    "clapnq": {
        "Type": f"{REWRITES_BASE_PATH}/elser_v1_fixed_type_predictions/clapnq_fixed_type_aware_v2_20260117_223328_REWRITES.json"

    },
    "govt":{
        "Type": f"{REWRITES_BASE_PATH}/elser_v1_fixed_type_predictions/govt_fixed_type_aware_v2_20260117_230421_REWRITES.json"
    },
    "cloud":{
        "Type": f"{REWRITES_BASE_PATH}/elser_v1_fixed_type_predictions/cloud_fixed_type_aware_v2_20260117_234154_REWRITES.json"
    },
    "fiqa":{
        "Type":f"{REWRITES_BASE_PATH}/elser_v1_fixed_type_predictions/fiqa_fixed_type_aware_v2_20260117_224418_REWRITES.json"
    }

}

# ---------- Config για ΟΛΑ τα corpora ----------
CORPORA_CONFIG = [
    {
        "name": "clapnq",
        "corpus_file": os.path.join(DATA_PATH, "corpora", "clapnq.jsonl"),
        "queries_file": os.path.join(DATA_PATH, "retrieval", "clapnq", "clapnq_lastturn.jsonl"),
        "history_file": os.path.join(DATA_PATH, "retrieval", "clapnq", "clapnq_questions.jsonl"),
        "emb_file": os.path.join(DATA_PATH, "embeddings_clapnq_cohere_v4_crossregion.npy"),
        "metadata_file": os.path.join(DATA_PATH, "retrieval", "clapnq", "dev_metadata.jsonl"),
        "collection_name": "mt-rag-clapnq-elser-512-100-20240503",
    },
    {
        "name": "govt",
        "corpus_file": os.path.join(DATA_PATH, "corpora", "govt.jsonl"),
        "queries_file": os.path.join(DATA_PATH, "retrieval", "govt", "govt_lastturn.jsonl"),
        "history_file": os.path.join(DATA_PATH, "retrieval", "govt", "govt_questions.jsonl"),
        "emb_file": os.path.join(DATA_PATH, "embeddings_govt_cohere_v4_crossregion.npy"),
        "metadata_file": os.path.join(DATA_PATH, "retrieval", "govt", "dev_metadata.jsonl"),
        "collection_name": "mt-rag-govt-elser-512-100-20240611",
    },
    {
        "name": "cloud",
        "corpus_file": os.path.join(DATA_PATH, "corpora", "cloud.jsonl"),
        "queries_file": os.path.join(DATA_PATH, "retrieval", "cloud", "cloud_lastturn.jsonl"),
        "history_file": os.path.join(DATA_PATH, "retrieval", "cloud", "cloud_questions.jsonl"),
        "emb_file": os.path.join(DATA_PATH, "embeddings_cloud_cohere_v4_crossregion.npy"),
        "metadata_file": os.path.join(DATA_PATH, "retrieval", "cloud", "dev_metadata.jsonl"),
        "collection_name": "mt-rag-ibmcloud-elser-512-100-20240502",
    },
    {
        "name": "fiqa",
        "corpus_file": os.path.join(DATA_PATH, "corpora", "fiqa.jsonl"),
        "queries_file": os.path.join(DATA_PATH, "retrieval", "fiqa", "fiqa_lastturn.jsonl"),
        "history_file": os.path.join(DATA_PATH, "retrieval", "fiqa", "fiqa_questions.jsonl"),
        "emb_file": os.path.join(DATA_PATH, "embeddings_fiqa_cohere_v4_crossregion.npy"),
        "metadata_file": os.path.join(DATA_PATH, "retrieval", "fiqa", "dev_metadata.jsonl"),
        "collection_name": "mt-rag-fiqa-beir-elser-512-100-20240501",
    },
]

print("\n" + "="*80)
print("CONFIGURATION SUMMARY - DIRECT COHERE API")
print("="*80)
print(f"Data Path: {DATA_PATH}")
print(f"Output Path: {OUTPUT_PATH}")
print(f"Cohere API: Direct (dashboard.cohere.com)")
print(f"Corpora: {[c['name'] for c in CORPORA_CONFIG]}")
print("Mode: Single query retrieval (rewritten query preferred)")
print("="*80 + "\n")

# ================================
# CELL 3: COHERE EMBEDDER + RETRIEVER V4
# ================================

class CohereEmbedderV4:
    """Embedder using direct Cohere API with embed-v4"""
    def __init__(self, api_key: str, model_id="embed-english-v3.0"):
        self.model_id = model_id
        # 🔥 CHANGED: Direct Cohere client without base_url
        self.client = cohere.Client(api_key=api_key)
        print(f"✅ Cohere Embedder initialized with model: {model_id}")

    def _sanitize_text(self, text: str) -> str:
        """Clean and sanitize input text"""
        if not text or not isinstance(text, str):
            return "empty document"
        text = text.replace('\x00', '').replace('\r\n', ' ').replace('\n', ' ').replace('\r', ' ')
        text = ''.join(ch for ch in text if ch.isprintable() or ch.isspace())
        text = ' '.join(text.split())
        return text if text.strip() else "empty document"

    def embed_query(self, query: str) -> np.ndarray:
        """Embed a single query using Cohere API"""
        query = self._sanitize_text(query)[:1500]
        max_retries = 3

        for attempt in range(max_retries):
            try:
                response = self.client.embed(
                    texts=[query],
                    model=self.model_id,
                    input_type="search_query",
                    embedding_types=["float"]
                )

                embedding = response.embeddings.float[0]
                return np.array(embedding, dtype=np.float32)

            except Exception as e:
                if attempt == max_retries - 1:
                    print(f"❌ Query embedding failed after {max_retries} attempts: {e}")
                    return np.zeros(1024, dtype=np.float32)
                time.sleep(0.5 * (attempt + 1))

    def embed_documents(self, texts: List[str], batch_size: int = 96) -> np.ndarray:
        """Embed multiple documents in batches"""
        all_embeddings = []

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            batch_clean = [self._sanitize_text(t)[:2048] for t in batch]

            try:
                response = self.client.embed(
                    texts=batch_clean,
                    model=self.model_id,
                    input_type="search_document",
                    embedding_types=["float"]
                )

                batch_embs = [np.array(emb, dtype=np.float32) for emb in response.embeddings.float]
                all_embeddings.extend(batch_embs)
                print(f"✅ Embedded batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}")

                time.sleep(0.2)

            except Exception as e:
                print(f"❌ Batch {i//batch_size + 1} failed: {e}")
                dim = 1024 if not all_embeddings else len(all_embeddings[0])
                all_embeddings.extend([np.zeros(dim, dtype=np.float32)] * len(batch))

        return np.array(all_embeddings, dtype=np.float32)

class CohereRetrieverV4:
    """Retriever with Cohere embeddings and FAISS index"""
    def __init__(self, docs: List[str], doc_ids: List[str], persist_file: str,
                 embedder: CohereEmbedderV4):
        self.docs = docs
        self.doc_ids = doc_ids
        self.embedder = embedder
        self.index = None
        self.embs = None

        if Path(persist_file).exists():
            print(f"📂 Loading embeddings from {persist_file}")
            self.embs = np.load(persist_file).astype(np.float32)
            print(f"✅ Loaded {len(self.embs)} embeddings (shape: {self.embs.shape})")
        else:
            print(f"⚠️ Embeddings file {persist_file} not found. Creating new embeddings...")
            self.embs = embedder.embed_documents(docs)
            np.save(persist_file, self.embs)
            print(f"✅ Saved new embeddings to {persist_file}")

        dim = self.embs.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        faiss.normalize_L2(self.embs)
        self.index.add(self.embs)
        print(f"✅ FAISS index built (dim={dim}, docs={len(self.docs)})")

    def retrieve(self, query: str, history: Optional[List[Dict]] = None, top_k: int = 100) -> List[Dict]:
        """Retrieve top-k documents for a query"""
        if self.index is None or self.embs is None:
            return []

        q_emb = self.embedder.embed_query(query)
        if np.all(q_emb == 0):
            return []

        q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-8)
        q_emb = q_emb.astype(np.float32).reshape(1, -1)

        D, I = self.index.search(q_emb, top_k)

        results = []
        for j, idx in enumerate(I[0]):
            if 0 <= idx < len(self.doc_ids):
                results.append({
                    "id": self.doc_ids[idx],
                    "text": self.docs[idx],
                    "score": float(D[0][j]),
                })
        return results

# ================================
# CELL 4: EVALUATION HELPERS
# ================================

def load_query_results(results_file: str) -> pd.DataFrame:
    """Load per-query results from evaluation output"""
    if not os.path.exists(results_file):
        return pd.DataFrame()

    records = []
    with open(results_file, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))

    return pd.DataFrame(records)

def summarize_group(df: pd.DataFrame, group_name: str):
    """Summarize metrics for a group of queries"""
    if df.empty:
        print(f"⚠️ {group_name}: No data")
        return

    metrics = ["recall@1", "recall@3", "recall@5", "ndcg@1", "ndcg@3", "ndcg@5"]
    available_metrics = [m for m in metrics if m in df.columns]

    if not available_metrics:
        print(f"⚠️ {group_name}: No metrics available")
        return

    print(f"\n{'='*60}")
    print(f"{group_name} (n={len(df)})")
    print('='*60)

    summary = df[available_metrics].mean()
    for metric, value in summary.items():
        print(f"{metric:15s}: {value:.4f}")

# ================================
# CELL 5: MAIN EXPERIMENT LOOP
# ================================

# 🔥 CHANGED: Initialize with direct Cohere API
cohere_embedder = CohereEmbedderV4(
    api_key=COHERE_API_KEY,
    model_id="embed-v4.0"  # or "embed-v-4-0" if using v4
)

EXPERIMENT_STRATEGIES = [
    "Type"

]

for cfg in CORPORA_CONFIG:
    corpus_name = cfg["name"]
    corpus_file = cfg["corpus_file"]
    queries_file = cfg["queries_file"]
    history_file = cfg["history_file"]
    emb_file = cfg["emb_file"]
    metadata_file = cfg["metadata_file"]
    collection_name = cfg["collection_name"]

    print("\n" + "#" * 80)
    print(f"### CORPUS: {corpus_name.upper()}")
    print("#" * 80)

    print(f"=== LOADING {corpus_name.upper()} DATA ===")
    corpus = load_corpus_jsonl(corpus_file)
    queries = load_queries_jsonl(queries_file)
    histories = load_history_from_questions(history_file)

    if not corpus or not queries:
        print(f"❌ Skipping {corpus_name} - missing corpus or queries")
        continue

    docs = [v["text"] for v in corpus.values()]
    doc_ids = list(corpus.keys())

    meta_df = load_task_metadata(metadata_file) if metadata_file else pd.DataFrame()
    if not meta_df.empty:
        print(f"✅ Metadata loaded for {corpus_name}: shape={meta_df.shape}")

    print(f"\n=== BUILDING RETRIEVER FOR {corpus_name.upper()} ===")
    # 🔥 CHANGED: Use CohereRetrieverV4 instead of AzureCohereRetrieverV4
    retriever = CohereRetrieverV4(
        docs=docs,
        doc_ids=doc_ids,
        persist_file=emb_file,
        embedder=cohere_embedder
    )

    if retriever.index is None:
        print(f"❌ Skipping {corpus_name} - retriever initialization failed")
        continue

    for strategy in EXPERIMENT_STRATEGIES:
        print(f"\n{'='*70}")
        print(f"{corpus_name.upper()} – STRATEGY: {strategy}")
        print('='*70)

        if corpus_name not in FILE_MAP or strategy not in FILE_MAP[corpus_name]:
            print(f"⚠️ Strategy {strategy} not available for {corpus_name}, skipping")
            continue

        rewrites_file = FILE_MAP[corpus_name][strategy]
        rewrites_dict = load_rewrites_from_json(rewrites_file)

        if not rewrites_dict:
            print(f"⚠️ No rewrites loaded for {corpus_name}/{strategy}, skipping")
            continue

        output_file = os.path.join(OUTPUT_PATH, f"{corpus_name}_cohere_{strategy}_predictions.jsonl")
        results_file = output_file.replace(".jsonl", "_results.jsonl")

        print(f"▶ Running retrieval for {corpus_name}/{strategy}...")

        predictions = []
        rewrites_used = 0
        originals_used = 0

        for task_id, query_data in queries.items():
            original_query = query_data["query"]
            history = histories.get(task_id, [])

            task_rewrites = rewrites_dict.get(task_id, [])

            if task_rewrites and len(task_rewrites) > 0:
                query_to_use = task_rewrites[0]
                rewrites_used += 1
            else:
                query_to_use = original_query
                originals_used += 1

            results = retriever.retrieve(query_to_use, history, top_k=100)

            pred_entry = {
                "task_id": task_id,
                "Collection": collection_name,
                "contexts": [
                    {
                        "document_id": doc["id"],
                        "score": doc["score"]
                    }
                    for doc in results[:100]
                ]
            }
            predictions.append(pred_entry)

        with open(output_file, 'w', encoding='utf-8') as f:
            for pred in predictions:
                f.write(json.dumps(pred) + '\n')

        print(f"✅ Saved {len(predictions)} predictions to {output_file}")
        print(f"   📊 Rewrites used: {rewrites_used}/{len(queries)} ({100*rewrites_used/len(queries):.1f}%)")
        print(f"   📊 Originals used: {originals_used}/{len(queries)} ({100*originals_used/len(queries):.1f}%)")

        if os.path.exists(EVAL_SCRIPT_PATH):
            print(f"▶ Running evaluation for {corpus_name}/{strategy}...")
            cmd = f"python3 {EVAL_SCRIPT_PATH} --input_file {output_file} --output_file {results_file}"
            result = os.system(cmd)
            if result != 0:
                print(f"⚠️ Evaluation failed for {corpus_name}/{strategy}")
            else:
                print(f"✅ Evaluation completed for {corpus_name}/{strategy}")

        csv_file = output_file.replace(".jsonl", "_results_aggregate.csv")
        if os.path.exists(csv_file):
            df = pd.read_csv(csv_file)
            df = df[df["collection"] != "all"]
            df["Pipeline"] = f"{corpus_name}_Cohere_{strategy}"
            print("\n" + "="*60)
            print(f"AGGREGATE RESULTS: {corpus_name}_Cohere_{strategy}")
            print("="*60)
            print(df.to_string(index=False))

        if os.path.exists(results_file) and not meta_df.empty:
            qdf = load_query_results(results_file)
            if not qdf.empty:
                merged_df = qdf.merge(meta_df, on="task_id", how="left")

                if "is_first_turn" in merged_df.columns:
                    summarize_group(
                        merged_df[merged_df["is_first_turn"] == True],
                        f"{corpus_name}/{strategy}: FIRST TURNS"
                    )
                    summarize_group(
                        merged_df[merged_df["is_first_turn"] == False],
                        f"{corpus_name}/{strategy}: LATER TURNS"
                    )

                if "is_standalone" in merged_df.columns:
                    summarize_group(
                        merged_df[merged_df["is_standalone"] == True],
                        f"{corpus_name}/{strategy}: STANDALONE"
                    )
                    summarize_group(
                        merged_df[merged_df["is_standalone"] == False],
                        f"{corpus_name}/{strategy}: NON-STANDALONE"
                    )

print("\n" + "="*80)
print("🎉 ALL EXPERIMENTS COMPLETED FOR ALL CORPORA!")
print("="*80)


In [ ]:
import os, json, glob

PRED_DIR = "/content/drive/MyDrive/Project/predictions_deepseek_singleprf_20260115_190813"
CONVERTED_DIR = os.path.join(PRED_DIR, "converted_for_eval")
os.makedirs(CONVERTED_DIR, exist_ok=True)

# Map από το δικό σου corpus_name -> το collection string που περιμένει το eval
COLLECTION_MAP = {
    "clapnq": "mt-rag-clapnq-elser-512-100-20240503",
    "govt":  "mt-rag-govt-elser-512-100-20240611",
    "fiqa":  "mt-rag-fiqa-beir-elser-512-100-20240501",
    "cloud": "mt-rag-ibmcloud-elser-512-100-20240502",
}

pred_files = sorted(glob.glob(os.path.join(PRED_DIR, "*.jsonl")))
pred_files = [p for p in pred_files if not p.endswith("_results.jsonl")]

print("Found:", len(pred_files), "prediction files")

for in_path in pred_files:
    base = os.path.basename(in_path).replace(".jsonl", "")
    out_path = os.path.join(CONVERTED_DIR, base + "_evalfmt.jsonl")

    n = 0
    with open(in_path, "r", encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for line in fin:
            if not line.strip():
                continue
            item = json.loads(line)

            qid = str(item.get("_id") or item.get("task_id") or "")
            corpus_name = item.get("collection")  # δικό σου πεδίο: clapnq/govt/cloud/fiqa
            if not qid or corpus_name not in COLLECTION_MAP:
                continue

            contexts = []
            for r in item.get("results", []):
                if "id" in r and "score" in r:
                    contexts.append({"document_id": r["id"], "score": float(r["score"])})

            out_item = {
                "task_id": qid,
                "Collection": COLLECTION_MAP[corpus_name],
                "contexts": contexts,
            }

            fout.write(json.dumps(out_item, ensure_ascii=False) + "\n")
            n += 1

    print(f"✅ Converted: {os.path.basename(in_path)} -> {os.path.basename(out_path)}  ({n} lines)")


## Reranking per rewriter

In [ ]:
import os, json, time, requests, subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import Dict, List, Tuple

# ==============================================================================
# 1. SETTINGS & CONFIGURATION
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
BASE_DIR = "/content/drive/MyDrive/Project"
PREDS_PATH = os.path.join(BASE_PATH, "predictions")

EVAL_SCRIPT_PATH = os.path.join(BASE_PATH, "run_retrieval_eval.py")

# Azure Cohere Credentials
AZURE_COHERE_ENDPOINT = os.environ["ENDPOINT"]
AZURE_COHERE_KEY = os.environ["_KEY"]

RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = os.path.join(BASE_PATH, "cohere_v4_rerank_results", RUN_TAG)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Δυναμική δημιουργία λίστας αρχείων βάσει των προηγούμενων αποτελεσμάτων
STRATEGIES = ["Anchor_Keyword", "Corpus_Specific", "Structural_Minimal", "COT", "Hyde", "Decomp"]
CORPORA = ["clapnq", "govt", "cloud", "fiqa"]

INPUT_FILES = {}
for corpus in CORPORA:
    INPUT_FILES[corpus] = {}
    for strategy in STRATEGIES:
        file_name = f"{corpus}_azure_cohere_{strategy}_predictions.jsonl"
        INPUT_FILES[corpus][strategy] = os.path.join(PREDS_PATH, file_name)

# Corpus-specific files (Παραμένουν ως έχουν για το Ground Truth)
CORPUS_CONFIG = {
    "clapnq": {
        "corpus_file": f"{BASE_PATH}/corpora/clapnq.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/clapnq/clapnq_lastturn.jsonl"
    },
    "fiqa": {
        "corpus_file": f"{BASE_PATH}/corpora/fiqa.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/fiqa/fiqa_lastturn.jsonl"
    },
    "govt": {
        "corpus_file": f"{BASE_PATH}/corpora/govt.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/govt/govt_lastturn.jsonl"
    },
    "cloud": {
        "corpus_file": f"{BASE_PATH}/corpora/cloud.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/cloud/cloud_lastturn.jsonl"
    }
}

TOP_K_RERANK = 100
COHERE_WEIGHTS = [0.1, 0.3, 0.5, 0.7, 0.9]

# ==============================================================================
# 2. HELPER FUNCTIONS (Παραμένουν ίδιες)
# ==============================================================================

def load_jsonl(path: str) -> List[dict]:
    data = []
    if not os.path.exists(path): return data
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip(): data.append(json.loads(line))
    return data

def load_corpus(path: str) -> Dict[str, str]:
    corpus = {}
    if not os.path.exists(path): return corpus
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            did = str(j.get("_id") or j.get("id"))
            text = (j.get("title", "") + " " + (j.get("text") or j.get("contents") or "")).strip()
            corpus[did] = text[:2000]
    return corpus

def load_queries(path: str) -> Dict[str, str]:
    queries = {}
    if not os.path.exists(path): return queries
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            qid = str(j.get("_id") or j.get("id") or j.get("task_id"))
            queries[qid] = j.get("text") or j.get("query")
    return queries

def azure_cohere_rerank(query: str, docs_text: List[str]) -> List[float]:
    if not query or not docs_text: return [0.0] * len(docs_text)
    headers = {"Content-Type": "application/json", "api-key": AZURE_COHERE_KEY}
    payload = {
        "model": "Cohere-rerank-v4.0-pro",
        "query": query, "documents": docs_text, "top_n": len(docs_text)
    }
    try:
        response = requests.post(AZURE_COHERE_ENDPOINT, headers=headers, json=payload, timeout=60)
        if response.status_code == 200:
            results = response.json()["results"]
            scores = [0.0] * len(docs_text)
            for r in results: scores[r["index"]] = float(r["relevance_score"])
            return scores
    except Exception as e: print(f"❌ Error: {e}")
    return [0.0] * len(docs_text)

def minmax_norm(scores: List[float]) -> np.ndarray:
    arr = np.array(scores, dtype=np.float32)
    if arr.max() == arr.min(): return np.zeros_like(arr)
    return (arr - arr.min()) / (arr.max() - arr.min())

def run_eval(input_jsonl: str) -> Tuple[float, float]:
    result_file = input_jsonl.replace(".jsonl", "_results.jsonl")
    csv_file = result_file.replace(".jsonl", "_aggregate.csv")
    cmd = f"python3 {EVAL_SCRIPT_PATH} --input_file {input_jsonl} --output_file {result_file}"
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        if "collection" in df.columns: df = df[df["collection"] != "all"]
        if df.empty: return 0.0, 0.0
        row = df.iloc[0]
        def parse_metric(val):
            if isinstance(val, str) and "[" in val:
                import ast
                return ast.literal_eval(val)[2]
            return val
        return parse_metric(row.get("Recall", 0.0)), parse_metric(row.get("nDCG", 0.0))
    return 0.0, 0.0

# ==============================================================================
# 3. MAIN PIPELINE
# ==============================================================================

performance_summary = []

for corpus_name in CORPORA:
    print(f"\n📚 CORPUS: {corpus_name.upper()}")
    corpus_cfg = CORPUS_CONFIG[corpus_name]
    corpus_map = load_corpus(corpus_cfg["corpus_file"])
    query_map = load_queries(corpus_cfg["queries_file"])

    for strategy_name, file_path in INPUT_FILES[corpus_name].items():
        if not os.path.exists(file_path): continue

        print(f"🎯 Reranking: {strategy_name}")
        preds = load_jsonl(file_path)
        cache = []
        for p in tqdm(preds, desc=f"Processing {strategy_name}", leave=False):
            qid = str(p["task_id"])
            q_txt = query_map.get(qid, "")
            top_ctxs = p["contexts"][:TOP_K_RERANK]
            rest_ctxs = p["contexts"][TOP_K_RERANK:]
            doc_texts = [corpus_map.get(str(c["document_id"]), "N/A") for c in top_ctxs]
            coh_scores = azure_cohere_rerank(q_txt, doc_texts)
            orig_scores = [float(c["score"]) for c in top_ctxs]
            cache.append({"task_id": qid, "Collection": p.get("Collection"), "top_ctxs": top_ctxs, "rest": rest_ctxs, "orig_scores": orig_scores, "coh_scores": coh_scores})

        # Eval Cohere-Only
        cohere_file = os.path.join(OUTPUT_DIR, f"{corpus_name}_{strategy_name}_COHERE_ONLY.jsonl")
        with open(cohere_file, "w") as f:
            for item in cache:
                new_ctxs = []
                for i, ctx in enumerate(item["top_ctxs"]):
                    nc = ctx.copy(); nc["score"] = float(item["coh_scores"][i]); new_ctxs.append(nc)
                new_ctxs.sort(key=lambda x: x["score"], reverse=True)
                f.write(json.dumps({"task_id": item["task_id"], "Collection": item["Collection"], "contexts": new_ctxs + item["rest"]}) + "\n")

        r5, n5 = run_eval(cohere_file)
        performance_summary.append({"Corpus": corpus_name, "Strategy": strategy_name, "Method": "Cohere-Only", "Recall@5": r5, "nDCG@5": n5})

        # Hybrid Fusion
        for cw in COHERE_WEIGHTS:
            ow = 1.0 - cw
            hybrid_file = os.path.join(OUTPUT_DIR, f"{corpus_name}_{strategy_name}_HYBRID_cw{cw}.jsonl")
            with open(hybrid_file, "w") as f:
                for item in cache:
                    n_orig, n_coh = minmax_norm(item["orig_scores"]), minmax_norm(item["coh_scores"])
                    combined = (ow * n_orig) + (cw * n_coh)
                    new_ctxs = []
                    for i, ctx in enumerate(item["top_ctxs"]):
                        nc = ctx.copy(); nc["score"] = float(combined[i]); new_ctxs.append(nc)
                    new_ctxs.sort(key=lambda x: x["score"], reverse=True)
                    f.write(json.dumps({"task_id": item["task_id"], "Collection": item["Collection"], "contexts": new_ctxs + item["rest"]}) + "\n")

            r5_h, n5_h = run_eval(hybrid_file)
            performance_summary.append({"Corpus": corpus_name, "Strategy": strategy_name, "Method": f"Hybrid (cw={cw})", "Recall@5": r5_h, "nDCG@5": n5_h})

# Final Report
df_final = pd.DataFrame(performance_summary)
display(df_final.sort_values(["Corpus", "nDCG@5"], ascending=[True, False]))
df_final.to_csv(os.path.join(OUTPUT_DIR, "reranking_report.csv"), index=False)

## Retrieval results and reranked results

In [ ]:
import json
import csv
import os
import pandas as pd
import numpy as np

# ==============================================================================
# 1. ΡΥΘΜΙΣΕΙΣ & ΔΥΝΑΜΙΚΟ FILE MAP (cw=0.5)
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
# Βεβαιώσου ότι αυτό το path είναι σωστό στο Drive σου
RESULTS_DIR = os.path.join(BASE_PATH, "cohere_v4_rerank_results/20260115_164434")

CORPORA = ["clapnq", "fiqa", "govt", "cloud"]
STRATEGIES = ["Anchor_Keyword", "Corpus_Specific", "Structural_Minimal", "COT", "Hyde", "Decomp"]
K_VALUES = [5, 10, 20, 50, 100]

FILE_MAP = {}
print("📂 Έλεγχος αρχείων στο RESULTS_DIR...")
for corpus in CORPORA:
    FILE_MAP[corpus] = {}
    for strategy in STRATEGIES:
        file_name = f"{corpus}_{strategy}_HYBRID_cw0.5.jsonl"
        full_path = os.path.join(RESULTS_DIR, file_name)
        if os.path.exists(full_path):
            FILE_MAP[corpus][strategy] = full_path
            print(f"  ✅ Βρέθηκε: {file_name}")

# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================
def load_qrels(corpus):
    path = os.path.join(BASE_PATH, "retrieval", corpus, "dev.tsv")
    qrels = {}
    if not os.path.exists(path):
        return qrels
    with open(path, 'r', encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        try: next(reader)
        except: return qrels
        for row in reader:
            if len(row) < 3: continue
            qid, docid, score = str(row[0]), str(row[1]), int(row[2])
            if score > 0:
                if qid not in qrels: qrels[qid] = set()
                qrels[qid].add(docid)
    return qrels

def calculate_recall(qrels, results_dict, k_values):
    scores = {k: 0.0 for k in k_values}
    num_queries = len(results_dict)
    if num_queries == 0:
        return {f"Recall@{k}": 0.0 for k in k_values}

    for qid, retrieved in results_dict.items():
        gold = qrels.get(qid, set())
        if not gold: continue
        for k in k_values:
            top_k = set(retrieved[:k])
            scores[k] += len(top_k.intersection(gold)) / len(gold)

    return {f"Recall@{k}": round(scores[k] / num_queries, 5) for k in k_values}

# ==============================================================================
# 3. ΜΑΖΙΚΗ ΑΝΑΛΥΣΗ
# ==============================================================================
all_rows = []

for corpus in CORPORA:
    print(f"\n🔍 Ανάλυση {corpus.upper()}...")
    qrels = load_qrels(corpus)
    if not qrels:
        print(f"  ⚠️ Δεν βρέθηκαν qrels για το {corpus}")
        continue

    strat_finds_ranks = {}
    strat_metrics = {}

    for strat_name, fpath in FILE_MAP[corpus].items():
        res_list = {}
        found_with_ranks = {}

        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    item = json.loads(line)
                    qid = str(item["task_id"])
                    if qid not in qrels: continue

                    retrieved = [str(c["document_id"]) for c in item.get("contexts", [])]
                    res_list[qid] = retrieved

                    gold_ids = qrels[qid]
                    for rank, did in enumerate(retrieved[:100], 1):
                        if did in gold_ids:
                            found_with_ranks[(qid, did)] = rank
                except: continue

        if res_list:
            strat_finds_ranks[strat_name] = found_with_ranks
            strat_metrics[strat_name] = calculate_recall(qrels, res_list, K_VALUES)
            print(f"  📊 {strat_name}: Επεξεργάστηκαν {len(res_list)} queries")

    # Unique Analysis
    for name in strat_finds_ranks.keys():
        found_ranks = strat_finds_ranks[name]
        others_found = set().union(*(d.keys() for k, d in strat_finds_ranks.items() if k != name))
        unique_pairs = set(found_ranks.keys()) - others_found

        all_ranks = list(found_ranks.values())
        unique_ranks = [found_ranks[p] for p in unique_pairs]

        row = {
            "Corpus": corpus,
            "Strategy": name,
            "UniqueFinds": len(unique_pairs),
            "MeanRank": round(np.mean(all_ranks), 2) if all_ranks else 0,
            "MeanUniqueRank": round(np.mean(unique_ranks), 2) if unique_ranks else 0
        }
        row.update(strat_metrics[name])
        all_rows.append(row)

# ==============================================================================
# 4. EXPORT & DISPLAY (Safe Check)
# ==============================================================================
if all_rows:
    df_final = pd.DataFrame(all_rows)
    recall_cols = [f"Recall@{k}" for k in K_VALUES]
    # Δυναμικός έλεγχος αν υπάρχουν οι στήλες πριν το subsetting
    available_cols = [c for c in ["Corpus", "Strategy"] + recall_cols + ["UniqueFinds", "MeanRank", "MeanUniqueRank"] if c in df_final.columns]
    df_final = df_final[available_cols]

    df_final = df_final.sort_values(["Corpus", "Recall@5"], ascending=[True, False])
    display(df_final)

    output_path = os.path.join(RESULTS_DIR, "detailed_metrics_comparison_cw05.csv")
    df_final.to_csv(output_path, index=False)
    print(f"\n✅ Επιτυχία! Αρχείο: {output_path}")
else:
    print("\n❌ Σφάλμα: Δεν συλλέχθηκαν δεδομένα. Ελέγξτε αν τα task_id στα JSONL ταυτίζονται με τα qid στα dev.tsv.")

## RRF results

In [ ]:
import json
import os
from collections import defaultdict
import pandas as pd

# ==============================================================================
# RRF SETTINGS
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
RESULTS_DIR = os.path.join(BASE_PATH, "cohere_v4_rerank_results/20260115_164434")
OUTPUT_DIR = os.path.join(BASE_PATH, "cohere_rrf_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

K_RRF = 20  # RRF constant

# Manual weights per strategy (adjust as needed)
STRATEGY_WEIGHTS = {
    "Anchor_Keyword": 0.0,
    "Corpus_Specific": 1.0,
    "Structural_Minimal": 1.0,
    "COT": 0.0,
    "Hyde": 0.2,
    "Decomp": 0.0
}

CORPORA = ["clapnq", "fiqa", "govt", "cloud"]
STRATEGIES = ["Anchor_Keyword", "Corpus_Specific", "Structural_Minimal", "COT", "Hyde", "Decomp"]

# ==============================================================================
# BUILD FILE MAP DYNAMICALLY
# ==============================================================================
FILE_MAP = {}
print("📂 Building file map from RESULTS_DIR...")
for corpus in CORPORA:
    FILE_MAP[corpus] = {}
    for strategy in STRATEGIES:
        file_name = f"{corpus}_{strategy}_HYBRID_cw0.5.jsonl"
        full_path = os.path.join(RESULTS_DIR, file_name)
        if os.path.exists(full_path):
            FILE_MAP[corpus][strategy] = full_path
            print(f"  ✅ Found: {file_name}")
        else:
            print(f"  ⚠️  Missing: {file_name}")


# ==============================================================================
# RRF FUSION FUNCTION
# ==============================================================================
def rrf_fusion(strategy_results, k=20, weights=None):
    if weights is None:
        weights = {s: 1.0 for s in strategy_results.keys()}

    all_qids = set()
    for strat_res in strategy_results.values():
        all_qids.update(strat_res.keys())

    fused = {}

    for qid in all_qids:
        doc_scores = defaultdict(float)

        for strat_name, qid_results in strategy_results.items():
            if qid not in qid_results:
                continue

            weight = weights.get(strat_name, 1.0)
            ranked_docs = qid_results[qid]

            for rank, doc_id in enumerate(ranked_docs, start=1):
                rrf_score = weight / (k + rank)
                doc_scores[doc_id] += rrf_score

        sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
        fused[qid] = sorted_docs

    return fused


# ==============================================================================
# LOAD STRATEGY RESULTS
# ==============================================================================
def load_strategy_results(corpus):
    strategy_results = {}
    query_metadata = {}

    for strat_name, fpath in FILE_MAP[corpus].items():
        if not os.path.exists(fpath):
            print(f"  ⚠️  File not found: {fpath}")
            continue

        strat_res = {}

        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    item = json.loads(line)
                    qid = str(item["task_id"])

                    if qid not in query_metadata:
                        query_metadata[qid] = {
                            "query": item.get("query", ""),
                            "Collection": item.get("Collection", corpus)
                        }

                    contexts = item.get("contexts", [])
                    ranked_docs = [str(c["document_id"]) for c in contexts]
                    strat_res[qid] = ranked_docs
                except:
                    continue

        strategy_results[strat_name] = strat_res
        print(f"  ✅ Loaded {strat_name}: {len(strat_res)} queries")

    return strategy_results, query_metadata


# ==============================================================================
# SAVE RRF RESULTS
# ==============================================================================
def save_rrf_results(corpus, fused_results, query_metadata, output_dir):
    output_path = os.path.join(output_dir, f"cohere_{corpus}_rrf.jsonl")

    with open(output_path, 'w', encoding='utf-8') as f:
        for qid, ranked_docs in fused_results.items():
            metadata = query_metadata.get(qid, {})

            contexts = []
            for doc_id, rrf_score in ranked_docs:
                contexts.append({
                    "document_id": doc_id,
                    "score": round(rrf_score, 6)  # ✅ "score" for evaluation script
                })

            output_item = {
                "task_id": qid,
                "query": metadata.get("query", ""),
                "Collection": metadata.get("Collection", corpus),
                "contexts": contexts
            }

            f.write(json.dumps(output_item, ensure_ascii=False) + "\n")

    print(f"  💾 Saved: {output_path} ({len(fused_results)} queries)")
    return output_path


# ==============================================================================
# MAIN EXECUTION - RRF FUSION
# ==============================================================================
print("\n" + "=" * 70)
print("🔀 RECIPROCAL RANK FUSION (RRF) - COHERE RESULTS")
print("=" * 70)
print(f"\nRRF constant k = {K_RRF}")
print(f"Strategy weights: {STRATEGY_WEIGHTS}\n")

saved_files = []

for corpus in CORPORA:
    print(f"\n{'='*70}")
    print(f"📊 Processing: {corpus.upper()}")
    print(f"{'='*70}")

    if corpus not in FILE_MAP or not FILE_MAP[corpus]:
        print(f"  ⚠️  No files found for {corpus}, skipping...")
        continue

    strategy_results, query_metadata = load_strategy_results(corpus)

    if not strategy_results:
        print(f"  ⚠️  No results loaded for {corpus}, skipping...")
        continue

    print(f"\n  🔀 Fusing {len(strategy_results)} strategies...")
    fused_results = rrf_fusion(strategy_results, k=K_RRF, weights=STRATEGY_WEIGHTS)
    print(f"  ✅ Fused {len(fused_results)} queries")

    output_path = save_rrf_results(corpus, fused_results, query_metadata, OUTPUT_DIR)
    saved_files.append(output_path)

print(f"\n{'='*70}")
print("✅ RRF FUSION COMPLETE!")
print(f"{'='*70}")
print(f"\nSaved {len(saved_files)} files to: {OUTPUT_DIR}")
for fpath in saved_files:
    print(f"  📄 {os.path.basename(fpath)}")


# ==============================================================================
# EVALUATION OF RRF RESULTS
# ==============================================================================
print(f"\n\n{'='*70}")
print("📈 RUNNING EVALUATION ON RRF RESULTS")
print(f"{'='*70}\n")

evaluation_script = "/content/drive/MyDrive/Project/run_retrieval_eval.py"

if not os.path.exists(evaluation_script):
    print(f"⚠️ Evaluation script not found: {evaluation_script}")
    print("Skipping evaluation...")
else:
    for corpus in CORPORA:
        input_path = os.path.join(OUTPUT_DIR, f"cohere_{corpus}_rrf.jsonl")
        output_path = os.path.join(OUTPUT_DIR, f"cohere_{corpus}_rrf_results.jsonl")

        if not os.path.exists(input_path):
            print(f"⚠️ Skipping {corpus} - file not found: {input_path}")
            continue

        print(f"\n{'='*50}")
        print(f"Running evaluation for {corpus} ...")
        print(f"Input: {input_path}")
        print(f"Output: {output_path}")
        print(f"{'='*50}")

        cmd = f"python {evaluation_script} --input_file {input_path} --output_file {output_path}"
        result = os.system(cmd)

        if result != 0:
            print(f"⚠️ Warning: Evaluation failed for {corpus}")
        else:
            print(f"✅ Evaluation completed for {corpus}")


# ==============================================================================
# AGGREGATE RESULTS COMPARISON
# ==============================================================================
print(f"\n\n{'='*70}")
print("📊 AGGREGATING EVALUATION RESULTS")
print(f"{'='*70}\n")

csv_files = [
    f"{OUTPUT_DIR}/cohere_clapnq_rrf_results_aggregate.csv",
    f"{OUTPUT_DIR}/cohere_fiqa_rrf_results_aggregate.csv",
    f"{OUTPUT_DIR}/cohere_govt_rrf_results_aggregate.csv",
    f"{OUTPUT_DIR}/cohere_cloud_rrf_results_aggregate.csv"
]

all_data = []

for file in csv_files:
    if os.path.exists(file):
        df = pd.read_csv(file)
        df = df[df["collection"] != "all"]  # Remove aggregate row if present
        corpus_name = file.split("/")[-1].replace("cohere_", "").replace("_rrf_results_aggregate.csv", "")
        df["Corpus"] = corpus_name
        all_data.append(df)
        print(f"✅ Loaded: {corpus_name}")
    else:
        print(f"⚠️ Not found: {file}")

if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df = final_df.set_index("Corpus")
    print("\n" + "="*70)
    print("📊 COHERE RRF EVALUATION RESULTS - ALL CORPORA")
    print("="*70)
    display(final_df)

    # Save combined results
    combined_output = f"{OUTPUT_DIR}/cohere_rrf_all_corpora_results.csv"
    final_df.to_csv(combined_output)
    print(f"\n💾 Combined results saved to: {combined_output}")
else:
    print("\n⚠️ No evaluation results found")

print("\n" + "="*70)
print("✅ ALL PROCESSING COMPLETE!")
print("="*70)


# **BM25 Pipeline**

In [ ]:
!pip -q install rank_bm25 tqdm pandas numpy requests

import os, json, re, time, string, threading, requests
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import Dict, List, Tuple, Optional
from rank_bm25 import BM25Okapi
from concurrent.futures import ThreadPoolExecutor, as_completed

# ----------------------------
# SETTINGS
# ----------------------------
BASE_PATH = "/content/drive/MyDrive/Project"

# Azure DeepSeek Configuration
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

EVAL_SCRIPT = f"{BASE_PATH}/run_retrieval_eval.py"

RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
OUT_DIR = f"{BASE_PATH}/predictions_bm25_twostage_deepseek_{RUN_TAG}"
EVAL_DIR = f"{BASE_PATH}/eval_outputs_bm25_twostage_deepseek_{RUN_TAG}"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

TOP_K = 100
HISTORY_TURNS = 6

# LLM parallelism
MAX_LLM_WORKERS = 8

# Stage 1 (initial retrieval)
STAGE1_K = 50

# PRF settings (evidence from Stage 1)
PRF_DOCS = 6
PRF_SNIPPET_CHARS = 900
PRF_TERMS = 10
MMR_LAMBDA = 0.72

# HyDE settings
HYDE_TERMS = 14
HYDE_TEMP = 0.25

CORPORA_CONFIG = [
    {
        "name": "clapnq",
        "collection": "mt-rag-clapnq-elser-512-100-20240503",
        "corpus_file": f"{BASE_PATH}/corpora/clapnq.jsonl",
        "questions_file": f"{BASE_PATH}/retrieval/clapnq/clapnq_questions.jsonl",
    },
    {
        "name": "govt",
        "collection": "mt-rag-govt-elser-512-100-20240611",
        "corpus_file": f"{BASE_PATH}/corpora/govt.jsonl",
        "questions_file": f"{BASE_PATH}/retrieval/govt/govt_questions.jsonl",
    },
    {
        "name": "cloud",
        "collection": "mt-rag-ibmcloud-elser-512-100-20240502",
        "corpus_file": f"{BASE_PATH}/corpora/cloud.jsonl",
        "questions_file": f"{BASE_PATH}/retrieval/cloud/cloud_questions.jsonl",
    },
    {
        "name": "fiqa",
        "collection": "mt-rag-fiqa-beir-elser-512-100-20240501",
        "corpus_file": f"{BASE_PATH}/corpora/fiqa.jsonl",
        "questions_file": f"{BASE_PATH}/retrieval/fiqa/fiqa_questions.jsonl",
    },
]

RUNS = [
    {"name": "A_corpus_based_twostage", "mode": "corpus_twostage"},
    {"name": "B_organizers_twostage", "mode": "organizers_twostage"},
    {"name": "C_hyde_twostage", "mode": "hyde_twostage"},
]

# ----------------------------
# COST TRACKER
# ----------------------------
class CostTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt_tokens, completion_tokens):
        with self.lock:
            self.total_input += int(prompt_tokens or 0)
            self.total_output += int(completion_tokens or 0)

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        return f"In: {self.total_input:,} | Out: {self.total_output:,} | Est. Cost: ${cost:.4f}"

tracker = CostTracker()

# ----------------------------
# AZURE DEEPSEEK LLM
# ----------------------------
_thread_local = threading.local()

def llm_invoke_text(system: str, user: str, max_tokens: int, temperature: float = 0.0) -> str:
    """Azure DeepSeek chat/completions call with retries + token tracking."""
    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-V3.2",
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ],
        "max_tokens": max_tokens,
        "temperature": temperature,
    }

    for attempt in range(7):
        try:
            r = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)

            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(3 * (2 ** attempt))
                continue

            r.raise_for_status()
            res_json = r.json()

            usage = res_json.get("usage", {})
            tracker.add_usage(usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0))

            return res_json["choices"][0]["message"]["content"].strip()

        except Exception:
            time.sleep(2 * (2 ** attempt))

    return ""

def healthcheck():
    print("🔍 Azure DeepSeek healthcheck...")
    try:
        txt = llm_invoke_text(
            system="You are a healthcheck bot.",
            user="Reply with exactly: OK",
            max_tokens=10,
            temperature=0.0
        )
        if "OK" in txt:
            print("✅ DeepSeek healthcheck OK")
        else:
            print(f"⚠️  Unexpected response: {txt}")
    except Exception as e:
        raise RuntimeError(f"❌ DeepSeek healthcheck failed: {e}")

print("✅ Using Azure DeepSeek V3.2")
healthcheck()

# ----------------------------
# JSON PARSING
# ----------------------------
_FENCE_RE = re.compile(r"^```(?:json)?\s*|\s*```$", re.IGNORECASE | re.MULTILINE)

def extract_first_json_object(text: str) -> str:
    if not text:
        raise ValueError("empty text")
    t = re.sub(_FENCE_RE, "", text).strip()
    start = t.find("{")
    if start < 0:
        raise ValueError(f"no '{{' found")
    in_str, esc, depth = False, False, 0
    for i in range(start, len(t)):
        ch = t[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        else:
            if ch == '"':
                in_str = True
                continue
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return t[start:i+1]
    raise ValueError("unterminated JSON object")

def parse_json_loose(text: str, tag: str) -> Dict:
    try:
        return json.loads(text)
    except Exception:
        s = extract_first_json_object(text)
        try:
            return json.loads(s)
        except Exception as e:
            raise RuntimeError(f"{tag} JSON parse failed") from e

def norm_ws(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())

# ----------------------------
# LOAD QUERIES + HISTORY
# ----------------------------
USER_LINE_RE = re.compile(r"^\s*\|user\|\s*:\s*(.*)\s*$", re.IGNORECASE)

def parse_user_only_history(text: str) -> Tuple[str, List[str]]:
    lines = [ln.strip() for ln in (text or "").splitlines() if ln.strip()]
    user_utts = []
    for ln in lines:
        m = USER_LINE_RE.match(ln)
        if m:
            user_utts.append(m.group(1).strip())
    if not user_utts:
        return (text or "").strip(), []
    return user_utts[-1], user_utts[:-1]

def load_questions_with_history(path: str) -> Dict[str, Dict]:
    qmap = {}
    if not os.path.exists(path):
        return qmap
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            j = json.loads(line)
            qid = j.get("_id")
            raw = (j.get("text") or "").strip()
            if not qid or not raw:
                continue
            q, hist = parse_user_only_history(raw)
            qmap[str(qid)] = {"query": q, "history": hist}
    return qmap

def history_to_text(hist: List[str]) -> str:
    last = (hist or [])[-HISTORY_TURNS:]
    return "\n".join([f"user: {h}" for h in last])

# ----------------------------
# LOAD CORPUS
# ----------------------------
def load_corpus_jsonl(path: str):
    docs, ids = [], []
    full = {}
    if not os.path.exists(path):
        return docs, ids, full
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            j = json.loads(line)
            did = j.get("_id") or j.get("doc_id") or j.get("id")
            if did is None:
                continue
            title = (j.get("title") or "").strip()
            text = (j.get("text") or j.get("contents") or j.get("content") or "").strip()
            if not title and not text:
                continue
            did = str(did)
            full_txt = (title + " " + text).strip()
            ids.append(did)
            docs.append(full_txt)
            full[did] = {"title": title, "text": text, "full": full_txt}
    return docs, ids, full

# ----------------------------
# BM25
# ----------------------------
_PUNCT_TABLE = str.maketrans("", "", string.punctuation)

def tokenize(s: str) -> List[str]:
    return (s or "").lower().translate(_PUNCT_TABLE).split()

class BM25Retriever:
    def __init__(self, docs: List[str], doc_ids: List[str]):
        self.doc_ids = doc_ids
        self.tokenized_corpus = [tokenize(d) for d in docs]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

    def retrieve(self, query: str, top_k: int = 100) -> List[Dict]:
        qtok = tokenize(query)
        scores = self.bm25.get_scores(qtok)
        top = np.argsort(scores)[::-1][:top_k]
        return [{"id": self.doc_ids[i], "score": float(scores[i]), "idx": int(i)} for i in top]

# ----------------------------
# MMR + EVIDENCE BUILDING
# ----------------------------
STOPLIKE = set("""
the a an and or of to for in on at by with from into about as is are was were be been being
what who when where why how do does did can could would should which
this that these those
""".split())

def drop_garbage_terms(terms: List[str]) -> List[str]:
    out = []
    seen = set()
    for t in terms:
        tt = norm_ws(str(t))
        if not tt:
            continue
        toks = tokenize(tt)
        if not toks:
            continue
        if all(tok in STOPLIKE for tok in toks):
            continue
        k = tt.lower()
        if k in seen:
            continue
        seen.add(k)
        out.append(tt)
    return out

def filter_terms_verbatim(terms: List[str], evidence: str) -> List[str]:
    ev = (evidence or "").lower()
    kept = []
    for t in drop_garbage_terms(terms):
        if t.lower() in ev:
            kept.append(t)
    return kept

def mmr_select(doc_ids: List[str], doc_texts: List[str], doc_scores: List[float], k: int, lam: float) -> List[int]:
    if not doc_ids:
        return []
    toks = [set(tokenize(t)) for t in doc_texts]
    selected = []
    candidates = list(range(len(doc_ids)))

    best0 = max(candidates, key=lambda i: doc_scores[i])
    selected.append(best0)
    candidates.remove(best0)

    def jacc(i, j):
        a, b = toks[i], toks[j]
        if not a or not b:
            return 0.0
        return len(a & b) / max(1, len(a | b))

    while candidates and len(selected) < k:
        def score(i):
            rel = doc_scores[i]
            div = max(jacc(i, s) for s in selected) if selected else 0.0
            return lam * rel - (1.0 - lam) * div
        nxt = max(candidates, key=score)
        selected.append(nxt)
        candidates.remove(nxt)
    return selected

def build_evidence(doc_ids: List[str], corpus_full: Dict[str, Dict]) -> str:
    chunks = []
    for did in doc_ids:
        rec = corpus_full.get(did, {})
        title = (rec.get("title") or "").strip()
        text = (rec.get("text") or "").strip()
        chunk = (title + "\n" + text).strip()[:PRF_SNIPPET_CHARS]
        if chunk:
            chunks.append(chunk)
    return "\n\n---\n\n".join(chunks)

def select_evidence_docs(stage1_hits: List[Dict], corpus_full: Dict[str, Dict]) -> Tuple[List[str], str]:
    """Apply MMR to select diverse evidence documents from Stage 1."""
    pool_ids = [h["id"] for h in stage1_hits]
    pool_scores = [h["score"] for h in stage1_hits]
    pool_texts = [corpus_full.get(did, {}).get("full", "") for did in pool_ids]

    sel_idx = mmr_select(pool_ids, pool_texts, pool_scores, PRF_DOCS, MMR_LAMBDA)
    sel_ids = [pool_ids[i] for i in sel_idx] if sel_idx else pool_ids[:PRF_DOCS]
    evidence = build_evidence(sel_ids, corpus_full)
    return sel_ids, evidence

# ----------------------------
# LLM REWRITERS (TWO-STAGE)
# ----------------------------
def llm_corpus_rewrite_with_evidence(orig_query: str, hist: List[str], corpus_name: str, evidence: str) -> Tuple[str, List[str]]:
    """Corpus-based rewrite using Stage 1 evidence."""
    system = (
        f"You rewrite queries for BM25 retrieval on '{corpus_name}' using retrieved evidence.\n"
        "Task: Generate a standalone rewrite + expansion terms based on the evidence.\n"
        "Return ONLY strict JSON: {\"query\":\"...\" , \"expansion_terms\":[\"...\"]}.\n\n"
        "Rules:\n"
        "- Use evidence to enrich the query with exact terms/entities.\n"
        "- expansion_terms must appear verbatim in evidence.\n"
        "- Prefer: proper nouns, titles, years, locations.\n"
        f"- Max {PRF_TERMS} expansion terms.\n"
        "- Keep query compact.\n"
    )

    hist_text = history_to_text(hist)
    user = (
        f"HISTORY:\n{hist_text}\n\n"
        f"QUERY:\n{orig_query}\n\n"
        f"EVIDENCE (from Stage 1):\n{evidence}\n\n"
        "JSON:"
    )

    txt = llm_invoke_text(system, user, 350, 0.0)
    j = parse_json_loose(txt, "CORPUS_REWRITE_EVIDENCE")

    rewritten = norm_ws(j.get("query") or orig_query)
    exp_terms = j.get("expansion_terms") or []
    if not isinstance(exp_terms, list):
        exp_terms = []
    exp_terms = filter_terms_verbatim([str(x).strip() for x in exp_terms], evidence)[:PRF_TERMS]

    return rewritten, exp_terms

def llm_organizers_rewrite_with_evidence(orig_query: str, hist: List[str], evidence: str) -> Tuple[str, List[str]]:
    """Organizers rewrite using Stage 1 evidence."""
    system = (
        "Rewrite the query into a standalone form using retrieved evidence.\n"
        "Return ONLY strict JSON: {\"query\":\"...\" , \"expansion_terms\":[\"...\"]}.\n\n"
        "Rules:\n"
        "- Minimal rewording; stay close to user's intent.\n"
        "- Use evidence to add exact entities/terms.\n"
        "- expansion_terms must appear verbatim in evidence.\n"
        f"- Max {PRF_TERMS} expansion terms.\n"
    )

    hist_text = history_to_text(hist)
    user = (
        f"HISTORY:\n{hist_text}\n\n"
        f"QUERY:\n{orig_query}\n\n"
        f"EVIDENCE:\n{evidence}\n\n"
        "JSON:"
    )

    txt = llm_invoke_text(system, user, 320, 0.0)
    j = parse_json_loose(txt, "ORGANIZERS_REWRITE_EVIDENCE")

    rewritten = norm_ws(j.get("query") or orig_query)
    exp_terms = j.get("expansion_terms") or []
    if not isinstance(exp_terms, list):
        exp_terms = []
    exp_terms = filter_terms_verbatim([str(x).strip() for x in exp_terms], evidence)[:PRF_TERMS]

    return rewritten, exp_terms

def llm_hyde_rewrite_with_evidence(orig_query: str, hist: List[str], evidence: str) -> Tuple[str, List[str], List[str]]:
    """HyDE: generate hypothetical terms + use evidence for expansion."""
    # First: generate HyDE terms (no evidence needed)
    system_hyde = (
        "Generate lexical expansion terms likely to appear in Wikipedia-like documents.\n"
        "Return ONLY strict JSON: {\"terms\":[\"...\"]}.\n\n"
        "Rules:\n"
        "- Prefer: proper nouns, alternative names, years, locations.\n"
        "- Avoid generic words.\n"
        f"- Max {HYDE_TERMS} terms, each 1-5 words.\n"
    )

    hist_text = history_to_text(hist)
    user_hyde = f"HISTORY:\n{hist_text}\n\nQUERY:\n{orig_query}\n\nJSON:"

    txt_hyde = llm_invoke_text(system_hyde, user_hyde, 220, HYDE_TEMP)
    j_hyde = parse_json_loose(txt_hyde, "HYDE_TERMS")
    hyde_terms = j_hyde.get("terms") or []
    if not isinstance(hyde_terms, list):
        hyde_terms = []
    hyde_terms = drop_garbage_terms([str(x).strip() for x in hyde_terms])[:HYDE_TERMS]

    # Second: use evidence for PRF expansion
    system_prf = (
        "Extract expansion terms from evidence for BM25 retrieval.\n"
        "Return ONLY strict JSON: {\"terms\":[\"...\"]}.\n\n"
        "Rules:\n"
        "- Every term MUST appear verbatim in evidence.\n"
        "- Prefer: proper nouns, titles, years.\n"
        f"- Max {PRF_TERMS} terms.\n"
    )

    user_prf = f"QUERY:\n{orig_query}\n\nEVIDENCE:\n{evidence}\n\nJSON:"

    txt_prf = llm_invoke_text(system_prf, user_prf, 220, 0.0)
    j_prf = parse_json_loose(txt_prf, "HYDE_PRF")
    prf_terms = j_prf.get("terms") or []
    if not isinstance(prf_terms, list):
        prf_terms = []
    prf_terms = filter_terms_verbatim([str(x).strip() for x in prf_terms], evidence)[:PRF_TERMS]

    return orig_query, hyde_terms, prf_terms

# ----------------------------
# EVAL
# ----------------------------
def run_eval(pred_path: str) -> Dict[str, Optional[str]]:
    pred_file = os.path.basename(pred_path)
    out_eval = os.path.join(EVAL_DIR, pred_file.replace(".jsonl", "_eval.jsonl"))
    cmd = f"python3 {EVAL_SCRIPT} --input_file {pred_path} --output_file {out_eval} > /dev/null 2>&1"
    rc = os.system(cmd)
    if rc != 0:
        return {"nDCG": None, "Recall": None}
    agg1 = out_eval.replace(".jsonl", "_aggregate.csv")
    agg2 = out_eval.replace(".jsonl", "_results_aggregate.csv")
    agg = agg1 if os.path.exists(agg1) else (agg2 if os.path.exists(agg2) else None)
    if not agg:
        return {"nDCG": None, "Recall": None}
    df = pd.read_csv(agg)
    if "collection" in df.columns:
        df = df[df["collection"] != "all"]
    if df.empty:
        return {"nDCG": None, "Recall": None}
    row = df.iloc[0].to_dict()
    return {"nDCG": row.get("nDCG"), "Recall": row.get("Recall")}

# ==============================================================================
# TWO-STAGE QUERY RUNNER
# ==============================================================================
def run_two_stage_query(qid: str, item: Dict, mode: str,
                         corpus_name: str,
                         bm25: BM25Retriever,
                         corpus_full: Dict[str, Dict]) -> Tuple[str, Dict, List[Dict]]:
    """
    Two-stage retrieval:
    1) Retrieve with original query (Stage 1)
    2) Use evidence to rewrite → re-retrieve (Stage 2)
    """
    orig_q = item["query"]
    hist = (item["history"] or [])[-HISTORY_TURNS:]

    # STAGE 1: Initial retrieval
    stage1_hits = bm25.retrieve(orig_q, top_k=STAGE1_K)

    # Select evidence docs using MMR
    evidence_doc_ids, evidence = select_evidence_docs(stage1_hits, corpus_full)

    # STAGE 2: Rewrite based on evidence + mode
    if mode == "corpus_twostage":
        rewritten, exp_terms = llm_corpus_rewrite_with_evidence(orig_q, hist, corpus_name, evidence)
        final_query = rewritten + (" " + " ".join(exp_terms) if exp_terms else "")
        meta = {
            "stage1_query": orig_q,
            "stage2_rewritten": rewritten,
            "expansion_terms": exp_terms,
            "evidence_docs": evidence_doc_ids
        }

    elif mode == "organizers_twostage":
        rewritten, exp_terms = llm_organizers_rewrite_with_evidence(orig_q, hist, evidence)
        final_query = rewritten + (" " + " ".join(exp_terms) if exp_terms else "")
        meta = {
            "stage1_query": orig_q,
            "stage2_rewritten": rewritten,
            "expansion_terms": exp_terms,
            "evidence_docs": evidence_doc_ids
        }

    elif mode == "hyde_twostage":
        base, hyde_terms, prf_terms = llm_hyde_rewrite_with_evidence(orig_q, hist, evidence)
        final_query = base
        if hyde_terms:
            final_query += " " + " ".join(hyde_terms[:10])
        if prf_terms:
            final_query += " " + " ".join(prf_terms)
        meta = {
            "stage1_query": orig_q,
            "hyde_terms": hyde_terms,
            "expansion_terms": prf_terms,
            "evidence_docs": evidence_doc_ids
        }

    else:
        raise ValueError(f"Unknown mode: {mode}")

    # STAGE 2: Final retrieval
    stage2_hits = bm25.retrieve(final_query, top_k=TOP_K)

    return final_query, meta, stage2_hits

# ==============================================================================
# MAIN LOOP
# ==============================================================================
print("\n🚀 RUNNING BM25 TWO-STAGE — ALL CORPORA — AZURE DEEPSEEK")
print("🏷️  RUN_TAG:", RUN_TAG)

all_rows = []

for cfg in CORPORA_CONFIG:
    corpus_name = cfg["name"]
    collection = cfg["collection"]

    print("\n" + "#"*92)
    print(f"### CORPUS: {corpus_name} | Collection: {collection}")
    print("#"*92)

    qmap = load_questions_with_history(cfg["questions_file"])
    if not qmap:
        print("❌ No queries loaded. Skipping.")
        continue

    items = list(qmap.items())
    print(f"✅ Loaded {len(items)} queries for {corpus_name}")

    docs, doc_ids, corpus_full = load_corpus_jsonl(cfg["corpus_file"])
    if not docs:
        print("❌ Empty corpus. Skipping.")
        continue

    print(f"   ⚙️  Building BM25 for {len(docs)} docs...")
    bm25 = BM25Retriever(docs, doc_ids)
    print("   ✅ BM25 ready.")

    for run in RUNS:
        run_name = run["name"]
        mode = run["mode"]

        print("\n" + "-"*92)
        print(f"▶️  RUN: {corpus_name} | {run_name}")

        pred_path = os.path.join(OUT_DIR, f"{corpus_name}_{run_name}_{RUN_TAG}_predictions.jsonl")
        rw_path = os.path.join(OUT_DIR, f"{corpus_name}_{run_name}_{RUN_TAG}_rewrites.jsonl")

        dbg_left = 2
        failures = 0

        with open(pred_path, "w", encoding="utf-8") as fp, open(rw_path, "w", encoding="utf-8") as fr:
            with ThreadPoolExecutor(max_workers=MAX_LLM_WORKERS) as ex:
                futs = {
                    ex.submit(run_two_stage_query, qid, item, mode, corpus_name, bm25, corpus_full): (qid, item)
                    for qid, item in items
                }

                for fut in tqdm(as_completed(futs), total=len(futs), desc=f"{corpus_name}/{run_name}", leave=False):
                    qid, item = futs[fut]
                    orig_q = item["query"]
                    hist_turns = len(item["history"])

                    try:
                        final_q, meta, stage2_hits = fut.result()
                    except Exception as e:
                        failures += 1
                        final_q = orig_q
                        meta = {"error": str(e)[:240], "fallback": True}
                        stage2_hits = bm25.retrieve(orig_q, top_k=TOP_K)

                    if dbg_left > 0:
                        dbg_left -= 1
                        print(f"\n[DEBUG {corpus_name}/{run_name}] qid={qid} hist_turns={hist_turns}")
                        print("  orig:", orig_q[:160])
                        print("  stage1:", meta.get("stage1_query", "")[:160])
                        if "stage2_rewritten" in meta:
                            print("  stage2_rewritten:", meta.get("stage2_rewritten")[:160])
                        if "expansion_terms" in meta:
                            print("  expansion:", meta.get("expansion_terms", [])[:8])
                        if "hyde_terms" in meta:
                            print("  hyde:", meta.get("hyde_terms", [])[:8])
                        print("  final:", final_q[:220])

                    fp.write(json.dumps({
                        "task_id": qid,
                        "Collection": collection,
                        "contexts": [{"document_id": h["id"], "score": h["score"]} for h in stage2_hits],
                        "metadata": {
                            "run": run_name,
                            "run_tag": RUN_TAG,
                            "original_query": orig_q,
                            "history_turns": hist_turns,
                            **meta
                        }
                    }, ensure_ascii=False) + "\n")

                    fr.write(json.dumps({
                        "task_id": qid,
                        "original_query": orig_q,
                        "final_query": final_q,
                        "history_turns": hist_turns,
                        "run": run_name,
                        "run_tag": RUN_TAG,
                        "meta": meta
                    }, ensure_ascii=False) + "\n")

        metrics = run_eval(pred_path)
        print("✅ Eval:", {"nDCG": metrics["nDCG"], "Recall": metrics["Recall"], "failures": failures})
        print(f"💰 {tracker.get_report()}")

        all_rows.append({
            "corpus": corpus_name,
            "Run": run_name,
            "count": len(items),
            "nDCG": metrics["nDCG"],
            "Recall": metrics["Recall"],
            "failures": failures,
            "pred_file": pred_path
        })

df = pd.DataFrame(all_rows)
print("\n" + "="*92)
print("📊 SUMMARY — BM25 TWO-STAGE (AZURE DEEPSEEK)")
print("="*92)
display(df[["corpus","Run","count","nDCG","Recall","failures","pred_file"]])

summary_csv = os.path.join(OUT_DIR, f"summary_bm25_twostage_deepseek_{RUN_TAG}.csv")
df.to_csv(summary_csv, index=False)

print("\n✅ Saved summary CSV:", summary_csv)
print("📌 Predictions dir:", OUT_DIR)
print("📌 Eval outputs dir:", EVAL_DIR)
print(f"💰 FINAL COST: {tracker.get_report()}")

In [ ]:
import os, json, time, requests, subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import Dict, List, Tuple

# ==============================================================================
# 1. SETTINGS & CONFIGURATION
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"

# Στοχεύουμε τον φάκελο με τα CONVERTED αρχεία
PREDS_PATH = os.path.join(BASE_PATH, "predictions_deepseek_singleprf_20260115_190813", "converted_for_eval")

EVAL_SCRIPT_PATH = os.path.join(BASE_PATH, "run_retrieval_eval.py")

# Azure Cohere Credentials
AZURE_COHERE_ENDPOINT = os.environ["ENDPOINT"]
AZURE_COHERE_KEY = os.environ["_KEY"]

RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
# Αποθήκευση αποτελεσμάτων σε νέο φάκελο rerank
OUTPUT_DIR = os.path.join(BASE_PATH, "cohere_v4_rerank_results", RUN_TAG)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Οι στρατηγικές όπως ονομάστηκαν από το DeepSeek run
STRATEGIES = [
    "A_corpus_based_prf_llm__hist_last6",
    "B_organizers_prf_llm__hist_last6",
    "C_hyde_prf_llm__hist_last6"
]
CORPORA = ["clapnq", "govt", "cloud", "fiqa"]

INPUT_FILES = {}
for corpus in CORPORA:
    INPUT_FILES[corpus] = {}
    for strategy in STRATEGIES:
        # Format: {corpus}_{strategy}_evalfmt.jsonl
        file_name = f"{corpus}_{strategy}_evalfmt.jsonl"
        INPUT_FILES[corpus][strategy] = os.path.join(PREDS_PATH, file_name)

# Ground Truth Files
CORPUS_CONFIG = {
    "clapnq": {
        "corpus_file": f"{BASE_PATH}/corpora/clapnq.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/clapnq/clapnq_lastturn.jsonl"
    },
    "fiqa": {
        "corpus_file": f"{BASE_PATH}/corpora/fiqa.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/fiqa/fiqa_lastturn.jsonl"
    },
    "govt": {
        "corpus_file": f"{BASE_PATH}/corpora/govt.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/govt/govt_lastturn.jsonl"
    },
    "cloud": {
        "corpus_file": f"{BASE_PATH}/corpora/cloud.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/cloud/cloud_lastturn.jsonl"
    }
}

TOP_K_RERANK = 100
COHERE_WEIGHTS = [0.1, 0.3, 0.5, 0.7, 0.9]

# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================

def load_jsonl(path: str) -> List[dict]:
    data = []
    if not os.path.exists(path): return data
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip(): data.append(json.loads(line))
    return data

def load_corpus(path: str) -> Dict[str, str]:
    corpus = {}
    if not os.path.exists(path): return corpus
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            did = str(j.get("_id") or j.get("id"))
            text = (j.get("title", "") + " " + (j.get("text") or j.get("contents") or "")).strip()
            corpus[did] = text[:2000]
    return corpus

def load_queries(path: str) -> Dict[str, str]:
    queries = {}
    if not os.path.exists(path): return queries
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            qid = str(j.get("_id") or j.get("id") or j.get("task_id"))
            queries[qid] = j.get("text") or j.get("query")
    return queries

def azure_cohere_rerank(query: str, docs_text: List[str]) -> List[float]:
    if not query or not docs_text: return [0.0] * len(docs_text)
    headers = {"Content-Type": "application/json", "api-key": AZURE_COHERE_KEY}
    payload = {
        "model": "Cohere-rerank-v4.0-pro",
        "query": query, "documents": docs_text, "top_n": len(docs_text)
    }
    try:
        response = requests.post(AZURE_COHERE_ENDPOINT, headers=headers, json=payload, timeout=60)
        if response.status_code == 200:
            results = response.json()["results"]
            scores = [0.0] * len(docs_text)
            for r in results: scores[r["index"]] = float(r["relevance_score"])
            return scores
    except Exception as e: print(f"❌ Error: {e}")
    return [0.0] * len(docs_text)

def minmax_norm(scores: List[float]) -> np.ndarray:
    arr = np.array(scores, dtype=np.float32)
    if len(arr) < 2 or arr.max() == arr.min(): return np.zeros_like(arr)
    return (arr - arr.min()) / (arr.max() - arr.min())

def run_eval(input_jsonl: str) -> Tuple[float, float]:
    result_file = input_jsonl.replace(".jsonl", "_results.jsonl")
    csv_file = result_file.replace(".jsonl", "_aggregate.csv")
    cmd = f"python3 {EVAL_SCRIPT_PATH} --input_file {input_jsonl} --output_file {result_file}"
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        if "collection" in df.columns: df = df[df["collection"] != "all"]
        if df.empty: return 0.0, 0.0
        row = df.iloc[0]
        def parse_metric(val):
            if isinstance(val, str) and "[" in val:
                import ast
                return ast.literal_eval(val)[2]
            return val
        return parse_metric(row.get("Recall", 0.0)), parse_metric(row.get("nDCG", 0.0))
    return 0.0, 0.0

# ==============================================================================
# 3. MAIN PIPELINE
# ==============================================================================

performance_summary = []

for corpus_name in CORPORA:
    print(f"\n📚 CORPUS: {corpus_name.upper()}")
    corpus_cfg = CORPUS_CONFIG[corpus_name]
    corpus_map = load_corpus(corpus_cfg["corpus_file"])
    query_map = load_queries(corpus_cfg["queries_file"])

    for strategy_name, file_path in INPUT_FILES[corpus_name].items():
        if not os.path.exists(file_path):
            print(f"⏩ Skipping {strategy_name} - Not found at {file_path}")
            continue

        print(f"🎯 Reranking: {strategy_name}")
        preds = load_jsonl(file_path)
        cache = []
        for p in tqdm(preds, desc=f"Processing {strategy_name}", leave=False):
            qid = str(p["task_id"])
            q_txt = query_map.get(qid, "")

            # Χειρισμός των contexts από το evalfmt
            all_ctxs = p.get("contexts", [])
            top_ctxs = all_ctxs[:TOP_K_RERANK]
            rest_ctxs = all_ctxs[TOP_K_RERANK:]

            doc_texts = [corpus_map.get(str(c["document_id"]), "N/A") for c in top_ctxs]
            coh_scores = azure_cohere_rerank(q_txt, doc_texts)
            orig_scores = [float(c["score"]) for c in top_ctxs]

            cache.append({
                "task_id": qid,
                "Collection": p.get("Collection"),
                "top_ctxs": top_ctxs,
                "rest": rest_ctxs,
                "orig_scores": orig_scores,
                "coh_scores": coh_scores
            })

        # Eval Cohere-Only
        cohere_file = os.path.join(OUTPUT_DIR, f"{corpus_name}_{strategy_name}_COHERE_ONLY.jsonl")
        with open(cohere_file, "w") as f:
            for item in cache:
                new_ctxs = []
                for i, ctx in enumerate(item["top_ctxs"]):
                    nc = ctx.copy()
                    nc["score"] = float(item["coh_scores"][i])
                    new_ctxs.append(nc)
                new_ctxs.sort(key=lambda x: x["score"], reverse=True)
                f.write(json.dumps({"task_id": item["task_id"], "Collection": item["Collection"], "contexts": new_ctxs + item["rest"]}) + "\n")

        r5, n5 = run_eval(cohere_file)
        performance_summary.append({"Corpus": corpus_name, "Strategy": strategy_name, "Method": "Cohere-Only", "Recall@5": r5, "nDCG@5": n5})

        # Hybrid Fusion
        for cw in COHERE_WEIGHTS:
            ow = 1.0 - cw
            hybrid_file = os.path.join(OUTPUT_DIR, f"{corpus_name}_{strategy_name}_HYBRID_cw{cw}.jsonl")
            with open(hybrid_file, "w") as f:
                for item in cache:
                    n_orig, n_coh = minmax_norm(item["orig_scores"]), minmax_norm(item["coh_scores"])
                    combined = (ow * n_orig) + (cw * n_coh)
                    new_ctxs = []
                    for i, ctx in enumerate(item["top_ctxs"]):
                        nc = ctx.copy()
                        nc["score"] = float(combined[i])
                        new_ctxs.append(nc)
                    new_ctxs.sort(key=lambda x: x["score"], reverse=True)
                    f.write(json.dumps({"task_id": item["task_id"], "Collection": item["Collection"], "contexts": new_ctxs + item["rest"]}) + "\n")

            r5_h, n5_h = run_eval(hybrid_file)
            performance_summary.append({"Corpus": corpus_name, "Strategy": strategy_name, "Method": f"Hybrid (cw={cw})", "Recall@5": r5_h, "nDCG@5": n5_h})

# Final Report
df_final = pd.DataFrame(performance_summary)
if not df_final.empty:
    from IPython.display import display
    display(df_final.sort_values(["Corpus", "nDCG@5"], ascending=[True, False]))
    df_final.to_csv(os.path.join(OUTPUT_DIR, "reranking_report.csv"), index=False)
    print(f"\n✅ All done! Results saved in: {OUTPUT_DIR}")

## Retrieval results and reranked results

In [ ]:
import json
import csv
import os
import pandas as pd
import numpy as np
from IPython.display import display

# ==============================================================================
# 1. ΡΥΘΜΙΣΕΙΣ & ΔΥΝΑΜΙΚΟ FILE MAP (cw=0.9)
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"

# ΠΡΟΣΟΧΗ: Αντικατάστησε το timestamp με αυτό του δικού σου φακέλου
# π.χ. RESULTS_DIR = os.path.join(BASE_PATH, "cohere_v4_rerank_results", "20260115_190813")
RESULTS_DIR = os.path.join(BASE_PATH, "cohere_v4_rerank_results", "20260116_084940")

CORPORA = ["clapnq", "fiqa", "govt", "cloud"]
STRATEGIES = [
    "A_corpus_based_prf_llm__hist_last6",
    "B_organizers_prf_llm__hist_last6",
    "C_hyde_prf_llm__hist_last6"
]
K_VALUES = [5, 10, 20, 50, 100]

FILE_MAP = {}
print(f"📂 Έλεγχος αρχείων στο: {RESULTS_DIR}")

for corpus in CORPORA:
    FILE_MAP[corpus] = {}
    for strategy in STRATEGIES:
        # Στοχεύουμε τα αρχεία με βάρος 0.9
        file_name = f"{corpus}_{strategy}_HYBRID_cw0.9.jsonl"
        full_path = os.path.join(RESULTS_DIR, file_name)
        if os.path.exists(full_path):
            FILE_MAP[corpus][strategy] = full_path
            print(f"  ✅ Βρέθηκε: {file_name}")
        else:
            print(f"  ❌ Λείπει: {file_name}")

# ==============================================================================
# 2. HELPER FUNCTIONS (QRELS & METRICS)
# ==============================================================================
def load_qrels(corpus):
    path = os.path.join(BASE_PATH, "retrieval", corpus, "dev.tsv")
    qrels = {}
    if not os.path.exists(path):
        return qrels
    with open(path, 'r', encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        try: next(reader)
        except: return qrels
        for row in reader:
            if len(row) < 3: continue
            qid, docid, score = str(row[0]), str(row[1]), int(row[2])
            if score > 0:
                if qid not in qrels: qrels[qid] = set()
                qrels[qid].add(docid)
    return qrels

def calculate_recall(qrels, results_dict, k_values):
    scores = {k: 0.0 for k in k_values}
    valid_queries = 0

    for qid, retrieved in results_dict.items():
        gold = qrels.get(qid, set())
        if not gold: continue
        valid_queries += 1
        for k in k_values:
            top_k = set(retrieved[:k])
            scores[k] += len(top_k.intersection(gold)) / len(gold)

    if valid_queries == 0: return {f"Recall@{k}": 0.0 for k in k_values}
    return {f"Recall@{k}": round(scores[k] / valid_queries, 5) for k in k_values}

# ==============================================================================
# 3. ΜΑΖΙΚΗ ΑΝΑΛΥΣΗ
# ==============================================================================
all_rows = []

for corpus in CORPORA:
    print(f"\n🔍 Ανάλυση {corpus.upper()}...")
    qrels = load_qrels(corpus)
    if not qrels:
        print(f"  ⚠️ Δεν βρέθηκαν qrels (ground truth) για το {corpus}")
        continue

    strat_finds_ranks = {}
    strat_metrics = {}

    for strat_name, fpath in FILE_MAP[corpus].items():
        res_list = {}
        found_with_ranks = {}

        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    item = json.loads(line)
                    qid = str(item["task_id"])
                    if qid not in qrels: continue

                    retrieved = [str(c["document_id"]) for c in item.get("contexts", [])]
                    res_list[qid] = retrieved

                    gold_ids = qrels[qid]
                    for rank, did in enumerate(retrieved, 1):
                        if did in gold_ids:
                            found_with_ranks[(qid, did)] = rank
                except: continue

        if res_list:
            strat_finds_ranks[strat_name] = found_with_ranks
            strat_metrics[strat_name] = calculate_recall(qrels, res_list, K_VALUES)
            print(f"  📊 {strat_name}: {len(res_list)} queries")

    # Unique Finds Analysis (Ποια στρατηγική βρήκε έγγραφα που δεν βρήκαν οι άλλες)
    for name in strat_finds_ranks.keys():
        found_ranks = strat_finds_ranks[name]
        others_found = set().union(*(d.keys() for k, d in strat_finds_ranks.items() if k != name))
        unique_pairs = set(found_ranks.keys()) - others_found

        all_ranks = list(found_ranks.values())
        unique_ranks = [found_ranks[p] for p in unique_pairs]

        row = {
            "Corpus": corpus,
            "Strategy": name,
            "UniqueFinds": len(unique_pairs),
            "MeanRank": round(np.mean(all_ranks), 2) if all_ranks else 0,
            "MeanUniqueRank": round(np.mean(unique_ranks), 2) if unique_ranks else 0
        }
        row.update(strat_metrics[name])
        all_rows.append(row)

# ==============================================================================
# 4. EXPORT & DISPLAY
# ==============================================================================
if all_rows:
    df_final = pd.DataFrame(all_rows)
    # Ταξινόμηση ανά Corpus και Recall@5
    df_final = df_final.sort_values(["Corpus", "Recall@5"], ascending=[True, False])

    display(df_final)

    output_path = os.path.join(RESULTS_DIR, "detailed_metrics_comparison_cw0.9.csv")
    df_final.to_csv(output_path, index=False)
    print(f"\n✅ Επιτυχία! Η αναλυτική σύγκριση αποθηκεύτηκε: {output_path}")
else:
    print("\n❌ Σφάλμα: Δεν βρέθηκαν δεδομένα για επεξεργασία.")

## RRF results

In [ ]:
import json
import os
from collections import defaultdict
import pandas as pd

# ==============================================================================
# RRF SETTINGS
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
RESULTS_DIR = os.path.join(BASE_PATH, "cohere_v4_rerank_results", "20260116_084940")
OUTPUT_DIR = os.path.join(BASE_PATH, "bm25_rrf_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

K_RRF = 20  # RRF constant

# Manual weights per strategy (adjust as needed)
STRATEGY_WEIGHTS = {
    "A_corpus_based_prf_llm__hist_last6": 0.2,
    "B_organizers_prf_llm__hist_last6": 0.6,
    "C_hyde_prf_llm__hist_last6": 2.0
}

CORPORA = ["clapnq", "fiqa", "govt", "cloud"]
STRATEGIES = [
    "A_corpus_based_prf_llm__hist_last6",
    "B_organizers_prf_llm__hist_last6",
    "C_hyde_prf_llm__hist_last6"
]

# ==============================================================================
# BUILD FILE MAP DYNAMICALLY
# ==============================================================================
FILE_MAP = {}
print("📂 Building file map from RESULTS_DIR...")
for corpus in CORPORA:
    FILE_MAP[corpus] = {}
    for strategy in STRATEGIES:
        file_name = f"{corpus}_{strategy}_HYBRID_cw0.9.jsonl"
        full_path = os.path.join(RESULTS_DIR, file_name)
        if os.path.exists(full_path):
            FILE_MAP[corpus][strategy] = full_path
            print(f"  ✅ Found: {file_name}")
        else:
            print(f"  ⚠️  Missing: {file_name}")


# ==============================================================================
# RRF FUSION FUNCTION
# ==============================================================================
def rrf_fusion(strategy_results, k=20, weights=None):
    if weights is None:
        weights = {s: 1.0 for s in strategy_results.keys()}

    all_qids = set()
    for strat_res in strategy_results.values():
        all_qids.update(strat_res.keys())

    fused = {}

    for qid in all_qids:
        doc_scores = defaultdict(float)

        for strat_name, qid_results in strategy_results.items():
            if qid not in qid_results:
                continue

            weight = weights.get(strat_name, 1.0)
            ranked_docs = qid_results[qid]

            for rank, doc_id in enumerate(ranked_docs, start=1):
                rrf_score = weight / (k + rank)
                doc_scores[doc_id] += rrf_score

        sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
        fused[qid] = sorted_docs

    return fused


# ==============================================================================
# LOAD STRATEGY RESULTS
# ==============================================================================
def load_strategy_results(corpus):
    strategy_results = {}
    query_metadata = {}

    for strat_name, fpath in FILE_MAP[corpus].items():
        if not os.path.exists(fpath):
            print(f"  ⚠️  File not found: {fpath}")
            continue

        strat_res = {}

        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    item = json.loads(line)
                    qid = str(item["task_id"])

                    if qid not in query_metadata:
                        query_metadata[qid] = {
                            "query": item.get("query", ""),
                            "Collection": item.get("Collection", corpus)
                        }

                    contexts = item.get("contexts", [])
                    ranked_docs = [str(c["document_id"]) for c in contexts]
                    strat_res[qid] = ranked_docs
                except:
                    continue

        strategy_results[strat_name] = strat_res
        print(f"  ✅ Loaded {strat_name}: {len(strat_res)} queries")

    return strategy_results, query_metadata


# ==============================================================================
# SAVE RRF RESULTS
# ==============================================================================
def save_rrf_results(corpus, fused_results, query_metadata, output_dir):
    output_path = os.path.join(output_dir, f"bm25_{corpus}_rrf.jsonl")

    with open(output_path, 'w', encoding='utf-8') as f:
        for qid, ranked_docs in fused_results.items():
            metadata = query_metadata.get(qid, {})

            contexts = []
            for doc_id, rrf_score in ranked_docs:
                contexts.append({
                    "document_id": doc_id,
                    "score": round(rrf_score, 6)  # ✅ "score" for evaluation script
                })

            output_item = {
                "task_id": qid,
                "query": metadata.get("query", ""),
                "Collection": metadata.get("Collection", corpus),
                "contexts": contexts
            }

            f.write(json.dumps(output_item, ensure_ascii=False) + "\n")

    print(f"  💾 Saved: {output_path} ({len(fused_results)} queries)")
    return output_path


# ==============================================================================
# MAIN EXECUTION - RRF FUSION
# ==============================================================================
print("\n" + "=" * 70)
print("🔀 RECIPROCAL RANK FUSION (RRF) - BM25 RESULTS (cw=0.9)")
print("=" * 70)
print(f"\nRRF constant k = {K_RRF}")
print(f"Strategy weights: {STRATEGY_WEIGHTS}\n")

saved_files = []

for corpus in CORPORA:
    print(f"\n{'='*70}")
    print(f"📊 Processing: {corpus.upper()}")
    print(f"{'='*70}")

    if corpus not in FILE_MAP or not FILE_MAP[corpus]:
        print(f"  ⚠️  No files found for {corpus}, skipping...")
        continue

    strategy_results, query_metadata = load_strategy_results(corpus)

    if not strategy_results:
        print(f"  ⚠️  No results loaded for {corpus}, skipping...")
        continue

    print(f"\n  🔀 Fusing {len(strategy_results)} strategies...")
    fused_results = rrf_fusion(strategy_results, k=K_RRF, weights=STRATEGY_WEIGHTS)
    print(f"  ✅ Fused {len(fused_results)} queries")

    output_path = save_rrf_results(corpus, fused_results, query_metadata, OUTPUT_DIR)
    saved_files.append(output_path)

print(f"\n{'='*70}")
print("✅ RRF FUSION COMPLETE!")
print(f"{'='*70}")
print(f"\nSaved {len(saved_files)} files to: {OUTPUT_DIR}")
for fpath in saved_files:
    print(f"  📄 {os.path.basename(fpath)}")


# ==============================================================================
# EVALUATION OF RRF RESULTS
# ==============================================================================
print(f"\n\n{'='*70}")
print("📈 RUNNING EVALUATION ON RRF RESULTS")
print(f"{'='*70}\n")

evaluation_script = "/content/drive/MyDrive/Project/run_retrieval_eval.py"

if not os.path.exists(evaluation_script):
    print(f"⚠️ Evaluation script not found: {evaluation_script}")
    print("Skipping evaluation...")
else:
    for corpus in CORPORA:
        input_path = os.path.join(OUTPUT_DIR, f"bm25_{corpus}_rrf.jsonl")
        output_path = os.path.join(OUTPUT_DIR, f"bm25_{corpus}_rrf_results.jsonl")

        if not os.path.exists(input_path):
            print(f"⚠️ Skipping {corpus} - file not found: {input_path}")
            continue

        print(f"\n{'='*50}")
        print(f"Running evaluation for {corpus} ...")
        print(f"Input: {input_path}")
        print(f"Output: {output_path}")
        print(f"{'='*50}")

        cmd = f"python {evaluation_script} --input_file {input_path} --output_file {output_path}"
        result = os.system(cmd)

        if result != 0:
            print(f"⚠️ Warning: Evaluation failed for {corpus}")
        else:
            print(f"✅ Evaluation completed for {corpus}")


# ==============================================================================
# AGGREGATE RESULTS COMPARISON
# ==============================================================================
print(f"\n\n{'='*70}")
print("📊 AGGREGATING EVALUATION RESULTS")
print(f"{'='*70}\n")

csv_files = [
    f"{OUTPUT_DIR}/bm25_clapnq_rrf_results_aggregate.csv",
    f"{OUTPUT_DIR}/bm25_fiqa_rrf_results_aggregate.csv",
    f"{OUTPUT_DIR}/bm25_govt_rrf_results_aggregate.csv",
    f"{OUTPUT_DIR}/bm25_cloud_rrf_results_aggregate.csv"
]

all_data = []

for file in csv_files:
    if os.path.exists(file):
        df = pd.read_csv(file)
        df = df[df["collection"] != "all"]  # Remove aggregate row if present
        corpus_name = file.split("/")[-1].replace("bm25_", "").replace("_rrf_results_aggregate.csv", "")
        df["Corpus"] = corpus_name
        all_data.append(df)
        print(f"✅ Loaded: {corpus_name}")
    else:
        print(f"⚠️ Not found: {file}")

if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df = final_df.set_index("Corpus")
    print("\n" + "="*70)
    print("📊 BM25 RRF EVALUATION RESULTS - ALL CORPORA (cw=0.9)")
    print("="*70)
    display(final_df)

    # Save combined results
    combined_output = f"{OUTPUT_DIR}/bm25_rrf_all_corpora_results.csv"
    final_df.to_csv(combined_output)
    print(f"\n💾 Combined results saved to: {combined_output}")
else:
    print("\n⚠️ No evaluation results found")

print("\n" + "="*70)
print("✅ ALL PROCESSING COMPLETE!")
print("="*70)


# **Splade Pipeline**

In [ ]:
# ==============================================================================
# SPLADE DETAILED ANALYSIS
# Recall@k, Unique Finds, Mean Ranks per Strategy
# ==============================================================================

import json
import csv
import os
import glob
import re
import pandas as pd
import numpy as np
from collections import defaultdict

# ==============================================================================
# 1. ΡΥΘΜΙΣΕΙΣ
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
SPLADE_DIR = f"{BASE_PATH}/splade_new_rewrites"

K_VALUES = [5, 10, 20, 50, 100]

# ==============================================================================
# 2. AUTO-DETECT SPLADE FILES
# ==============================================================================
print("="*90)
print("📂 SPLADE DETAILED ANALYSIS")
print("="*90)
print(f"Scanning: {SPLADE_DIR}\n")

# Find all SPLADE prediction files
splade_files = glob.glob(f"{SPLADE_DIR}/*.jsonl")
splade_files = [
    f for f in splade_files
    if not any(x in os.path.basename(f) for x in ["_eval", "_results", "rewrites"])
]

print(f"✅ Found {len(splade_files)} SPLADE prediction files\n")

# Organize files by corpus and strategy
FILE_MAP = defaultdict(dict)

for fpath in splade_files:
    basename = os.path.basename(fpath).replace(".jsonl", "")

    # Parse: {corpus}_splade_{strategy}_{timestamp}
    parts = basename.split("_splade_")

    if len(parts) != 2:
        print(f"⚠️  Cannot parse: {basename}")
        continue

    corpus = parts[0]
    strategy_timestamp = parts[1]

    # Remove timestamp
    strategy = re.sub(r'_\d{8}_\d{6}$', '', strategy_timestamp)

    FILE_MAP[corpus][strategy] = fpath

# Display detected files
print("📋 Detected files by corpus and strategy:\n")
for corpus in sorted(FILE_MAP.keys()):
    print(f"  {corpus.upper()}:")
    for strategy in sorted(FILE_MAP[corpus].keys()):
        print(f"    • {strategy}")
print()

# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================
def load_qrels(corpus):
    """Load ground truth relevance judgments"""
    path = os.path.join(BASE_PATH, "retrieval", corpus, "dev.tsv")
    qrels = {}

    if not os.path.exists(path):
        print(f"⚠️  Warning: qrels not found for {corpus} at {path}")
        return qrels

    with open(path, 'r', encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        next(reader)  # Skip header
        for row in reader:
            qid, docid, score = row[0], row[1], int(row[2])
            if score > 0:
                if qid not in qrels:
                    qrels[qid] = set()
                qrels[qid].add(docid)

    return qrels


def calculate_recall(qrels, results_dict, k_values):
    """Calculate Recall@k for multiple k values"""
    scores = {k: 0.0 for k in k_values}
    num_queries = len(results_dict)

    if num_queries == 0:
        return {f"Recall@{k}": 0.0 for k in k_values}

    for qid, retrieved in results_dict.items():
        gold = qrels.get(qid, set())
        if not gold:
            continue

        for k in k_values:
            top_k = set(retrieved[:k])
            scores[k] += len(top_k.intersection(gold)) / len(gold)

    return {f"Recall@{k}": round(scores[k] / num_queries, 5) for k in k_values}


# ==============================================================================
# 4. MAIN ANALYSIS WITH RANKS
# ==============================================================================
all_rows = []

for corpus in sorted(FILE_MAP.keys()):
    print(f"Processing {corpus.upper()}...")

    # Load qrels
    qrels = load_qrels(corpus)

    if not qrels:
        print(f"  ⚠️  Skipping {corpus} - no qrels\n")
        continue

    # Load corpus texts (for debugging)
    texts = {}
    corpus_file = os.path.join(BASE_PATH, "corpora", f"{corpus}.jsonl")
    if os.path.exists(corpus_file):
        with open(corpus_file, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    j = json.loads(line)
                    doc_id = j.get('_id') or j.get('id') or j.get('document_id')
                    text = j.get('text', '')
                    if doc_id:
                        texts[doc_id] = text
                except:
                    continue

    # Store ranks for each strategy
    strat_finds_ranks = {}  # strategy -> {(qid, did): rank}
    strat_metrics = {}

    # Process each strategy
    for strat_name, fpath in FILE_MAP[corpus].items():
        if not os.path.exists(fpath):
            continue

        res_list = {}
        found_with_ranks = {}  # (qid, did) -> rank

        # Load predictions
        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    item = json.loads(line)
                    qid = item["task_id"]

                    if qid not in qrels:
                        continue

                    # Get retrieved document IDs
                    retrieved = [c["document_id"] for c in item.get("contexts", [])]
                    res_list[qid] = retrieved

                    # Track ranks for gold documents found in top-100
                    gold_ids = qrels[qid]
                    for rank, did in enumerate(retrieved[:100], 1):
                        if did in gold_ids:
                            found_with_ranks[(qid, did)] = rank

                except:
                    continue

        # Store results
        strat_finds_ranks[strat_name] = found_with_ranks
        strat_metrics[strat_name] = calculate_recall(qrels, res_list, K_VALUES)

        print(f"  ✅ {strat_name}: {len(res_list)} queries")

    # Unique Finds & Mean Ranks Analysis
    for name, found_ranks in strat_finds_ranks.items():
        # Find unique pairs for this strategy
        others_found = set().union(
            *(d.keys() for k, d in strat_finds_ranks.items() if k != name)
        )
        unique_pairs = set(found_ranks.keys()) - others_found

        # Calculate mean ranks
        all_ranks = list(found_ranks.values())
        unique_ranks = [found_ranks[p] for p in unique_pairs]

        mean_rank = round(np.mean(all_ranks), 2) if all_ranks else 0
        mean_unique_rank = round(np.mean(unique_ranks), 2) if unique_ranks else 0

        # Build row
        row = {
            "Corpus": corpus,
            "Strategy": name,
            "UniqueFinds": len(unique_pairs),
            "MeanRank": mean_rank,
            "MeanUniqueRank": mean_unique_rank
        }
        row.update(strat_metrics[name])
        all_rows.append(row)

    print()

# ==============================================================================
# 5. CREATE DATAFRAME & DISPLAY
# ==============================================================================
if not all_rows:
    print("❌ No results to display!")
else:
    df_final = pd.DataFrame(all_rows)

    # Column order
    cols = (
        ["Corpus", "Strategy"] +
        [f"Recall@{k}" for k in K_VALUES] +
        ["UniqueFinds", "MeanRank", "MeanUniqueRank"]
    )

    df_final = df_final[cols]
    df_final = df_final.sort_values(["Corpus", "Strategy"])

    # Display
    print("="*90)
    print("📊 SPLADE DETAILED RESULTS")
    print("="*90 + "\n")

    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.float_format', '{:.5f}'.format)

    display(df_final)

    # Save
    output_csv = f"{SPLADE_DIR}/SPLADE_detailed_retrieval_comparison.csv"
    df_final.to_csv(output_csv, index=False)

    print(f"\n💾 Saved to: {output_csv}")

    # ==============================================================================
    # 6. SUMMARY STATISTICS
    # ==============================================================================
    print("\n" + "="*90)
    print("📊 SUMMARY BY STRATEGY")
    print("="*90 + "\n")

    metric_cols = [f"Recall@{k}" for k in K_VALUES] + ["UniqueFinds", "MeanRank", "MeanUniqueRank"]
    strategy_avg = df_final.groupby("Strategy")[metric_cols].mean().sort_index()

    display(strategy_avg)

    # Save
    strategy_csv = f"{SPLADE_DIR}/SPLADE_strategy_summary.csv"
    strategy_avg.to_csv(strategy_csv)
    print(f"\n💾 Saved to: {strategy_csv}")

    print("\n" + "="*90)
    print("📊 SUMMARY BY CORPUS")
    print("="*90 + "\n")

    corpus_avg = df_final.groupby("Corpus")[metric_cols].mean().sort_index()

    display(corpus_avg)

    # Save
    corpus_csv = f"{SPLADE_DIR}/SPLADE_corpus_summary.csv"
    corpus_avg.to_csv(corpus_csv)
    print(f"\n💾 Saved to: {corpus_csv}")

    # ==============================================================================
    # 7. BEST PERFORMERS
    # ==============================================================================
    print("\n" + "="*90)
    print("🏆 BEST STRATEGY PER METRIC")
    print("="*90 + "\n")

    for metric in [f"Recall@{k}" for k in K_VALUES]:
        if metric not in strategy_avg.columns:
            continue

        best_strategy = strategy_avg[metric].idxmax()
        best_val = strategy_avg[metric].max()
        worst_strategy = strategy_avg[metric].idxmin()
        worst_val = strategy_avg[metric].min()

        improvement = ((best_val / worst_val) - 1) * 100 if worst_val > 0 else 0

        print(f"{metric}:")
        print(f"  🥇 Best:  {best_strategy:<25} {best_val:.5f}")
        print(f"  🥉 Worst: {worst_strategy:<25} {worst_val:.5f}")
        print(f"  📊 Gain:  {improvement:+.1f}%\n")

    # Unique Finds
    print("UniqueFinds:")
    best_unique = strategy_avg["UniqueFinds"].idxmax()
    print(f"  🥇 Most unique: {best_unique:<25} {strategy_avg.loc[best_unique, 'UniqueFinds']:.0f} finds\n")

    # Mean Rank
    print("MeanRank (lower is better):")
    best_rank = strategy_avg["MeanRank"].idxmin()
    print(f"  🥇 Best rank: {best_rank:<25} {strategy_avg.loc[best_rank, 'MeanRank']:.2f}\n")

    # ==============================================================================
    # 8. PER-CORPUS BREAKDOWN
    # ==============================================================================
    print("="*90)
    print("📊 DETAILED BREAKDOWN BY CORPUS")
    print("="*90 + "\n")

    for corpus in sorted(df_final["Corpus"].unique()):
        print(f"🎯 {corpus.upper()}")
        print("-" * 90)

        corpus_df = df_final[df_final["Corpus"] == corpus]

        # Display key columns
        display_cols = ["Strategy", "Recall@10", "Recall@100", "UniqueFinds", "MeanRank"]
        display_df = corpus_df[display_cols].set_index("Strategy").sort_index()

        display(display_df)
        print()

    print("="*90)
    print("✅ ANALYSIS COMPLETE")
    print("="*90)

print("\n📁 Output files:")
print(f"  • {output_csv}")
print(f"  • {strategy_csv}")
print(f"  • {corpus_csv}")


## Reranking per rewriter

In [ ]:
import os, json, time, requests, subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import Dict, List, Tuple


# ==============================================================================
# 1. SETTINGS & CONFIGURATION
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
BASE_DIR = "/content/drive/MyDrive/Project"


EVAL_SCRIPT_PATH = os.path.join(BASE_PATH, "run_retrieval_eval.py")


# Azure Cohere Credentials
AZURE_COHERE_ENDPOINT = os.environ["ENDPOINT"]
AZURE_COHERE_KEY = os.environ["_KEY"]


RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = os.path.join(BASE_PATH, "splade_cohere_v4_rerank_results", RUN_TAG)
os.makedirs(OUTPUT_DIR, exist_ok=True)


# 🔥 NEW: SPLADE Rewrites Input Files
SPLADE_DIR = f"{BASE_DIR}/splade_new_rewrites"

INPUT_FILES = {
    "clapnq": {
        "Anchor_Keyword": f"{SPLADE_DIR}/clapnq_splade_anchor_keyword_20260115_000206.jsonl",
        "Corpus_Specific": f"{SPLADE_DIR}/clapnq_splade_corpus_specific_20260115_000206.jsonl",
        "Structural_Minimal": f"{SPLADE_DIR}/clapnq_splade_structural_minimal_20260115_000206.jsonl",
        "COT": f"{SPLADE_DIR}/clapnq_splade_cot_20260115_000206.jsonl",
        "Hyde": f"{SPLADE_DIR}/clapnq_splade_hyde_20260115_000206.jsonl",
        "Decomp": f"{SPLADE_DIR}/clapnq_splade_decomposition_20260115_000206.jsonl"
    },
    "fiqa": {
        "Anchor_Keyword": f"{SPLADE_DIR}/fiqa_splade_anchor_keyword_20260115_000206.jsonl",
        "Corpus_Specific": f"{SPLADE_DIR}/fiqa_splade_corpus_specific_20260115_000206.jsonl",
        "Structural_Minimal": f"{SPLADE_DIR}/fiqa_splade_structural_minimal_20260115_000206.jsonl",
        "COT": f"{SPLADE_DIR}/fiqa_splade_cot_20260115_000206.jsonl",
        "Hyde": f"{SPLADE_DIR}/fiqa_splade_hyde_20260115_000206.jsonl",
        "Decomp": f"{SPLADE_DIR}/fiqa_splade_decomposition_20260115_000206.jsonl"
    },
    "govt": {
        "Anchor_Keyword": f"{SPLADE_DIR}/govt_splade_anchor_keyword_20260115_000206.jsonl",
        "Corpus_Specific": f"{SPLADE_DIR}/govt_splade_corpus_specific_20260115_000206.jsonl",
        "Structural_Minimal": f"{SPLADE_DIR}/govt_splade_structural_minimal_20260115_000206.jsonl",
        "COT": f"{SPLADE_DIR}/govt_splade_cot_20260115_000206.jsonl",
        "Hyde": f"{SPLADE_DIR}/govt_splade_hyde_20260115_000206.jsonl",
        "Decomp": f"{SPLADE_DIR}/govt_splade_decomposition_20260115_000206.jsonl"
    },
    "cloud": {
        "Anchor_Keyword": f"{SPLADE_DIR}/cloud_splade_anchor_keyword_20260115_000206.jsonl",
        "Corpus_Specific": f"{SPLADE_DIR}/cloud_splade_corpus_specific_20260115_000206.jsonl",
        "Structural_Minimal": f"{SPLADE_DIR}/cloud_splade_structural_minimal_20260115_000206.jsonl",
        "COT": f"{SPLADE_DIR}/cloud_splade_cot_20260115_000206.jsonl",
        "Hyde": f"{SPLADE_DIR}/cloud_splade_hyde_20260115_000206.jsonl",
        "Decomp": f"{SPLADE_DIR}/cloud_splade_decomposition_20260115_001028.jsonl"  # Different timestamp
    }
}


# Corpus-specific files (unchanged)
CORPUS_CONFIG = {
    "clapnq": {
        "corpus_file": f"{BASE_PATH}/corpora/clapnq.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/clapnq/clapnq_lastturn.jsonl"
    },
    "fiqa": {
        "corpus_file": f"{BASE_PATH}/corpora/fiqa.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/fiqa/fiqa_lastturn.jsonl"
    },
    "govt": {
        "corpus_file": f"{BASE_PATH}/corpora/govt.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/govt/govt_lastturn.jsonl"
    },
    "cloud": {
        "corpus_file": f"{BASE_PATH}/corpora/cloud.jsonl",
        "queries_file": f"{BASE_PATH}/retrieval/cloud/cloud_lastturn.jsonl"
    }
}


TOP_K_RERANK = 100
COHERE_WEIGHTS = [0.1, 0.3, 0.5, 0.7, 0.9]  # Grid search weights


# ==============================================================================
# 2. HELPER FUNCTIONS (IO & PROCESSING)
# ==============================================================================


def load_jsonl(path: str) -> List[dict]:
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data


def load_corpus(path: str) -> Dict[str, str]:
    """Load corpus documents with combined title + text."""
    corpus = {}
    if not os.path.exists(path):
        print(f"⚠️  Corpus file not found: {path}")
        return corpus

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            did = str(j.get("_id") or j.get("id"))
            # Combine title and text for better reranking
            text = (j.get("title", "") + " " + (j.get("text") or j.get("contents") or "")).strip()
            corpus[did] = text[:2000]  # Token limit for cost efficiency
    return corpus


def load_queries(path: str) -> Dict[str, str]:
    """Load query texts by ID."""
    queries = {}
    if not os.path.exists(path):
        print(f"⚠️  Queries file not found: {path}")
        return queries

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            qid = str(j.get("_id") or j.get("id") or j.get("task_id"))
            queries[qid] = j.get("text") or j.get("query")
    return queries


def azure_cohere_rerank(query: str, docs_text: List[str]) -> List[float]:
    """Call Cohere v4 Reranker via Azure Endpoint."""
    if not query or not docs_text:
        return [0.0] * len(docs_text)

    headers = {"Content-Type": "application/json", "api-key": AZURE_COHERE_KEY}
    payload = {
        "model": "Cohere-rerank-v4.0-pro",
        "query": query,
        "documents": docs_text,
        "top_n": len(docs_text)
    }

    try:
        response = requests.post(AZURE_COHERE_ENDPOINT, headers=headers, json=payload, timeout=60)
        if response.status_code == 200:
            results = response.json()["results"]
            scores = [0.0] * len(docs_text)
            for r in results:
                scores[r["index"]] = float(r["relevance_score"])
            return scores
        else:
            print(f"⚠️  API Error: {response.status_code} - {response.text[:200]}")
    except Exception as e:
        print(f"❌ Connection Error: {e}")

    return [0.0] * len(docs_text)


def minmax_norm(scores: List[float]) -> np.ndarray:
    """Normalize scores to [0, 1] range."""
    arr = np.array(scores, dtype=np.float32)
    if arr.max() == arr.min():
        return np.zeros_like(arr)
    return (arr - arr.min()) / (arr.max() - arr.min())


def run_eval(input_jsonl: str) -> Tuple[float, float]:
    """Run evaluation script and return Recall@5 and nDCG@5."""
    result_file = input_jsonl.replace(".jsonl", "_results.jsonl")
    csv_file = result_file.replace(".jsonl", "_aggregate.csv")

    cmd = f"python3 {EVAL_SCRIPT_PATH} --input_file {input_jsonl} --output_file {result_file}"
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        if "collection" in df.columns:
            df = df[df["collection"] != "all"]
        if df.empty:
            return 0.0, 0.0
        row = df.iloc[0]

        # Parse metric (handle string list format)
        def parse_metric(val):
            if isinstance(val, str) and "[" in val:
                import ast
                return ast.literal_eval(val)[2]  # Index 2 is usually @5
            return val

        return parse_metric(row.get("Recall", 0.0)), parse_metric(row.get("nDCG", 0.0))

    return 0.0, 0.0


# ==============================================================================
# 3. MAIN RERANKING & FUSION PIPELINE
# ==============================================================================


print("="*80)
print("🚀 COHERE V4 RERANKING - SPLADE REWRITES (All 4 Corpora)")
print("="*80)
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Run Tag: {RUN_TAG}\n")


performance_summary = []


for corpus_name, files in INPUT_FILES.items():
    print(f"\n{'='*80}")
    print(f"📚 CORPUS: {corpus_name.upper()}")
    print(f"{'='*80}")

    # Load corpus-specific data
    corpus_cfg = CORPUS_CONFIG[corpus_name]
    corpus_map = load_corpus(corpus_cfg["corpus_file"])
    query_map = load_queries(corpus_cfg["queries_file"])

    if not corpus_map or not query_map:
        print(f"⚠️  Skipping {corpus_name} - missing corpus or queries\n")
        continue

    print(f"✅ Loaded {len(corpus_map)} docs and {len(query_map)} queries")

    for strategy_name, file_path in files.items():
        if not os.path.exists(file_path):
            print(f"⚠️  File not found: {file_path}")
            continue

        print(f"\n🎯 Processing: {strategy_name}")
        preds = load_jsonl(file_path)

        # --- STEP 1: Compute Cohere Scores (once per query) ---
        cache = []
        for p in tqdm(preds, desc=f"{corpus_name}/{strategy_name} Rerank", leave=False):
            qid = str(p["task_id"])
            q_txt = query_map.get(qid, "")

            top_ctxs = p["contexts"][:TOP_K_RERANK]
            rest_ctxs = p["contexts"][TOP_K_RERANK:]

            # Get document texts from corpus
            doc_texts = [corpus_map.get(str(c["document_id"]), "N/A") for c in top_ctxs]

            # Call Azure Cohere
            coh_scores = azure_cohere_rerank(q_txt, doc_texts)
            orig_scores = [float(c["score"]) for c in top_ctxs]

            cache.append({
                "task_id": qid,
                "Collection": p.get("Collection"),
                "top_ctxs": top_ctxs,
                "rest": rest_ctxs,
                "orig_scores": orig_scores,
                "coh_scores": coh_scores
            })

        # --- STEP 2: Evaluate Cohere-Only (cw=1.0) ---
        cohere_only_file = os.path.join(OUTPUT_DIR, f"{corpus_name}_{strategy_name}_COHERE_ONLY.jsonl")
        with open(cohere_only_file, "w", encoding="utf-8") as f:
            for item in cache:
                new_ctxs = []
                for i, ctx in enumerate(item["top_ctxs"]):
                    nc = ctx.copy()
                    nc["score"] = float(item["coh_scores"][i])
                    new_ctxs.append(nc)
                new_ctxs.sort(key=lambda x: x["score"], reverse=True)
                f.write(json.dumps({
                    "task_id": item["task_id"],
                    "Collection": item["Collection"],
                    "contexts": new_ctxs + item["rest"]
                }) + "\n")

        r5, n5 = run_eval(cohere_only_file)
        performance_summary.append({
            "Corpus": corpus_name,
            "Strategy": strategy_name,
            "Method": "Cohere-Only",
            "Recall@5": r5,
            "nDCG@5": n5
        })
        print(f"  ✅ Cohere-Only: Recall@5={r5:.4f}, nDCG@5={n5:.4f}")

        # --- STEP 3: Grid Search Hybrid Fusion ---
        best_recall = r5
        best_ndcg = n5
        best_weight = 1.0

        for cw in COHERE_WEIGHTS:
            ow = 1.0 - cw
            hybrid_file = os.path.join(OUTPUT_DIR, f"{corpus_name}_{strategy_name}_HYBRID_cw{cw}.jsonl")

            with open(hybrid_file, "w", encoding="utf-8") as f:
                for item in cache:
                    n_orig = minmax_norm(item["orig_scores"])
                    n_coh = minmax_norm(item["coh_scores"])
                    # Hybrid scoring formula
                    combined = (ow * n_orig) + (cw * n_coh)

                    new_ctxs = []
                    for i, ctx in enumerate(item["top_ctxs"]):
                        nc = ctx.copy()
                        nc["score"] = float(combined[i])
                        new_ctxs.append(nc)
                    new_ctxs.sort(key=lambda x: x["score"], reverse=True)
                    f.write(json.dumps({
                        "task_id": item["task_id"],
                        "Collection": item["Collection"],
                        "contexts": new_ctxs + item["rest"]
                    }) + "\n")

            r5, n5 = run_eval(hybrid_file)
            performance_summary.append({
                "Corpus": corpus_name,
                "Strategy": strategy_name,
                "Method": f"Hybrid (cw={cw})",
                "Recall@5": r5,
                "nDCG@5": n5
            })

            if n5 > best_ndcg:
                best_ndcg = n5
                best_recall = r5
                best_weight = cw

        print(f"  🏆 Best Hybrid: cw={best_weight}, Recall@5={best_recall:.4f}, nDCG@5={best_ndcg:.4f}")


# ==============================================================================
# 4. FINAL RESULTS DISPLAY
# ==============================================================================
df_final = pd.DataFrame(performance_summary)


print("\n" + "="*80)
print("🏆 FINAL RERANKING & HYBRID FUSION RESULTS - SPLADE")
print("="*80 + "\n")


# Display sorted by corpus, strategy, and nDCG
display(df_final.sort_values(["Corpus", "Strategy", "nDCG@5"], ascending=[True, True, False]))


# Save detailed report
report_file = os.path.join(OUTPUT_DIR, "splade_reranking_final_report.csv")
df_final.to_csv(report_file, index=False)
print(f"\n💾 Saved detailed report: {report_file}")


# Summary statistics by corpus
print("\n" + "="*80)
print("📊 SUMMARY BY CORPUS")
print("="*80 + "\n")
summary_by_corpus = df_final.groupby("Corpus")[["Recall@5", "nDCG@5"]].agg(["mean", "max"])
display(summary_by_corpus)


# Best method per corpus
print("\n" + "="*80)
print("🥇 BEST METHOD PER CORPUS")
print("="*80 + "\n")
for corpus in df_final["Corpus"].unique():
    corpus_df = df_final[df_final["Corpus"] == corpus]
    best_row = corpus_df.loc[corpus_df["nDCG@5"].idxmax()]
    print(f"{corpus.upper()}:")
    print(f"  Strategy: {best_row['Strategy']}")
    print(f"  Method: {best_row['Method']}")
    print(f"  Recall@5: {best_row['Recall@5']:.4f}")
    print(f"  nDCG@5: {best_row['nDCG@5']:.4f}\n")


print("="*80)
print("✅ SPLADE RERANKING COMPLETE!")
print(f"📁 All results saved to: {OUTPUT_DIR}")
print("="*80)


## Retrieval results and reranked results

In [ ]:
import json
import csv
import os
import pandas as pd
import numpy as np
from collections import defaultdict

# ==============================================================================
# 1. ΡΥΘΜΙΣΕΙΣ & FILE MAP (20260115_184516)
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
# Ο φάκελος των τελευταίων αποτελεσμάτων
RESULTS_DIR = f"{BASE_PATH}/splade_cohere_v4_rerank_results/20260115_184516"

FILE_MAP = {
    "clapnq": {
        "Corpus_Specific": f"{RESULTS_DIR}/clapnq_Corpus_Specific_HYBRID_cw0.5.jsonl",
        "Structural_Minimal": f"{RESULTS_DIR}/clapnq_Structural_Minimal_HYBRID_cw0.5.jsonl",
        "COT": f"{RESULTS_DIR}/clapnq_COT_HYBRID_cw0.5.jsonl",
        "Hyde": f"{RESULTS_DIR}/clapnq_Hyde_HYBRID_cw0.5.jsonl",
    },
    "fiqa": {
        "Corpus_Specific": f"{RESULTS_DIR}/fiqa_Corpus_Specific_HYBRID_cw0.5.jsonl",
        "Structural_Minimal": f"{RESULTS_DIR}/fiqa_Structural_Minimal_HYBRID_cw0.5.jsonl",
        "COT": f"{RESULTS_DIR}/fiqa_COT_HYBRID_cw0.5.jsonl",
        "Hyde": f"{RESULTS_DIR}/fiqa_Hyde_HYBRID_cw0.5.jsonl",
    },
    "govt": {
        "Anchor_Keyword": f"{RESULTS_DIR}/govt_Anchor_Keyword_HYBRID_cw0.5.jsonl",
        "Corpus_Specific": f"{RESULTS_DIR}/govt_Corpus_Specific_HYBRID_cw0.5.jsonl",
        "Structural_Minimal": f"{RESULTS_DIR}/govt_Structural_Minimal_HYBRID_cw0.5.jsonl",
        "Hyde": f"{RESULTS_DIR}/govt_Hyde_HYBRID_cw0.5.jsonl",
    },
    "cloud": {
        "Corpus_Specific": f"{RESULTS_DIR}/cloud_Corpus_Specific_HYBRID_cw0.5.jsonl",
        "Structural_Minimal": f"{RESULTS_DIR}/cloud_Structural_Minimal_HYBRID_cw0.5.jsonl",
        "Hyde": f"{RESULTS_DIR}/cloud_Hyde_HYBRID_cw0.5.jsonl",
        "Decomp": f"{RESULTS_DIR}/cloud_Decomp_HYBRID_cw0.5.jsonl"
    }
}

K_VALUES = [5, 10, 20, 50, 100]

# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================
def load_qrels(corpus):
    path = os.path.join(BASE_PATH, "retrieval", corpus, "dev.tsv")
    qrels = {}
    if not os.path.exists(path):
        print(f"⚠️  QRELS NOT FOUND: {path}")
        return qrels

    with open(path, 'r', encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        try:
            next(reader) # skip header
        except StopIteration:
            return qrels
        for row in reader:
            if len(row) < 3: continue
            qid, docid, score = row[0], row[1], int(row[2])
            if score > 0:
                if qid not in qrels: qrels[qid] = set()
                qrels[qid].add(docid)
    print(f"  ✅ Loaded {len(qrels)} queries with gold docs for {corpus}")
    return qrels

def calculate_recall(qrels, results_dict, k_values, corpus_name="", strat_name=""):
    scores = {k: 0.0 for k in k_values}
    num_queries = len(results_dict)
    if num_queries == 0: return {f"Recall@{k}": 0.0 for k in k_values}

    matched_queries = 0
    for qid, retrieved in results_dict.items():
        gold = qrels.get(qid, set())
        if not gold: continue
        matched_queries += 1
        for k in k_values:
            top_k = set(retrieved[:k])
            scores[k] += len(top_k.intersection(gold)) / len(gold)

    return {f"Recall@{k}": round(scores[k] / num_queries, 5) for k in k_values}

# ==============================================================================
# 3. ΜΑΖΙΚΗ ΑΝΑΛΥΣΗ
# ==============================================================================
all_rows = []

for corpus in ["clapnq", "fiqa", "govt", "cloud"]:
    print(f"\n🔍 Processing {corpus.upper()}...")
    qrels = load_qrels(corpus)
    if not qrels: continue

    strat_finds_ranks = {}
    strat_metrics = {}

    for strat_name, fpath in FILE_MAP[corpus].items():
        if not os.path.exists(fpath):
            print(f"  ❌ File not found: {strat_name}")
            continue

        res_list = {}
        found_with_ranks = {}

        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                item = json.loads(line)
                qid = str(item["task_id"])
                if qid not in qrels: continue

                retrieved = [str(c["document_id"]) for c in item.get("contexts", [])]
                res_list[qid] = retrieved
                gold_ids = qrels[qid]

                for rank, did in enumerate(retrieved[:100], 1):
                    if did in gold_ids:
                        found_with_ranks[(qid, did)] = rank

        strat_finds_ranks[strat_name] = found_with_ranks
        strat_metrics[strat_name] = calculate_recall(qrels, res_list, K_VALUES, corpus, strat_name)

    # Unique Analysis & Mean Ranks
    for name, found_ranks in strat_finds_ranks.items():
        others_found = set().union(*(d.keys() for k, d in strat_finds_ranks.items() if k != name))
        unique_pairs = set(found_ranks.keys()) - others_found

        all_ranks = list(found_ranks.values())
        unique_ranks = [found_ranks[p] for p in unique_pairs]

        mean_rank = round(np.mean(all_ranks), 2) if all_ranks else 0
        mean_unique_rank = round(np.mean(unique_ranks), 2) if unique_ranks else 0

        row = {
            "Corpus": corpus,
            "Strategy": name,
            "UniqueFinds": len(unique_pairs),
            "MeanRank": mean_rank,
            "MeanUniqueRank": mean_unique_rank
        }
        row.update(strat_metrics[name])
        all_rows.append(row)

# ==============================================================================
# 4. EXPORT & DISPLAY
# ==============================================================================
df_final = pd.DataFrame(all_rows)
cols = ["Corpus", "Strategy"] + [f"Recall@{k}" for k in K_VALUES] + ["UniqueFinds", "MeanRank", "MeanUniqueRank"]
df_final = df_final[cols]

display(df_final)

output_path = f"{RESULTS_DIR}/detailed_metrics_comparison_cw05.csv"
df_final.to_csv(output_path, index=False)
print(f"\n✅ Η ανάλυση ολοκληρώθηκε! Αποθηκεύτηκε στο: {output_path}")

## RRF results

In [ ]:
import json
import os
from collections import defaultdict
import pandas as pd

# ==============================================================================
# RRF SETTINGS
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
RESULTS_DIR = f"{BASE_PATH}/splade_cohere_v4_rerank_results/20260115_184516"
OUTPUT_DIR = os.path.join(BASE_PATH, "splade_rrf_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

K_RRF = 20  # RRF constant

# Manual weights per strategy (adjust as needed)
STRATEGY_WEIGHTS = {
    "Anchor_Keyword": 0.0,
    "Corpus_Specific": 1.0,
    "Structural_Minimal": 1.0,
    "COT": 0.0,
    "Hyde": 0.2,
    "Decomp": 0.0
}

# ==============================================================================
# FILE MAP (από το input σου)
# ==============================================================================
FILE_MAP = {
    "clapnq": {
        "Corpus_Specific": f"{RESULTS_DIR}/clapnq_Corpus_Specific_HYBRID_cw0.5.jsonl",
        "Structural_Minimal": f"{RESULTS_DIR}/clapnq_Structural_Minimal_HYBRID_cw0.5.jsonl",
        "COT": f"{RESULTS_DIR}/clapnq_COT_HYBRID_cw0.5.jsonl",
        "Hyde": f"{RESULTS_DIR}/clapnq_Hyde_HYBRID_cw0.5.jsonl",
    },
    "fiqa": {
        "Corpus_Specific": f"{RESULTS_DIR}/fiqa_Corpus_Specific_HYBRID_cw0.5.jsonl",
        "Structural_Minimal": f"{RESULTS_DIR}/fiqa_Structural_Minimal_HYBRID_cw0.5.jsonl",
        "COT": f"{RESULTS_DIR}/fiqa_COT_HYBRID_cw0.5.jsonl",
        "Hyde": f"{RESULTS_DIR}/fiqa_Hyde_HYBRID_cw0.5.jsonl",
    },
    "govt": {
        "Anchor_Keyword": f"{RESULTS_DIR}/govt_Anchor_Keyword_HYBRID_cw0.5.jsonl",
        "Corpus_Specific": f"{RESULTS_DIR}/govt_Corpus_Specific_HYBRID_cw0.5.jsonl",
        "Structural_Minimal": f"{RESULTS_DIR}/govt_Structural_Minimal_HYBRID_cw0.5.jsonl",
        "Hyde": f"{RESULTS_DIR}/govt_Hyde_HYBRID_cw0.5.jsonl",
    },
    "cloud": {
        "Corpus_Specific": f"{RESULTS_DIR}/cloud_Corpus_Specific_HYBRID_cw0.5.jsonl",
        "Structural_Minimal": f"{RESULTS_DIR}/cloud_Structural_Minimal_HYBRID_cw0.5.jsonl",
        "Hyde": f"{RESULTS_DIR}/cloud_Hyde_HYBRID_cw0.5.jsonl",
        "Decomp": f"{RESULTS_DIR}/cloud_Decomp_HYBRID_cw0.5.jsonl"
    }
}

CORPORA = ["clapnq", "fiqa", "govt", "cloud"]


# ==============================================================================
# RRF FUSION FUNCTION
# ==============================================================================
def rrf_fusion(strategy_results, k=20, weights=None):
    if weights is None:
        weights = {s: 1.0 for s in strategy_results.keys()}

    all_qids = set()
    for strat_res in strategy_results.values():
        all_qids.update(strat_res.keys())

    fused = {}

    for qid in all_qids:
        doc_scores = defaultdict(float)

        for strat_name, qid_results in strategy_results.items():
            if qid not in qid_results:
                continue

            weight = weights.get(strat_name, 1.0)
            ranked_docs = qid_results[qid]

            for rank, doc_id in enumerate(ranked_docs, start=1):
                rrf_score = weight / (k + rank)
                doc_scores[doc_id] += rrf_score

        sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
        fused[qid] = sorted_docs

    return fused


# ==============================================================================
# LOAD STRATEGY RESULTS
# ==============================================================================
def load_strategy_results(corpus):
    strategy_results = {}
    query_metadata = {}

    for strat_name, fpath in FILE_MAP[corpus].items():
        if not os.path.exists(fpath):
            print(f"  ⚠️  File not found: {fpath}")
            continue

        strat_res = {}

        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    item = json.loads(line)
                    qid = str(item["task_id"])

                    if qid not in query_metadata:
                        query_metadata[qid] = {
                            "query": item.get("query", ""),
                            "Collection": item.get("Collection", corpus)
                        }

                    contexts = item.get("contexts", [])
                    ranked_docs = [str(c["document_id"]) for c in contexts]
                    strat_res[qid] = ranked_docs
                except:
                    continue

        strategy_results[strat_name] = strat_res
        print(f"  ✅ Loaded {strat_name}: {len(strat_res)} queries")

    return strategy_results, query_metadata


# ==============================================================================
# SAVE RRF RESULTS
# ==============================================================================
def save_rrf_results(corpus, fused_results, query_metadata, output_dir):
    output_path = os.path.join(output_dir, f"splade_{corpus}_rrf.jsonl")

    with open(output_path, 'w', encoding='utf-8') as f:
        for qid, ranked_docs in fused_results.items():
            metadata = query_metadata.get(qid, {})

            contexts = []
            for doc_id, rrf_score in ranked_docs:
                contexts.append({
                    "document_id": doc_id,
                    "score": round(rrf_score, 6)  # ✅ "score" for evaluation script
                })

            output_item = {
                "task_id": qid,
                "query": metadata.get("query", ""),
                "Collection": metadata.get("Collection", corpus),
                "contexts": contexts
            }

            f.write(json.dumps(output_item, ensure_ascii=False) + "\n")

    print(f"  💾 Saved: {output_path} ({len(fused_results)} queries)")
    return output_path


# ==============================================================================
# MAIN EXECUTION - RRF FUSION
# ==============================================================================
print("\n" + "=" * 70)
print("🔀 RECIPROCAL RANK FUSION (RRF) - SPLADE RESULTS")
print("=" * 70)
print(f"\nRRF constant k = {K_RRF}")
print(f"Strategy weights: {STRATEGY_WEIGHTS}\n")

saved_files = []

for corpus in CORPORA:
    print(f"\n{'='*70}")
    print(f"📊 Processing: {corpus.upper()}")
    print(f"{'='*70}")

    if corpus not in FILE_MAP or not FILE_MAP[corpus]:
        print(f"  ⚠️  No files found for {corpus}, skipping...")
        continue

    strategy_results, query_metadata = load_strategy_results(corpus)

    if not strategy_results:
        print(f"  ⚠️  No results loaded for {corpus}, skipping...")
        continue

    print(f"\n  🔀 Fusing {len(strategy_results)} strategies...")
    fused_results = rrf_fusion(strategy_results, k=K_RRF, weights=STRATEGY_WEIGHTS)
    print(f"  ✅ Fused {len(fused_results)} queries")

    output_path = save_rrf_results(corpus, fused_results, query_metadata, OUTPUT_DIR)
    saved_files.append(output_path)

print(f"\n{'='*70}")
print("✅ RRF FUSION COMPLETE!")
print(f"{'='*70}")
print(f"\nSaved {len(saved_files)} files to: {OUTPUT_DIR}")
for fpath in saved_files:
    print(f"  📄 {os.path.basename(fpath)}")


# ==============================================================================
# EVALUATION OF RRF RESULTS
# ==============================================================================
print(f"\n\n{'='*70}")
print("📈 RUNNING EVALUATION ON RRF RESULTS")
print(f"{'='*70}\n")

evaluation_script = "/content/drive/MyDrive/Project/run_retrieval_eval.py"

if not os.path.exists(evaluation_script):
    print(f"⚠️ Evaluation script not found: {evaluation_script}")
    print("Skipping evaluation...")
else:
    for corpus in CORPORA:
        input_path = os.path.join(OUTPUT_DIR, f"splade_{corpus}_rrf.jsonl")
        output_path = os.path.join(OUTPUT_DIR, f"splade_{corpus}_rrf_results.jsonl")

        if not os.path.exists(input_path):
            print(f"⚠️ Skipping {corpus} - file not found: {input_path}")
            continue

        print(f"\n{'='*50}")
        print(f"Running evaluation for {corpus} ...")
        print(f"Input: {input_path}")
        print(f"Output: {output_path}")
        print(f"{'='*50}")

        cmd = f"python {evaluation_script} --input_file {input_path} --output_file {output_path}"
        result = os.system(cmd)

        if result != 0:
            print(f"⚠️ Warning: Evaluation failed for {corpus}")
        else:
            print(f"✅ Evaluation completed for {corpus}")


# ==============================================================================
# AGGREGATE RESULTS COMPARISON
# ==============================================================================
print(f"\n\n{'='*70}")
print("📊 AGGREGATING EVALUATION RESULTS")
print(f"{'='*70}\n")

csv_files = [
    f"{OUTPUT_DIR}/splade_clapnq_rrf_results_aggregate.csv",
    f"{OUTPUT_DIR}/splade_fiqa_rrf_results_aggregate.csv",
    f"{OUTPUT_DIR}/splade_govt_rrf_results_aggregate.csv",
    f"{OUTPUT_DIR}/splade_cloud_rrf_results_aggregate.csv"
]

all_data = []

for file in csv_files:
    if os.path.exists(file):
        df = pd.read_csv(file)
        df = df[df["collection"] != "all"]  # Remove aggregate row if present
        corpus_name = file.split("/")[-1].replace("splade_", "").replace("_rrf_results_aggregate.csv", "")
        df["Corpus"] = corpus_name
        all_data.append(df)
        print(f"✅ Loaded: {corpus_name}")
    else:
        print(f"⚠️ Not found: {file}")

if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df = final_df.set_index("Corpus")
    print("\n" + "="*70)
    print("📊 SPLADE RRF EVALUATION RESULTS - ALL CORPORA")
    print("="*70)
    display(final_df)

    # Save combined results
    combined_output = f"{OUTPUT_DIR}/splade_rrf_all_corpora_results.csv"
    final_df.to_csv(combined_output)
    print(f"\n💾 Combined results saved to: {combined_output}")
else:
    print("\n⚠️ No evaluation results found")

print("\n" + "="*70)
print("✅ ALL PROCESSING COMPLETE!")
print("="*70)


In [ ]:
import os

# ΔΙΟΡΘΩΣΗ: Αφαίρεση διπλασιασμού
predictions_files = [

  "IMPROVED_LLM_clapnq.jsonl",
  "IMPROVED_hybrid_0.30_clapnq.jsonl"





]

output_dir = "/content/drive/MyDrive/Project"
os.makedirs(output_dir, exist_ok=True)

for pred_file in predictions_files:
    input_path = os.path.join("/content/drive/MyDrive/Project/deepseek_experiments_improved/20260122_121745_IMPROVED_MULTIDOMAIN/clapnq", pred_file)
    output_path = os.path.join(output_dir, pred_file.replace(".jsonl", "_results.jsonl"))

    print(f"\n{'='*50}")
    print(f"Running evaluation for {pred_file} ...")
    print(f"Input: {input_path}")
    print(f"Output: {output_path}")
    print(f"{'='*50}")

    cmd = f"python /content/drive/MyDrive/Project/run_retrieval_eval.py --input_file {input_path} --output_file {output_path}"
    result = os.system(cmd)

    if result != 0:
        print(f"⚠️ Warning: Evaluation failed for {pred_file}")
    else:
        print(f"✅ Evaluation completed for {pred_file}")


# **RRF results of all retrievers**

In [ ]:
import json
import csv
import os
import pandas as pd
import numpy as np
from collections import defaultdict

# ==============================================================================
# 1. ΡΥΘΜΙΣΕΙΣ & FILE MAP (20260115_184516)
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
# Ο φάκελος των τελευταίων αποτελεσμάτων
RESULTS_DIR = f"{BASE_PATH}/splade_cohere_v4_rerank_results/20260115_184516"

FILE_MAP = {
    "clapnq": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_clapnq_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_clapnq_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_clapnq_rrf.jsonl",
        "BM25": f"{BASE_PATH}/bm25_rrf_results/bm25_clapnq_rrf.jsonl"
    },
    "fiqa": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_fiqa_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_fiqa_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_fiqa_rrf.jsonl",
        "BM25": f"{BASE_PATH}/bm25_rrf_results/bm25_fiqa_rrf.jsonl"
    },
    "govt": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_govt_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_govt_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_govt_rrf.jsonl",
        "BM25": f"{BASE_PATH}/bm25_rrf_results/bm25_govt_rrf.jsonl"
    },
    "cloud": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_cloud_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_cloud_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_cloud_rrf.jsonl",
        "BM25": f"{BASE_PATH}/bm25_rrf_results/bm25_cloud_rrf.jsonl"
    }
}

K_VALUES = [5, 10, 20, 50, 100]

# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================
def load_qrels(corpus):
    path = os.path.join(BASE_PATH, "retrieval", corpus, "dev.tsv")
    qrels = {}
    if not os.path.exists(path):
        print(f"⚠️  QRELS NOT FOUND: {path}")
        return qrels

    with open(path, 'r', encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        try:
            next(reader) # skip header
        except StopIteration:
            return qrels
        for row in reader:
            if len(row) < 3: continue
            qid, docid, score = row[0], row[1], int(row[2])
            if score > 0:
                if qid not in qrels: qrels[qid] = set()
                qrels[qid].add(docid)
    print(f"  ✅ Loaded {len(qrels)} queries with gold docs for {corpus}")
    return qrels

def calculate_recall(qrels, results_dict, k_values, corpus_name="", strat_name=""):
    scores = {k: 0.0 for k in k_values}
    num_queries = len(results_dict)
    if num_queries == 0: return {f"Recall@{k}": 0.0 for k in k_values}

    matched_queries = 0
    for qid, retrieved in results_dict.items():
        gold = qrels.get(qid, set())
        if not gold: continue
        matched_queries += 1
        for k in k_values:
            top_k = set(retrieved[:k])
            scores[k] += len(top_k.intersection(gold)) / len(gold)

    return {f"Recall@{k}": round(scores[k] / num_queries, 5) for k in k_values}

# ==============================================================================
# 3. ΜΑΖΙΚΗ ΑΝΑΛΥΣΗ
# ==============================================================================
all_rows = []

for corpus in ["clapnq", "fiqa", "govt", "cloud"]:
    print(f"\n🔍 Processing {corpus.upper()}...")
    qrels = load_qrels(corpus)
    if not qrels: continue

    strat_finds_ranks = {}
    strat_metrics = {}

    for strat_name, fpath in FILE_MAP[corpus].items():
        if not os.path.exists(fpath):
            print(f"  ❌ File not found: {strat_name}")
            continue

        res_list = {}
        found_with_ranks = {}

        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                item = json.loads(line)
                qid = str(item["task_id"])
                if qid not in qrels: continue

                retrieved = [str(c["document_id"]) for c in item.get("contexts", [])]
                res_list[qid] = retrieved
                gold_ids = qrels[qid]

                for rank, did in enumerate(retrieved[:100], 1):
                    if did in gold_ids:
                        found_with_ranks[(qid, did)] = rank

        strat_finds_ranks[strat_name] = found_with_ranks
        strat_metrics[strat_name] = calculate_recall(qrels, res_list, K_VALUES, corpus, strat_name)

    # Unique Analysis & Mean Ranks
    for name, found_ranks in strat_finds_ranks.items():
        others_found = set().union(*(d.keys() for k, d in strat_finds_ranks.items() if k != name))
        unique_pairs = set(found_ranks.keys()) - others_found

        all_ranks = list(found_ranks.values())
        unique_ranks = [found_ranks[p] for p in unique_pairs]

        mean_rank = round(np.mean(all_ranks), 2) if all_ranks else 0
        mean_unique_rank = round(np.mean(unique_ranks), 2) if unique_ranks else 0

        row = {
            "Corpus": corpus,
            "Strategy": name,
            "UniqueFinds": len(unique_pairs),
            "MeanRank": mean_rank,
            "MeanUniqueRank": mean_unique_rank
        }
        row.update(strat_metrics[name])
        all_rows.append(row)

# ==============================================================================
# 4. EXPORT & DISPLAY
# ==============================================================================
df_final = pd.DataFrame(all_rows)
cols = ["Corpus", "Strategy"] + [f"Recall@{k}" for k in K_VALUES] + ["UniqueFinds", "MeanRank", "MeanUniqueRank"]
df_final = df_final[cols]

display(df_final)

output_path = f"{RESULTS_DIR}/detailed_metrics_comparison_cw05.csv"
df_final.to_csv(output_path, index=False)
print(f"\n✅ Η ανάλυση ολοκληρώθηκε! Αποθηκεύτηκε στο: {output_path}")

In [ ]:
import json
import os
from collections import defaultdict
import pandas as pd

# ==============================================================================
# FINAL RRF SETTINGS - COMBINING ALL RETRIEVERS
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
OUTPUT_DIR = os.path.join(BASE_PATH, "final_rrf_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

K_RRF = 60  # RRF constant

# Manual weights per retriever (adjust as needed)
RETRIEVER_WEIGHTS = {
    "ELSER": 3.0,
    "Cohere": 0.8,
    "SPLADE": 1.5,
    "BM25": 0.8
}

CORPORA = ["clapnq", "fiqa", "govt", "cloud"]

# ==============================================================================
# INPUT FILES MAP - FUSED RESULTS FROM EACH RETRIEVER
# ==============================================================================
RETRIEVER_FILES = {
    "clapnq": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_clapnq_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_clapnq_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_clapnq_rrf.jsonl",
    },
    "fiqa": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_fiqa_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_fiqa_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_fiqa_rrf.jsonl",
    },
    "govt": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_govt_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_govt_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_govt_rrf.jsonl",
    },
    "cloud": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_cloud_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_cloud_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_cloud_rrf.jsonl",
    }
}


# ==============================================================================
# RRF FUSION FUNCTION
# ==============================================================================
def rrf_fusion(retriever_results, k=20, weights=None):
    """
    Perform RRF fusion across multiple retrievers.

    Args:
        retriever_results: dict {retriever_name: {qid: [doc_ids in rank order]}}
        k: RRF constant
        weights: dict {retriever_name: weight}

    Returns:
        dict {qid: [(doc_id, rrf_score), ...] sorted by score desc}
    """
    if weights is None:
        weights = {r: 1.0 for r in retriever_results.keys()}

    all_qids = set()
    for ret_res in retriever_results.values():
        all_qids.update(ret_res.keys())

    fused = {}

    for qid in all_qids:
        doc_scores = defaultdict(float)

        for ret_name, qid_results in retriever_results.items():
            if qid not in qid_results:
                continue

            weight = weights.get(ret_name, 1.0)
            ranked_docs = qid_results[qid]

            for rank, doc_id in enumerate(ranked_docs, start=1):
                rrf_score = weight / (k + rank)
                doc_scores[doc_id] += rrf_score

        sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
        fused[qid] = sorted_docs

    return fused


# ==============================================================================
# LOAD RETRIEVER RESULTS
# ==============================================================================
def load_retriever_results(corpus):
    """Load fused results from all retrievers for a corpus."""
    retriever_results = {}
    query_metadata = {}

    for ret_name, fpath in RETRIEVER_FILES[corpus].items():
        if not os.path.exists(fpath):
            print(f"  ⚠️  File not found: {ret_name} - {fpath}")
            continue

        ret_res = {}

        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    item = json.loads(line)
                    qid = str(item["task_id"])

                    # Store metadata from first retriever encounter
                    if qid not in query_metadata:
                        query_metadata[qid] = {
                            "query": item.get("query", ""),
                            "Collection": item.get("Collection", corpus)
                        }

                    # Extract ranked document IDs
                    contexts = item.get("contexts", [])
                    ranked_docs = [str(c["document_id"]) for c in contexts]
                    ret_res[qid] = ranked_docs
                except:
                    continue

        retriever_results[ret_name] = ret_res
        print(f"  ✅ Loaded {ret_name}: {len(ret_res)} queries")

    return retriever_results, query_metadata


# ==============================================================================
# SAVE FINAL RRF RESULTS
# ==============================================================================
def save_final_rrf_results(corpus, fused_results, query_metadata, output_dir):
    """Save final RRF fused results."""
    output_path = os.path.join(output_dir, f"final_rrf_{corpus}.jsonl")

    with open(output_path, 'w', encoding='utf-8') as f:
        for qid, ranked_docs in fused_results.items():
            metadata = query_metadata.get(qid, {})

            contexts = []
            for doc_id, rrf_score in ranked_docs:
                contexts.append({
                    "document_id": doc_id,
                    "score": round(rrf_score, 6)
                })

            output_item = {
                "task_id": qid,
                "query": metadata.get("query", ""),
                "Collection": metadata.get("Collection", corpus),
                "contexts": contexts
            }

            f.write(json.dumps(output_item, ensure_ascii=False) + "\n")

    print(f"  💾 Saved: {output_path} ({len(fused_results)} queries)")
    return output_path


# ==============================================================================
# MAIN EXECUTION - FINAL RRF FUSION
# ==============================================================================
print("\n" + "=" * 70)
print("🔀 FINAL RRF FUSION - COMBINING ALL RETRIEVERS")
print("=" * 70)
print(f"\nRRF constant k = {K_RRF}")
print(f"Retriever weights: {RETRIEVER_WEIGHTS}\n")

saved_files = []

for corpus in CORPORA:
    print(f"\n{'='*70}")
    print(f"📊 Processing: {corpus.upper()}")
    print(f"{'='*70}")

    retriever_results, query_metadata = load_retriever_results(corpus)

    if not retriever_results:
        print(f"  ⚠️  No results loaded for {corpus}, skipping...")
        continue

    print(f"\n  🔀 Fusing {len(retriever_results)} retrievers...")
    fused_results = rrf_fusion(retriever_results, k=K_RRF, weights=RETRIEVER_WEIGHTS)
    print(f"  ✅ Fused {len(fused_results)} queries")

    output_path = save_final_rrf_results(corpus, fused_results, query_metadata, OUTPUT_DIR)
    saved_files.append(output_path)

print(f"\n{'='*70}")
print("✅ FINAL RRF FUSION COMPLETE!")
print(f"{'='*70}")
print(f"\nSaved {len(saved_files)} files to: {OUTPUT_DIR}")
for fpath in saved_files:
    print(f"  📄 {os.path.basename(fpath)}")


# ==============================================================================
# EVALUATION OF FINAL RRF RESULTS
# ==============================================================================
print(f"\n\n{'='*70}")
print("📈 RUNNING EVALUATION ON FINAL RRF RESULTS")
print(f"{'='*70}\n")

evaluation_script = "/content/drive/MyDrive/Project/run_retrieval_eval.py"

if not os.path.exists(evaluation_script):
    print(f"⚠️ Evaluation script not found: {evaluation_script}")
    print("Skipping evaluation...")
else:
    for corpus in CORPORA:
        input_path = os.path.join(OUTPUT_DIR, f"final_rrf_{corpus}.jsonl")
        output_path = os.path.join(OUTPUT_DIR, f"final_rrf_{corpus}_results.jsonl")

        if not os.path.exists(input_path):
            print(f"⚠️ Skipping {corpus} - file not found: {input_path}")
            continue

        print(f"\n{'='*50}")
        print(f"Running evaluation for {corpus} ...")
        print(f"Input: {input_path}")
        print(f"Output: {output_path}")
        print(f"{'='*50}")

        cmd = f"python {evaluation_script} --input_file {input_path} --output_file {output_path}"
        result = os.system(cmd)

        if result != 0:
            print(f"⚠️ Warning: Evaluation failed for {corpus}")
        else:
            print(f"✅ Evaluation completed for {corpus}")


# ==============================================================================
# AGGREGATE RESULTS COMPARISON
# ==============================================================================
print(f"\n\n{'='*70}")
print("📊 AGGREGATING FINAL EVALUATION RESULTS")
print(f"{'='*70}\n")

csv_files = [
    f"{OUTPUT_DIR}/final_rrf_clapnq_results_aggregate.csv",
    f"{OUTPUT_DIR}/final_rrf_fiqa_results_aggregate.csv",
    f"{OUTPUT_DIR}/final_rrf_govt_results_aggregate.csv",
    f"{OUTPUT_DIR}/final_rrf_cloud_results_aggregate.csv"
]

all_data = []

for file in csv_files:
    if os.path.exists(file):
        df = pd.read_csv(file)
        df = df[df["collection"] != "all"]  # Remove aggregate row if present
        corpus_name = file.split("/")[-1].replace("final_rrf_", "").replace("_results_aggregate.csv", "")
        df["Corpus"] = corpus_name
        all_data.append(df)
        print(f"✅ Loaded: {corpus_name}")
    else:
        print(f"⚠️ Not found: {file}")

if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df = final_df.set_index("Corpus")
    print("\n" + "="*70)
    print("📊 FINAL RRF EVALUATION RESULTS - ALL CORPORA")
    print("   (Combining ELSER + Cohere + SPLADE + BM25)")
    print("="*70)
    display(final_df)

    # Save combined results
    combined_output = f"{OUTPUT_DIR}/final_rrf_all_corpora_results.csv"
    final_df.to_csv(combined_output)
    print(f"\n💾 Combined results saved to: {combined_output}")
else:
    print("\n⚠️ No evaluation results found")

print("\n" + "="*70)
print("✅ ALL PROCESSING COMPLETE!")
print("="*70)
print(f"\n🎯 Final RRF results combining all 4 retrievers saved to:")
print(f"   {OUTPUT_DIR}/")


In [ ]:
import json
import csv
import os
import pandas as pd
import numpy as np
from collections import defaultdict

# ==============================================================================
# 1. ΡΥΘΜΙΣΕΙΣ
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
OUTPUT_DIR = os.path.join(BASE_PATH, "rrf_elser_cohere_experiments")
os.makedirs(OUTPUT_DIR, exist_ok=True)

K_RRF = 60
K_VALUES = [5, 10, 20, 50, 100, 150, 200]

FILE_MAP = {
    "clapnq": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_clapnq_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_clapnq_rrf.jsonl",
        "Splade":f"{BASE_PATH}/splade_rrf_results/splade_clapnq_rrf.jsonl"
    },
    "fiqa": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_fiqa_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_fiqa_rrf.jsonl",
        "Splade":f"{BASE_PATH}/splade_rrf_results/splade_fiqa_rrf"
    },
    "govt": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_govt_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_govt_rrf.jsonl",
        "Splade":f"{BASE_PATH}/splade_rrf_results/splade_govt_rrf.jsonl"
    },
    "cloud": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_cloud_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_cloud_rrf.jsonl",
        "Splade":f"{BASE_PATH}/splade_rrf_results/splade_cloud_rrf.jsonl"
    }
}

CORPORA = ["clapnq", "fiqa", "govt", "cloud"]
RETRIEVERS = ["ELSER", "Cohere","Splade"]

# ==============================================================================
# 2. WEIGHT CONFIGURATIONS - ELSER + COHERE ONLY
# ==============================================================================
# Rule: ELSER >= Cohere (ELSER is usually better)
WEIGHT_CONFIGS = [
    # Baseline
    {"name": "Equal", "ELSER": 2.0, "Cohere": 1.0,"Splade":0.5},


]


# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================
def load_qrels(corpus):
    path = os.path.join(BASE_PATH, "retrieval", corpus, "dev.tsv")
    qrels = {}
    if not os.path.exists(path):
        return qrels
    with open(path, 'r', encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        try:
            next(reader)
        except:
            return qrels
        for row in reader:
            if len(row) < 3:
                continue
            qid, docid, score = str(row[0]), str(row[1]), int(row[2])
            if score > 0:
                if qid not in qrels:
                    qrels[qid] = set()
                qrels[qid].add(docid)
    return qrels


def rrf_fusion(retriever_results, k=20, weights=None):
    if weights is None:
        weights = {r: 1.0 for r in retriever_results.keys()}

    all_qids = set()
    for ret_res in retriever_results.values():
        all_qids.update(ret_res.keys())

    fused = {}
    for qid in all_qids:
        doc_scores = defaultdict(float)
        for ret_name, qid_results in retriever_results.items():
            if qid not in qid_results:
                continue
            weight = weights.get(ret_name, 1.0)
            ranked_docs = qid_results[qid]
            for rank, doc_id in enumerate(ranked_docs, start=1):
                rrf_score = weight / (k + rank)
                doc_scores[doc_id] += rrf_score
        sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
        fused[qid] = [doc_id for doc_id, _ in sorted_docs]
    return fused


def load_retriever_results(corpus, retriever_name):
    fpath = FILE_MAP[corpus][retriever_name]
    if not os.path.exists(fpath):
        return {}
    results = {}
    with open(fpath, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                item = json.loads(line)
                qid = str(item["task_id"])
                contexts = item.get("contexts", [])
                ranked_docs = [str(c["document_id"]) for c in contexts]
                results[qid] = ranked_docs
            except:
                continue
    return results


def calculate_recall(qrels, results_dict, k_values):
    scores = {k: 0.0 for k in k_values}
    num_queries = len(results_dict)
    if num_queries == 0:
        return {f"Recall@{k}": 0.0 for k in k_values}
    for qid, retrieved in results_dict.items():
        gold = qrels.get(qid, set())
        if not gold:
            continue
        for k in k_values:
            top_k = set(retrieved[:k])
            scores[k] += len(top_k.intersection(gold)) / len(gold)
    return {f"Recall@{k}": round(scores[k] / num_queries, 5) for k in k_values}


# ==============================================================================
# 4. MAIN EXPERIMENT - ELSER + COHERE ONLY, RANKED BY RECALL@5
# ==============================================================================
all_results = []

print("="*70)
print("🔬 ELSER + COHERE RRF EXPERIMENTS - 2-WAY COMBINATION")
print("="*70)
print(f"\\nTesting {len(WEIGHT_CONFIGS)} weight configurations")
print(f"Total experiments: {len(WEIGHT_CONFIGS) * len(CORPORA)}\\n")

for corpus in CORPORA:
    print(f"\\n{'='*70}")
    print(f"📊 Processing: {corpus.upper()}")
    print(f"{'='*70}")

    qrels = load_qrels(corpus)
    if not qrels:
        print(f"  ⚠️ No qrels found for {corpus}")
        continue

    print("  Loading retriever data...")
    retriever_data = {}
    for retriever in RETRIEVERS:
        retriever_data[retriever] = load_retriever_results(corpus, retriever)
        print(f"    ✅ {retriever}: {len(retriever_data[retriever])} queries")

    print(f"\\n  🔀 Testing ELSER + Cohere with {len(WEIGHT_CONFIGS)} weight configs...")

    for i, weight_cfg in enumerate(WEIGHT_CONFIGS, 1):
        weights = {r: weight_cfg[r] for r in RETRIEVERS}

        fused = rrf_fusion(retriever_data, k=K_RRF, weights=weights)
        metrics = calculate_recall(qrels, fused, K_VALUES)

        row = {
            "Corpus": corpus,
            "WeightConfig": weight_cfg["name"],
            "W_ELSER": weight_cfg["ELSER"],
            "W_Cohere": weight_cfg["Cohere"],
            "ELSER_Cohere_Ratio": round(weight_cfg["ELSER"] / weight_cfg["Cohere"], 2)
        }
        row.update(metrics)
        all_results.append(row)

        if i % 10 == 0:
            print(f"    Progress: {i}/{len(WEIGHT_CONFIGS)} configs tested")


print(f"\\n{'='*70}")
print("✅ EXPERIMENTS COMPLETE!")
print(f"{'='*70}")


# ==============================================================================
# 5. ANALYZE AND EXPORT - RANKED BY RECALL@5
# ==============================================================================
df = pd.DataFrame(all_results)

weight_cols = ["W_ELSER", "W_Cohere", "ELSER_Cohere_Ratio"]
recall_cols = [f"Recall@{k}" for k in K_VALUES]
base_cols = ["Corpus", "WeightConfig"]
df = df[base_cols + weight_cols + recall_cols]

# Sort by Recall@5 DESCENDING (primary), then Recall@10, Recall@20
df = df.sort_values(["Recall@5", "Recall@10", "Recall@20"], ascending=False)

full_output = os.path.join(OUTPUT_DIR, "elser_cohere_full_results.csv")
df.to_csv(full_output, index=False)
print(f"\\n✅ Full results saved: {full_output}\\n")


# ==============================================================================
# 6. TOP RESULTS PER CORPUS (BY RECALL@5)
# ==============================================================================
print("="*70)
print("🏆 TOP 15 CONFIGURATIONS PER CORPUS (Ranked by Recall@5)")
print("="*70)

top_per_corpus_list = []
for corpus in CORPORA:
    print(f"\\n📊 {corpus.upper()} - Top 15 by Recall@5")
    print("-"*70)
    corpus_df = df[df["Corpus"] == corpus].head(15)
    display_cols = ["WeightConfig", "W_ELSER", "W_Cohere", "ELSER_Cohere_Ratio",
                    "Recall@5", "Recall@10", "Recall@100", "Recall@200"]
    print(corpus_df[display_cols].to_string(index=False))
    top_per_corpus_list.append(corpus_df)

top_df = pd.concat(top_per_corpus_list, ignore_index=True)
top_output = os.path.join(OUTPUT_DIR, "elser_cohere_top15_per_corpus.csv")
top_df.to_csv(top_output, index=False)
print(f"\\n✅ Top 15 per corpus saved: {top_output}")


# ==============================================================================
# 7. OVERALL BEST (AVERAGE ACROSS CORPORA, RANKED BY RECALL@5)
# ==============================================================================
print(f"\\n{'='*70}")
print("🎯 BEST CONFIGURATIONS OVERALL (Average Recall@5 across all corpora)")
print(f"{'='*70}\\n")

avg_results = df.groupby("WeightConfig").agg({
    "Recall@5": "mean",
    "Recall@10": "mean",
    "Recall@20": "mean",
    "Recall@50": "mean",
    "Recall@100": "mean",
    "Recall@150": "mean",
    "Recall@200": "mean",
    "W_ELSER": "first",
    "W_Cohere": "first",
    "ELSER_Cohere_Ratio": "first"
}).round(5).reset_index()

avg_results = avg_results.sort_values(["Recall@5", "Recall@10"], ascending=False)

print("Top 20 Overall (by avg Recall@5):")
print("-"*70)
print(avg_results.head(20).to_string(index=False))

avg_output = os.path.join(OUTPUT_DIR, "elser_cohere_averaged_rankings.csv")
avg_results.to_csv(avg_output, index=False)
print(f"\\n✅ Averaged rankings saved: {avg_output}")


# ==============================================================================
# 8. SUMMARY STATISTICS
# ==============================================================================
print(f"\\n{'='*70}")
print("📊 SUMMARY STATISTICS")
print(f"{'='*70}\\n")

print(f"Total experiments: {len(df)}")
print(f"Number of weight configs tested: {len(WEIGHT_CONFIGS)}")
print(f"Number of corpora: {len(CORPORA)}\\n")

print("Recall@5 Statistics:")
print(f"  Mean:   {df['Recall@5'].mean():.5f}")
print(f"  Median: {df['Recall@5'].median():.5f}")
print(f"  Max:    {df['Recall@5'].max():.5f}")
print(f"  Min:    {df['Recall@5'].min():.5f}\\n")

best_overall = avg_results.iloc[0]
print(f"🏆 Best Configuration Overall: {best_overall['WeightConfig']}")
print(f"   Weights: ELSER={best_overall['W_ELSER']}, Cohere={best_overall['W_Cohere']}")
print(f"   Ratio: {best_overall['ELSER_Cohere_Ratio']}:1")
print(f"   Avg Recall@5:   {best_overall['Recall@5']:.5f}")
print(f"   Avg Recall@10:  {best_overall['Recall@10']:.5f}")
print(f"   Avg Recall@100: {best_overall['Recall@100']:.5f}")
print(f"   Avg Recall@200: {best_overall['Recall@200']:.5f}")

print(f"\\n{'='*70}")
print("✅ ALL ANALYSIS COMPLETE!")
print(f"{'='*70}")
print(f"\\nResults saved to: {OUTPUT_DIR}/")

In [ ]:
import json
import csv
import os
import pandas as pd
import numpy as np
from collections import defaultdict

# ==============================================================================
# 1. ΡΥΘΜΙΣΕΙΣ & FILE MAP (20260115_184516)
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
# Ο φάκελος των τελευταίων αποτελεσμάτων
RESULTS_DIR = f"{BASE_PATH}/splade_cohere_v4_rerank_results/20260115_184516"

FILE_MAP = {
    "clapnq": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_clapnq_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_clapnq_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_clapnq_rrf.jsonl",
        "BM25": f"{BASE_PATH}/bm25_rrf_results/bm25_clapnq_rrf.jsonl"
    },
    "fiqa": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_fiqa_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_fiqa_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_fiqa_rrf.jsonl",
        "BM25": f"{BASE_PATH}/bm25_rrf_results/bm25_fiqa_rrf.jsonl"
    },
    "govt": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_govt_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_govt_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_govt_rrf.jsonl",
        "BM25": f"{BASE_PATH}/bm25_rrf_results/bm25_govt_rrf.jsonl"
    },
    "cloud": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_cloud_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_cloud_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_cloud_rrf.jsonl",
        "BM25": f"{BASE_PATH}/bm25_rrf_results/bm25_cloud_rrf.jsonl"
    }
}

K_VALUES = [5, 10, 20, 50, 100]

# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================
def load_qrels(corpus):
    path = os.path.join(BASE_PATH, "retrieval", corpus, "dev.tsv")
    qrels = {}
    if not os.path.exists(path):
        print(f"⚠️  QRELS NOT FOUND: {path}")
        return qrels

    with open(path, 'r', encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        try:
            next(reader) # skip header
        except StopIteration:
            return qrels
        for row in reader:
            if len(row) < 3: continue
            qid, docid, score = row[0], row[1], int(row[2])
            if score > 0:
                if qid not in qrels: qrels[qid] = set()
                qrels[qid].add(docid)
    print(f"  ✅ Loaded {len(qrels)} queries with gold docs for {corpus}")
    return qrels

def calculate_recall(qrels, results_dict, k_values, corpus_name="", strat_name=""):
    scores = {k: 0.0 for k in k_values}
    num_queries = len(results_dict)
    if num_queries == 0: return {f"Recall@{k}": 0.0 for k in k_values}

    matched_queries = 0
    for qid, retrieved in results_dict.items():
        gold = qrels.get(qid, set())
        if not gold: continue
        matched_queries += 1
        for k in k_values:
            top_k = set(retrieved[:k])
            scores[k] += len(top_k.intersection(gold)) / len(gold)

    return {f"Recall@{k}": round(scores[k] / num_queries, 5) for k in k_values}

# ==============================================================================
# 3. ΜΑΖΙΚΗ ΑΝΑΛΥΣΗ
# ==============================================================================
all_rows = []

for corpus in ["clapnq", "fiqa", "govt", "cloud"]:
    print(f"\n🔍 Processing {corpus.upper()}...")
    qrels = load_qrels(corpus)
    if not qrels: continue

    strat_finds_ranks = {}
    strat_metrics = {}

    for strat_name, fpath in FILE_MAP[corpus].items():
        if not os.path.exists(fpath):
            print(f"  ❌ File not found: {strat_name}")
            continue

        res_list = {}
        found_with_ranks = {}

        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                item = json.loads(line)
                qid = str(item["task_id"])
                if qid not in qrels: continue

                retrieved = [str(c["document_id"]) for c in item.get("contexts", [])]
                res_list[qid] = retrieved
                gold_ids = qrels[qid]

                for rank, did in enumerate(retrieved[:100], 1):
                    if did in gold_ids:
                        found_with_ranks[(qid, did)] = rank

        strat_finds_ranks[strat_name] = found_with_ranks
        strat_metrics[strat_name] = calculate_recall(qrels, res_list, K_VALUES, corpus, strat_name)

    # Unique Analysis & Mean Ranks
    for name, found_ranks in strat_finds_ranks.items():
        others_found = set().union(*(d.keys() for k, d in strat_finds_ranks.items() if k != name))
        unique_pairs = set(found_ranks.keys()) - others_found

        all_ranks = list(found_ranks.values())
        unique_ranks = [found_ranks[p] for p in unique_pairs]

        mean_rank = round(np.mean(all_ranks), 2) if all_ranks else 0
        mean_unique_rank = round(np.mean(unique_ranks), 2) if unique_ranks else 0

        row = {
            "Corpus": corpus,
            "Strategy": name,
            "UniqueFinds": len(unique_pairs),
            "MeanRank": mean_rank,
            "MeanUniqueRank": mean_unique_rank
        }
        row.update(strat_metrics[name])
        all_rows.append(row)

# ==============================================================================
# 4. EXPORT & DISPLAY
# ==============================================================================
df_final = pd.DataFrame(all_rows)
cols = ["Corpus", "Strategy"] + [f"Recall@{k}" for k in K_VALUES] + ["UniqueFinds", "MeanRank", "MeanUniqueRank"]
df_final = df_final[cols]

display(df_final)

output_path = f"{RESULTS_DIR}/detailed_metrics_comparison_cw05.csv"
df_final.to_csv(output_path, index=False)
print(f"\n✅ Η ανάλυση ολοκληρώθηκε! Αποθηκεύτηκε στο: {output_path}")

In [ ]:
import json, csv, os
import pandas as pd

# ==============================================================================
# 1. ANALYSIS FUNCTION: WHERE IS THE ADDED VALUE?
# ==============================================================================
def analyze_added_value_distribution():
    dist_rows = []

    for domain in DOMAINS:
        qrels = load_qrels(domain)
        # Φορτώνουμε rankings (χωρίς περιορισμό K για να δούμε που βρίσκονται τα χαμένα)
        rankings = {name: load_ranks(FILE_MAP[domain][name], qrels) for name in MODELS}

        for name in ["Cohere", "SPLADE", "BM25"]:
            # Κατανομή θέσεων για τα έγγραφα που ΔΕΝ βρήκε ο ELSER στο δικό του Top-20
            # αλλά βρήκε το συγκεκριμένο μοντέλο κάπου στο δικό του Top-100
            ranks_found = []

            for qid, gold_set in qrels.items():
                elser_top20 = set(rankings["ELSER"].get(qid, [])[:20])
                model_full_list = rankings[name].get(qid, [])

                # Gold docs που έχασε ο ELSER στο Top-20
                missed_by_elser = gold_set - elser_top20

                if missed_by_elser:
                    # Που τα έχει το τρέχον μοντέλο;
                    for rank, did in enumerate(model_full_list, 1):
                        if did in missed_by_elser:
                            ranks_found.append(rank)

            # Ομαδοποίηση κατανομής
            r_top5 = len([r for r in ranks_found if r <= 5])
            r_top10 = len([r for r in ranks_found if 5 < r <= 10])
            r_top20 = len([r for r in ranks_found if 10 < r <= 20])
            r_top50 = len([r for r in ranks_found if 20 < r <= 50])
            r_deep = len([r for r in ranks_found if r > 50])

            dist_rows.append({
                "Domain": domain,
                "Model": name,
                "In_Top5": r_top5,
                "In_6_10": r_top10,
                "In_11_20": r_top20,
                "In_21_50": r_top50,
                "Beyond_50": r_deep,
                "Total_Recoverable": len(ranks_found)
            })

    return pd.DataFrame(dist_rows)

# ==============================================================================
# 2. RUN ANALYSIS
# ==============================================================================
df_dist = analyze_added_value_distribution()
print("📊 DISTRIBUTION OF ADDED VALUE DOCUMENTS (Docs ELSER missed at Top-20)")
print(df_dist.to_string(index=False))

In [ ]:
import json, csv, os, time
import pandas as pd
import numpy as np

# ==============================================================================
# ΣΥΝΑΡΤΗΣΗ: ΑΝΑΛΥΣΗ WEAK ENSEMBLE (3-MODELS) VS ELSER
# ==============================================================================
def analyze_weak_ensemble_value():
    results = []

    for domain in DOMAINS:
        qrels = load_qrels(domain)
        # Φόρτωση rankings
        rankings = {name: load_ranks(FILE_MAP[domain][name], qrels) for name in MODELS}

        # 1. Δημιουργία του Weak Ensemble (RRF των 3)
        # Χρησιμοποιούμε k=60 και ίσα βάρη για να δούμε τη συλλογική τους ισχύ
        weak_weights = {"SPLADE": 0.33, "Cohere": 0.33, "BM25": 0.34}
        weak_qid_scores = {}

        for name, w in weak_weights.items():
            for qid, docs in rankings[name].items():
                if qid not in weak_qid_scores: weak_qid_scores[qid] = {}
                for rank, did in enumerate(docs, 1):
                    weak_qid_scores[qid][did] = weak_qid_scores[qid].get(did, 0.0) + w * (1.0 / (60 + rank))

        # Ταξινόμηση Ensemble
        weak_ensemble = {qid: [d for d, s in sorted(docs.items(), key=lambda x: x[1], reverse=True)]
                         for qid, docs in weak_qid_scores.items()}

        # 2. Σύγκριση Ensemble vs ELSER
        ranks_found = []
        for qid, gold_set in qrels.items():
            elser_top20 = set(rankings["ELSER"].get(qid, [])[:20])
            ensemble_list = weak_ensemble.get(qid, [])

            # Gold που έχασε ο ELSER στο Top-20
            missed_by_elser = gold_set - elser_top20

            if missed_by_elser:
                for rank, did in enumerate(ensemble_list, 1):
                    if did in missed_by_elser:
                        ranks_found.append(rank)

        # 3. Καταγραφή Κατανομής
        results.append({
            "Domain": domain,
            "Ensemble": "SPL+COH+BM25",
            "In_Top5": len([r for r in ranks_found if r <= 5]),
            "In_6_10": len([r for r in ranks_found if 5 < r <= 10]),
            "In_11_20": len([r for r in ranks_found if 10 < r <= 20]),
            "In_21_50": len([r for r in ranks_found if 20 < r <= 50]),
            "Beyond_50": len([r for r in ranks_found if r > 50]),
            "Total_Unique_Recoverable": len(ranks_found)
        })

    return pd.DataFrame(results)

# ==============================================================================
# ΕΚΤΕΛΕΣΗ
# ==============================================================================
df_ensemble = analyze_weak_ensemble_value()
print("📊 WEAK ENSEMBLE (3-MODELS) ADDED VALUE VS ELSER TOP-20")
print(df_ensemble.to_string(index=False))

In [ ]:
import json, csv, os
import pandas as pd
import numpy as np

# ==============================================================================
# 1. ΚΑΘΟΛΙΚΕΣ ΡΥΘΜΙΣΕΙΣ & FILE_MAP
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
DOMAINS = ["clapnq", "fiqa", "govt", "cloud"]
MODELS = ["ELSER", "Cohere", "SPLADE", "BM25"]

FILE_MAP = {
    "clapnq": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_clapnq_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_clapnq_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_clapnq_rrf.jsonl",
        "BM25": f"{BASE_PATH}/bm25_rrf_results/bm25_clapnq_rrf.jsonl"
    },
    "fiqa": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_fiqa_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_fiqa_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_fiqa_rrf.jsonl",
        "BM25": f"{BASE_PATH}/bm25_rrf_results/bm25_fiqa_rrf.jsonl"
    },
    "govt": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_govt_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_govt_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_govt_rrf.jsonl",
        "BM25": f"{BASE_PATH}/bm25_rrf_results/bm25_govt_rrf.jsonl"
    },
    "cloud": {
        "ELSER": f"{BASE_PATH}/rrf_results/elserv1_cloud_rrf.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_cloud_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_cloud_rrf.jsonl",
        "BM25": f"{BASE_PATH}/bm25_rrf_results/bm25_cloud_rrf.jsonl"
    }
}

# ==============================================================================
# 2. ΒΟΗΘΗΤΙΚΕΣ ΣΥΝΑΡΤΗΣΕΙΣ
# ==============================================================================
def load_qrels(domain):
    path = os.path.join(BASE_PATH, "retrieval", domain, "dev.tsv")
    qrels = {}
    if not os.path.exists(path): return qrels
    with open(path, 'r', encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        next(reader, None)
        for row in reader:
            qid, did, score = row[0], row[1], int(row[2])
            if score > 0:
                if qid not in qrels: qrels[qid] = set()
                qrels[qid].add(did)
    return qrels

def load_ranks(path, qrels):
    data = {}
    if not os.path.exists(path): return data
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            qid = str(item["task_id"])
            if qid not in qrels: continue
            data[qid] = [str(c["document_id"]) for c in item.get("contexts", [])]
    return data

# ==============================================================================
# 3. ΑΝΑΛΥΣΗ ΣΤΡΑΤΗΓΙΚΩΝ RESCUE (40 & 50 DOCS)
# ==============================================================================
def analyze_extended_rescue_strategies():
    results = []

    for domain in DOMAINS:
        qrels = load_qrels(domain)
        rankings = {name: load_ranks(FILE_MAP[domain][name], qrels) for name in MODELS}

        # Κατασκευή του Weak Ensemble (SPLADE + Cohere + BM25)
        weak_weights = {"SPLADE": 0.33, "Cohere": 0.33, "BM25": 0.34}
        weak_qid_scores = {}
        for name, w in weak_weights.items():
            for qid, docs in rankings[name].items():
                if qid not in weak_qid_scores: weak_qid_scores[qid] = {}
                for rank, did in enumerate(docs, 1):
                    weak_qid_scores[qid][did] = weak_qid_scores[qid].get(did, 0.0) + w * (1.0 / (60 + rank))

        weak_ensemble = {qid: [d for d, s in sorted(docs.items(), key=lambda x: x[1], reverse=True)]
                         for qid, docs in weak_qid_scores.items()}

        # Μετρητές Gold εγγράφων
        stats = {
            "E_26_40": 0, "Ens_Res_15": 0,  # Σενάριο A (Pool 40)
            "E_26_50": 0, "Ens_Res_25": 0,  # Σενάριο B (Pool 50 - Keep ELSER 25)
            "E_11_50": 0, "Ens_Res_40": 0   # Σενάριο C (Pool 50 - Keep ELSER 10)
        }

        for qid, gold_set in qrels.items():
            elser_full = rankings["ELSER"].get(qid, [])
            ens_full = weak_ensemble.get(qid, [])

            # --- Scenario A: Pool 40 (Keep ELSER 1-25) ---
            e_top25_set = set(elser_full[:25])
            stats["E_26_40"] += len(set(elser_full[25:40]) & gold_set)
            stats["Ens_Res_15"] += len(set([d for d in ens_full if d not in e_top25_set][:15]) & gold_set)

            # --- Scenario B: Pool 50 (Keep ELSER 1-25) ---
            stats["E_26_50"] += len(set(elser_full[25:50]) & gold_set)
            stats["Ens_Res_25"] += len(set([d for d in ens_full if d not in e_top25_set][:25]) & gold_set)

            # --- Scenario C: Pool 50 (Keep ELSER 1-10) ---
            e_top10_set = set(elser_full[:10])
            stats["E_11_50"] += len(set(elser_full[10:50]) & gold_set)
            stats["Ens_Res_40"] += len(set([d for d in ens_full if d not in e_top10_set][:40]) & gold_set)

        # Καταγραφή των 3 στρατηγικών για κάθε domain
        results.append({"Domain": domain, "Type": "Pool_40 (ELSER:1-25 + Ens:15)", "Lost_ELSER": stats["E_26_40"], "Gain_Ens": stats["Ens_Res_15"], "Net": stats["Ens_Res_15"] - stats["E_26_40"]})
        results.append({"Domain": domain, "Type": "Pool_50 (ELSER:1-25 + Ens:25)", "Lost_ELSER": stats["E_26_50"], "Gain_Ens": stats["Ens_Res_25"], "Net": stats["Ens_Res_25"] - stats["E_26_50"]})
        results.append({"Domain": domain, "Type": "Pool_50 (ELSER:1-10 + Ens:40)", "Lost_ELSER": stats["E_11_50"], "Gain_Ens": stats["Ens_Res_40"], "Net": stats["Ens_Res_40"] - stats["E_11_50"]})

    return pd.DataFrame(results)

# ==============================================================================
# 4. ΕΚΤΕΛΕΣΗ & ΕΜΦΑΝΙΣΗ
# ==============================================================================
df_strategies = analyze_extended_rescue_strategies()

# Φιλτράρισμα και εμφάνιση ανά Domain για ευκολότερη σύγκριση
for domain in DOMAINS:
    print(f"\n📊 COMPARISON FOR {domain.upper()}")
    print("-" * 90)
    subset = df_strategies[df_strategies["Domain"] == domain]
    print(subset[["Type", "Lost_ELSER", "Gain_Ens", "Net"]].to_string(index=False))

In [ ]:
import os, json, csv, itertools
import pandas as pd
import numpy as np
from tqdm import tqdm

# ==============================================================================
# 1. SETUP & DATA LOADING
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
DOMAINS = ["clapnq", "fiqa", "govt", "cloud"]
MODELS = ["ELSER", "Cohere", "SPLADE"]

# Φόρτωση των αρχείων όπως τα όρισες
RETRIEVER_FILES = {
    "clapnq": {
        "ELSER": f"{BASE_PATH}/final_nested_ensembles/clapnq_nested_rrf_final.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_clapnq_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_clapnq_rrf.jsonl",
    },
    "fiqa": {
        "ELSER": f"{BASE_PATH}/final_nested_ensembles/fiqa_nested_rrf_final.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_fiqa_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_fiqa_rrf.jsonl",
    },
    "govt": {
        "ELSER": f"{BASE_PATH}/final_nested_ensembles/govt_nested_rrf_final.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_govt_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_govt_rrf.jsonl",
    },
    "cloud": {
        "ELSER": f"{BASE_PATH}/final_nested_ensembles/cloud_nested_rrf_final.jsonl",
        "Cohere": f"{BASE_PATH}/cohere_rrf_results/cohere_cloud_rrf.jsonl",
        "SPLADE": f"{BASE_PATH}/splade_rrf_results/splade_cloud_rrf.jsonl",
    }
}

def load_qrels(domain):
    path = os.path.join(BASE_PATH, "retrieval", domain, "dev.tsv")
    qrels = {}
    if not os.path.exists(path): return qrels
    with open(path, 'r', encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        next(reader)
        for row in reader:
            qid, did, score = str(row[0]), str(row[1]), int(row[2])
            if score > 0:
                if qid not in qrels: qrels[qid] = set()
                qrels[qid].add(did)
    return qrels

def pre_load_rankings(domain, qrels):
    data = {m: {} for m in MODELS}
    for m in MODELS:
        path = RETRIEVER_FILES[domain][m]
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    item = json.loads(line)
                    qid = str(item["task_id"])
                    if qid in qrels:
                        data[m][qid] = [str(c["document_id"]) for c in item.get("contexts", [])]
    return data

# ==============================================================================
# 2. GRID GENERATION (Περιορισμός: ELSER > Others)
# ==============================================================================
def generate_weight_grid():
    grid = []
    # Δοκιμάζουμε βάρος ELSER από 0.40 έως 0.80 με βήμα 0.05
    for w_elser in np.round(np.arange(0.40, 0.81, 0.05), 2):
        remaining = np.round(1.0 - w_elser, 2)
        # Μοιράζουμε το υπόλοιπο σε Cohere και SPLADE
        # Δοκιμάζουμε διαφορετικές αναλογίες για το remaining
        for ratio in np.round(np.arange(0.1, 0.91, 0.1), 1):
            w_cohere = np.round(remaining * ratio, 3)
            w_splade = np.round(remaining - w_cohere, 3)

            # Έλεγχος αν ο ELSER παραμένει ο μεγαλύτερος
            if w_elser > w_cohere and w_elser > w_splade:
                grid.append({"ELSER": w_elser, "Cohere": w_cohere, "SPLADE": w_splade})
    return grid

# ==============================================================================
# 3. SEARCH EXECUTION
# ==============================================================================
K_VALUES = [20, 40, 60, 80, 100]
weight_grid = generate_weight_grid()

best_overall_configs = []

for domain in DOMAINS:
    print(f"\n🔎 Searching best RRF for: {domain.upper()}...")
    qrels = load_qrels(domain)
    rank_cache = pre_load_rankings(domain, qrels)

    qids = list(rank_cache["ELSER"].keys())
    best_r5 = -1.0
    best_cfg = {}

    for k in tqdm(K_VALUES, desc=f"Scanning k-values"):
        for weights in weight_grid:
            # Υπολογισμός RRF
            recall_sum = 0
            for qid in qids:
                doc_scores = {}
                for m in MODELS:
                    w = weights[m]
                    for rank, did in enumerate(rank_cache[m].get(qid, []), 1):
                        doc_scores[did] = doc_scores.get(did, 0.0) + w * (1.0 / (k + rank))

                # Top-5 Recall calculation
                top_5 = [d for d, s in sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:5]]
                gold = qrels.get(qid, set())
                if gold:
                    recall_sum += len(set(top_5) & gold) / len(gold)

            current_r5 = recall_sum / len(qids)
            if current_r5 > best_r5:
                best_r5 = current_r5
                best_cfg = {"k": k, "weights": weights.copy(), "Recall@5": round(best_r5, 5)}

    best_overall_configs.append({"Domain": domain, **best_cfg})
    print(f"🏆 Best for {domain}: R@5={best_cfg['Recall@5']} (k={best_cfg['k']}, Weights={best_cfg['weights']})")

# ==============================================================================
# 4. FINAL REPORT
# ==============================================================================
print("\n" + "="*80)
print("📊 FINAL GRID SEARCH RESULTS (ELSER > COHERE, SPLADE)")
print("="*80)
df_results = pd.DataFrame(best_overall_configs)
print(df_results.to_string(index=False))

# Αποθήκευση των βέλτιστων παραμέτρων
df_results.to_csv(f"{BASE_PATH}/best_elser_hybrid_configs.csv", index=False)

# **Final Reranked Results**

In [ ]:
import os, json, time, requests, subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import Dict, List, Tuple


# ==============================================================================
# 1. SET YOUR FILE PATHS HERE 📍
# ==============================================================================
# Η διαδρομή του αρχείου που θέλεις να επεξεργαστείς
INPUT_FILE = "/content/drive/MyDrive/Project/elser_v1_structural_6U3A_predictions/cloud_elser_minimal_optimized_20260112_122934.jsonl"


# Οι απαραίτητες διαδρομές για το συγκεκριμένο αρχείο
CORPUS_FILE = "/content/drive/MyDrive/Project/corpora/cloud.jsonl"
QUERIES_FILE = "/content/drive/MyDrive/Project/retrieval/cloud/cloud_lastturn.jsonl"
REWRITES_FILE = "/content/drive/MyDrive/Project/elser_v1_structural_6U3A_predictions/cloud_elser_minimal_optimized_20260112_122934_REWRITES.json"


# Φάκελος Εξόδου
BASE_PATH = "/content/drive/MyDrive/Project"
OUTPUT_DIR = os.path.join(BASE_PATH, "cohere_v4_rewrite1_rerank", time.strftime("%Y%m%d_%H%M%S"))
os.makedirs(OUTPUT_DIR, exist_ok=True)


# Azure Cohere Credentials
AZURE_COHERE_ENDPOINT = os.environ["ENDPOINT"]
AZURE_COHERE_KEY = os.environ["_KEY"]


EVAL_SCRIPT_PATH = os.path.join(BASE_PATH, "run_retrieval_eval.py")
TOP_K_RERANK = 100
COHERE_WEIGHTS = [0.1, 0.3, 0.5, 0.7, 0.9]


# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================


def load_jsonl(path: str) -> List[dict]:
    data = []
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        return data
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data


def load_json(path: str) -> Dict:
    """Load JSON file (for rewrites)"""
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_corpus(path: str) -> Dict[str, str]:
    corpus = {}
    if not os.path.exists(path): return corpus
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            did = str(j.get("_id") or j.get("id"))
            text = (j.get("title", "") + " " + (j.get("text") or j.get("contents") or "")).strip()
            corpus[did] = text[:2000]
    return corpus


def load_queries(path: str) -> Dict[str, str]:
    """Load original queries (fallback only)"""
    queries = {}
    if not os.path.exists(path): return queries
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            qid = str(j.get("_id") or j.get("id") or j.get("task_id"))
            queries[qid] = j.get("text") or j.get("query")
    return queries


def load_rewrites(path: str) -> Dict[str, str]:
    """
    Load rewrites from JSON file and create mapping

    Returns:
        Dict mapping task_id -> rewritten_query
    """
    rewrites_map = {}

    if not os.path.exists(path):
        print(f"⚠️ Rewrites file not found: {path}")
        return rewrites_map

    try:
        rewrites_data = load_json(path)

        # Handle both list and dict formats
        if isinstance(rewrites_data, list):
            for item in rewrites_data:
                qid = item.get("qid") or item.get("task_id")
                rewrite = item.get("rewritten_query") or item.get("rewrite")
                if qid and rewrite:
                    rewrites_map[str(qid)] = rewrite
        elif isinstance(rewrites_data, dict):
            # If it's a dict, assume keys are qids
            rewrites_map = {str(k): str(v) for k, v in rewrites_data.items()}

        print(f"✅ Loaded {len(rewrites_map)} rewrites")

    except Exception as e:
        print(f"❌ Error loading rewrites: {e}")

    return rewrites_map


def azure_cohere_rerank(query: str, docs_text: List[str]) -> List[float]:
    if not query or not docs_text: return [0.0] * len(docs_text)
    headers = {"Content-Type": "application/json", "api-key": AZURE_COHERE_KEY}
    payload = {
        "model": "Cohere-rerank-v4.0-pro",
        "query": query,
        "documents": docs_text,
        "top_n": len(docs_text)
    }
    try:
        response = requests.post(AZURE_COHERE_ENDPOINT, headers=headers, json=payload, timeout=60)
        if response.status_code == 200:
            results = response.json()["results"]
            scores = [0.0] * len(docs_text)
            for r in results:
                scores[r["index"]] = float(r["relevance_score"])
            return scores
    except Exception as e:
        print(f"❌ Cohere API Error: {e}")
    return [0.0] * len(docs_text)


def minmax_norm(scores: List[float]) -> np.ndarray:
    arr = np.array(scores, dtype=np.float32)
    if arr.max() == arr.min(): return np.zeros_like(arr)
    return (arr - arr.min()) / (arr.max() - arr.min())


def run_eval(input_jsonl: str) -> Tuple[float, float]:
    result_file = input_jsonl.replace(".jsonl", "_results.jsonl")
    csv_file = result_file.replace(".jsonl", "_aggregate.csv")
    cmd = f"python3 {EVAL_SCRIPT_PATH} --input_file {input_jsonl} --output_file {result_file}"
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        if "collection" in df.columns:
            df = df[df["collection"] != "all"]
        if df.empty:
            return 0.0, 0.0
        row = df.iloc[0]
        def parse_metric(val):
            if isinstance(val, str) and "[" in val:
                import ast
                return ast.literal_eval(val)[2]
            return val
        return parse_metric(row.get("Recall", 0.0)), parse_metric(row.get("nDCG", 0.0))
    return 0.0, 0.0


# ==============================================================================
# 3. MAIN PIPELINE WITH REWRITES
# ==============================================================================


def main():
    print("="*80)
    print("🔥 COHERE V4 RERANKING WITH REWRITES")
    print("="*80)
    print(f"\n📂 Input: {os.path.basename(INPUT_FILE)}")
    print(f"📚 Corpus: {os.path.basename(CORPUS_FILE)}")
    print(f"📝 Rewrites: {os.path.basename(REWRITES_FILE)}")
    print(f"📊 Reranking top {TOP_K_RERANK} documents per query")
    print(f"💾 Output: {OUTPUT_DIR}\n")

    # Φόρτωση δεδομένων
    print("📥 Loading data...")
    corpus_map = load_corpus(CORPUS_FILE)
    original_queries = load_queries(QUERIES_FILE)  # Fallback
    rewrites_map = load_rewrites(REWRITES_FILE)    # Primary source
    preds = load_jsonl(INPUT_FILE)

    if not preds:
        print("❌ Error: Input file not found or empty.")
        return

    print(f"✅ Loaded {len(corpus_map)} corpus documents")
    print(f"✅ Loaded {len(original_queries)} original queries")
    print(f"✅ Loaded {len(rewrites_map)} rewritten queries")
    print(f"✅ Loaded {len(preds)} predictions\n")

    # Statistics
    queries_with_rewrites = 0
    queries_with_fallback = 0

    performance_summary = []
    cache = []

    # Step 1: Run Reranking and cache results
    print("🎯 Running Cohere reranking...")
    for p in tqdm(preds, desc="Reranking", ncols=100):
        qid = str(p["task_id"])

        # Try to get rewritten query first, fallback to original
        if qid in rewrites_map:
            q_txt = rewrites_map[qid]
            queries_with_rewrites += 1
        else:
            q_txt = original_queries.get(qid, "")
            queries_with_fallback += 1

        top_ctxs = p["contexts"][:TOP_K_RERANK]
        rest_ctxs = p["contexts"][TOP_K_RERANK:]

        doc_texts = [corpus_map.get(str(c["document_id"]), "N/A") for c in top_ctxs]
        coh_scores = azure_cohere_rerank(q_txt, doc_texts)
        orig_scores = [float(c["score"]) for c in top_ctxs]

        cache.append({
            "task_id": qid,
            "Collection": p.get("Collection"),
            "top_ctxs": top_ctxs,
            "rest": rest_ctxs,
            "orig_scores": orig_scores,
            "coh_scores": coh_scores,
            "query_used": q_txt,
            "used_rewrite": qid in rewrites_map
        })

    print(f"\n📊 Query source statistics:")
    print(f"   • Queries with rewrites: {queries_with_rewrites} ({queries_with_rewrites/len(preds)*100:.1f}%)")
    print(f"   • Queries using fallback: {queries_with_fallback} ({queries_with_fallback/len(preds)*100:.1f}%)\n")

    # Step 2: Evaluation - Cohere Only
    print("📊 Evaluating: Cohere-Only...")
    cohere_file = os.path.join(OUTPUT_DIR, "COHERE_ONLY.jsonl")
    with open(cohere_file, "w") as f:
        for item in cache:
            new_ctxs = []
            for i, ctx in enumerate(item["top_ctxs"]):
                nc = ctx.copy()
                nc["score"] = float(item["coh_scores"][i])
                new_ctxs.append(nc)
            new_ctxs.sort(key=lambda x: x["score"], reverse=True)
            f.write(json.dumps({
                "task_id": item["task_id"],
                "Collection": item["Collection"],
                "contexts": new_ctxs + item["rest"]
            }) + "\n")

    r5, n5 = run_eval(cohere_file)
    performance_summary.append({
        "Method": "Cohere-Only",
        "Weight": "1.0",
        "Recall@5": r5,
        "nDCG@5": n5
    })
    print(f"   Recall@5: {r5:.4f} | nDCG@5: {n5:.4f}")

    # Step 3: Evaluation - Hybrid Fusion
    for cw in COHERE_WEIGHTS:
        ow = 1.0 - cw
        print(f"\n📊 Evaluating: Hybrid (Cohere={cw}, Original={ow})...")
        hybrid_file = os.path.join(OUTPUT_DIR, f"HYBRID_cw{cw}.jsonl")

        with open(hybrid_file, "w") as f:
            for item in cache:
                n_orig = minmax_norm(item["orig_scores"])
                n_coh = minmax_norm(item["coh_scores"])
                combined = (ow * n_orig) + (cw * n_coh)

                new_ctxs = []
                for i, ctx in enumerate(item["top_ctxs"]):
                    nc = ctx.copy()
                    nc["score"] = float(combined[i])
                    new_ctxs.append(nc)
                new_ctxs.sort(key=lambda x: x["score"], reverse=True)

                f.write(json.dumps({
                    "task_id": item["task_id"],
                    "Collection": item["Collection"],
                    "contexts": new_ctxs + item["rest"]
                }) + "\n")

        r5_h, n5_h = run_eval(hybrid_file)
        performance_summary.append({
            "Method": "Hybrid",
            "Weight": f"cw={cw}",
            "Recall@5": r5_h,
            "nDCG@5": n5_h
        })
        print(f"   Recall@5: {r5_h:.4f} | nDCG@5: {n5_h:.4f}")

    # Step 4: Original-Only (baseline)
    print(f"\n📊 Evaluating: Original-Only (baseline)...")
    orig_file = os.path.join(OUTPUT_DIR, "ORIGINAL_ONLY.jsonl")
    with open(orig_file, "w") as f:
        for item in cache:
            new_ctxs = []
            for i, ctx in enumerate(item["top_ctxs"]):
                nc = ctx.copy()
                nc["score"] = float(item["orig_scores"][i])
                new_ctxs.append(nc)
            new_ctxs.sort(key=lambda x: x["score"], reverse=True)
            f.write(json.dumps({
                "task_id": item["task_id"],
                "Collection": item["Collection"],
                "contexts": new_ctxs + item["rest"]
            }) + "\n")

    r5_orig, n5_orig = run_eval(orig_file)
    performance_summary.append({
        "Method": "Original-Only",
        "Weight": "0.0",
        "Recall@5": r5_orig,
        "nDCG@5": n5_orig
    })
    print(f"   Recall@5: {r5_orig:.4f} | nDCG@5: {n5_orig:.4f}")

    # Final Report
    print("\n" + "="*80)
    print("🏆 FINAL RESULTS SUMMARY (WITH REWRITES)")
    print("="*80)
    df_final = pd.DataFrame(performance_summary)
    df_final = df_final.sort_values("nDCG@5", ascending=False)
    print(df_final.to_string(index=False))

    # Save detailed report
    report_path = os.path.join(OUTPUT_DIR, "reranking_with_rewrites_report.csv")
    df_final.to_csv(report_path, index=False)

    # Calculate improvement
    baseline_recall = r5_orig
    best_row = df_final.iloc[0]
    improvement = best_row["Recall@5"] - baseline_recall

    print(f"\n🎯 BEST CONFIGURATION:")
    print(f"   Method: {best_row['Method']}")
    print(f"   Weight: {best_row['Weight']}")
    print(f"   Recall@5: {best_row['Recall@5']:.4f} ({improvement:+.4f} vs baseline)")
    print(f"   nDCG@5: {best_row['nDCG@5']:.4f}")

    print(f"\n✅ Report saved: {report_path}")
    print(f"\n📁 All results saved to: {OUTPUT_DIR}")

    # Save query mapping for debugging
    debug_file = os.path.join(OUTPUT_DIR, "query_mapping_debug.json")
    debug_data = []
    for item in cache[:10]:  # First 10 queries
        debug_data.append({
            "task_id": item["task_id"],
            "query_used": item["query_used"],
            "used_rewrite": item["used_rewrite"]
        })
    with open(debug_file, "w", encoding="utf-8") as f:
        json.dump(debug_data, f, indent=2, ensure_ascii=False)
    print(f"\n🔍 Debug info (first 10 queries): {debug_file}")


if __name__ == "__main__":
    main()

In [ ]:
import json
import csv
import os
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from typing import List, Dict, Set

# ==============================================================================
# 1. ΡΥΘΜΙΣΕΙΣ & PATHS
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"
CORPUS = "clapnq"  # Το corpus που εξετάζουμε

# PATHS Αρχείων
QRELS_PATH = os.path.join(BASE_PATH, "retrieval", CORPUS, "dev.tsv")

# Αρχικά αποτελέσματα (πριν το reranking)
INITIAL_FILE = os.path.join(BASE_PATH, f"rrf_results/elserv1_clapnq_rrf.jsonl")

# Αποτελέσματα από το DeepSeek Reranker (Βάλτε το σωστό timestamp/φάκελο)
# Π.χ. "/content/drive/MyDrive/Project/deepseek_stratified_rerank_results/20260118_042324/clapnq_Structural_Minimal_STRATIFIED.jsonl"
RERANKED_FILE = os.path.join(BASE_PATH, "deepseek_reranked_custom/20260118_150723/elserv1_clapnq_rrf_DEEPSEEK_RERANKED.jsonl")

# Βάρη για δοκιμή (Alpha: το βάρος του LLM Reranked σκορ)
ALPHAS = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
K_VALUES = [5, 10, 20]

# ==============================================================================
# 2. HELPER FUNCTIONS (QRELS & RECALL)
# ==============================================================================

def load_qrels(path: str) -> Dict[str, Set[str]]:
    """Φορτώνει τα gold labels από το dev.tsv"""
    qrels = {}
    if not os.path.exists(path):
        print(f"❌ QRELS file not found: {path}")
        return {}
    with open(path, 'r', encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        next(reader) # skip header
        for row in reader:
            qid, docid, score = row[0], row[1], int(row[2])
            if score > 0:
                if qid not in qrels: qrels[qid] = set()
                qrels[qid].add(docid)
    return qrels

def load_results_to_dict(path: str) -> Dict[str, Dict[str, float]]:
    """Φορτώνει jsonl σε λεξικό {task_id: {doc_id: score}}"""
    data = {}
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        return {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            qid = str(j["task_id"])
            # Κανονικοποίηση scores αν χρειάζεται ή απλή ανάγνωση
            data[qid] = {ctx["document_id"]: float(ctx["score"]) for ctx in j["contexts"]}
    return data

def calculate_internal_recall(qrels: Dict[str, Set[str]], results: Dict[str, List[str]], k_values: List[int]):
    """Υπολογίζει το Recall@K εσωτερικά"""
    scores = {k: 0.0 for k in k_values}
    num_queries = len(results)

    for qid, retrieved in results.items():
        gold = qrels.get(qid, set())
        if not gold: continue

        for k in k_values:
            top_k = set(retrieved[:k])
            hits = len(top_k.intersection(gold))
            scores[k] += hits / len(gold)

    return {f"Recall@{k}": round(scores[k] / num_queries, 5) for k in k_values}

# ==============================================================================
# 3. FUSION & EVALUATION LOOP
# ==============================================================================

def main():
    print(f"🚀 Starting Fusion Analysis for Corpus: {CORPUS}")

    # 1. Load Everything
    qrels = load_qrels(QRELS_PATH)
    initial_scores_map = load_results_to_dict(INITIAL_FILE)
    rerank_scores_map = load_results_to_dict(RERANKED_FILE)

    if not qrels or not initial_scores_map or not rerank_scores_map:
        print("❌ Error loading files. Check paths.")
        return

    print(f"✅ Data loaded. Queries in Qrels: {len(qrels)}, Queries in Results: {len(initial_scores_map)}")

    summary_table = []

    # 2. Iterate through Alphas
    for alpha in ALPHAS:
        fused_results_ids = {} # {qid: [doc_id1, doc_id2, ...]}

        for qid in initial_scores_map.keys():
            init_dict = initial_scores_map[qid]
            rerank_dict = rerank_scores_map.get(qid, {})

            # Ένωση όλων των document IDs που βρέθηκαν
            all_dids = set(init_dict.keys()).union(set(rerank_dict.keys()))

            combined = []
            for did in all_dids:
                s_init = init_dict.get(did, 0.0)
                s_rerank = rerank_dict.get(did, 0.0)

                # Formula: Alpha * Rerank_Score + (1-Alpha) * Initial_Score
                fused_score = (alpha * s_rerank) + ((1 - alpha) * s_init)
                combined.append((did, fused_score))

            # Ταξινόμηση βάσει του νέου σκορ (Descending)
            combined.sort(key=lambda x: x[1], reverse=True)
            fused_results_ids[qid] = [x[0] for x in combined]

        # 3. Calculate Metrics
        metrics = calculate_internal_recall(qrels, fused_results_ids, K_VALUES)
        metrics["Alpha"] = alpha
        summary_table.append(metrics)

        print(f"Done Alpha={alpha:.1f} | R@5: {metrics['Recall@5']:.4f}")

    # ==================================================
    # 4. FINAL RESULTS DISPLAY
    # ==================================================
    df_results = pd.DataFrame(summary_table)
    # Αναδιάταξη στηλών
    cols = ["Alpha"] + [f"Recall@{k}" for k in K_VALUES]
    df_results = df_results[cols]

    print("\n" + "="*60)
    print(f"{'WEIGHTED FUSION RECALL ANALYSIS (' + CORPUS.upper() + ')':^60}")
    print("="*60)
    print(df_results.to_string(index=False))
    print("="*60)

    # Εύρεση βέλτιστου Alpha
    best_idx = df_results["Recall@5"].idxmax()
    print(f"\n🌟 Best Alpha for Recall@5: {df_results.iloc[best_idx]['Alpha']}")
    print(f"   Max Recall@5: {df_results.iloc[best_idx]['Recall@5']:.4f}")

if __name__ == "__main__":
    main()

In [ ]:
import os, json, time, requests, subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import Dict, List, Tuple


# ==============================================================================
# 1. CONFIGURATION - ALL 4 DOMAINS
# ==============================================================================

BASE_PATH = "/content/drive/MyDrive/Project"

# Domain configurations
DOMAIN_CONFIGS = {
    "clapnq": {
        "predictions": f"{BASE_PATH}/final_nested_ensembles/clapnq_nested_rrf_final.jsonl",
        "corpus": f"{BASE_PATH}/corpora/clapnq.jsonl",
        "queries": f"{BASE_PATH}/retrieval/clapnq/clapnq_lastturn.jsonl",
        "rewrites": f"{BASE_PATH}/elser_v1_structural_6U3A_predictions/clapnq_elser_minimal_optimized_20260112_114149_REWRITES.json"
    },
    "fiqa": {
        "predictions": f"{BASE_PATH}/final_nested_ensembles/fiqa_nested_rrf_final.jsonl",
        "corpus": f"{BASE_PATH}/corpora/fiqa.jsonl",
        "queries": f"{BASE_PATH}/retrieval/fiqa/fiqa_lastturn.jsonl",
        "rewrites": f"{BASE_PATH}/elser_v1_structural_6U3A_predictions/fiqa_elser_minimal_optimized_20260112_115931_REWRITES.json"
    },
    "govt": {
        "predictions": f"{BASE_PATH}/final_nested_ensembles/govt_nested_rrf_final.jsonl",
        "corpus": f"{BASE_PATH}/corpora/govt.jsonl",
        "queries": f"{BASE_PATH}/retrieval/govt/govt_lastturn.jsonl",
        "rewrites": f"{BASE_PATH}/elser_v1_structural_6U3A_predictions/govt_elser_minimal_optimized_20260112_121434_REWRITES.json"
    },
    "cloud": {
        "predictions": f"{BASE_PATH}/final_nested_ensembles/cloud_nested_rrf_final.jsonl",
        "corpus": f"{BASE_PATH}/corpora/cloud.jsonl",
        "queries": f"{BASE_PATH}/retrieval/cloud/cloud_lastturn.jsonl",
        "rewrites": f"{BASE_PATH}/elser_v1_structural_6U3A_predictions/cloud_elser_minimal_optimized_20260112_122934_REWRITES.json"
    }
}

# Output directory
RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_BASE = os.path.join(BASE_PATH, "cohere_v4_nested_ensemble_rerank", RUN_TAG)
os.makedirs(OUTPUT_BASE, exist_ok=True)

# Azure Cohere Credentials
AZURE_COHERE_ENDPOINT = os.environ["ENDPOINT"]
AZURE_COHERE_KEY = os.environ["_KEY"]

EVAL_SCRIPT_PATH = os.path.join(BASE_PATH, "run_retrieval_eval.py")
TOP_K_RERANK = 100
COHERE_WEIGHTS = [0.1, 0.3, 0.5, 0.7, 0.9]


# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================

def load_jsonl(path: str) -> List[dict]:
    data = []
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        return data
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data


def load_json(path: str) -> Dict:
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_corpus(path: str) -> Dict[str, str]:
    corpus = {}
    if not os.path.exists(path): return corpus
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            did = str(j.get("_id") or j.get("id"))
            text = (j.get("title", "") + " " + (j.get("text") or j.get("contents") or "")).strip()
            corpus[did] = text[:2000]
    return corpus


def load_queries(path: str) -> Dict[str, str]:
    queries = {}
    if not os.path.exists(path): return queries
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            qid = str(j.get("_id") or j.get("id") or j.get("task_id"))
            queries[qid] = j.get("text") or j.get("query")
    return queries


def load_rewrites(path: str) -> Dict[str, str]:
    rewrites_map = {}
    if not os.path.exists(path):
        print(f"⚠️ Rewrites file not found: {path}")
        return rewrites_map

    try:
        rewrites_data = load_json(path)

        if isinstance(rewrites_data, list):
            for item in rewrites_data:
                qid = item.get("qid") or item.get("task_id")
                rewrite = item.get("rewritten_query") or item.get("rewrite")
                if qid and rewrite:
                    rewrites_map[str(qid)] = rewrite
        elif isinstance(rewrites_data, dict):
            rewrites_map = {str(k): str(v) for k, v in rewrites_data.items()}

        print(f"   ✅ Loaded {len(rewrites_map)} rewrites")
    except Exception as e:
        print(f"   ❌ Error loading rewrites: {e}")

    return rewrites_map


def azure_cohere_rerank(query: str, docs_text: List[str]) -> List[float]:
    if not query or not docs_text: return [0.0] * len(docs_text)
    headers = {"Content-Type": "application/json", "api-key": AZURE_COHERE_KEY}
    payload = {
        "model": "Cohere-rerank-v4.0-pro",
        "query": query,
        "documents": docs_text,
        "top_n": len(docs_text)
    }
    try:
        response = requests.post(AZURE_COHERE_ENDPOINT, headers=headers, json=payload, timeout=60)
        if response.status_code == 200:
            results = response.json()["results"]
            scores = [0.0] * len(docs_text)
            for r in results:
                scores[r["index"]] = float(r["relevance_score"])
            return scores
        elif response.status_code == 429:
            time.sleep(2)
            return azure_cohere_rerank(query, docs_text)
    except Exception as e:
        print(f"❌ Cohere API Error: {e}")
    return [0.0] * len(docs_text)


def minmax_norm(scores: List[float]) -> np.ndarray:
    arr = np.array(scores, dtype=np.float32)
    if arr.max() == arr.min(): return np.zeros_like(arr)
    return (arr - arr.min()) / (arr.max() - arr.min())


def run_eval(input_jsonl: str) -> Tuple[float, float]:
    result_file = input_jsonl.replace(".jsonl", "_results.jsonl")
    csv_file = result_file.replace(".jsonl", "_aggregate.csv")
    cmd = f"python3 {EVAL_SCRIPT_PATH} --input_file {input_jsonl} --output_file {result_file}"
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        if "collection" in df.columns:
            df = df[df["collection"] != "all"]
        if df.empty:
            return 0.0, 0.0
        row = df.iloc[0]

        def parse_metric(val):
            if isinstance(val, str) and "[" in val:
                import ast
                return ast.literal_eval(val)[2]
            return val

        return parse_metric(row.get("Recall", 0.0)), parse_metric(row.get("nDCG", 0.0))
    return 0.0, 0.0


# ==============================================================================
# 3. PROCESS SINGLE DOMAIN
# ==============================================================================

def process_domain(domain_name: str, config: Dict) -> pd.DataFrame:
    """Process a single domain and return performance results"""

    print("\n" + "="*80)
    print(f"🎯 PROCESSING DOMAIN: {domain_name.upper()}")
    print("="*80)

    # Create domain-specific output directory
    domain_output = os.path.join(OUTPUT_BASE, domain_name)
    os.makedirs(domain_output, exist_ok=True)

    # Load data
    print("📥 Loading data...")
    corpus_map = load_corpus(config["corpus"])
    original_queries = load_queries(config["queries"])
    rewrites_map = load_rewrites(config["rewrites"])
    preds = load_jsonl(config["predictions"])

    if not preds:
        print(f"❌ Error: No predictions found for {domain_name}")
        return pd.DataFrame()

    print(f"✅ Loaded {len(corpus_map)} corpus documents")
    print(f"✅ Loaded {len(original_queries)} original queries")
    print(f"✅ Loaded {len(preds)} predictions")

    # Statistics
    queries_with_rewrites = sum(1 for p in preds if str(p["task_id"]) in rewrites_map)
    queries_with_fallback = len(preds) - queries_with_rewrites

    print(f"\n📊 Query source statistics:")
    print(f"   • Queries with rewrites: {queries_with_rewrites} ({queries_with_rewrites/len(preds)*100:.1f}%)")
    print(f"   • Queries using fallback: {queries_with_fallback} ({queries_with_fallback/len(preds)*100:.1f}%)")

    performance_summary = []
    cache = []

    # Step 1: Run Reranking
    print("\n🎯 Running Cohere reranking...")
    for p in tqdm(preds, desc=f"Reranking {domain_name}", ncols=100):
        qid = str(p["task_id"])

        # Try rewritten query first, fallback to original
        if qid in rewrites_map:
            q_txt = rewrites_map[qid]
        else:
            q_txt = original_queries.get(qid, "")

        top_ctxs = p["contexts"][:TOP_K_RERANK]
        rest_ctxs = p["contexts"][TOP_K_RERANK:]

        doc_texts = [corpus_map.get(str(c["document_id"]), "N/A") for c in top_ctxs]
        coh_scores = azure_cohere_rerank(q_txt, doc_texts)
        orig_scores = [float(c["score"]) for c in top_ctxs]

        cache.append({
            "task_id": qid,
            "Collection": p.get("Collection"),
            "top_ctxs": top_ctxs,
            "rest": rest_ctxs,
            "orig_scores": orig_scores,
            "coh_scores": coh_scores
        })

    # Step 2: Cohere-Only
    print("\n📊 Evaluating: Cohere-Only...")
    cohere_file = os.path.join(domain_output, f"{domain_name}_COHERE_ONLY.jsonl")
    with open(cohere_file, "w") as f:
        for item in cache:
            new_ctxs = []
            for i, ctx in enumerate(item["top_ctxs"]):
                nc = ctx.copy()
                nc["score"] = float(item["coh_scores"][i])
                new_ctxs.append(nc)
            new_ctxs.sort(key=lambda x: x["score"], reverse=True)
            f.write(json.dumps({
                "task_id": item["task_id"],
                "Collection": item["Collection"],
                "contexts": new_ctxs + item["rest"]
            }) + "\n")

    r5, n5 = run_eval(cohere_file)
    performance_summary.append({
        "Domain": domain_name,
        "Method": "Cohere-Only",
        "Weight": "1.0",
        "Recall@5": r5,
        "nDCG@5": n5
    })
    print(f"   Recall@5: {r5:.4f} | nDCG@5: {n5:.4f}")

    # Step 3: Hybrid Fusion
    for cw in COHERE_WEIGHTS:
        ow = 1.0 - cw
        print(f"\n📊 Evaluating: Hybrid (Cohere={cw}, Original={ow})...")
        hybrid_file = os.path.join(domain_output, f"{domain_name}_HYBRID_cw{cw}.jsonl")

        with open(hybrid_file, "w") as f:
            for item in cache:
                n_orig = minmax_norm(item["orig_scores"])
                n_coh = minmax_norm(item["coh_scores"])
                combined = (ow * n_orig) + (cw * n_coh)

                new_ctxs = []
                for i, ctx in enumerate(item["top_ctxs"]):
                    nc = ctx.copy()
                    nc["score"] = float(combined[i])
                    new_ctxs.append(nc)
                new_ctxs.sort(key=lambda x: x["score"], reverse=True)

                f.write(json.dumps({
                    "task_id": item["task_id"],
                    "Collection": item["Collection"],
                    "contexts": new_ctxs + item["rest"]
                }) + "\n")

        r5_h, n5_h = run_eval(hybrid_file)
        performance_summary.append({
            "Domain": domain_name,
            "Method": "Hybrid",
            "Weight": f"cw={cw}",
            "Recall@5": r5_h,
            "nDCG@5": n5_h
        })
        print(f"   Recall@5: {r5_h:.4f} | nDCG@5: {n5_h:.4f}")

    # Step 4: Original-Only (baseline)
    print(f"\n📊 Evaluating: Original-Only (baseline)...")
    orig_file = os.path.join(domain_output, f"{domain_name}_ORIGINAL_ONLY.jsonl")
    with open(orig_file, "w") as f:
        for item in cache:
            new_ctxs = []
            for i, ctx in enumerate(item["top_ctxs"]):
                nc = ctx.copy()
                nc["score"] = float(item["orig_scores"][i])
                new_ctxs.append(nc)
            new_ctxs.sort(key=lambda x: x["score"], reverse=True)
            f.write(json.dumps({
                "task_id": item["task_id"],
                "Collection": item["Collection"],
                "contexts": new_ctxs + item["rest"]
            }) + "\n")

    r5_orig, n5_orig = run_eval(orig_file)
    performance_summary.append({
        "Domain": domain_name,
        "Method": "Original-Only",
        "Weight": "0.0",
        "Recall@5": r5_orig,
        "nDCG@5": n5_orig
    })
    print(f"   Recall@5: {r5_orig:.4f} | nDCG@5: {n5_orig:.4f}")

    # Domain summary
    df_domain = pd.DataFrame(performance_summary)
    df_domain = df_domain.sort_values("nDCG@5", ascending=False)

    print(f"\n🏆 BEST RESULT FOR {domain_name.upper()}:")
    best_row = df_domain.iloc[0]
    improvement = best_row["Recall@5"] - r5_orig
    print(f"   Method: {best_row['Method']}")
    print(f"   Weight: {best_row['Weight']}")
    print(f"   Recall@5: {best_row['Recall@5']:.4f} ({improvement:+.4f} vs baseline)")
    print(f"   nDCG@5: {best_row['nDCG@5']:.4f}")

    # Save domain report
    report_path = os.path.join(domain_output, f"{domain_name}_report.csv")
    df_domain.to_csv(report_path, index=False)
    print(f"\n✅ Domain report saved: {report_path}")

    return df_domain


# ==============================================================================
# 4. MAIN - BATCH PROCESSING ALL DOMAINS
# ==============================================================================

def main():
    print("="*80)
    print("🔥 COHERE V4 RERANKING - NESTED ENSEMBLE (ALL 4 DOMAINS)")
    print("="*80)
    print(f"\n📂 Processing domains: {', '.join(DOMAIN_CONFIGS.keys())}")
    print(f"💾 Output base: {OUTPUT_BASE}\n")

    all_results = []

    # Process each domain
    for domain_name, config in DOMAIN_CONFIGS.items():
        try:
            df_domain = process_domain(domain_name, config)
            all_results.append(df_domain)
        except Exception as e:
            print(f"\n❌ Error processing {domain_name}: {e}")
            import traceback
            traceback.print_exc()

    # Combine all results
    if all_results:
        df_combined = pd.concat(all_results, ignore_index=True)

        # Save combined report
        combined_report_path = os.path.join(OUTPUT_BASE, "ALL_DOMAINS_COMBINED_REPORT.csv")
        df_combined.to_csv(combined_report_path, index=False)

        # Final summary
        print("\n" + "="*80)
        print("🎉 ALL DOMAINS PROCESSING COMPLETE")
        print("="*80)

        print("\n📊 BEST RESULTS PER DOMAIN:")
        print("="*80)
        for domain in DOMAIN_CONFIGS.keys():
            df_domain = df_combined[df_combined["Domain"] == domain]
            if not df_domain.empty:
                best = df_domain.loc[df_domain["nDCG@5"].idxmax()]
                baseline = df_domain[df_domain["Method"] == "Original-Only"]["Recall@5"].values[0]
                improvement = best["Recall@5"] - baseline

                print(f"\n{domain.upper()}:")
                print(f"   Method: {best['Method']} | Weight: {best['Weight']}")
                print(f"   Recall@5: {best['Recall@5']:.4f} ({improvement:+.4f})")
                print(f"   nDCG@5: {best['nDCG@5']:.4f}")

        print(f"\n✅ Combined report: {combined_report_path}")
        print(f"📁 All outputs: {OUTPUT_BASE}")


if __name__ == "__main__":
    main()


In [ ]:
import os, json, time, requests, subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import Dict, List, Tuple



# ==============================================================================
# 1. CONFIGURATION - ALL 4 DOMAINS
# ==============================================================================


BASE_PATH = "/content/drive/MyDrive/Project"


# Domain configurations
DOMAIN_CONFIGS = {
    "clapnq": {
        "predictions": f"{BASE_PATH}/final_nested_ensembles/clapnq_nested_rrf_final.jsonl",
        "corpus": f"{BASE_PATH}/corpora/clapnq.jsonl",
        "queries": f"{BASE_PATH}/retrieval/clapnq/clapnq_lastturn.jsonl",
        "collection": "mt-rag-clapnq-elser-512-100-20240503"
    },
    "fiqa": {
        "predictions": f"{BASE_PATH}/final_nested_ensembles/fiqa_nested_rrf_final.jsonl",
        "corpus": f"{BASE_PATH}/corpora/fiqa.jsonl",
        "queries": f"{BASE_PATH}/retrieval/fiqa/fiqa_lastturn.jsonl",
        "collection": "mt-rag-fiqa-beir-elser-512-100-20240501"
    },
    "govt": {
        "predictions": f"{BASE_PATH}/final_nested_ensembles/govt_nested_rrf_final.jsonl",
        "corpus": f"{BASE_PATH}/corpora/govt.jsonl",
        "queries": f"{BASE_PATH}/retrieval/govt/govt_lastturn.jsonl",
        "collection": "mt-rag-govt-elser-512-100-20240611"
    },
    "cloud": {
        "predictions": f"{BASE_PATH}/final_nested_ensembles/cloud_nested_rrf_final.jsonl",
        "corpus": f"{BASE_PATH}/corpora/cloud.jsonl",
        "queries": f"{BASE_PATH}/retrieval/cloud/cloud_lastturn.jsonl",
        "collection": "mt-rag-ibmcloud-elser-512-100-20240502"
    }
}


# Output directory
RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_BASE = os.path.join(BASE_PATH, "cohere_v4_nested_ensemble_rerank", RUN_TAG)
os.makedirs(OUTPUT_BASE, exist_ok=True)


# Azure Cohere Credentials
AZURE_COHERE_ENDPOINT = os.environ["ENDPOINT"]
AZURE_COHERE_KEY = os.environ["_KEY"]


EVAL_SCRIPT_PATH = os.path.join(BASE_PATH, "run_retrieval_eval.py")
TOP_K_RERANK = 100
COHERE_WEIGHTS = [0.1, 0.3, 0.5, 0.7, 0.9]



# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================


def load_jsonl(path: str) -> List[dict]:
    data = []
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        return data
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data



def load_corpus(path: str) -> Dict[str, str]:
    corpus = {}
    if not os.path.exists(path): return corpus
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            did = str(j.get("_id") or j.get("id"))
            text = (j.get("title", "") + " " + (j.get("text") or j.get("contents") or "")).strip()
            corpus[did] = text[:2000]
    return corpus



def load_queries(path: str) -> Dict[str, str]:
    """Load queries from lastturn JSONL files"""
    queries = {}
    if not os.path.exists(path):
        print(f"⚠️ Queries file not found: {path}")
        return queries

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            qid = str(j.get("_id") or j.get("id") or j.get("task_id"))
            # Extract text and remove the "|user|: " prefix if present
            raw_text = j.get("text") or j.get("query", "")
            if raw_text.startswith("|user|: "):
                queries[qid] = raw_text.replace("|user|: ", "", 1)
            else:
                queries[qid] = raw_text

    return queries



def azure_cohere_rerank(query: str, docs_text: List[str]) -> List[float]:
    if not query or not docs_text: return [0.0] * len(docs_text)
    headers = {"Content-Type": "application/json", "api-key": AZURE_COHERE_KEY}
    payload = {
        "model": "Cohere-rerank-v4.0-pro",
        "query": query,
        "documents": docs_text,
        "top_n": len(docs_text)
    }
    try:
        response = requests.post(AZURE_COHERE_ENDPOINT, headers=headers, json=payload, timeout=60)
        if response.status_code == 200:
            results = response.json()["results"]
            scores = [0.0] * len(docs_text)
            for r in results:
                scores[r["index"]] = float(r["relevance_score"])
            return scores
        elif response.status_code == 429:
            time.sleep(2)
            return azure_cohere_rerank(query, docs_text)
    except Exception as e:
        print(f"❌ Cohere API Error: {e}")
    return [0.0] * len(docs_text)



def minmax_norm(scores: List[float]) -> np.ndarray:
    arr = np.array(scores, dtype=np.float32)
    if arr.max() == arr.min(): return np.zeros_like(arr)
    return (arr - arr.min()) / (arr.max() - arr.min())



def run_eval(input_jsonl: str) -> Tuple[float, float]:
    result_file = input_jsonl.replace(".jsonl", "_results.jsonl")
    csv_file = result_file.replace(".jsonl", "_aggregate.csv")
    cmd = f"python3 {EVAL_SCRIPT_PATH} --input_file {input_jsonl} --output_file {result_file}"
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        if "collection" in df.columns:
            df = df[df["collection"] != "all"]
        if df.empty:
            return 0.0, 0.0
        row = df.iloc[0]

        def parse_metric(val):
            if isinstance(val, str) and "[" in val:
                import ast
                return ast.literal_eval(val)[2]
            return val

        return parse_metric(row.get("Recall", 0.0)), parse_metric(row.get("nDCG", 0.0))
    return 0.0, 0.0



# ==============================================================================
# 3. PROCESS SINGLE DOMAIN
# ==============================================================================


def process_domain(domain_name: str, config: Dict) -> pd.DataFrame:
    """Process a single domain and return performance results"""

    print("\n" + "="*80)
    print(f"🎯 PROCESSING DOMAIN: {domain_name.upper()}")
    print("="*80)

    # Create domain-specific output directory
    domain_output = os.path.join(OUTPUT_BASE, domain_name)
    os.makedirs(domain_output, exist_ok=True)

    # Load data
    print("📥 Loading data...")
    corpus_map = load_corpus(config["corpus"])
    original_queries = load_queries(config["queries"])
    preds = load_jsonl(config["predictions"])
    collection_name = config["collection"]

    if not preds:
        print(f"❌ Error: No predictions found for {domain_name}")
        return pd.DataFrame()

    print(f"✅ Loaded {len(corpus_map)} corpus documents")
    print(f"✅ Loaded {len(original_queries)} original queries")
    print(f"✅ Loaded {len(preds)} predictions")
    print(f"✅ Collection name: {collection_name}")

    # Statistics
    queries_found = sum(1 for p in preds if str(p["task_id"]) in original_queries)
    queries_missing = len(preds) - queries_found

    print(f"\n📊 Query coverage:")
    print(f"   • Queries found: {queries_found} ({queries_found/len(preds)*100:.1f}%)")
    print(f"   • Queries missing: {queries_missing} ({queries_missing/len(preds)*100:.1f}%)")

    performance_summary = []
    cache = []

    # Step 1: Run Reranking
    print("\n🎯 Running Cohere reranking with ORIGINAL queries...")
    for p in tqdm(preds, desc=f"Reranking {domain_name}", ncols=100):
        qid = str(p["task_id"])

        # Use original query
        q_txt = original_queries.get(qid, "")

        if not q_txt:
            print(f"⚠️  Warning: No query found for task_id: {qid}")

        top_ctxs = p["contexts"][:TOP_K_RERANK]
        rest_ctxs = p["contexts"][TOP_K_RERANK:]

        doc_texts = [corpus_map.get(str(c["document_id"]), "N/A") for c in top_ctxs]
        coh_scores = azure_cohere_rerank(q_txt, doc_texts)
        orig_scores = [float(c["score"]) for c in top_ctxs]

        cache.append({
            "task_id": qid,
            "Collection": collection_name,
            "top_ctxs": top_ctxs,
            "rest": rest_ctxs,
            "orig_scores": orig_scores,
            "coh_scores": coh_scores
        })

    # Step 2: Cohere-Only
    print("\n📊 Evaluating: Cohere-Only...")
    cohere_file = os.path.join(domain_output, f"{domain_name}_COHERE_ONLY.jsonl")
    with open(cohere_file, "w") as f:
        for item in cache:
            new_ctxs = []
            for i, ctx in enumerate(item["top_ctxs"]):
                nc = ctx.copy()
                nc["score"] = float(item["coh_scores"][i])
                new_ctxs.append(nc)
            new_ctxs.sort(key=lambda x: x["score"], reverse=True)
            f.write(json.dumps({
                "task_id": item["task_id"],
                "Collection": item["Collection"],
                "contexts": new_ctxs + item["rest"]
            }) + "\n")

    r5, n5 = run_eval(cohere_file)
    performance_summary.append({
        "Domain": domain_name,
        "Method": "Cohere-Only",
        "Weight": "1.0",
        "Recall@5": r5,
        "nDCG@5": n5
    })
    print(f"   Recall@5: {r5:.4f} | nDCG@5: {n5:.4f}")

    # Step 3: Hybrid Fusion
    for cw in COHERE_WEIGHTS:
        ow = 1.0 - cw
        print(f"\n📊 Evaluating: Hybrid (Cohere={cw}, Original={ow})...")
        hybrid_file = os.path.join(domain_output, f"{domain_name}_HYBRID_cw{cw}.jsonl")

        with open(hybrid_file, "w") as f:
            for item in cache:
                n_orig = minmax_norm(item["orig_scores"])
                n_coh = minmax_norm(item["coh_scores"])
                combined = (ow * n_orig) + (cw * n_coh)

                new_ctxs = []
                for i, ctx in enumerate(item["top_ctxs"]):
                    nc = ctx.copy()
                    nc["score"] = float(combined[i])
                    new_ctxs.append(nc)
                new_ctxs.sort(key=lambda x: x["score"], reverse=True)

                f.write(json.dumps({
                    "task_id": item["task_id"],
                    "Collection": item["Collection"],
                    "contexts": new_ctxs + item["rest"]
                }) + "\n")

        r5_h, n5_h = run_eval(hybrid_file)
        performance_summary.append({
            "Domain": domain_name,
            "Method": "Hybrid",
            "Weight": f"cw={cw}",
            "Recall@5": r5_h,
            "nDCG@5": n5_h
        })
        print(f"   Recall@5: {r5_h:.4f} | nDCG@5: {n5_h:.4f}")

    # Step 4: Original-Only (baseline)
    print(f"\n📊 Evaluating: Original-Only (baseline)...")
    orig_file = os.path.join(domain_output, f"{domain_name}_ORIGINAL_ONLY.jsonl")
    with open(orig_file, "w") as f:
        for item in cache:
            new_ctxs = []
            for i, ctx in enumerate(item["top_ctxs"]):
                nc = ctx.copy()
                nc["score"] = float(item["orig_scores"][i])
                new_ctxs.append(nc)
            new_ctxs.sort(key=lambda x: x["score"], reverse=True)
            f.write(json.dumps({
                "task_id": item["task_id"],
                "Collection": item["Collection"],
                "contexts": new_ctxs + item["rest"]
            }) + "\n")

    r5_orig, n5_orig = run_eval(orig_file)
    performance_summary.append({
        "Domain": domain_name,
        "Method": "Original-Only",
        "Weight": "0.0",
        "Recall@5": r5_orig,
        "nDCG@5": n5_orig
    })
    print(f"   Recall@5: {r5_orig:.4f} | nDCG@5: {n5_orig:.4f}")

    # Domain summary
    df_domain = pd.DataFrame(performance_summary)
    df_domain = df_domain.sort_values("nDCG@5", ascending=False)

    print(f"\n🏆 BEST RESULT FOR {domain_name.upper()}:")
    best_row = df_domain.iloc[0]
    improvement = best_row["Recall@5"] - r5_orig
    print(f"   Method: {best_row['Method']}")
    print(f"   Weight: {best_row['Weight']}")
    print(f"   Recall@5: {best_row['Recall@5']:.4f} ({improvement:+.4f} vs baseline)")
    print(f"   nDCG@5: {best_row['nDCG@5']:.4f}")

    # Save domain report
    report_path = os.path.join(domain_output, f"{domain_name}_report.csv")
    df_domain.to_csv(report_path, index=False)
    print(f"\n✅ Domain report saved: {report_path}")

    return df_domain



# ==============================================================================
# 4. MAIN - BATCH PROCESSING ALL DOMAINS
# ==============================================================================


def main():
    print("="*80)
    print("🔥 COHERE V4 RERANKING - NESTED ENSEMBLE (ALL 4 DOMAINS)")
    print("🔹 Using ORIGINAL queries (no rewrites)")
    print("="*80)
    print(f"\n📂 Processing domains: {', '.join(DOMAIN_CONFIGS.keys())}")
    print(f"💾 Output base: {OUTPUT_BASE}\n")

    all_results = []

    # Process each domain
    for domain_name, config in DOMAIN_CONFIGS.items():
        try:
            df_domain = process_domain(domain_name, config)
            all_results.append(df_domain)
        except Exception as e:
            print(f"\n❌ Error processing {domain_name}: {e}")
            import traceback
            traceback.print_exc()

    # Combine all results
    if all_results:
        df_combined = pd.concat(all_results, ignore_index=True)

        # Save combined report
        combined_report_path = os.path.join(OUTPUT_BASE, "ALL_DOMAINS_COMBINED_REPORT.csv")
        df_combined.to_csv(combined_report_path, index=False)

        # Final summary
        print("\n" + "="*80)
        print("🎉 ALL DOMAINS PROCESSING COMPLETE")
        print("="*80)

        print("\n📊 BEST RESULTS PER DOMAIN:")
        print("="*80)
        for domain in DOMAIN_CONFIGS.keys():
            df_domain = df_combined[df_combined["Domain"] == domain]
            if not df_domain.empty:
                best = df_domain.loc[df_domain["nDCG@5"].idxmax()]
                baseline = df_domain[df_domain["Method"] == "Original-Only"]["Recall@5"].values[0]
                improvement = best["Recall@5"] - baseline

                print(f"\n{domain.upper()}:")
                print(f"   Method: {best['Method']} | Weight: {best['Weight']}")
                print(f"   Recall@5: {best['Recall@5']:.4f} ({improvement:+.4f})")
                print(f"   nDCG@5: {best['nDCG@5']:.4f}")

        print(f"\n✅ Combined report: {combined_report_path}")
        print(f"📁 All outputs: {OUTPUT_BASE}")



if __name__ == "__main__":
    main()

In [ ]:
import os
import json
from glob import glob

# ==============================================================================
# CONFIGURATION
# ==============================================================================

RUN_DIR = "/content/drive/MyDrive/Project/cohere_v4_nested_ensemble_rerank/20260119_034313"

# Domain to collection name mapping
DOMAIN_TO_COLLECTION = {
    "clapnq": "mt-rag-clapnq-elser-512-100-20240503",
    "fiqa": "mt-rag-fiqa-beir-elser-512-100-20240501",
    "govt": "mt-rag-govt-elser-512-100-20240611",
    "cloud": "mt-rag-ibmcloud-elser-512-100-20240502"
}

DOMAINS = ["clapnq", "fiqa", "govt", "cloud"]

# ==============================================================================
# FIX COLLECTION NAMES
# ==============================================================================

def fix_collection_in_file(filepath, collection_name):
    """Fix Collection field in a JSONL file"""
    print(f"   🔧 Fixing: {os.path.basename(filepath)}")

    # Read all lines
    lines = []
    with open(filepath, 'r') as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                data['Collection'] = collection_name
                lines.append(json.dumps(data) + '\n')

    # Write back
    with open(filepath, 'w') as f:
        f.writelines(lines)

    print(f"      ✅ Fixed {len(lines)} lines")

def main():
    print("="*80)
    print("🔧 FIXING COLLECTION NAMES IN ALL FILES")
    print("="*80)
    print(f"📂 Run directory: {RUN_DIR}\n")

    total_fixed = 0

    for domain in DOMAINS:
        print(f"\n{'='*80}")
        print(f"🎯 DOMAIN: {domain.upper()}")
        print("="*80)

        domain_dir = os.path.join(RUN_DIR, domain)

        if not os.path.exists(domain_dir):
            print(f"   ⚠️  Directory not found: {domain_dir}")
            continue

        collection_name = DOMAIN_TO_COLLECTION[domain]
        print(f"   📝 Collection name: {collection_name}")

        # Find all JSONL files (excluding results files)
        jsonl_files = [f for f in glob(os.path.join(domain_dir, "*.jsonl"))
                       if "_results.jsonl" not in f]

        if not jsonl_files:
            print(f"   ⚠️  No JSONL files found")
            continue

        print(f"   📊 Found {len(jsonl_files)} files\n")

        for jsonl_file in sorted(jsonl_files):
            try:
                fix_collection_in_file(jsonl_file, collection_name)
                total_fixed += 1
            except Exception as e:
                print(f"      ❌ Error: {e}")

    print(f"\n{'='*80}")
    print("🎉 COLLECTION NAMES FIXED!")
    print("="*80)
    print(f"\n✅ Total files fixed: {total_fixed}")
    print(f"\n📋 Now you can run the original evaluation:")
    print(f"   !python rerun_evaluation.py")
    print("="*80)

if __name__ == "__main__":
    main()

In [ ]:
import os
import json
import subprocess
import pandas as pd
from glob import glob

# ==============================================================================
# CONFIGURATION
# ==============================================================================

BASE_PATH = "/content/drive/MyDrive/Project"
RUN_DIR = "/content/drive/MyDrive/Project/cohere_v4_nested_ensemble_rerank/20260119_034313"
EVAL_SCRIPT_PATH = os.path.join(BASE_PATH, "run_retrieval_eval.py")

DOMAINS = ["clapnq", "fiqa", "govt", "cloud"]

# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================

def run_eval(input_jsonl: str):
    """Run evaluation on a single JSONL file"""
    result_file = input_jsonl.replace(".jsonl", "_results.jsonl")
    csv_file = result_file.replace(".jsonl", "_aggregate.csv")

    print(f"   📊 Evaluating: {os.path.basename(input_jsonl)}")

    cmd = f"python3 {EVAL_SCRIPT_PATH} --input_file {input_jsonl} --output_file {result_file}"
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        if "collection" in df.columns:
            df = df[df["collection"] != "all"]
        if df.empty:
            return 0.0, 0.0
        row = df.iloc[0]

        def parse_metric(val):
            if isinstance(val, str) and "[" in val:
                import ast
                return ast.literal_eval(val)[2]
            return val

        recall = parse_metric(row.get("Recall", 0.0))
        ndcg = parse_metric(row.get("nDCG", 0.0))
        print(f"      ✅ Recall@5: {recall:.4f} | nDCG@5: {ndcg:.4f}")
        return recall, ndcg
    else:
        print(f"      ❌ CSV not found: {csv_file}")
        return 0.0, 0.0

# ==============================================================================
# MAIN EVALUATION
# ==============================================================================

def main():
    print("="*80)
    print("🔄 RE-RUNNING EVALUATION FOR ALL GENERATED FILES")
    print("="*80)
    print(f"📂 Run directory: {RUN_DIR}\n")

    all_results = []

    for domain in DOMAINS:
        print(f"\n{'='*80}")
        print(f"🎯 DOMAIN: {domain.upper()}")
        print("="*80)

        domain_dir = os.path.join(RUN_DIR, domain)

        if not os.path.exists(domain_dir):
            print(f"   ⚠️  Directory not found: {domain_dir}")
            continue

        # Find all JSONL files (excluding results files)
        jsonl_files = [f for f in glob(os.path.join(domain_dir, "*.jsonl"))
                       if "_results.jsonl" not in f]

        if not jsonl_files:
            print(f"   ⚠️  No JSONL files found in {domain_dir}")
            continue

        print(f"   Found {len(jsonl_files)} files to evaluate\n")

        for jsonl_file in sorted(jsonl_files):
            filename = os.path.basename(jsonl_file)

            # Determine method and weight
            if "COHERE_ONLY" in filename:
                method = "Cohere-Only"
                weight = "1.0"
            elif "ORIGINAL_ONLY" in filename:
                method = "Original-Only"
                weight = "0.0"
            elif "HYBRID" in filename:
                method = "Hybrid"
                # Extract weight from filename
                import re
                match = re.search(r'cw([0-9.]+)', filename)
                weight = f"cw={match.group(1)}" if match else "unknown"
            else:
                method = "Unknown"
                weight = "N/A"

            recall, ndcg = run_eval(jsonl_file)

            all_results.append({
                "Domain": domain,
                "Method": method,
                "Weight": weight,
                "Recall@5": recall,
                "nDCG@5": ndcg,
                "File": filename
            })

    # Create results DataFrame
    if all_results:
        df_results = pd.DataFrame(all_results)

        # Save combined report
        output_csv = os.path.join(RUN_DIR, "EVALUATION_RESULTS_COMPLETE.csv")
        df_results.to_csv(output_csv, index=False)

        print(f"\n{'='*80}")
        print("🎉 EVALUATION COMPLETE")
        print("="*80)

        # Display summary per domain
        print("\n📊 BEST RESULTS PER DOMAIN:")
        print("="*80)

        for domain in DOMAINS:
            df_domain = df_results[df_results["Domain"] == domain]
            if not df_domain.empty:
                # Find best by nDCG
                best = df_domain.loc[df_domain["nDCG@5"].idxmax()]
                # Find baseline
                baseline_rows = df_domain[df_domain["Method"] == "Original-Only"]
                baseline_recall = baseline_rows["Recall@5"].values[0] if len(baseline_rows) > 0 else 0.0

                improvement = best["Recall@5"] - baseline_recall

                print(f"\n{domain.upper()}:")
                print(f"   Method: {best['Method']} | Weight: {best['Weight']}")
                print(f"   Recall@5: {best['Recall@5']:.4f} ({improvement:+.4f} vs baseline)")
                print(f"   nDCG@5: {best['nDCG@5']:.4f}")

        # Display full results table
        print(f"\n{'='*80}")
        print("📋 FULL RESULTS TABLE:")
        print("="*80)
        print(df_results[["Domain", "Method", "Weight", "Recall@5", "nDCG@5"]].to_string(index=False))

        print(f"\n✅ Results saved to: {output_csv}")
        print(f"📁 All files in: {RUN_DIR}")
    else:
        print("\n❌ No results generated!")

if __name__ == "__main__":
    main()

In [ ]:
import os, json, csv
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from typing import Dict, List, Tuple


# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
BASE_PATH = "/content/drive/MyDrive/Project"

# Τα 3 αρχεία που θέλεις να συνδυάσεις
FILES = {
    "Original": f"{BASE_PATH}/final_nested_ensembles/fiqa_nested_rrf_final.jsonl",
    "R1_Rerank": f"{BASE_PATH}/deepseek_two_stage_experiments/20260119_140058_TWO_STAGE_4STRATEGIES/fiqa/s1_coarsetofine/S1_CoarseToFine_hybrid_0.3.jsonl",
    "Cohere_Rerank": f"{BASE_PATH}/deepseek_two_stage_experiments/20260119_140058_TWO_STAGE_4STRATEGIES/fiqa/s1_coarsetofine/S1_CoarseToFine_hybrid_0.5.jsonl"
}

OUTPUT_DIR = f"{BASE_PATH}/triple_ensemble_test"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DOMAIN = "fiqa"
K_VALUES = [5, 10, 20, 50, 100]
TOP_K_RERANK = 20  # Μόνο top-20 reranking (όπως R1 hybrid)


# ==============================================================================
# 2. WEIGHT COMBINATIONS TO TEST
# ==============================================================================
WEIGHT_CONFIGS = [
    # Baseline
    (1.0, 0.0, 0.0, "Original_Only"),
    (0.0, 1.0, 0.0, "R1_Only"),
    (0.0, 0.0, 1.0, "Cohere_Only"),

    # Pairwise
    (0.7, 0.3, 0.0, "Original_R1_50_50"),
    (0.7, 0.0, 0.3, "Original_Cohere_50_50"),
    (0.0, 0.5, 0.5, "R1_Cohere_50_50"),

    # Triple Equal
    (0.33, 0.33, 0.34, "Triple_Equal"),

    # Triple Cohere-focused
    (0.2, 0.2, 0.6, "Triple_Cohere_Heavy"),
    (0.3, 0.2, 0.5, "Triple_Cohere_Medium"),
    (0.25, 0.25, 0.5, "Triple_Cohere_Balanced"),

    # Triple Original-focused
    (0.5, 0.2, 0.3, "Triple_Original_Heavy"),
    (0.6, 0.1, 0.3, "Triple_Original_Dominant"),

    # Triple Balanced
    (0.4, 0.3, 0.3, "Triple_Balanced_1"),
    (0.3, 0.4, 0.3, "Triple_Balanced_2"),
    (0.3, 0.3, 0.4, "Triple_Balanced_3"),
]


# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================
def load_qrels(domain: str) -> Dict[str, set]:
    """Load ground truth qrels"""
    path = os.path.join(BASE_PATH, "retrieval", domain, "dev.tsv")
    qrels = {}
    if not os.path.exists(path):
        print(f"⚠️ Qrels not found: {path}")
        return qrels

    with open(path, 'r', encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        next(reader)
        for row in reader:
            qid, did, score = row[0], row[1], int(row[2])
            if score > 0:
                if qid not in qrels:
                    qrels[qid] = set()
                qrels[qid].add(did)
    return qrels


def load_predictions(path: str) -> Dict[str, List[Tuple[str, float]]]:
    """Load predictions as {qid: [(doc_id, score), ...]}"""
    data = {}
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        return data

    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            qid = str(item["task_id"])
            contexts = item.get("contexts", [])
            data[qid] = [(str(c["document_id"]), float(c.get("score", 0.0))) for c in contexts]
    return data


def minmax_norm(scores: List[float]) -> np.ndarray:
    """MinMax normalization to 0-1"""
    arr = np.array(scores, dtype=np.float32)
    if arr.max() == arr.min():
        return np.zeros_like(arr)
    return (arr - arr.min()) / (arr.max() - arr.min())


def combine_rankings_local(orig_data, r1_data, cohere_data, w_orig, w_r1, w_cohere, top_k=20):
    """
    LOCAL RERANKING (R1 Hybrid style):
    - Rerank μόνο τα top-K docs
    - Τα υπόλοιπα μένουν στις αρχικές θέσεις

    Returns: {qid: [doc_id1, doc_id2, ...]}
    """
    combined = {}

    # Get all query IDs from original (baseline)
    all_qids = set(orig_data.keys())

    for qid in all_qids:
        # Get original ranking (this is our reference)
        orig_ranking = [doc_id for doc_id, score in orig_data.get(qid, [])]

        if not orig_ranking:
            combined[qid] = []
            continue

        # Split into top-K and rest
        top_k_docs = orig_ranking[:top_k]
        rest_docs = orig_ranking[top_k:]

        # Get scores for top-K from all sources
        orig_dict = dict(orig_data.get(qid, []))
        r1_dict = dict(r1_data.get(qid, []))
        cohere_dict = dict(cohere_data.get(qid, []))

        # Extract scores only for top-K docs
        orig_scores = [orig_dict.get(d, 0.0) for d in top_k_docs]
        r1_scores = [r1_dict.get(d, 0.0) for d in top_k_docs]
        cohere_scores = [cohere_dict.get(d, 0.0) for d in top_k_docs]

        # Normalize scores
        norm_orig = minmax_norm(orig_scores)
        norm_r1 = minmax_norm(r1_scores)
        norm_cohere = minmax_norm(cohere_scores)

        # Combine with weights
        doc_scores = {}
        for i, doc_id in enumerate(top_k_docs):
            combined_score = (
                w_orig * norm_orig[i] +
                w_r1 * norm_r1[i] +
                w_cohere * norm_cohere[i]
            )
            doc_scores[doc_id] = float(combined_score)

        # Sort top-K by combined score
        reranked_top_k = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
        reranked_top_k_ids = [doc_id for doc_id, score in reranked_top_k]

        # Combine: reranked top-K + unchanged rest
        combined[qid] = reranked_top_k_ids + rest_docs

    return combined


def calculate_metrics(results: Dict[str, List[str]], qrels: Dict[str, set], k_values: List[int]) -> Dict:
    """Calculate Recall@K metrics"""
    metrics = {}

    for k in k_values:
        total_recall = 0.0
        count = 0

        for qid, retrieved in results.items():
            gold = qrels.get(qid, set())
            if not gold:
                continue

            top_k = set(retrieved[:k])
            recall = len(top_k.intersection(gold)) / len(gold)
            total_recall += recall
            count += 1

        metrics[f"Recall@{k}"] = round(total_recall / count, 5) if count > 0 else 0.0

    return metrics


# ==============================================================================
# 4. MAIN EXECUTION
# ==============================================================================
def main():
    print("=" * 80)
    print("🔬 TRIPLE ENSEMBLE - LOCAL RERANKING (R1 Hybrid Style)")
    print("=" * 80)
    print(f"📊 Reranking scope: Top-{TOP_K_RERANK} documents")
    print(f"📊 Testing {len(WEIGHT_CONFIGS)} weight combinations")
    print(f"📊 Domain: {DOMAIN}\n")

    # Load qrels
    print("📥 Loading ground truth qrels...")
    qrels = load_qrels(DOMAIN)
    print(f"✅ Loaded {len(qrels)} queries with relevance judgments\n")

    # Load predictions
    print("📥 Loading predictions...")
    orig_data = load_predictions(FILES["Original"])
    r1_data = load_predictions(FILES["R1_Rerank"])
    cohere_data = load_predictions(FILES["Cohere_Rerank"])

    print(f"✅ Original: {len(orig_data)} queries")
    print(f"✅ R1 Rerank: {len(r1_data)} queries")
    print(f"✅ Cohere Rerank: {len(cohere_data)} queries\n")

    # Test all weight combinations
    all_results = []

    print("🔄 Testing weight combinations with local reranking...\n")
    for w_orig, w_r1, w_cohere, name in tqdm(WEIGHT_CONFIGS, desc="Evaluating"):
        # Combine rankings (LOCAL - top-K only)
        combined = combine_rankings_local(
            orig_data, r1_data, cohere_data,
            w_orig, w_r1, w_cohere,
            top_k=TOP_K_RERANK
        )

        # Calculate metrics
        metrics = calculate_metrics(combined, qrels, K_VALUES)
        metrics["Configuration"] = name
        metrics["Weight_Original"] = w_orig
        metrics["Weight_R1"] = w_r1
        metrics["Weight_Cohere"] = w_cohere

        all_results.append(metrics)

    # Create results DataFrame
    df_results = pd.DataFrame(all_results)

    # Reorder columns
    cols = ["Configuration", "Weight_Original", "Weight_R1", "Weight_Cohere"] + [f"Recall@{k}" for k in K_VALUES]
    df_results = df_results[cols]

    # Sort by Recall@5
    df_results = df_results.sort_values("Recall@5", ascending=False)

    # Display results
    print("\n" + "=" * 120)
    print("🏆 FINAL RESULTS - SORTED BY Recall@5 (LOCAL RERANKING)")
    print("=" * 120)
    print(df_results.to_string(index=False))

    # Save CSV report
    report_path = os.path.join(OUTPUT_DIR, "local_ensemble_comparison_report.csv")
    df_results.to_csv(report_path, index=False)

    # Best configuration
    best = df_results.iloc[0]
    baseline = df_results[df_results["Configuration"] == "Original_Only"].iloc[0]

    improvement_r5 = best["Recall@5"] - baseline["Recall@5"]
    improvement_r10 = best["Recall@10"] - baseline["Recall@10"]

    print(f"\n{'=' * 120}")
    print("🥇 BEST CONFIGURATION")
    print(f"{'=' * 120}")
    print(f"Name: {best['Configuration']}")
    print(f"Weights: Original={best['Weight_Original']:.2f} | R1={best['Weight_R1']:.2f} | Cohere={best['Weight_Cohere']:.2f}")
    print(f"\nMetrics:")
    print(f"  Recall@5:   {best['Recall@5']:.5f} ({improvement_r5:+.5f} vs baseline)")
    print(f"  Recall@10:  {best['Recall@10']:.5f} ({improvement_r10:+.5f} vs baseline)")
    print(f"  Recall@20:  {best['Recall@20']:.5f}")
    print(f"  Recall@50:  {best['Recall@50']:.5f}")
    print(f"  Recall@100: {best['Recall@100']:.5f}")

    print(f"\n✅ Report saved: {report_path}")

    # Top 5 configurations
    print(f"\n{'=' * 120}")
    print("📊 TOP 5 CONFIGURATIONS")
    print(f"{'=' * 120}")
    top5_cols = ["Configuration", "Weight_Original", "Weight_R1", "Weight_Cohere", "Recall@5", "Recall@10"]
    print(df_results.head(5)[top5_cols].to_string(index=False))

    # Baseline comparison
    print(f"\n{'=' * 120}")
    print("📈 SINGLE-SOURCE BASELINE COMPARISON")
    print(f"{'=' * 120}")
    baselines = df_results[df_results["Configuration"].isin(["Original_Only", "R1_Only", "Cohere_Only"])]
    print(baselines[top5_cols].to_string(index=False))

    # Analysis note
    print(f"\n{'=' * 120}")
    print("💡 ANALYSIS NOTE")
    print(f"{'=' * 120}")
    print(f"Method: LOCAL RERANKING (like R1 Hybrid)")
    print(f"  - Reranks only top-{TOP_K_RERANK} documents")
    print(f"  - Documents {TOP_K_RERANK+1}-100 maintain original positions")
    print(f"  - Fast and focused, but limited coverage")
    print(f"\nExpected improvement: +2-4% vs baseline")
    print(f"Actual improvement: {improvement_r5:+.5f} ({improvement_r5/baseline['Recall@5']*100:+.2f}%)")


if __name__ == "__main__":
    main()


In [ ]:
import os, json, time, requests, subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import Dict, List, Tuple


# ==============================================================================
# 1. SET YOUR FILE PATHS HERE 📍
# ==============================================================================
# Η διαδρομή του αρχείου που θέλεις να επεξεργαστείς
INPUT_FILE = "/content/drive/MyDrive/Project/elser_v1_cot_hyde_predictions/cloud_elser_hyde_6U3A_20260114_165333.jsonl"


# Οι απαραίτητες διαδρομές για το συγκεκριμένο αρχείο
CORPUS_FILE = "/content/drive/MyDrive/Project/corpora/cloud.jsonl"
QUERIES_FILE = "/content/drive/MyDrive/Project/retrieval/cloud/cloud_lastturn.jsonl"
REWRITES_FILE = "/content/drive/MyDrive/Project/elser_v1_cot_hyde_predictions/cloud_elser_hyde_6U3A_20260114_165333_REWRITES.json"


# Φάκελος Εξόδου
BASE_PATH = "/content/drive/MyDrive/Project"
OUTPUT_DIR = os.path.join(BASE_PATH, "cohere_v4_rewrite2_rerank", time.strftime("%Y%m%d_%H%M%S"))
os.makedirs(OUTPUT_DIR, exist_ok=True)


# Azure Cohere Credentials
AZURE_COHERE_ENDPOINT = os.environ["ENDPOINT"]
AZURE_COHERE_KEY = os.environ["_KEY"]


EVAL_SCRIPT_PATH = os.path.join(BASE_PATH, "run_retrieval_eval.py")
TOP_K_RERANK = 100
COHERE_WEIGHTS = [0.1, 0.3, 0.5, 0.7, 0.9]


# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================


def load_jsonl(path: str) -> List[dict]:
    data = []
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        return data
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data


def load_json(path: str) -> Dict:
    """Load JSON file (for rewrites)"""
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_corpus(path: str) -> Dict[str, str]:
    corpus = {}
    if not os.path.exists(path): return corpus
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            did = str(j.get("_id") or j.get("id"))
            text = (j.get("title", "") + " " + (j.get("text") or j.get("contents") or "")).strip()
            corpus[did] = text[:2000]
    return corpus


def load_queries(path: str) -> Dict[str, str]:
    """Load original queries (fallback only)"""
    queries = {}
    if not os.path.exists(path): return queries
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            qid = str(j.get("_id") or j.get("id") or j.get("task_id"))
            queries[qid] = j.get("text") or j.get("query")
    return queries


def load_rewrites(path: str) -> Dict[str, str]:
    """
    Load rewrites from JSON file and create mapping

    Returns:
        Dict mapping task_id -> rewritten_query
    """
    rewrites_map = {}

    if not os.path.exists(path):
        print(f"⚠️ Rewrites file not found: {path}")
        return rewrites_map

    try:
        rewrites_data = load_json(path)

        # Handle both list and dict formats
        if isinstance(rewrites_data, list):
            for item in rewrites_data:
                qid = item.get("qid") or item.get("task_id")
                rewrite = item.get("rewritten_query") or item.get("rewrite")
                if qid and rewrite:
                    rewrites_map[str(qid)] = rewrite
        elif isinstance(rewrites_data, dict):
            # If it's a dict, assume keys are qids
            rewrites_map = {str(k): str(v) for k, v in rewrites_data.items()}

        print(f"✅ Loaded {len(rewrites_map)} rewrites")

    except Exception as e:
        print(f"❌ Error loading rewrites: {e}")

    return rewrites_map


def azure_cohere_rerank(query: str, docs_text: List[str]) -> List[float]:
    if not query or not docs_text: return [0.0] * len(docs_text)
    headers = {"Content-Type": "application/json", "api-key": AZURE_COHERE_KEY}
    payload = {
        "model": "Cohere-rerank-v4.0-pro",
        "query": query,
        "documents": docs_text,
        "top_n": len(docs_text)
    }
    try:
        response = requests.post(AZURE_COHERE_ENDPOINT, headers=headers, json=payload, timeout=60)
        if response.status_code == 200:
            results = response.json()["results"]
            scores = [0.0] * len(docs_text)
            for r in results:
                scores[r["index"]] = float(r["relevance_score"])
            return scores
    except Exception as e:
        print(f"❌ Cohere API Error: {e}")
    return [0.0] * len(docs_text)


def minmax_norm(scores: List[float]) -> np.ndarray:
    arr = np.array(scores, dtype=np.float32)
    if arr.max() == arr.min(): return np.zeros_like(arr)
    return (arr - arr.min()) / (arr.max() - arr.min())


def run_eval(input_jsonl: str) -> Tuple[float, float]:
    result_file = input_jsonl.replace(".jsonl", "_results.jsonl")
    csv_file = result_file.replace(".jsonl", "_aggregate.csv")
    cmd = f"python3 {EVAL_SCRIPT_PATH} --input_file {input_jsonl} --output_file {result_file}"
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        if "collection" in df.columns:
            df = df[df["collection"] != "all"]
        if df.empty:
            return 0.0, 0.0
        row = df.iloc[0]
        def parse_metric(val):
            if isinstance(val, str) and "[" in val:
                import ast
                return ast.literal_eval(val)[2]
            return val
        return parse_metric(row.get("Recall", 0.0)), parse_metric(row.get("nDCG", 0.0))
    return 0.0, 0.0


# ==============================================================================
# 3. MAIN PIPELINE WITH REWRITES
# ==============================================================================


def main():
    print("="*80)
    print("🔥 COHERE V4 RERANKING WITH REWRITES")
    print("="*80)
    print(f"\n📂 Input: {os.path.basename(INPUT_FILE)}")
    print(f"📚 Corpus: {os.path.basename(CORPUS_FILE)}")
    print(f"📝 Rewrites: {os.path.basename(REWRITES_FILE)}")
    print(f"📊 Reranking top {TOP_K_RERANK} documents per query")
    print(f"💾 Output: {OUTPUT_DIR}\n")

    # Φόρτωση δεδομένων
    print("📥 Loading data...")
    corpus_map = load_corpus(CORPUS_FILE)
    original_queries = load_queries(QUERIES_FILE)  # Fallback
    rewrites_map = load_rewrites(REWRITES_FILE)    # Primary source
    preds = load_jsonl(INPUT_FILE)

    if not preds:
        print("❌ Error: Input file not found or empty.")
        return

    print(f"✅ Loaded {len(corpus_map)} corpus documents")
    print(f"✅ Loaded {len(original_queries)} original queries")
    print(f"✅ Loaded {len(rewrites_map)} rewritten queries")
    print(f"✅ Loaded {len(preds)} predictions\n")

    # Statistics
    queries_with_rewrites = 0
    queries_with_fallback = 0

    performance_summary = []
    cache = []

    # Step 1: Run Reranking and cache results
    print("🎯 Running Cohere reranking...")
    for p in tqdm(preds, desc="Reranking", ncols=100):
        qid = str(p["task_id"])

        # Try to get rewritten query first, fallback to original
        if qid in rewrites_map:
            q_txt = rewrites_map[qid]
            queries_with_rewrites += 1
        else:
            q_txt = original_queries.get(qid, "")
            queries_with_fallback += 1

        top_ctxs = p["contexts"][:TOP_K_RERANK]
        rest_ctxs = p["contexts"][TOP_K_RERANK:]

        doc_texts = [corpus_map.get(str(c["document_id"]), "N/A") for c in top_ctxs]
        coh_scores = azure_cohere_rerank(q_txt, doc_texts)
        orig_scores = [float(c["score"]) for c in top_ctxs]

        cache.append({
            "task_id": qid,
            "Collection": p.get("Collection"),
            "top_ctxs": top_ctxs,
            "rest": rest_ctxs,
            "orig_scores": orig_scores,
            "coh_scores": coh_scores,
            "query_used": q_txt,
            "used_rewrite": qid in rewrites_map
        })

    print(f"\n📊 Query source statistics:")
    print(f"   • Queries with rewrites: {queries_with_rewrites} ({queries_with_rewrites/len(preds)*100:.1f}%)")
    print(f"   • Queries using fallback: {queries_with_fallback} ({queries_with_fallback/len(preds)*100:.1f}%)\n")

    # Step 2: Evaluation - Cohere Only
    print("📊 Evaluating: Cohere-Only...")
    cohere_file = os.path.join(OUTPUT_DIR, "COHERE_ONLY.jsonl")
    with open(cohere_file, "w") as f:
        for item in cache:
            new_ctxs = []
            for i, ctx in enumerate(item["top_ctxs"]):
                nc = ctx.copy()
                nc["score"] = float(item["coh_scores"][i])
                new_ctxs.append(nc)
            new_ctxs.sort(key=lambda x: x["score"], reverse=True)
            f.write(json.dumps({
                "task_id": item["task_id"],
                "Collection": item["Collection"],
                "contexts": new_ctxs + item["rest"]
            }) + "\n")

    r5, n5 = run_eval(cohere_file)
    performance_summary.append({
        "Method": "Cohere-Only",
        "Weight": "1.0",
        "Recall@5": r5,
        "nDCG@5": n5
    })
    print(f"   Recall@5: {r5:.4f} | nDCG@5: {n5:.4f}")

    # Step 3: Evaluation - Hybrid Fusion
    for cw in COHERE_WEIGHTS:
        ow = 1.0 - cw
        print(f"\n📊 Evaluating: Hybrid (Cohere={cw}, Original={ow})...")
        hybrid_file = os.path.join(OUTPUT_DIR, f"HYBRID_cw{cw}.jsonl")

        with open(hybrid_file, "w") as f:
            for item in cache:
                n_orig = minmax_norm(item["orig_scores"])
                n_coh = minmax_norm(item["coh_scores"])
                combined = (ow * n_orig) + (cw * n_coh)

                new_ctxs = []
                for i, ctx in enumerate(item["top_ctxs"]):
                    nc = ctx.copy()
                    nc["score"] = float(combined[i])
                    new_ctxs.append(nc)
                new_ctxs.sort(key=lambda x: x["score"], reverse=True)

                f.write(json.dumps({
                    "task_id": item["task_id"],
                    "Collection": item["Collection"],
                    "contexts": new_ctxs + item["rest"]
                }) + "\n")

        r5_h, n5_h = run_eval(hybrid_file)
        performance_summary.append({
            "Method": "Hybrid",
            "Weight": f"cw={cw}",
            "Recall@5": r5_h,
            "nDCG@5": n5_h
        })
        print(f"   Recall@5: {r5_h:.4f} | nDCG@5: {n5_h:.4f}")

    # Step 4: Original-Only (baseline)
    print(f"\n📊 Evaluating: Original-Only (baseline)...")
    orig_file = os.path.join(OUTPUT_DIR, "ORIGINAL_ONLY.jsonl")
    with open(orig_file, "w") as f:
        for item in cache:
            new_ctxs = []
            for i, ctx in enumerate(item["top_ctxs"]):
                nc = ctx.copy()
                nc["score"] = float(item["orig_scores"][i])
                new_ctxs.append(nc)
            new_ctxs.sort(key=lambda x: x["score"], reverse=True)
            f.write(json.dumps({
                "task_id": item["task_id"],
                "Collection": item["Collection"],
                "contexts": new_ctxs + item["rest"]
            }) + "\n")

    r5_orig, n5_orig = run_eval(orig_file)
    performance_summary.append({
        "Method": "Original-Only",
        "Weight": "0.0",
        "Recall@5": r5_orig,
        "nDCG@5": n5_orig
    })
    print(f"   Recall@5: {r5_orig:.4f} | nDCG@5: {n5_orig:.4f}")

    # Final Report
    print("\n" + "="*80)
    print("🏆 FINAL RESULTS SUMMARY (WITH REWRITES)")
    print("="*80)
    df_final = pd.DataFrame(performance_summary)
    df_final = df_final.sort_values("nDCG@5", ascending=False)
    print(df_final.to_string(index=False))

    # Save detailed report
    report_path = os.path.join(OUTPUT_DIR, "reranking_with_rewrites_report.csv")
    df_final.to_csv(report_path, index=False)

    # Calculate improvement
    baseline_recall = r5_orig
    best_row = df_final.iloc[0]
    improvement = best_row["Recall@5"] - baseline_recall

    print(f"\n🎯 BEST CONFIGURATION:")
    print(f"   Method: {best_row['Method']}")
    print(f"   Weight: {best_row['Weight']}")
    print(f"   Recall@5: {best_row['Recall@5']:.4f} ({improvement:+.4f} vs baseline)")
    print(f"   nDCG@5: {best_row['nDCG@5']:.4f}")

    print(f"\n✅ Report saved: {report_path}")
    print(f"\n📁 All results saved to: {OUTPUT_DIR}")

    # Save query mapping for debugging
    debug_file = os.path.join(OUTPUT_DIR, "query_mapping_debug.json")
    debug_data = []
    for item in cache[:10]:  # First 10 queries
        debug_data.append({
            "task_id": item["task_id"],
            "query_used": item["query_used"],
            "used_rewrite": item["used_rewrite"]
        })
    with open(debug_file, "w", encoding="utf-8") as f:
        json.dump(debug_data, f, indent=2, ensure_ascii=False)
    print(f"\n🔍 Debug info (first 10 queries): {debug_file}")


if __name__ == "__main__":
    main()

# **Generation**

In [ ]:
# ================================================================
# HYBRID v5.7 — "v5.6 + User-Satisfaction Judge"
# ================================================================
# 🎯 v5.7 ADDITIONS:
#    • User-Satisfaction Judge (Stage 6b)
#    • Judges from USER perspective: "Does this satisfy what I asked?"
#    • Triggers on 30% of non-IDK answers
#    • All v5.6 features maintained
# ================================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install -q -c /content/drive/MyDrive/Project/constraints.txt -r /content/drive/MyDrive/Project/requirements.txt
!pip install -q pandas numpy huggingface_hub tqdm requests

import os
import sys
import json
import re
import time
import random
import requests
import threading
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm
from huggingface_hub import login
from concurrent.futures import ThreadPoolExecutor, as_completed

# =========================
# HF LOGIN
# =========================
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if HF_TOKEN:
    login(token=HF_TOKEN)

# =========================
# AZURE DEEPSEEK CONFIGURATION
# =========================
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

# =========================
# 🎯 v5.7 CONFIGURATION
# =========================
N_PASSAGES = 5

# Pipeline stages
ENABLE_SPAN_EXTRACTION = True
ENABLE_SPAN_RETRY = True
ENABLE_MULTI_SAMPLING = True
ENABLE_SELECTIVE_JUDGE = True          # Stage 6a: Technical Judge
ENABLE_USER_SATISFACTION_JUDGE = True  # 🔥 NEW Stage 6b: User Judge
ENABLE_EXTRACTIVE_POLISH = True
ENABLE_COMPRESSION = False

# Answerability
ANSWERABILITY_BIAS = "ANSWERABLE"

# Retry
RETRY_IF_IDK_ON_ANSWERABLE = True
NUM_RETRIES = 2

# Span extraction
MIN_SPANS = 1
MAX_SPANS = 8
SPAN_RETRY_ATTEMPTS = 2

# Generation
NUM_CANDIDATES = 2
GENERATION_TEMPERATURE_1 = 0.0
GENERATION_TEMPERATURE_2 = 0.1

# Word count
SUGGESTED_MIN = 80
SUGGESTED_TARGET = 90
SUGGESTED_MAX = 105

SUGGESTED_PARTIAL_MIN = 60
SUGGESTED_PARTIAL_TARGET = 70
SUGGESTED_PARTIAL_MAX = 85

# Extractiveness
TARGET_EXTRACTIVENESS = 0.32

# Soft trim
SOFT_TRIM_THRESHOLD = 300

# Technical Judge (Stage 6a)
JUDGE_WC_MIN = 50
JUDGE_WC_MAX = 150
JUDGE_EXTR_MIN = 0.10

# 🔥 NEW: User-Satisfaction Judge (Stage 6b)
USER_SAT_JUDGE_SAMPLE_RATE = 0.30           # Run on 30% of non-IDK answers
USER_SAT_CONFIDENCE_THRESHOLD = ["HIGH", "MEDIUM"]  # Regen on HIGH/MEDIUM confidence

# IDK & Conversational
IDK_PHRASE = "I do not have specific information"
MAX_WORDS_CONV = 50

# Debug
DEBUG_MODE = True
MAX_WORKERS = 5

# Set random seed for reproducibility
random.seed(42)

# =========================
# STATS TRACKER
# =========================
class StatsTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.answerability_counts = {"ANSWERABLE": 0, "PARTIAL": 0, "UNANSWERABLE": 0, "CONVERSATIONAL": 0}
        self.idk_count = 0
        self.full_answer_count = 0
        self.answerable_to_idk_count = 0
        self.span_retry_count = 0
        self.judge_trigger_count = 0
        self.judge_regenerate_count = 0
        # 🔥 NEW stats
        self.user_sat_judge_count = 0
        self.user_sat_unsatisfied_count = 0
        self.user_sat_regenerate_count = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt, completion):
        with self.lock:
            self.total_input += prompt
            self.total_output += completion

    def add_answerability(self, verdict: str):
        with self.lock:
            self.answerability_counts[verdict] = self.answerability_counts.get(verdict, 0) + 1

    def add_prediction(self, is_idk: bool, answerability: str = None):
        with self.lock:
            if is_idk:
                self.idk_count += 1
                if answerability in ["ANSWERABLE", "PARTIAL"]:
                    self.answerable_to_idk_count += 1
            else:
                self.full_answer_count += 1

    def add_span_retry(self):
        with self.lock:
            self.span_retry_count += 1

    def add_judge_trigger(self, regenerated: bool = False):
        with self.lock:
            self.judge_trigger_count += 1
            if regenerated:
                self.judge_regenerate_count += 1

    def add_user_sat_judge(self, unsatisfied: bool = False, regenerated: bool = False):
        with self.lock:
            self.user_sat_judge_count += 1
            if unsatisfied:
                self.user_sat_unsatisfied_count += 1
            if regenerated:
                self.user_sat_regenerate_count += 1

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        total = sum(self.answerability_counts.values())

        report = f"\n{'='*70}\n"
        report += "📊 GENERATION STATISTICS (v5.7 - User-Satisfaction Judge)\n"
        report += f"{'='*70}\n"

        report += f"\n💰 Token Usage:\n"
        report += f"   Input:  {self.total_input:,}\n"
        report += f"   Output: {self.total_output:,}\n"
        report += f"   Cost:   ${cost:.4f}\n"

        report += f"\n🎯 Answerability Predictions (Total: {total}):\n"
        for verdict, count in sorted(self.answerability_counts.items()):
            pct = (count / total * 100) if total > 0 else 0
            report += f"   {verdict:15s}: {count:4d} ({pct:5.1f}%)\n"

        total_preds = self.idk_count + self.full_answer_count
        report += f"\n📝 Final Predictions (Total: {total_preds}):\n"
        idk_pct = (self.idk_count / total_preds * 100) if total_preds > 0 else 0
        ans_pct = (self.full_answer_count / total_preds * 100) if total_preds > 0 else 0
        report += f"   IDK Responses:   {self.idk_count:4d} ({idk_pct:5.1f}%)\n"
        report += f"   Full Answers:    {self.full_answer_count:4d} ({ans_pct:5.1f}%)\n"

        if self.span_retry_count > 0:
            report += f"\n🔄 Span Extraction Retries: {self.span_retry_count}\n"

        if self.judge_trigger_count > 0:
            judge_rate = (self.judge_trigger_count / total_preds * 100) if total_preds > 0 else 0
            regen_rate = (self.judge_regenerate_count / self.judge_trigger_count * 100) if self.judge_trigger_count > 0 else 0
            report += f"\n👨‍⚖️ Technical Judge (Stage 6a):\n"
            report += f"   Triggered:    {self.judge_trigger_count:4d} ({judge_rate:5.1f}% of tasks)\n"
            report += f"   Regenerated:  {self.judge_regenerate_count:4d} ({regen_rate:5.1f}% of judged)\n"

        # 🔥 NEW stats
        if self.user_sat_judge_count > 0:
            usr_judge_rate = (self.user_sat_judge_count / total_preds * 100) if total_preds > 0 else 0
            unsat_rate = (self.user_sat_unsatisfied_count / self.user_sat_judge_count * 100) if self.user_sat_judge_count > 0 else 0
            regen_rate = (self.user_sat_regenerate_count / self.user_sat_unsatisfied_count * 100) if self.user_sat_unsatisfied_count > 0 else 0
            report += f"\n👤 User-Satisfaction Judge (Stage 6b):\n"
            report += f"   Evaluated:    {self.user_sat_judge_count:4d} ({usr_judge_rate:5.1f}% of tasks)\n"
            report += f"   Unsatisfied:  {self.user_sat_unsatisfied_count:4d} ({unsat_rate:5.1f}% of evaluated)\n"
            report += f"   Regenerated:  {self.user_sat_regenerate_count:4d} ({regen_rate:5.1f}% of unsatisfied)\n"

        if self.answerable_to_idk_count > 0:
            answerable_total = self.answerability_counts.get("ANSWERABLE", 0) + self.answerability_counts.get("PARTIAL", 0)
            false_refusal_rate = (self.answerable_to_idk_count / answerable_total * 100) if answerable_total > 0 else 0
            report += f"\n⚠️  ANSWERABLE → IDK Cases: {self.answerable_to_idk_count} ({false_refusal_rate:.1f}%)\n"

        report += f"\n📖 v5.7 Features:\n"
        report += f"   MAX_SPANS: 8\n"
        report += f"   Target Extractiveness: 32%\n"
        report += f"   Technical Judge: ENABLED\n"
        report += f"   🔥 User-Satisfaction Judge: ENABLED ({int(USER_SAT_JUDGE_SAMPLE_RATE*100)}% sample)\n"

        report += f"{'='*70}\n"
        return report

tracker = StatsTracker()

# =========================
# HELPERS (SAME as v5.6)
# =========================
def word_count(text: str) -> int:
    return len([w for w in (text or "").strip().split() if w])

def safe_strip_quotes(s: str) -> str:
    s = (s or "").strip()
    if len(s) >= 2 and ((s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'")):
        return s[1:-1].strip()
    return s

def clean_model_output(text: str) -> str:
    t = safe_strip_quotes((text or "").strip())
    t = re.sub(r"^(Agent:|Assistant:)\s*", "", t.strip(), flags=re.IGNORECASE)
    t = re.sub(r"\n{3,}", "\n\n", t)
    t = re.sub(r"[ \t]+", " ", t)
    return t.strip()

def soft_trim_to_word_limit_by_sentences(text: str, max_words: int) -> str:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    kept = []
    for sentence in sentences:
        candidate = " ".join(kept + [sentence])
        if word_count(candidate) <= max_words:
            kept.append(sentence)
        else:
            break

    result = " ".join(kept).strip()

    if not result and sentences:
        result = " ".join(sentences[:3])

    return result

def is_pure_idk(text: str) -> bool:
    t = (text or "").strip()
    if not t:
        return False
    low = t.lower()
    canonical = IDK_PHRASE.lower()
    if re.sub(r"[.\s]+", " ", low).strip() == re.sub(r"[.\s]+", " ", canonical).strip():
        return True
    idk_indicators = [
        "i don't know", "i do not know",
        "do not have enough information", "don't have enough information",
        "do not have specific information", "don't have specific information",
        "insufficient information", "not enough information",
    ]
    if any(ind in low for ind in idk_indicators) and word_count(t) <= 24:
        return True
    return False

def normalize_idk_partial_safe(text: str) -> str:
    t = (text or "").strip()
    if not t:
        return t
    if is_pure_idk(t):
        return IDK_PHRASE
    return t

def get_ngrams(text: str, n: int) -> set:
    words = text.lower().split()
    return set(' '.join(words[i:i+n]) for i in range(len(words)-n+1))

def calculate_extractiveness(answer: str, contexts: List[Dict]) -> float:
    answer_grams = get_ngrams(answer, 4)
    if not answer_grams:
        return 0.0

    passage_text = ' '.join([c.get('text', '') for c in contexts])
    passage_grams = get_ngrams(passage_text, 4)

    overlap = len(answer_grams & passage_grams)
    return overlap / len(answer_grams) if answer_grams else 0.0

# =========================
# MAIN CLASS v5.7
# =========================
class MTRAGGenerationV57:
    def __init__(self):
        print("🔧 v5.7 — User-Satisfaction Judge + v5.6 Features")
        print("✅ DeepSeek via Azure initialized")
        print(f"⚡ Workers: {MAX_WORKERS}")
        print(f"✅ Technical Judge (Stage 6a): ENABLED")
        print(f"🔥 User-Satisfaction Judge (Stage 6b): ENABLED ({int(USER_SAT_JUDGE_SAMPLE_RATE*100)}% sample)")
        print(f"✅ All v5.6 features maintained")

    def load_tasks(self, input_file: str, max_tasks: int = None) -> List[Dict]:
        print(f"\n📂 Loading: {input_file}")
        tasks = []
        with open(input_file, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    tasks.append(json.loads(line))
        if max_tasks:
            tasks = tasks[:max_tasks]
        print(f"✅ Loaded {len(tasks)} tasks")
        return tasks

    def save_predictions(self, tasks: List[Dict], output_file: str):
        print(f"\n💾 Saving: {output_file}")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        with open(output_file, "w", encoding="utf-8") as f:
            for task in tasks:
                f.write(json.dumps(task, ensure_ascii=False) + "\n")
        print(f"✅ Saved {len(tasks)} predictions")

    def get_current_question(self, task: Dict) -> str:
        inputs = task.get("input", []) or []
        if not inputs:
            return ""
        for inp in reversed(inputs):
            if inp.get("speaker", "").lower() == "user":
                return (inp.get("text", "") or "").strip()
        return ""

    def is_conversational(self, question: str) -> bool:
        q_low = question.lower().strip()
        conversational_patterns = [
            r'^hi\b', r'^hello\b', r'^hey\b', r'^thanks?\b', r'^thank you\b',
            r'^ok\b', r'^okay\b', r'^i see\b', r'^got it\b', r'^alright\b',
        ]
        return any(re.match(p, q_low) for p in conversational_patterns)

    def build_history_lines(self, inputs: List[Dict], current_question: str) -> List[str]:
        """Build history (USER turns only)"""
        history_lines = []
        for inp in inputs:
            speaker = inp.get("speaker", "").lower()
            text = inp.get("text", "").strip()
            if not text:
                continue
            if speaker == "user" and text == current_question:
                break
            if speaker == "user":
                history_lines.append(f"User: {text}")
        return history_lines

    # =========================
    # ANSWERABILITY (SAME as v5.6)
    # =========================
    def predict_answerability(self, question: str, contexts: List[Dict], history_lines: List[str]) -> str:
        if not contexts:
            return "UNANSWERABLE"

        q_keywords = set(re.findall(r'\b[A-Z][a-z]+|\b\d+|\b\w{4,}\b', question))
        passage_text = " ".join([c.get("text", "").lower() for c in contexts])
        q_keywords_lower = {k.lower() for k in q_keywords if len(k) > 2}

        if q_keywords_lower:
            overlap_ratio = sum(1 for kw in q_keywords_lower if kw in passage_text) / len(q_keywords_lower)
            if overlap_ratio > 0.40:
                return "ANSWERABLE"

        passages_block = "DOCUMENTS:\n"
        for i, ctx in enumerate(contexts, 1):
            text = (ctx.get("text", "") or "").strip()
            passages_block += f"[PASSAGE {i}]\n{text}\n\n"

        history_block = ""
        if history_lines:
            history_block = "CONVERSATION HISTORY:\n" + "\n".join(history_lines[-2:]).strip() + "\n\n"

        system_msg = "You judge answerability. BIAS towards ANSWERABLE."
        user_msg = f"""ANSWERABILITY JUDGMENT:

ANSWERABLE: Docs contain relevant facts (even partial)
PARTIAL: Docs contain some info but missing key details
UNANSWERABLE: Temporal mismatch OR completely missing info

Bias: When uncertain → ANSWERABLE

{passages_block}
{history_block}QUESTION: {question}

Return JSON: {{"answerability": "ANSWERABLE"|"PARTIAL"|"UNANSWERABLE", "confidence": "HIGH"|"MEDIUM"|"LOW"}}"""

        try:
            response = self.generate_response_with_system(system_msg, user_msg, temperature=0.0, max_tokens=150)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                answerability = result.get("answerability", "").strip().upper()
                confidence = result.get("confidence", "MEDIUM").strip().upper()

                if answerability == "UNANSWERABLE" and confidence == "LOW":
                    return "ANSWERABLE"

                if answerability in {"ANSWERABLE", "PARTIAL", "UNANSWERABLE"}:
                    return answerability
        except:
            pass

        return "ANSWERABLE" if contexts else "UNANSWERABLE"

    # =========================
    # SPAN EXTRACTION (SAME as v5.6)
    # =========================
    def extract_relevant_spans_with_retry(
        self,
        question: str,
        contexts: List[Dict],
        history_lines: List[str],
        answerability: str
    ) -> List[Dict]:

        for attempt in range(1, SPAN_RETRY_ATTEMPTS + 1):
            if attempt > 1:
                if DEBUG_MODE:
                    print(f"   🔄 Span retry {attempt}/{SPAN_RETRY_ATTEMPTS}")
                tracker.add_span_retry()

            spans = self.extract_relevant_spans_cot(question, contexts, history_lines, attempt)

            if spans and len(spans) > 0:
                return spans

            if answerability == "ANSWERABLE" and attempt < SPAN_RETRY_ATTEMPTS:
                continue

            break

        if DEBUG_MODE:
            print(f"   ⚠️  Smart fallback (keyword-based)")
        return self.smart_fallback(question, contexts)

    def extract_relevant_spans_cot(
        self,
        question: str,
        contexts: List[Dict],
        history_lines: List[str],
        attempt: int = 1
    ) -> List[Dict]:

        passages_block = ""
        for i, ctx in enumerate(contexts[:5], 1):
            text = (ctx.get("text", "") or "").strip()

            relevance_note = ""
            if i == 1:
                relevance_note = " [MOST RELEVANT]"
            elif i == 2:
                relevance_note = " [HIGHLY RELEVANT]"

            passages_block += f"[PASSAGE {i}]{relevance_note}\n{text}\n\n"

        history_block = ""
        if history_lines:
            recent = "\n".join(history_lines[-3:])
            history_block = f"CONVERSATION HISTORY:\n{recent}\n\n"

        urgency = "The answer IS in the passages - look carefully!" if attempt > 1 else ""

        span_prompt = f"""TASK: Extract sentences that answer the question.

{urgency}

{history_block}QUESTION: {question}

{passages_block}

RULES:
1. PRIORITIZE earlier passages ([MOST RELEVANT])
2. Extract VERBATIM sentences (exact quotes)
3. COMPLETENESS PRIORITY:
   - Include ALL sentences with proper nouns (names, places, organizations)
   - Include ALL sentences with numbers (dates, quantities, measurements)
   - Include ALL sentences needed for complete answer
4. Better to include 1 extra sentence than miss key entity
5. Max {MAX_SPANS} sentences (use all {MAX_SPANS} if needed for completeness)
6. If unanswerable, return empty extracted_spans: []

Return JSON:
{{
  "reasoning": "Brief explanation of what facts are being extracted",
  "extracted_spans": [
    {{"passage_id": 1, "sentence": "exact sentence"}},
    {{"passage_id": 2, "sentence": "another sentence"}}
  ]
}}
"""

        try:
            response = self.generate_response(span_prompt, temperature=0.0, max_tokens=800)

            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                spans = result.get("extracted_spans", [])
                reasoning = result.get("reasoning", "")

                if DEBUG_MODE and attempt == 1:
                    print(f"\n🔍 Extracted {len(spans)} spans")
                    if reasoning:
                        print(f"   Reasoning: {reasoning[:80]}...")

                if spans:
                    filtered = []
                    for span in spans[:MAX_SPANS]:
                        filtered.append({
                            "text": span["sentence"],
                            "document_id": f"span_from_passage_{span['passage_id']}",
                            "source": "cot_extracted",
                            "reference": True
                        })

                    if filtered:
                        return filtered

        except Exception as e:
            if DEBUG_MODE:
                print(f"⚠️  Span extraction failed: {e}")

        return []

    def smart_fallback(self, question: str, contexts: List[Dict]) -> List[Dict]:
        q_keywords = set(re.findall(r'\b[A-Z][a-z]+|\b\d+|\b\w{5,}\b', question))
        q_keywords_lower = {k.lower() for k in q_keywords if len(k) > 3}

        scored = []
        for idx, ctx in enumerate(contexts[:5]):
            text = ctx.get("text", "").lower()
            overlap = sum(1 for kw in q_keywords_lower if kw in text)
            position_bonus = (5 - idx) * 0.5
            score = overlap + position_bonus
            scored.append((score, ctx))

        scored.sort(reverse=True, key=lambda x: x[0])

        return [ctx for _, ctx in scored[:3]]

    # =========================
    # GENERATION (SAME as v5.6)
    # =========================
    def create_generation_prompt_v57(
        self,
        spans: List[Dict],
        question: str,
        history_lines: List[str],
        answerability: str
    ) -> str:

        if answerability == "PARTIAL":
            word_suggestion = f"typically {SUGGESTED_PARTIAL_TARGET}w (simple questions may be shorter)"
        else:
            word_suggestion = f"typically {SUGGESTED_TARGET}w (adapt to question complexity)"

        spans_block = "EXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            text = span.get("text", "").strip()
            spans_block += f"[{i}] {text}\n"
        spans_block += "\n"

        history_block = ""
        if history_lines:
            recent = "\n".join(history_lines[-3:])
            history_block = f"CONTEXT:\n{recent}\n\n"

        instruction = f"""Generate a natural answer using the EXTRACTED FACTS.

REQUIREMENTS:
• Length: {word_suggestion}
• Include ALL facts from EXTRACTED FACTS:
  ✓ EVERY name (person, place, organization)
  ✓ EVERY number (quantity, date, year, percentage, step number)
  ✓ EVERY key term from question
• Use EXACT wording for:
  - Proper nouns: copy verbatim
  - Numbers: copy exactly as written (including "1.", "2." for lists)
  - Technical terms: copy verbatim
• Paraphrase only: connecting words ("and", "also", "while")
• Target ~{int(TARGET_EXTRACTIVENESS*100)}% verbatim overlap
• If EXTRACTED FACTS incomplete: "{IDK_PHRASE}"

🚫 CRITICAL: Missing ANY entity/number from EXTRACTED FACTS = FAILURE

{history_block}{spans_block}QUESTION:
{question}

ANSWER:"""

        return instruction

    # =========================
    # MULTI-SAMPLING (SAME as v5.6)
    # =========================
    def generate_candidates(
        self,
        spans: List[Dict],
        question: str,
        history_lines: List[str],
        answerability: str
    ) -> List[str]:

        prompt = self.create_generation_prompt_v57(spans, question, history_lines, answerability)

        candidates = []

        cand1 = self.generate_response(prompt, temperature=GENERATION_TEMPERATURE_1, max_tokens=450)
        cand1 = normalize_idk_partial_safe(clean_model_output(cand1))
        candidates.append(cand1)

        if ENABLE_MULTI_SAMPLING and NUM_CANDIDATES >= 2:
            cand2 = self.generate_response(prompt, temperature=GENERATION_TEMPERATURE_2, max_tokens=450)
            cand2 = normalize_idk_partial_safe(clean_model_output(cand2))
            candidates.append(cand2)

        return candidates

    # =========================
    # CANDIDATE SELECTION (SAME as v5.6)
    # =========================
    def score_answer_quality(self, answer: str, answerability: str) -> float:

        if is_pure_idk(answer):
            return 0.0

        score = 0.0
        wc = word_count(answer)

        if answerability == "PARTIAL":
            target = SUGGESTED_PARTIAL_TARGET
            min_acceptable = 40
            max_acceptable = 100
        else:
            target = SUGGESTED_TARGET
            min_acceptable = 60
            max_acceptable = 130

        if min_acceptable <= wc <= max_acceptable:
            distance = abs(wc - target)
            score += max(0, 1.0 - (distance / 40.0)) * 0.3
        elif wc < min_acceptable:
            score += (wc / min_acceptable) * 0.15
        else:
            score += max(0, 1.0 - ((wc - max_acceptable) / 60.0)) * 0.15

        entities = len(re.findall(r'\b[A-Z][a-z]+', answer))
        score += min(entities / 10.0, 0.3)

        numbers = len(re.findall(r'\b\d+', answer))
        score += min(numbers / 5.0, 0.2)

        sentences = re.split(r'[.!?]+', answer)
        complete = [s for s in sentences if len(s.strip()) > 10]
        score += min(len(complete) / 5.0, 0.2)

        return score

    def select_best_candidate(self, candidates: List[str], answerability: str) -> str:

        if len(candidates) == 1:
            return candidates[0]

        scored = []
        for cand in candidates:
            score = self.score_answer_quality(cand, answerability)
            scored.append((score, cand))

        scored.sort(reverse=True, key=lambda x: x[0])
        return scored[0][1]

    # =========================
    # STAGE 6a: TECHNICAL JUDGE (SAME as v5.6)
    # =========================
    def should_use_judge(
        self,
        answer: str,
        answerability: str,
        spans: List[Dict],
        retry_count: int = 0
    ) -> Tuple[bool, str]:

        wc = word_count(answer)

        if answerability in ["ANSWERABLE", "PARTIAL"] and is_pure_idk(answer):
            return True, "false_refusal"

        if answerability == "ANSWERABLE":
            if wc < JUDGE_WC_MIN:
                return True, "too_short"
            if wc > JUDGE_WC_MAX:
                return True, "too_long"

        if not is_pure_idk(answer) and wc > 30:
            extr = calculate_extractiveness(answer, spans)
            if extr < JUDGE_EXTR_MIN:
                return True, "low_extractiveness"

        if retry_count >= NUM_RETRIES and is_pure_idk(answer):
            return True, "retry_exhausted"

        return False, ""

    def judge_answer_lightweight(
        self,
        answer: str,
        question: str,
        spans: List[Dict],
        trigger_reason: str
    ) -> Dict:

        spans_block = "EXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            spans_block += f"[{i}] {span.get('text', '')}\n"

        if "refusal" in trigger_reason:
            focus = "Can this question be answered using EXTRACTED FACTS?"
        elif "short" in trigger_reason:
            focus = "Are all key facts included? Is it complete?"
        elif "long" in trigger_reason:
            focus = "Is this concise? Any redundancy?"
        elif "extractiveness" in trigger_reason:
            focus = "Is this grounded in EXTRACTED FACTS (not hallucinating)?"
        else:
            focus = "Is this a good quality answer?"

        judge_prompt = f"""Quick quality check.

QUESTION: {question}

{spans_block}

ANSWER:
{answer}

FOCUS: {focus}

Return JSON:
{{
  "verdict": "ACCEPT" | "REGENERATE",
  "issue": "What's wrong (if REGENERATE)",
  "suggestion": "How to fix (if REGENERATE)"
}}

Rules:
• ACCEPT if answer is reasonable
• REGENERATE only if clear problem
"""

        try:
            response = self.generate_response(judge_prompt, temperature=0.0, max_tokens=200)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                return {
                    "verdict": result.get("verdict", "ACCEPT").upper(),
                    "issue": result.get("issue", ""),
                    "suggestion": result.get("suggestion", "")
                }
        except:
            pass

        return {"verdict": "ACCEPT", "issue": "", "suggestion": ""}

    # =========================
    # 🔥 STAGE 6b: USER-SATISFACTION JUDGE (NEW!)
    # =========================
    def user_satisfaction_judge(
        self,
        question: str,
        answer: str,
        contexts: List[Dict],
        history_lines: List[str]
    ) -> Dict:
        """Judge from USER perspective: Does this satisfy what I asked?"""

        # Build context
        passages_preview = ""
        for i, ctx in enumerate(contexts[:5]):
            text = ctx.get('text', '')[:200]
            passages_preview += f"[{i+1}] {text}...\n"

        history_block = ""
        if history_lines:
            history_block = "CONVERSATION:\n" + "\n".join(history_lines[-3:]) + "\n\n"

        judge_prompt = f"""You are a USER who asked a question. Judge if the answer satisfies YOUR question.

{history_block}YOUR QUESTION:
{question}

AVAILABLE INFORMATION (what the system had access to):
{passages_preview}

SYSTEM'S ANSWER:
{answer}

TASK: As a user, would you be SATISFIED with this answer?

Consider:
1. **Intent Match**: Does the answer address what I was REALLY asking?
   - Not just "does it mention the same entities?"
   - But "does it answer the SPIRIT of my question?"

2. **Completeness for MY Needs**:
   - If I asked "how", does it explain the process?
   - If I asked "why", does it give reasons?
   - If I asked "what", does it give the definition/description?
   - If I asked for comparison, does it compare?

3. **Clarity**:
   - Can I understand the answer without re-reading?
   - Is it organized logically?

4. **Relevance**:
   - Does it focus on what I care about?
   - Or does it give me irrelevant details?

5. **Context Awareness**:
   - Given the conversation history, is this appropriate?
   - Does it answer MY follow-up properly?

IMPORTANT:
- Ignore minor wording differences
- Ignore if answer is slightly shorter/longer than ideal
- Focus on: "Did I get what I was looking for?"

Return JSON:
{{
  "satisfaction": "SATISFIED" | "UNSATISFIED",
  "confidence": "HIGH" | "MEDIUM" | "LOW",
  "reason": "Brief explanation of why satisfied/unsatisfied",
  "missing_aspect": "What the user wanted but didn't get (if unsatisfied)",
  "suggestion": "How to better answer this question (if unsatisfied)"
}}

Examples:

Q: "How do I install the SDK?"
A: "The SDK is a software development kit for iOS and Android."
→ UNSATISFIED (didn't explain HOW to install, just defined what SDK is)

Q: "How do I install the SDK?"
A: "Install via CocoaPods: add 'pod OpenWhisk' to Podfile, run 'pod install'."
→ SATISFIED (explains the process step-by-step)

Q: "What is the capital of France?"
A: "The capital of France is Paris, which is also its largest city."
→ SATISFIED (answers question, extra info is fine)

Q: "Why was the request rejected?"
A: "The request was rejected in 2023 by the committee."
→ UNSATISFIED (says WHEN and WHO, but not WHY)

Now judge the actual answer above.
"""

        try:
            response = self.generate_response(judge_prompt, temperature=0.0, max_tokens=300)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                return {
                    "satisfaction": result.get("satisfaction", "SATISFIED").upper(),
                    "confidence": result.get("confidence", "MEDIUM").upper(),
                    "reason": result.get("reason", ""),
                    "missing_aspect": result.get("missing_aspect", ""),
                    "suggestion": result.get("suggestion", "")
                }
        except Exception as e:
            if DEBUG_MODE:
                print(f"⚠️  User-satisfaction judge failed: {e}")

        return {"satisfaction": "SATISFIED", "confidence": "LOW", "reason": "Judge failed"}

    # =========================
    # POLISH (SAME as v5.6)
    # =========================
    def build_extractive_polish_prompt(self, spans: List[Dict], raw_answer: str) -> str:

        spans_block = "EXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            spans_block += f"[{i}] {span.get('text', '')}\n"

        polish_prompt = f"""{spans_block}

CURRENT ANSWER:
{raw_answer}

TASK: Revise to achieve ~{int(TARGET_EXTRACTIVENESS*100)}% verbatim overlap with EXTRACTED FACTS.

RULES:
• Keep EXACT: names, numbers, dates, locations (4-6 word spans)
• Paraphrase: connecting words, transitions
• Target: ~{int(TARGET_EXTRACTIVENESS*100)}% of answer = exact phrases from FACTS
• Maintain natural flow
• Keep same length ({word_count(raw_answer)}w)

REVISED ANSWER:"""

        return polish_prompt

    # =========================
    # NORMALIZATION (SAME as v5.6)
    # =========================
    def soft_normalize(self, text: str, answerability: str) -> str:

        text = re.sub(r'\s+', ' ', text.strip())
        text = re.sub(r'\n{3,}', '\n\n', text)

        if is_pure_idk(text):
            return IDK_PHRASE

        wc = word_count(text)

        if wc > SOFT_TRIM_THRESHOLD:
            if DEBUG_MODE:
                print(f"   ⚠️  Very long ({wc}w) - soft trimming to 250w")
            text = soft_trim_to_word_limit_by_sentences(text, 250)

        return text

    # =========================
    # RETRY (SAME as v5.6)
    # =========================
    def create_answerable_retry_prompt(
        self,
        spans: List[Dict],
        history_lines: List[str],
        question: str,
        answerability: str
    ) -> str:

        word_suggestion = f"{SUGGESTED_TARGET}w" if answerability == "ANSWERABLE" else f"{SUGGESTED_PARTIAL_TARGET}w"

        spans_block = "EXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            spans_block += f"[{i}] {span.get('text', '')}\n"

        history_block = ""
        if history_lines:
            history_block = f"CONTEXT:\n{' '.join(history_lines[-2:])}\n\n"

        return f"""The answer IS in the EXTRACTED FACTS - find it!

{history_block}{spans_block}

QUESTION: {question}

Write typically {word_suggestion} using the facts above.
🚫 DO NOT output '{IDK_PHRASE}' - the answer is present!

ANSWER:"""

    # =========================
    # CONVERSATIONAL (SAME as v5.6)
    # =========================
    def create_conversational_prompt(self, history_lines: List[str], question: str) -> str:
        instruction = f"Reply naturally and briefly. Keep it under {MAX_WORDS_CONV} words."
        history_block = ""
        if history_lines:
            history_block = "\n".join(history_lines[-4:]).strip() + "\n"
        return f"{instruction}\n\n{history_block}User: {question}\nAssistant:"

    # =========================
    # LLM CALLS (SAME as v5.6)
    # =========================
    def generate_response_with_system(
        self,
        system: str,
        user: str,
        temperature: float = 0.0,
        max_tokens: int = 360
    ) -> str:
        headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
        payload = {
            "model": "DeepSeek-V3.2",
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user}
            ],
            "max_tokens": max_tokens,
            "temperature": temperature
        }
        for attempt in range(7):
            try:
                response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
                if response.status_code == 200:
                    res_json = response.json()
                    usage = res_json.get('usage', {})
                    tracker.add_usage(usage.get('prompt_tokens', 0), usage.get('completion_tokens', 0))
                    return res_json['choices'][0]['message']['content'].strip()
                elif response.status_code == 429:
                    time.sleep(5 * (2 ** attempt))
                    continue
            except Exception as e:
                time.sleep(5)
        return ""

    def generate_response(self, prompt: str, temperature: float = 0.0, max_tokens: int = 360) -> str:
        return self.generate_response_with_system("You are a helpful assistant.", prompt, temperature, max_tokens)

    # =========================
    # 🔥 ENHANCED PROCESS TASK (with Stage 6b)
    # =========================
    def process_task(self, task: Dict) -> Dict:
        inputs = task.get("input", []) or []
        q = self.get_current_question(task)

        history_lines = self.build_history_lines(inputs, q)

        contexts_original = task.get("contexts", [])[:N_PASSAGES]

        # Conversational
        if self.is_conversational(q):
            prompt = self.create_conversational_prompt(history_lines, q)
            raw = self.generate_response(prompt, temperature=0.3, max_tokens=160)
            raw = clean_model_output(raw)
            if word_count(raw) > MAX_WORDS_CONV:
                raw = soft_trim_to_word_limit_by_sentences(raw, MAX_WORDS_CONV)

            task["contexts"] = contexts_original
            task["predictions"] = [{"text": raw}]
            task["debug_info"] = {"type": "CONVERSATIONAL"}
            tracker.add_answerability("CONVERSATIONAL")
            tracker.add_prediction(False)
            return task

        # Answerability
        ans = self.predict_answerability(q, contexts_original, history_lines)
        tracker.add_answerability(ans)

        debug_steps = {"answerability": ans} if DEBUG_MODE else {}

        if ans == "UNANSWERABLE":
            task["contexts"] = contexts_original
            task["predictions"] = [{"text": IDK_PHRASE}]
            task["debug_info"] = {"type": "UNANSWERABLE", "steps": debug_steps}
            tracker.add_prediction(True, ans)
            return task

        # Span extraction
        if ENABLE_SPAN_EXTRACTION:
            spans = self.extract_relevant_spans_with_retry(q, contexts_original, history_lines, ans)
        else:
            spans = contexts_original[:3]

        if DEBUG_MODE:
            debug_steps["spans_extracted"] = len(spans)

        if not spans:
            task["contexts"] = contexts_original
            task["predictions"] = [{"text": IDK_PHRASE}]
            task["debug_info"] = {"type": "NO_SPANS", "steps": debug_steps}
            tracker.add_prediction(True, ans)
            return task

        # Generate candidates
        candidates = self.generate_candidates(spans, q, history_lines, ans)
        final_answer = self.select_best_candidate(candidates, ans)

        # Retry if IDK on ANSWERABLE
        retry_count = 0
        if RETRY_IF_IDK_ON_ANSWERABLE and ans in {"ANSWERABLE", "PARTIAL"} and final_answer == IDK_PHRASE:
            for retry_num in range(1, NUM_RETRIES + 1):
                retry_prompt = self.create_answerable_retry_prompt(spans, history_lines, q, ans)
                raw2 = self.generate_response(retry_prompt, temperature=0.0 if retry_num == 1 else 0.15, max_tokens=450)
                raw2 = normalize_idk_partial_safe(clean_model_output(raw2))

                if raw2 and raw2 != IDK_PHRASE and word_count(raw2) >= 40:
                    final_answer = raw2
                    retry_count = retry_num
                    if DEBUG_MODE:
                        debug_steps[f"retry_{retry_num}"] = "✅"
                    break

        # STAGE 6a: Technical Judge
        if ENABLE_SELECTIVE_JUDGE and final_answer != IDK_PHRASE:
            should_judge, trigger_reason = self.should_use_judge(final_answer, ans, spans, retry_count)

            if should_judge:
                if DEBUG_MODE:
                    print(f"   🔍 Technical judge triggered: {trigger_reason}")

                tracker.add_judge_trigger(regenerated=False)
                judgment = self.judge_answer_lightweight(final_answer, q, spans, trigger_reason)

                if judgment["verdict"] == "REGENERATE":
                    if DEBUG_MODE:
                        print(f"   🔄 Regenerating: {judgment['issue'][:60]}...")

                    regen_prompt = f"""Previous answer had issues: {judgment['issue']}

Suggestion: {judgment['suggestion']}

{self.create_generation_prompt_v57(spans, q, history_lines, ans)}"""

                    regenerated = self.generate_response(regen_prompt, temperature=0.15, max_tokens=450)
                    regenerated = normalize_idk_partial_safe(clean_model_output(regenerated))

                    if regenerated and regenerated != IDK_PHRASE:
                        final_answer = regenerated
                        tracker.add_judge_trigger(regenerated=True)
                        if DEBUG_MODE:
                            debug_steps["tech_judge_regenerated"] = "✅"

        # 🔥 STAGE 6b: User-Satisfaction Judge
        if ENABLE_USER_SATISFACTION_JUDGE and final_answer != IDK_PHRASE:
            # Sample based on configured rate
            if random.random() < USER_SAT_JUDGE_SAMPLE_RATE:
                sat_judgment = self.user_satisfaction_judge(q, final_answer, contexts_original, history_lines)

                unsatisfied = (sat_judgment['satisfaction'] == 'UNSATISFIED')
                tracker.add_user_sat_judge(unsatisfied=unsatisfied, regenerated=False)

                if DEBUG_MODE:
                    print(f"   👤 User satisfaction: {sat_judgment['satisfaction']} ({sat_judgment['confidence']})")
                    if unsatisfied:
                        print(f"      Reason: {sat_judgment['reason'][:80]}...")

                # Regenerate if unsatisfied with high/medium confidence
                if unsatisfied and sat_judgment['confidence'] in USER_SAT_CONFIDENCE_THRESHOLD:
                    if DEBUG_MODE:
                        print(f"   🔄 Regenerating for user satisfaction...")

                    regen_prompt = f"""Previous answer didn't satisfy the user's INTENT.

USER'S QUESTION: {q}

PREVIOUS ANSWER: {final_answer}

ISSUE: {sat_judgment['reason']}
MISSING: {sat_judgment['missing_aspect']}
HOW TO FIX: {sat_judgment['suggestion']}

{self.create_generation_prompt_v57(spans, q, history_lines, ans)}

Focus on answering what the user REALLY wants to know."""

                    regenerated = self.generate_response(regen_prompt, temperature=0.2, max_tokens=450)
                    regenerated = normalize_idk_partial_safe(clean_model_output(regenerated))

                    if regenerated and regenerated != IDK_PHRASE:
                        final_answer = regenerated
                        tracker.add_user_sat_judge(unsatisfied=False, regenerated=True)
                        if DEBUG_MODE:
                            debug_steps["user_sat_regenerated"] = "✅"

        # Polish
        if ENABLE_EXTRACTIVE_POLISH and final_answer != IDK_PHRASE:
            polish_prompt = self.build_extractive_polish_prompt(spans, final_answer)
            polished = self.generate_response(polish_prompt, temperature=0.0, max_tokens=400)
            polished = normalize_idk_partial_safe(clean_model_output(polished))

            if polished and polished != IDK_PHRASE:
                final_answer = polished
                if DEBUG_MODE:
                    debug_steps["polished"] = "✅"

        # Soft normalization
        final_answer = self.soft_normalize(final_answer, ans)

        if DEBUG_MODE:
            debug_steps["final_wc"] = word_count(final_answer)

        task["contexts"] = contexts_original
        task["contexts_used"] = [s.get("text", "")[:100] for s in spans]
        task["predictions"] = [{"text": final_answer}]
        task["debug_info"] = {
            "type": ans,
            "answer_preview": final_answer[:80],
            "steps": debug_steps
        }

        tracker.add_prediction(final_answer == IDK_PHRASE, ans)
        return task

    def run_generation(self, input_file: str, output_file: str, max_tasks: int = None):
        print("\n" + "=" * 70)
        print("📝 v5.7 — User-Satisfaction Judge + v5.6 Features")
        print("=" * 70)
        tasks = self.load_tasks(input_file, max_tasks)

        processed = [None] * len(tasks)

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_idx = {executor.submit(self.process_task, task): idx for idx, task in enumerate(tasks)}

            for future in tqdm(as_completed(future_to_idx), total=len(tasks), desc="v5.7"):
                idx = future_to_idx[future]
                try:
                    out_task = future.result()
                    processed[idx] = out_task

                    wc = word_count(out_task["predictions"][0]["text"])
                    is_idk = (out_task["predictions"][0]["text"] == IDK_PHRASE)
                    debug = out_task.get("debug_info", {})
                    ans_type = debug.get("type", "?")

                    tqdm.write(f"[{idx+1}] {wc}w | {ans_type}")
                except Exception as e:
                    tqdm.write(f"❌ Task {idx+1} error: {e}")
                    processed[idx] = tasks[idx]

        print(tracker.get_report())
        self.save_predictions(processed, output_file)
        print("\n✅ v5.7 COMPLETE!")
        return output_file

# ================================================================
# RUN
# ================================================================
INPUT_FILE = "/content/drive/MyDrive/Project/generation/RAG.jsonl"
PREDICTIONS_OUTPUT = "/content/drive/MyDrive/Project/outputs/deepseek_RAG_v5_7.jsonl"

generator = MTRAGGenerationV57()
generator.run_generation(INPUT_FILE, PREDICTIONS_OUTPUT, max_tasks=None)

print("\n🏆 v5.7 COMPLETE!")
print("\n📊 New Features:")
print("  • User-Satisfaction Judge (Stage 6b)")
print("  • Judges from USER perspective")
print("  • 30% sample rate")
print("  • Regenerates on HIGH/MEDIUM confidence unsatisfied")
print("\n🎯 Expected: 0.61-0.615 harmonic mean (+2-3% from v5.6)")


# **Task B**

In [ ]:
import os
import sys
import json
import re
import time
import random
import requests
import threading
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


# ============================================================================
# CONFIGURATION - HYBRID: DeepSeek Extraction + GPT-4o-mini Generation
# ============================================================================

# DeepSeek V3 (for span extraction & judges - cheaper)
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

# GPT-4o-mini (for natural generation - better instruction following)
AZURE_GPT_ENDPOINT = os.environ["ENDPOINT"]
AZURE_GPT_KEY = os.environ["_KEY"]


# 🧪 EXPERIMENT PARAMETERS
EXPERIMENT_NAME = "hybrid_deepseek_gpt4o"
MAX_SPANS = 8
N_PASSAGES = 5


# 🎯 TESTING MODE
TEST_MODE = True
TEST_SAMPLE_SIZE = None


# Pipeline stages
ENABLE_SPAN_EXTRACTION = True
ENABLE_SPAN_RETRY = True
ENABLE_DUAL_GENERATION = True
ENABLE_TECHNICAL_JUDGE = True
ENABLE_USER_SATISFACTION_JUDGE = True
ENABLE_FINAL_MICRO_ADJUSTMENTS = True


# 🎯 STRICT ANSWERABILITY RULE
STRICT_ANSWERABILITY_MODE = True


# Span extraction
MIN_SPANS = 1
SPAN_RETRY_ATTEMPTS = 2


# Generation
GENERATION_TEMP_V1 = 0.0
GENERATION_TEMP_V2 = 0.1


# Word count targets
SUGGESTED_MIN = 80
SUGGESTED_TARGET = 90
SUGGESTED_MAX = 105


# Judge settings
JUDGE_WC_MIN = 50
JUDGE_WC_MAX = 150


# User satisfaction judge
USER_SAT_JUDGE_SAMPLE_RATE = 0.50


# Conversational and Unanswerable responses
MAX_WORDS_CONV = 50
MAX_WORDS_UNANSWERABLE = 25


# Adjustment criteria
ADJUSTMENT_MIN_WORD_DIFF = 10
ADJUSTMENT_MAX_LENGTH = 150
GENERATION_TEMP_ADJUSTMENT = 0.1


DEBUG_MODE = True
MAX_WORKERS = 5


random.seed(42)


# ============================================================================
# STATISTICS TRACKER
# ============================================================================


class StatsTracker:
    def __init__(self):
        self.total_input_ds = 0
        self.total_output_ds = 0
        self.total_input_gpt = 0
        self.total_output_gpt = 0
        self.idk_count = 0
        self.full_answer_count = 0
        self.conversational_count = 0
        self.span_retry_count = 0
        self.dual_generation_count = 0
        self.chose_v1 = 0
        self.chose_v2 = 0
        self.tech_judge_v1_wins = 0
        self.tech_judge_v2_wins = 0
        self.usersat_judge_v1_wins = 0
        self.usersat_judge_v2_wins = 0
        self.micro_adjustments_count = 0
        self.strict_idk_enforced = 0
        self.lock = threading.Lock()


    def add_usage_ds(self, prompt_tokens, completion_tokens):
        with self.lock:
            self.total_input_ds += prompt_tokens
            self.total_output_ds += completion_tokens


    def add_usage_gpt(self, prompt_tokens, completion_tokens):
        with self.lock:
            self.total_input_gpt += prompt_tokens
            self.total_output_gpt += completion_tokens


    def add_prediction(self, is_idk: bool):
        with self.lock:
            if is_idk:
                self.idk_count += 1
            else:
                self.full_answer_count += 1


    def add_conversational(self):
        with self.lock:
            self.conversational_count += 1


    def add_span_retry(self):
        with self.lock:
            self.span_retry_count += 1


    def add_dual_generation(self):
        with self.lock:
            self.dual_generation_count += 1


    def add_choice(self, chose_v1: bool):
        with self.lock:
            if chose_v1:
                self.chose_v1 += 1
            else:
                self.chose_v2 += 1


    def add_tech_judge_result(self, v1_better: bool):
        with self.lock:
            if v1_better:
                self.tech_judge_v1_wins += 1
            else:
                self.tech_judge_v2_wins += 1


    def add_usersat_judge_result(self, v1_better: bool):
        with self.lock:
            if v1_better:
                self.usersat_judge_v1_wins += 1
            else:
                self.usersat_judge_v2_wins += 1


    def add_micro_adjustment(self):
        with self.lock:
            self.micro_adjustments_count += 1


    def add_strict_idk(self):
        with self.lock:
            self.strict_idk_enforced += 1


    def get_report(self):
        cost_ds = (self.total_input_ds * 0.20 / 1e6) + (self.total_output_ds * 0.50 / 1e6)
        cost_gpt = (self.total_input_gpt * 0.15 / 1e6) + (self.total_output_gpt * 0.60 / 1e6)
        total_cost = cost_ds + cost_gpt

        report = "=" * 70 + "\n"
        report += "🎯 HYBRID APPROACH (DeepSeek Extraction + GPT-4o-mini Generation)\n"
        report += "=" * 70 + "\n"
        report += f"Token Usage:\n"
        report += f"  DeepSeek V3:\n"
        report += f"    Input: {self.total_input_ds:,} (${cost_ds:.4f})\n"
        report += f"    Output: {self.total_output_ds:,}\n"
        report += f"  GPT-4o-mini:\n"
        report += f"    Input: {self.total_input_gpt:,} (${cost_gpt:.4f})\n"
        report += f"    Output: {self.total_output_gpt:,}\n"
        report += f"  Total Cost: ${total_cost:.4f}\n"

        total_preds = self.idk_count + self.full_answer_count + self.conversational_count
        report += f"\nPredictions (Total: {total_preds}):\n"

        if total_preds > 0:
            idk_pct = (self.idk_count / total_preds * 100)
            ans_pct = (self.full_answer_count / total_preds * 100)
            conv_pct = (self.conversational_count / total_preds * 100)

            report += f"  Full Answers: {self.full_answer_count:4d} ({ans_pct:5.1f}%)\n"
            report += f"  IDK Responses: {self.idk_count:4d} ({idk_pct:5.1f}%)\n"
            report += f"  Conversational: {self.conversational_count:4d} ({conv_pct:5.1f}%)\n"

        if self.strict_idk_enforced > 0:
            report += f"\n🛡️ Strict IDK Enforced: {self.strict_idk_enforced}\n"

        if self.span_retry_count > 0:
            report += f"\nSpan Extraction Retries: {self.span_retry_count}\n"

        if self.dual_generation_count > 0:
            report += f"\n🆕 Dual Generation:\n"
            report += f"  Generated Both Versions: {self.dual_generation_count:4d}\n"

            total_choices = self.chose_v1 + self.chose_v2
            if total_choices > 0:
                v1_pct = (self.chose_v1 / total_choices * 100)
                v2_pct = (self.chose_v2 / total_choices * 100)
                report += f"\n  Final Choice:\n"
                report += f"    V1 (temp=0.0): {self.chose_v1:4d} ({v1_pct:5.1f}%)\n"
                report += f"    V2 (temp=0.1): {self.chose_v2:4d} ({v2_pct:5.1f}%)\n"

        tech_total = self.tech_judge_v1_wins + self.tech_judge_v2_wins
        if tech_total > 0:
            report += f"\n📊 Technical Judge Results:\n"
            report += f"  V1 Wins: {self.tech_judge_v1_wins:4d} ({self.tech_judge_v1_wins/tech_total*100:5.1f}%)\n"
            report += f"  V2 Wins: {self.tech_judge_v2_wins:4d} ({self.tech_judge_v2_wins/tech_total*100:5.1f}%)\n"

        usersat_total = self.usersat_judge_v1_wins + self.usersat_judge_v2_wins
        if usersat_total > 0:
            report += f"\n👤 User Satisfaction Judge Results:\n"
            report += f"  V1 Wins: {self.usersat_judge_v1_wins:4d} ({self.usersat_judge_v1_wins/usersat_total*100:5.1f}%)\n"
            report += f"  V2 Wins: {self.usersat_judge_v2_wins:4d} ({self.usersat_judge_v2_wins/usersat_total*100:5.1f}%)\n"

        if self.micro_adjustments_count > 0:
            report += f"\n🔧 Micro-Adjustments Applied: {self.micro_adjustments_count}\n"

        report += "\n" + "=" * 70
        return report


tracker = StatsTracker()


# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================


def word_count(text: str) -> int:
    return len([w for w in (text or "").strip().split() if w])


def safe_strip_quotes(s: str) -> str:
    s = (s or "").strip()
    if len(s) >= 2 and ((s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'")):
        return s[1:-1].strip()
    return s


def clean_model_output(text: str) -> str:
    t = safe_strip_quotes((text or "").strip())
    t = re.sub(r'(Agent|Assistant):', '', t.strip(), flags=re.IGNORECASE)
    t = re.sub(r'\s{3,}', ' ', t)
    t = re.sub(r'\n+', '\n', t)
    return t.strip()


def is_pure_idk(text: str) -> bool:
    t = (text or "").strip()
    if not t:
        return False
    low = t.lower()
    idk_indicators = [
        "i don't know", "i do not know", "do not have enough information",
        "don't have enough information", "do not have specific information",
        "don't have specific information", "insufficient information",
        "i cannot answer", "unable to answer"
    ]
    if any(ind in low for ind in idk_indicators) and word_count(t) <= 30:
        return True
    return False


def normalize_idk_partial_safe(text: str) -> str:
    t = (text or "").strip()
    if not t:
        return t
    if is_pure_idk(t):
        return t
    return t


def get_ngrams(text: str, n: int) -> set:
    words = text.lower().split()
    return set(" ".join(words[i:i+n]) for i in range(len(words)-n+1))


def calculate_extractiveness(answer: str, contexts: List[Dict]) -> float:
    answer_grams = get_ngrams(answer, 4)
    if not answer_grams:
        return 0.0
    passage_text = " ".join(c.get("text", "") for c in contexts)
    passage_grams = get_ngrams(passage_text, 4)
    overlap = len(answer_grams & passage_grams)
    return overlap / len(answer_grams) if answer_grams else 0.0


# ============================================================================
# MAIN GENERATION CLASS - HYBRID
# ============================================================================


class HybridGeneration:
    def __init__(self):
        print("=" * 70)
        print("🎯 HYBRID APPROACH")
        print("=" * 70)
        print("Pipeline:")
        print("  1️⃣  DeepSeek V3 → Span Extraction (cheap, good at structured tasks)")
        print("  2️⃣  GPT-4o-mini → Natural Generation V1 + V2 (better instruction following)")
        print("  3️⃣  DeepSeek V3 → Technical Judge (cheap)")
        print("  4️⃣  GPT-4o-mini → User Satisfaction Judge (better at user perspective)")
        print("  5️⃣  GPT-4o-mini → Micro-adjustments")
        print(f"\n⚙ Workers: {MAX_WORKERS}")
        print(f"⚙ N_PASSAGES: {N_PASSAGES}")
        print(f"⚙ MAX_SPANS: {MAX_SPANS}")
        print("=" * 70)

    # ========================================================================
    # ANSWERABILITY LOADING & SAVING
    # ========================================================================

    def load_tasks_with_ground_truth(
        self,
        input_file: str,
        reference_file: str = None,
        max_tasks: int = None
    ) -> List[Dict]:
        print(f"📂 Loading {input_file}...")
        tasks = []
        with open(input_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    tasks.append(json.loads(line))

        if reference_file and os.path.exists(reference_file):
            print(f"📂 Loading ground truth from {reference_file}...")
            ref_tasks = []
            with open(reference_file, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip():
                        ref_tasks.append(json.loads(line))

            answerability_map = {}
            question_type_map = {}
            multiturn_map = {}

            for ref_task in ref_tasks:
                task_id = ref_task.get('task_id', '')

                ans = ref_task.get('answerability', ref_task.get('Answerability', ['ANSWERABLE']))
                if isinstance(ans, list) and len(ans) > 0:
                    answerability_map[task_id] = ans[0]
                elif isinstance(ans, str):
                    answerability_map[task_id] = ans

                qt = ref_task.get('Question Type', [])
                if isinstance(qt, list):
                    question_type_map[task_id] = qt

                mt = ref_task.get('Multi-Turn', [])
                if isinstance(mt, list):
                    multiturn_map[task_id] = mt

            for task in tasks:
                task_id = task.get('task_id', '')
                if task_id in answerability_map:
                    task['_ground_truth_answerability'] = answerability_map[task_id]
                    task['_ground_truth_question_type'] = question_type_map.get(task_id, [])
                    task['_ground_truth_multiturn'] = multiturn_map.get(task_id, [])

            print(f"✓ Merged {len(answerability_map)} ground truth labels")

        if max_tasks:
            tasks = tasks[:max_tasks]

        print(f"✓ Loaded {len(tasks)} tasks")
        return tasks

    def _extract_answerability(self, task: Dict) -> Optional[str]:
        if '_ground_truth_answerability' in task:
            return task['_ground_truth_answerability']

        if 'answerability' in task:
            ans = task['answerability']
            if isinstance(ans, list) and len(ans) > 0:
                return ans[0]
            return ans

        if 'Answerability' in task:
            ans = task['Answerability']
            if isinstance(ans, list) and len(ans) > 0:
                return ans[0]
            return ans

        targets = task.get('targets', [])
        if targets and len(targets) > 0:
            enrichments = targets[0].get('enrichments', {})
            if 'answerability' in enrichments:
                ans = enrichments['answerability']
                if isinstance(ans, list) and len(ans) > 0:
                    return ans[0]
                return ans

        return None

    def save_predictions(self, tasks: List[Dict], output_file: str):
        print(f"💾 Saving {output_file}...")
        os.makedirs(os.path.dirname(output_file) if os.path.dirname(output_file) else '.', exist_ok=True)

        with open(output_file, 'w', encoding='utf-8') as f:
            for task in tasks:
                answerability = self._extract_answerability(task)

                if answerability:
                    task['answerability'] = [answerability]

                if '_ground_truth_question_type' in task:
                    task['Question Type'] = task['_ground_truth_question_type']

                if '_ground_truth_multiturn' in task:
                    task['Multi-Turn'] = task['_ground_truth_multiturn']

                task_copy = task.copy()
                for key in list(task_copy.keys()):
                    if key.startswith('_ground_truth_'):
                        del task_copy[key]

                f.write(json.dumps(task_copy, ensure_ascii=False) + '\n')

        print(f"✓ Saved {len(tasks)} predictions")

    def load_tasks(self, input_file: str, max_tasks: int = None) -> List[Dict]:
        return self.load_tasks_with_ground_truth(input_file, reference_file=None, max_tasks=max_tasks)

    def get_current_question(self, task: Dict) -> str:
        inputs = task.get("input", []) or []
        if not inputs:
            return ""
        for inp in reversed(inputs):
            if inp.get("speaker", "").lower() == "user":
                return (inp.get("text", "") or "").strip()
        return ""

    def is_conversational(self, question: str) -> bool:
        q_low = question.lower().strip()
        conversational_patterns = [
            r'\bthanks?\b', r'\bthank you\b', r'\bok\b',
            r'\balright\b', r'^yes\??$', r'^no\??$',
            r'\bwhat about\b', r'\bhow about\b',
            r'\bgot it\b', r'\bi see\b', r'\bgreat\b'
        ]
        return any(re.search(p, q_low) for p in conversational_patterns)


    def build_history_lines(self, inputs: List[Dict], current_question: str) -> List[str]:
        history_lines = []
        for inp in inputs:
            speaker = inp.get("speaker", "").lower()
            text = inp.get("text", "").strip()

            if not text:
                continue

            if speaker == "user" and text == current_question:
                break

            if speaker == "user":
                history_lines.append(f"User: {text}")
            elif speaker == "agent":
                history_lines.append(f"Assistant: {text}")

        return history_lines


    # ========================================================================
    # API CALLS - HYBRID
    # ========================================================================

    def generate_response_deepseek(
        self,
        system: str,
        user: str,
        temperature: float = 0.0,
        max_tokens: int = 360
    ) -> str:
        """DeepSeek V3 API call"""
        headers = {
            "Content-Type": "application/json",
            "api-key": AZURE_DS_KEY
        }

        payload = {
            "model": "DeepSeek-V3.2",
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user}
            ],
            "max_tokens": max_tokens,
            "temperature": temperature
        }

        for attempt in range(7):
            try:
                response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
                if response.status_code == 200:
                    res_json = response.json()
                    usage = res_json.get("usage", {})
                    tracker.add_usage_ds(usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0))
                    return res_json["choices"][0]["message"]["content"].strip()
                elif response.status_code == 429:
                    time.sleep(5 * (2 ** attempt))
                    continue
            except Exception as e:
                if attempt < 6:
                    time.sleep(5)
                    continue
                else:
                    if DEBUG_MODE:
                        print(f"  ⚠ DeepSeek API error: {e}")
                    return ""

        return ""

    def generate_response_gpt(
        self,
        system: str,
        user: str,
        temperature: float = 0.0,
        max_tokens: int = 360
    ) -> str:
        """GPT-4o-mini API call"""
        headers = {
            "Content-Type": "application/json",
            "api-key": AZURE_GPT_KEY
        }

        payload = {
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user}
            ],
            "max_tokens": max_tokens,
            "temperature": temperature
        }

        for attempt in range(7):
            try:
                response = requests.post(AZURE_GPT_ENDPOINT, headers=headers, json=payload, timeout=60)
                if response.status_code == 200:
                    res_json = response.json()
                    usage = res_json.get("usage", {})
                    tracker.add_usage_gpt(usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0))
                    return res_json["choices"][0]["message"]["content"].strip()
                elif response.status_code == 429:
                    time.sleep(5 * (2 ** attempt))
                    continue
            except Exception as e:
                if attempt < 6:
                    time.sleep(5)
                    continue
                else:
                    if DEBUG_MODE:
                        print(f"  ⚠ GPT API error: {e}")
                    return ""

        return ""


    # ========================================================================
    # STAGE 0: STRICT TRIAGE
    # ========================================================================

    def generate_conversational_response(self, question: str, history_lines: List[str]) -> str:
        history_block = ""
        if history_lines:
            recent = "\n".join(history_lines[-6:])
            history_block = f"\nCONVERSATION HISTORY:\n{recent}\n"

        prompt = f"""You are having a natural conversation with a user.

{history_block}USER: {question}

Generate a brief, friendly, conversational response (under {MAX_WORDS_CONV} words).
- Match the tone of the conversation
- Be natural and human-like
- Acknowledge what they said appropriately

RESPONSE:"""

        return self.generate_response_gpt("You are a helpful assistant.", prompt, temperature=0.3, max_tokens=160)

    def generate_strict_idk_response(self, question: str) -> str:
        """STRICT IDK: Short, clear response when NO contexts available"""
        prompt = f"""NO information is available to answer this question.

USER QUESTION: {question}

Generate a SHORT, clear response (under {MAX_WORDS_UNANSWERABLE} words) that:
- States clearly that the information is not available
- Is concise and direct
- Uses SIMPLE language

🚫 STRICTLY FORBIDDEN - DO NOT USE THESE PHRASES:
- "I don't know"
- "I do not know"
- "I'm not sure"
- "I am not sure"
- "I'm uncertain"
- "I cannot say"

✅ USE INSTEAD:
- "The information is not available."
- "No information found."
- "This information is not accessible."

RESPONSE:"""

        return self.generate_response_gpt("You are a helpful assistant.", prompt, temperature=0.0, max_tokens=100)


    # ========================================================================
    # STAGE 1: SPAN EXTRACTION (DeepSeek V3)
    # ========================================================================

    def extract_relevant_spans_with_retry(
        self,
        question: str,
        contexts: List[Dict],
        history_lines: List[str]
    ) -> List[Dict]:
        for attempt in range(1, SPAN_RETRY_ATTEMPTS + 1):
            if attempt > 1:
                if DEBUG_MODE:
                    print(f"  ├─ Span retry {attempt}/{SPAN_RETRY_ATTEMPTS}")
                tracker.add_span_retry()

            spans = self.extract_relevant_spans_cot(question, contexts, history_lines, attempt)
            if spans and len(spans) >= MIN_SPANS:
                return spans

        if DEBUG_MODE:
            print("  ├─ Fallback to top 3 passages")
        return contexts[:3]

    def extract_relevant_spans_cot(
        self,
        question: str,
        contexts: List[Dict],
        history_lines: List[str],
        attempt: int = 1
    ) -> List[Dict]:
        passages_block = ""
        for i, ctx in enumerate(contexts[:N_PASSAGES], 1):
            text = ctx.get("text", "").strip()
            relevance_note = " [MOST RELEVANT]" if i == 1 else " [HIGHLY RELEVANT]" if i == 2 else ""
            passages_block += f"\n\nPASSAGE {i}{relevance_note}:\n{text}"

        history_block = ""
        if history_lines:
            recent = "\n".join(history_lines[-6:])
            history_block = f"\n\nCONVERSATION HISTORY:\n{recent}\n"

        urgency = "⚠️ CRITICAL: The answer MUST be in the passages - extract it!" if attempt > 1 else ""

        span_prompt = f"""TASK: Extract sentences that answer the question.

🎯 CERTAINTY RULE: Since passages are provided, the answer MUST exist in them.

{urgency}
{history_block}QUESTION: {question}
{passages_block}

STRICT EXTRACTION RULES:
1. The answer IS in the passages - find it
2. Extract VERBATIM sentences (copy exactly)
3. PRIORITIZE earlier passages
4. Include ALL sentences with:
   - Proper nouns (names, places, organizations)
   - Numbers (dates, quantities, measurements)
   - Key facts needed for complete answer
5. Better to include 1 extra sentence than miss key entity
6. Max {MAX_SPANS} sentences
7. Empty extractedSpans ONLY if truly nothing relevant exists

Return JSON:
{{
  "reasoning": "Brief explanation",
  "extractedSpans": [
    {{"passageId": 1, "sentence": "exact sentence"}}
  ]
}}"""

        try:
            response = self.generate_response_deepseek("You are a helpful assistant.", span_prompt, temperature=0.0, max_tokens=800)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                spans = result.get("extractedSpans", [])

                if DEBUG_MODE and attempt == 1:
                    print(f"  ├─ Extracted {len(spans)} spans (DeepSeek)")

                if spans:
                    filtered = []
                    for span in spans[:MAX_SPANS]:
                        filtered.append({
                            "text": span["sentence"],
                            "document_id": f"passage_{span['passageId']}",
                            "source": "cot_extracted",
                            "reference": True
                        })
                    return filtered
        except Exception as e:
            if DEBUG_MODE:
                print(f"  ⚠ Span extraction failed: {e}")

        return []

    # ========================================================================
    # STAGE 2: NATURAL GENERATION (GPT-4o-mini)
    # ========================================================================

    def generate_natural_answer(
        self,
        spans: List[Dict],
        question: str,
        history_lines: List[str],
        temperature: float
    ) -> str:
        """Natural generation with GPT-4o-mini"""
        spans_block = "\n\nEXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            text = span.get("text", "").strip()
            spans_block += f"{i}. {text}\n"

        history_block = ""
        if history_lines:
            recent = "\n".join(history_lines[-6:])
            history_block = f"\nCONVERSATION HISTORY:\n{recent}\n"

        instruction = f"""Generate a natural, accurate answer using ONLY the EXTRACTED FACTS below.

🛡️ CRITICAL RULES - GROUNDING:
✅ USE ONLY information from EXTRACTED FACTS
✅ Answer MUST be 100% based on the facts provided
✅ If facts don't fully answer the question, acknowledge limitation briefly
❌ ABSOLUTELY NO information from general knowledge
❌ ABSOLUTELY NO assumptions beyond the facts
❌ ABSOLUTELY NO inferences not directly stated

🎯 QUALITY GUIDELINES:
❌ FORBIDDEN PHRASES - DO NOT USE:
   - "based on", "according to", "the documents state", "the information shows"
   - "I don't know", "I'm not sure", "I cannot say"
   - "It appears", "It seems", "Possibly", "Maybe"

✅ WRITE DIRECTLY: State facts naturally as definite statements (but ONLY from EXTRACTED FACTS)

📝 EXTRACTIVENESS REQUIREMENTS:
✅ COPY VERBATIM: Proper nouns, numbers, dates, technical terms, specific claims
✅ Use EXACT 4-6 word phrases for key facts
✅ Paraphrase ONLY connecting words and transitions
✅ Target: ~30-40% verbatim overlap with facts
   - Below 20%: Risk of hallucination
   - Above 60%: Sounds robotic
   - Sweet spot (30-40%): Natural + safe

📏 LENGTH & COMPLETENESS:
- Target: {SUGGESTED_TARGET}w (range: {SUGGESTED_MIN}-{SUGGESTED_MAX}w)
- Include ALL key facts from EXTRACTED FACTS
- Be complete, natural, and conversational

{history_block}{spans_block}
QUESTION: {question}

ANSWER (using ONLY the EXTRACTED FACTS above):"""

        answer = self.generate_response_gpt("You are a helpful assistant.", instruction, temperature=temperature, max_tokens=450)
        return normalize_idk_partial_safe(clean_model_output(answer))


    # ========================================================================
    # STAGE 3: JUDGES
    # ========================================================================

    def technical_judge_comparison(
        self,
        v1: str,
        v2: str,
        question: str,
        spans: List[Dict]
    ) -> Dict:
        """Technical Judge with DeepSeek V3 (cheaper)"""
        v1_wc = word_count(v1)
        v2_wc = word_count(v2)
        v1_extr = calculate_extractiveness(v1, spans)
        v2_extr = calculate_extractiveness(v2, spans)

        spans_block = "\nEXTRACTED FACTS:\n"
        for i, span in enumerate(spans[:5], 1):
            spans_block += f"{i}. {span.get('text', '')[:100]}...\n"

        judge_prompt = f"""Technical quality comparison.

QUESTION: {question}
{spans_block}

VERSION A:
{v1}
Stats: {v1_wc}w, {v1_extr:.0%} extractiveness

VERSION B:
{v2}
Stats: {v2_wc}w, {v2_extr:.0%} extractiveness

EVALUATE:
1. Completeness (all key facts present?)
2. Length (50-150w ideal)
3. Naturalness (sounds human?)
4. Factual grounding (no hallucinations? uses exact phrases for key facts?)
5. Extractiveness balance (30-40% is ideal)

Return JSON:
{{
  "winner": "A|B",
  "reason": "Brief explanation",
  "score_A": 0-10,
  "score_B": 0-10
}}"""

        try:
            response = self.generate_response_deepseek("You are a helpful assistant.", judge_prompt, temperature=0.0, max_tokens=200)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                winner = result.get("winner", "B").upper()
                tracker.add_tech_judge_result(v1_better=(winner == "A"))
                return {
                    "winner": winner,
                    "reason": result.get("reason", ""),
                    "score_A": result.get("score_A", 5),
                    "score_B": result.get("score_B", 5)
                }
        except:
            pass

        tracker.add_tech_judge_result(v1_better=False)
        return {"winner": "B", "reason": "Default", "score_A": 5, "score_B": 6}

    def user_satisfaction_judge_comparison(
        self,
        v1: str,
        v2: str,
        question: str,
        contexts: List[Dict],
        history_lines: List[str]
    ) -> Dict:
        """User Satisfaction Judge with GPT-4o-mini (better at user perspective)"""
        history_block = ""
        if history_lines:
            history_block = f"\nCONTEXT: {' | '.join(history_lines[-4:])}\n"

        judge_prompt = f"""You are a USER who asked a question. Which answer do you prefer?

{history_block}YOUR QUESTION: {question}

ANSWER A:
{v1}

ANSWER B:
{v2}

As a user, consider:
1. Clarity
2. Completeness
3. Naturalness
4. Directness

Return JSON:
{{
  "preferred": "A|B",
  "reason": "Brief explanation",
  "confidence": "HIGH|MEDIUM|LOW"
}}"""

        try:
            response = self.generate_response_gpt("You are a helpful assistant.", judge_prompt, temperature=0.0, max_tokens=200)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                preferred = result.get("preferred", "A").upper()
                tracker.add_usersat_judge_result(v1_better=(preferred == "A"))
                return {
                    "preferred": preferred,
                    "reason": result.get("reason", ""),
                    "confidence": result.get("confidence", "MEDIUM")
                }
        except:
            pass

        tracker.add_usersat_judge_result(v1_better=True)
        return {"preferred": "A", "reason": "Default", "confidence": "LOW"}


    # ========================================================================
    # STAGE 4: PICK BEST VERSION
    # ========================================================================

    def pick_best_version(
        self,
        v1: str,
        v2: str,
        tech_judgment: Dict,
        usersat_judgment: Optional[Dict]
    ) -> str:
        scores = {"A": 0, "B": 0}

        tech_score_A = tech_judgment.get("score_A", 5)
        tech_score_B = tech_judgment.get("score_B", 5)
        scores["A"] += tech_score_A * 0.4
        scores["B"] += tech_score_B * 0.4

        if usersat_judgment:
            usersat_preferred = usersat_judgment.get("preferred", "A")
            usersat_conf = usersat_judgment.get("confidence", "MEDIUM")

            conf_weight = {"HIGH": 1.0, "MEDIUM": 0.7, "LOW": 0.4}.get(usersat_conf, 0.7)
            if usersat_preferred == "A":
                scores["A"] += 6.0 * conf_weight
            else:
                scores["B"] += 6.0 * conf_weight
        else:
            scores["A"] += 3.0

        chosen = "A" if scores["A"] >= scores["B"] else "B"

        if DEBUG_MODE:
            print(f"  ├─ Scores: A={scores['A']:.1f}, B={scores['B']:.1f} → Chose {chosen}")

        tracker.add_choice(chose_v1=(chosen == "A"))

        return v1 if chosen == "A" else v2

    # ========================================================================
    # STAGE 5: MICRO-ADJUSTMENTS (GPT-4o-mini)
    # ========================================================================

    def apply_micro_adjustments(
        self,
        answer: str,
        question: str,
        spans: List[Dict]
    ) -> str:
        if not ENABLE_FINAL_MICRO_ADJUSTMENTS or is_pure_idk(answer):
            return answer

        wc = word_count(answer)

        needs_adjustment = False
        adjustment_reason = ""

        if wc < JUDGE_WC_MIN:
            needs_adjustment = True
            adjustment_reason = f"Too short ({wc}w) - expand slightly"
        elif wc > ADJUSTMENT_MAX_LENGTH:
            needs_adjustment = True
            adjustment_reason = f"Too long ({wc}w) - trim to {ADJUSTMENT_MAX_LENGTH}w"

        if not needs_adjustment:
            return answer

        if DEBUG_MODE:
            print(f"  ├─ Micro-adjustment: {adjustment_reason}")

        adjustment_prompt = f"""Make SMALL adjustments to this answer.

ISSUE: {adjustment_reason}

CURRENT ANSWER:
{answer}

TASK: Fix the issue with MINIMAL changes.
- Keep all facts
- Maintain naturalness
- Use exact phrases for key facts
- NO meta-phrases
- NO uncertain phrases

ADJUSTED ANSWER:"""

        adjusted = self.generate_response_gpt("You are a helpful assistant.", adjustment_prompt, temperature=GENERATION_TEMP_ADJUSTMENT, max_tokens=400)
        adjusted = normalize_idk_partial_safe(clean_model_output(adjusted))

        if adjusted and not is_pure_idk(adjusted) and word_count(adjusted) >= 30:
            tracker.add_micro_adjustment()
            return adjusted

        return answer


    # ========================================================================
    # MAIN PIPELINE
    # ========================================================================

    def process_task(self, task: Dict) -> Dict:
        """
        🎯 HYBRID PIPELINE:
        0. STRICT TRIAGE
        1. DeepSeek → Extract spans
        2. GPT-4o-mini → Generate 2 versions
        3. DeepSeek → Technical Judge
        4. GPT-4o-mini → User Satisfaction Judge
        5. GPT-4o-mini → Micro-adjust
        """
        inputs = task.get("input", []) or []
        question = self.get_current_question(task)
        history_lines = self.build_history_lines(inputs, current_question=question)

        contexts_original = task.get("contexts", [])[:N_PASSAGES]

        debug_steps = {} if DEBUG_MODE else {}

        # ========== STAGE 0: STRICT TRIAGE ==========
        if not contexts_original or len(contexts_original) == 0:
            if self.is_conversational(question):
                raw = self.generate_conversational_response(question, history_lines)
                raw = clean_model_output(raw)
                task["contexts"] = []
                task["predictions"] = [{"text": raw}]
                task["debug_info"] = {"type": "CONVERSATIONAL"}
                tracker.add_conversational()
                return task
            else:
                idk_response = self.generate_strict_idk_response(question)
                idk_response = clean_model_output(idk_response)
                task["contexts"] = []
                task["predictions"] = [{"text": idk_response}]
                task["debug_info"] = {"type": "STRICT_IDK", "reason": "no_contexts"}
                tracker.add_prediction(is_idk=True)
                tracker.add_strict_idk()
                return task

        # ========== HAS CONTEXTS → MANDATORY ANSWER GENERATION ==========
        debug_steps["has_contexts"] = True

        # STAGE 1: Extract spans (DeepSeek)
        if ENABLE_SPAN_EXTRACTION:
            spans = self.extract_relevant_spans_with_retry(question, contexts_original, history_lines)
        else:
            spans = contexts_original[:3]

        if not spans:
            spans = contexts_original[:3]

        debug_steps["spans_extracted"] = len(spans)

        # STAGE 2: Dual generation (GPT-4o-mini)
        if ENABLE_DUAL_GENERATION:
            v1 = self.generate_natural_answer(spans, question, history_lines, GENERATION_TEMP_V1)
            v2 = self.generate_natural_answer(spans, question, history_lines, GENERATION_TEMP_V2)

            tracker.add_dual_generation()
            debug_steps["dual_generation"] = True

            if DEBUG_MODE:
                v1_extr = calculate_extractiveness(v1, spans)
                v2_extr = calculate_extractiveness(v2, spans)
                print(f"  ├─ V1 (GPT, temp=0.0): {word_count(v1)}w, {v1_extr:.0%} extr")
                print(f"  ├─ V2 (GPT, temp=0.1): {word_count(v2)}w, {v2_extr:.0%} extr")

            if is_pure_idk(v1) and is_pure_idk(v2):
                force_answer = f"Based on the available information: {spans[0].get('text', '')[:100]}"
                if DEBUG_MODE:
                    print(f"  ⚠️ Both V1/V2 were IDK, forcing answer from spans")
                task["contexts"] = contexts_original
                task["predictions"] = [{"text": force_answer}]
                task["debug_info"] = {"type": "FORCED_ANSWER", "steps": debug_steps}
                tracker.add_prediction(is_idk=False)
                return task

            # STAGE 3: Judge both
            tech_judgment = None
            usersat_judgment = None

            if ENABLE_TECHNICAL_JUDGE:
                tech_judgment = self.technical_judge_comparison(v1, v2, question, spans)
                debug_steps["tech_judge_winner"] = tech_judgment.get("winner", "?")

            if ENABLE_USER_SATISFACTION_JUDGE and random.random() < USER_SAT_JUDGE_SAMPLE_RATE:
                usersat_judgment = self.user_satisfaction_judge_comparison(
                    v1, v2, question, contexts_original, history_lines
                )
                debug_steps["usersat_judge_preferred"] = usersat_judgment.get("preferred", "?")

            # STAGE 4: Pick best
            final_answer = self.pick_best_version(v1, v2, tech_judgment or {}, usersat_judgment)
            debug_steps["final_choice"] = "v1" if final_answer == v1 else "v2"
        else:
            final_answer = self.generate_natural_answer(spans, question, history_lines, GENERATION_TEMP_V1)

        # STAGE 5: Micro-adjustments (GPT-4o-mini)
        if ENABLE_FINAL_MICRO_ADJUSTMENTS:
            adjusted = self.apply_micro_adjustments(final_answer, question, spans)
            if adjusted != final_answer:
                debug_steps["micro_adjusted"] = True
                final_answer = adjusted

        # Finalize
        final_answer = final_answer.strip()
        debug_steps["final_wc"] = word_count(final_answer)
        debug_steps["final_extractiveness"] = calculate_extractiveness(final_answer, spans)

        is_idk = is_pure_idk(final_answer)

        task["contexts"] = contexts_original
        task["contexts_used"] = [s.get("text", "")[:100] for s in spans[:5]]
        task["predictions"] = [{"text": final_answer}]
        task["debug_info"] = {
            "type": "IDK" if is_idk else "FULL_ANSWER",
            "answer_preview": final_answer[:80],
            "steps": debug_steps
        }

        tracker.add_prediction(is_idk=is_idk)
        return task

    # ========================================================================
    # EXECUTION
    # ========================================================================

    def run_generation(self, input_file: str, output_file: str, max_tasks: int = None, reference_file: str = None):
        print("=" * 70)
        print("🎯 HYBRID - Starting...")
        print("=" * 70)

        if reference_file:
            tasks = self.load_tasks_with_ground_truth(input_file, reference_file, max_tasks)
        else:
            tasks = self.load_tasks(input_file, max_tasks)

        print(f"\n🚀 Processing {len(tasks)} tasks with {MAX_WORKERS} workers...\n")

        processed = [None] * len(tasks)

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_idx = {executor.submit(self.process_task, task): idx for idx, task in enumerate(tasks)}

            for future in tqdm(as_completed(future_to_idx), total=len(tasks), desc="Hybrid Pipeline"):
                idx = future_to_idx[future]
                try:
                    out_task = future.result()
                    processed[idx] = out_task

                    pred = out_task["predictions"][0]
                    wc = word_count(pred["text"])
                    debug = out_task.get("debug_info", {})
                    ans_type = debug.get("type", "?")
                    extr = debug.get("steps", {}).get("final_extractiveness", 0)
                    tqdm.write(f"{idx+1}: {wc}w, {extr:.0%} extr → {ans_type}")
                except Exception as e:
                    tqdm.write(f"⚠ Task {idx+1} error: {e}")
                    processed[idx] = tasks[idx]

        print(tracker.get_report())
        self.save_predictions(processed, output_file)

        print("=" * 70)
        print("✅ HYBRID COMPLETE!")
        print("=" * 70)

        return output_file


# ============================================================================
# MAIN ENTRY POINT
# ============================================================================

if __name__ == "__main__":
    if TEST_MODE:
        INPUT_FILE = "/content/drive/MyDrive/Project/generation/reference.jsonl"
        REFERENCE_FILE = "/content/drive/MyDrive/Project/generation/reference.jsonl"
        PREDICTIONS_OUTPUT = f"/content/drive/MyDrive/Project/outputs/test_hybrid_deepseek_gpt4o_pass{N_PASSAGES}_spans{MAX_SPANS}.jsonl"
        max_tasks = TEST_SAMPLE_SIZE

        print("=" * 70)
        print(f"🧪 TESTING MODE: {EXPERIMENT_NAME}")
        print(f"   Extraction: DeepSeek V3")
        print(f"   Generation: GPT-4o-mini")
        print(f"   Sample Size: {TEST_SAMPLE_SIZE} tasks")
        print(f"   Est. Cost: $0.65")
        print("=" * 70)
    else:
        INPUT_FILE = "/content/drive/MyDrive/Project/generation/reference.jsonl"
        REFERENCE_FILE = "/content/drive/MyDrive/Project/generation/reference.jsonl"
        PREDICTIONS_OUTPUT = f"/content/drive/MyDrive/Project/outputs/taskb_hybrid_deepseek_gpt4o_pass{N_PASSAGES}_spans{MAX_SPANS}.jsonl"
        max_tasks = None

        print("=" * 70)
        print("🚀 PRODUCTION MODE: Full Dataset (842 tasks)")
        print(f"   Extraction: DeepSeek V3")
        print(f"   Generation: GPT-4o-mini")
        print(f"   Est. Cost: $0.85")
        print("=" * 70)

    generator = HybridGeneration()
    generator.run_generation(INPUT_FILE, PREDICTIONS_OUTPUT, max_tasks=max_tasks, reference_file=REFERENCE_FILE)

    print("=" * 70)
    print("🎉 EXPERIMENT COMPLETE!")
    print("=" * 70)


In [ ]:
import os
import sys
import json
import re
import time
import random
import requests
import threading
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ============================================================================
# CONFIGURATION - Task B (Refined Pipeline: Extract → Generate 2 → Judge → Pick)
# ============================================================================

AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

# Core settings
N_PASSAGES = 5

# Pipeline stages (Refined)
ENABLE_SPAN_EXTRACTION = True
ENABLE_SPAN_RETRY = True
ENABLE_IDK_CALIBRATION = True
ENABLE_DUAL_GENERATION = True          # 🆕 Generate both prepolish + postpolish
ENABLE_TECHNICAL_JUDGE = True          # Judge both versions
ENABLE_USER_SATISFACTION_JUDGE = True  # Judge both versions
ENABLE_FINAL_MICRO_ADJUSTMENTS = True  # 🆕 Small final tweaks

# Span extraction
MIN_SPANS = 1
MAX_SPANS = 8
SPAN_RETRY_ATTEMPTS = 2

# Generation
GENERATION_TEMP_PREPOLISH = 0.0    # More conservative
GENERATION_TEMP_POSTPOLISH = 0.0   # Same (polish = deterministic)
GENERATION_TEMP_ADJUSTMENT = 0.1   # Slightly creative for micro-adjustments

# Word count targets
SUGGESTED_MIN = 80
SUGGESTED_TARGET = 90
SUGGESTED_MAX = 105

# Extractiveness
TARGET_EXTRACTIVENESS = 0.35  # 35% verbatim overlap

# Judge settings
JUDGE_WC_MIN = 50
JUDGE_WC_MAX = 150
JUDGE_EXTR_MIN = 0.15

# User satisfaction judge
USER_SAT_JUDGE_SAMPLE_RATE = 0.50  # Judge 50% of tasks

# IDK Calibration
IDK_COVERAGE_THRESHOLD_FACTOID = 0.65
IDK_COVERAGE_THRESHOLD_EXPLANATION = 0.35
IDK_COVERAGE_THRESHOLD_GENERAL = 0.45
IDK_MIN_SPANS_FOR_RETRY = 3
IDK_PHRASE = "I do not have specific information"

# Conversational
MAX_WORDS_CONV = 50

# Adjustment criteria
ADJUSTMENT_MIN_WORD_DIFF = 10  # Min word count difference to trigger adjustment
ADJUSTMENT_MAX_LENGTH = 150    # Max length after adjustments

DEBUG_MODE = True
MAX_WORKERS = 5

random.seed(42)

# ============================================================================
# STATISTICS TRACKER
# ============================================================================

class StatsTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.idk_count = 0
        self.full_answer_count = 0
        self.conversational_count = 0
        self.span_retry_count = 0
        self.dual_generation_count = 0
        self.chose_prepolish = 0
        self.chose_postpolish = 0
        self.tech_judge_prepolish_wins = 0
        self.tech_judge_postpolish_wins = 0
        self.usersat_judge_prepolish_wins = 0
        self.usersat_judge_postpolish_wins = 0
        self.micro_adjustments_count = 0
        self.idk_calibration_success = 0
        self.idk_calibration_count = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt_tokens, completion_tokens):
        with self.lock:
            self.total_input += prompt_tokens
            self.total_output += completion_tokens

    def add_prediction(self, is_idk: bool):
        with self.lock:
            if is_idk:
                self.idk_count += 1
            else:
                self.full_answer_count += 1

    def add_conversational(self):
        with self.lock:
            self.conversational_count += 1

    def add_span_retry(self):
        with self.lock:
            self.span_retry_count += 1

    def add_dual_generation(self):
        with self.lock:
            self.dual_generation_count += 1

    def add_choice(self, chose_prepolish: bool):
        with self.lock:
            if chose_prepolish:
                self.chose_prepolish += 1
            else:
                self.chose_postpolish += 1

    def add_tech_judge_result(self, prepolish_better: bool):
        with self.lock:
            if prepolish_better:
                self.tech_judge_prepolish_wins += 1
            else:
                self.tech_judge_postpolish_wins += 1

    def add_usersat_judge_result(self, prepolish_better: bool):
        with self.lock:
            if prepolish_better:
                self.usersat_judge_prepolish_wins += 1
            else:
                self.usersat_judge_postpolish_wins += 1

    def add_micro_adjustment(self):
        with self.lock:
            self.micro_adjustments_count += 1

    def add_idk_calibration(self, success: bool):
        with self.lock:
            self.idk_calibration_count += 1
            if success:
                self.idk_calibration_success += 1

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        report = "=" * 70 + "\n"
        report += "🎯 TASK B - REFINED PIPELINE (Extract → Generate 2 → Judge → Pick)\n"
        report += "=" * 70 + "\n"
        report += f"Token Usage:\n"
        report += f"  Input: {self.total_input:,}\n"
        report += f"  Output: {self.total_output:,}\n"
        report += f"  Cost: ${cost:.4f}\n"

        total_preds = self.idk_count + self.full_answer_count + self.conversational_count
        report += f"\nPredictions (Total: {total_preds}):\n"

        if total_preds > 0:
            idk_pct = (self.idk_count / total_preds * 100)
            ans_pct = (self.full_answer_count / total_preds * 100)
            conv_pct = (self.conversational_count / total_preds * 100)

            report += f"  Full Answers: {self.full_answer_count:4d} ({ans_pct:5.1f}%)\n"
            report += f"  IDK Responses: {self.idk_count:4d} ({idk_pct:5.1f}%)\n"
            report += f"  Conversational: {self.conversational_count:4d} ({conv_pct:5.1f}%)\n"

        if self.span_retry_count > 0:
            report += f"\nSpan Extraction Retries: {self.span_retry_count}\n"

        if self.idk_calibration_count > 0:
            success_rate = (self.idk_calibration_success / self.idk_calibration_count) * 100
            report += f"\nIDK Calibration:\n"
            report += f"  Triggered: {self.idk_calibration_count:4d}\n"
            report += f"  Successful: {self.idk_calibration_success:4d} ({success_rate:5.1f}%)\n"

        if self.dual_generation_count > 0:
            report += f"\n🆕 Dual Generation (Prepolish + Postpolish):\n"
            report += f"  Generated Both Versions: {self.dual_generation_count:4d}\n"

            total_choices = self.chose_prepolish + self.chose_postpolish
            if total_choices > 0:
                pre_pct = (self.chose_prepolish / total_choices * 100)
                post_pct = (self.chose_postpolish / total_choices * 100)
                report += f"\n  Final Choice:\n"
                report += f"    Prepolish (Natural): {self.chose_prepolish:4d} ({pre_pct:5.1f}%)\n"
                report += f"    Postpolish (Polished): {self.chose_postpolish:4d} ({post_pct:5.1f}%)\n"

        tech_total = self.tech_judge_prepolish_wins + self.tech_judge_postpolish_wins
        if tech_total > 0:
            report += f"\n📊 Technical Judge Results:\n"
            report += f"  Prepolish Wins: {self.tech_judge_prepolish_wins:4d} ({self.tech_judge_prepolish_wins/tech_total*100:5.1f}%)\n"
            report += f"  Postpolish Wins: {self.tech_judge_postpolish_wins:4d} ({self.tech_judge_postpolish_wins/tech_total*100:5.1f}%)\n"

        usersat_total = self.usersat_judge_prepolish_wins + self.usersat_judge_postpolish_wins
        if usersat_total > 0:
            report += f"\n👤 User Satisfaction Judge Results:\n"
            report += f"  Prepolish Wins: {self.usersat_judge_prepolish_wins:4d} ({self.usersat_judge_prepolish_wins/usersat_total*100:5.1f}%)\n"
            report += f"  Postpolish Wins: {self.usersat_judge_postpolish_wins:4d} ({self.usersat_judge_postpolish_wins/usersat_total*100:5.1f}%)\n"

        if self.micro_adjustments_count > 0:
            report += f"\n🔧 Micro-Adjustments Applied: {self.micro_adjustments_count}\n"

        report += "\n" + "=" * 70
        return report

tracker = StatsTracker()

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def word_count(text: str) -> int:
    return len([w for w in (text or "").strip().split() if w])

def safe_strip_quotes(s: str) -> str:
    s = (s or "").strip()
    if len(s) >= 2 and ((s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'")):
        return s[1:-1].strip()
    return s

def clean_model_output(text: str) -> str:
    t = safe_strip_quotes((text or "").strip())
    t = re.sub(r'(Agent|Assistant):', '', t.strip(), flags=re.IGNORECASE)
    t = re.sub(r'\s{3,}', ' ', t)
    t = re.sub(r'\n+', '\n', t)
    return t.strip()

def is_pure_idk(text: str) -> bool:
    t = (text or "").strip()
    if not t:
        return False
    low = t.lower()
    canonical = IDK_PHRASE.lower()
    if re.sub(r'[^\w\s]', '', low.strip()) == re.sub(r'[^\w\s]', '', canonical.strip()):
        return True
    idk_indicators = [
        "i don't know", "i do not know", "do not have enough information",
        "don't have enough information", "do not have specific information",
        "don't have specific information", "insufficient information"
    ]
    if any(ind in low for ind in idk_indicators) and word_count(t) <= 24:
        return True
    return False

def normalize_idk_partial_safe(text: str) -> str:
    t = (text or "").strip()
    if not t:
        return t
    if is_pure_idk(t):
        return IDK_PHRASE
    return t

def get_ngrams(text: str, n: int) -> set:
    words = text.lower().split()
    return set(" ".join(words[i:i+n]) for i in range(len(words)-n+1))

def calculate_extractiveness(answer: str, contexts: List[Dict]) -> float:
    answer_grams = get_ngrams(answer, 4)
    if not answer_grams:
        return 0.0
    passage_text = " ".join(c.get("text", "") for c in contexts)
    passage_grams = get_ngrams(passage_text, 4)
    overlap = len(answer_grams & passage_grams)
    return overlap / len(answer_grams) if answer_grams else 0.0

def calculate_keyword_coverage(question: str, spans: List[Dict]) -> float:
    q_keywords = set(re.findall(r'\w{4,}', question.lower()))
    if not q_keywords:
        return 0.0
    span_text = " ".join(s.get("text", "").lower() for s in spans)
    overlap = sum(1 for kw in q_keywords if kw in span_text)
    return overlap / len(q_keywords)

def detect_question_type(question: str) -> str:
    q_lower = question.lower().strip()

    factoid_patterns = [
        r'\bwho\b', r'\bwhat\b', r'\bwhen\b', r'\bwhere\b',
        r'\bhow many\b', r'\bhow much\b', r'\bwhich\b',
        r'\bname\b', r'\blist\b'
    ]
    if any(re.search(p, q_lower) for p in factoid_patterns):
        return "factoid"

    explanation_patterns = [
        r'\bwhy\b', r'\bhow does\b', r'\bhow did\b',
        r'\bexplain\b', r'\bdescribe\b', r'\bthe reason\b'
    ]
    if any(re.search(p, q_lower) for p in explanation_patterns):
        return "explanation"

    return "general"

def get_adaptive_idk_threshold(question: str) -> float:
    q_type = detect_question_type(question)
    thresholds = {
        "factoid": IDK_COVERAGE_THRESHOLD_FACTOID,
        "explanation": IDK_COVERAGE_THRESHOLD_EXPLANATION,
        "general": IDK_COVERAGE_THRESHOLD_GENERAL
    }
    return thresholds.get(q_type, IDK_COVERAGE_THRESHOLD_GENERAL)

# ============================================================================
# MAIN GENERATION CLASS - REFINED PIPELINE
# ============================================================================

class TaskBGenerationRefined:
    def __init__(self):
        print("=" * 70)
        print("🎯 Task B - REFINED PIPELINE")
        print("=" * 70)
        print("Pipeline Flow:")
        print("  1️⃣  Extract relevant spans")
        print("  2️⃣  Generate TWO responses:")
        print("      • Prepolish (natural)")
        print("      • Postpolish (extractively polished)")
        print("  3️⃣  Judge BOTH with:")
        print("      • Technical Judge (quality metrics)")
        print("      • User Satisfaction Judge (50% sample)")
        print("  4️⃣  Pick best version")
        print("  5️⃣  Apply micro-adjustments if needed")
        print(f"\n⚙ Workers: {MAX_WORKERS}")
        print("=" * 70)

    def load_tasks(self, input_file: str, max_tasks: int = None) -> List[Dict]:
        print(f"📂 Loading {input_file}...")
        tasks = []
        with open(input_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    tasks.append(json.loads(line))
        if max_tasks:
            tasks = tasks[:max_tasks]
        print(f"✓ Loaded {len(tasks)} tasks")
        return tasks

    def save_predictions(self, tasks: List[Dict], output_file: str):
        print(f"💾 Saving {output_file}...")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        with open(output_file, 'w', encoding='utf-8') as f:
            for task in tasks:
                f.write(json.dumps(task, ensure_ascii=False) + '\n')
        print(f"✓ Saved {len(tasks)} predictions")

    def get_current_question(self, task: Dict) -> str:
        inputs = task.get("input", []) or []
        if not inputs:
            return ""
        for inp in reversed(inputs):
            if inp.get("speaker", "").lower() == "user":
                return (inp.get("text", "") or "").strip()
        return ""

    def is_conversational(self, question: str) -> bool:
        q_low = question.lower().strip()
        conversational_patterns = [
            r'\bthanks?\b', r'\bthank you\b', r'\bok\b',
            r'\balright\b', r'^yes\??$', r'^no\??$',
            r'\bwhat about\b', r'\bhow about\b'
        ]
        return any(re.match(p, q_low) for p in conversational_patterns)

    def build_history_lines(self, inputs: List[Dict], current_question: str) -> List[str]:
        history_lines = []
        for inp in inputs:
            speaker = inp.get("speaker", "").lower()
            text = inp.get("text", "").strip()
            if not text:
                continue
            if speaker == "user" and text == current_question:
                break
            if speaker == "user":
                history_lines.append(f"User: {text}")
        return history_lines

    # ========================================================================
    # STEP 1: SPAN EXTRACTION
    # ========================================================================

    def extract_relevant_spans_with_retry(
        self,
        question: str,
        contexts: List[Dict],
        history_lines: List[str]
    ) -> List[Dict]:
        """Εξάγει spans με retry logic"""
        for attempt in range(1, SPAN_RETRY_ATTEMPTS + 1):
            if attempt > 1:
                if DEBUG_MODE:
                    print(f"  ├─ Span retry {attempt}/{SPAN_RETRY_ATTEMPTS}")
                tracker.add_span_retry()

            spans = self.extract_relevant_spans_cot(question, contexts, history_lines, attempt)
            if spans and len(spans) >= MIN_SPANS:
                return spans

        if DEBUG_MODE:
            print("  ├─ Fallback to top 3 passages")
        return contexts[:3]

    def extract_relevant_spans_cot(
        self,
        question: str,
        contexts: List[Dict],
        history_lines: List[str],
        attempt: int = 1
    ) -> List[Dict]:
        """Εξάγει relevant spans με CoT"""
        passages_block = ""
        for i, ctx in enumerate(contexts[:N_PASSAGES], 1):
            text = ctx.get("text", "") or "".strip()
            relevance_note = " [MOST RELEVANT]" if i == 1 else " [HIGHLY RELEVANT]" if i == 2 else ""
            passages_block += f"\n\nPASSAGE {i}{relevance_note}:\n{text}"

        history_block = ""
        if history_lines:
            recent = "\n".join(history_lines[-3:])
            history_block = f"\n\nCONVERSATION HISTORY:\n{recent}\n"

        urgency = "Look carefully - the answer IS in the passages!" if attempt > 1 else ""

        span_prompt = f"""TASK: Extract sentences that answer the question.

{urgency}
{history_block}QUESTION: {question}
{passages_block}

RULES:
1. PRIORITIZE earlier passages
2. Extract VERBATIM sentences
3. Include ALL sentences with:
   - Proper nouns (names, places, organizations)
   - Numbers (dates, quantities, measurements)
   - Key facts needed for complete answer
4. Better to include 1 extra sentence than miss key entity
5. Max {MAX_SPANS} sentences
6. If unanswerable, return empty extractedSpans

Return JSON:
{{
  "reasoning": "Brief explanation",
  "extractedSpans": [
    {{"passageId": 1, "sentence": "exact sentence"}}
  ]
}}"""

        try:
            response = self.generate_response(span_prompt, temperature=0.0, max_tokens=800)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                spans = result.get("extractedSpans", [])

                if DEBUG_MODE and attempt == 1:
                    print(f"  ├─ Extracted {len(spans)} spans")

                if spans:
                    filtered = []
                    for span in spans[:MAX_SPANS]:
                        filtered.append({
                            "text": span["sentence"],
                            "documentId": f"passage_{span['passageId']}",
                            "source": "cot_extracted",
                            "reference": True
                        })
                    return filtered
        except Exception as e:
            if DEBUG_MODE:
                print(f"  ⚠ Span extraction failed: {e}")

        return []

    # ========================================================================
    # STEP 2: DUAL GENERATION (Prepolish + Postpolish)
    # ========================================================================

    def create_generation_prompt(
        self,
        spans: List[Dict],
        question: str,
        history_lines: List[str]
    ) -> str:
        """Δημιουργεί generation prompt"""
        spans_block = "\n\nEXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            text = span.get("text", "").strip()
            spans_block += f"{i}. {text}\n"

        history_block = ""
        if history_lines:
            recent = "\n".join(history_lines[-3:])
            history_block = f"\nCONTEXT:\n{recent}\n"

        instruction = f"""Generate a natural answer using the EXTRACTED FACTS.

CRITICAL - NO META-PHRASES:
❌ FORBIDDEN: "based on the extracted facts", "according to the information", "the documents state"
✅ CORRECT: Write directly as if you know the facts

REQUIREMENTS:
- Length: {SUGGESTED_TARGET}w (range: {SUGGESTED_MIN}-{SUGGESTED_MAX}w)
- Include ALL facts: names, numbers, dates
- Use EXACT wording for:
  * Proper nouns (copy verbatim)
  * Numbers (copy exactly)
  * Technical terms (copy verbatim)
- Paraphrase connecting words only
- Natural, conversational tone

{history_block}{spans_block}
QUESTION: {question}

ANSWER:"""

        return instruction

    def generate_prepolish_answer(
        self,
        spans: List[Dict],
        question: str,
        history_lines: List[str]
    ) -> str:
        """Παράγει PREPOLISH (natural) version"""
        prompt = self.create_generation_prompt(spans, question, history_lines)
        answer = self.generate_response(prompt, temperature=GENERATION_TEMP_PREPOLISH, max_tokens=450)
        return normalize_idk_partial_safe(clean_model_output(answer))

    def generate_postpolish_answer(
        self,
        spans: List[Dict],
        prepolish_answer: str
    ) -> str:
        """Παράγει POSTPOLISH (extractively polished) version"""
        if is_pure_idk(prepolish_answer):
            return prepolish_answer

        spans_block = "\nEXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            spans_block += f"{i}. {span.get('text', '')}\\n"

        polish_prompt = f"""{spans_block}
CURRENT ANSWER:
{prepolish_answer}

TASK: Revise to achieve {int(TARGET_EXTRACTIVENESS*100)}% verbatim overlap with EXTRACTED FACTS.

RULES:
- Keep EXACT phrases for names, numbers, dates (4-6 word spans)
- Paraphrase connecting words only
- Target {int(TARGET_EXTRACTIVENESS*100)}% of answer = exact phrases from FACTS
- Maintain natural flow
- Keep same length (~{word_count(prepolish_answer)}w)
- NO meta-phrases

REVISED ANSWER:"""

        polished = self.generate_response(polish_prompt, temperature=GENERATION_TEMP_POSTPOLISH, max_tokens=450)
        return normalize_idk_partial_safe(clean_model_output(polished))

    def calibrate_idk_response(
        self,
        answer: str,
        question: str,
        spans: List[Dict],
        history_lines: List[str]
    ) -> str:
        """IDK calibration - δοκιμάζει για partial answer"""
        if not ENABLE_IDK_CALIBRATION or answer != IDK_PHRASE:
            return answer

        if len(spans) < IDK_MIN_SPANS_FOR_RETRY:
            return answer

        coverage = calculate_keyword_coverage(question, spans)
        threshold = get_adaptive_idk_threshold(question)

        if coverage < threshold:
            return answer

        if DEBUG_MODE:
            print(f"  ├─ IDK calibration: coverage={coverage:.2f} > threshold={threshold:.2f}")

        retry_prompt = f"""The EXTRACTED FACTS contain useful information.

DO NOT refuse - provide what you CAN say based on the facts.
Even PARTIAL answers are valuable.

{self.create_generation_prompt(spans, question, history_lines)}"""

        retried = self.generate_response(retry_prompt, temperature=0.15, max_tokens=450)
        retried = normalize_idk_partial_safe(clean_model_output(retried))

        if retried and retried != IDK_PHRASE and word_count(retried) >= 30:
            if DEBUG_MODE:
                print(f"  ├─ ✅ Calibration success ({word_count(retried)}w)")
            tracker.add_idk_calibration(success=True)
            return retried
        else:
            tracker.add_idk_calibration(success=False)

        return answer

    # ========================================================================
    # STEP 3: JUDGE BOTH VERSIONS
    # ========================================================================

    def technical_judge_comparison(
        self,
        prepolish: str,
        postpolish: str,
        question: str,
        spans: List[Dict]
    ) -> Dict:
        """Technical Judge - συγκρίνει prepolish vs postpolish"""
        pre_wc = word_count(prepolish)
        post_wc = word_count(postpolish)
        pre_extr = calculate_extractiveness(prepolish, spans)
        post_extr = calculate_extractiveness(postpolish, spans)

        spans_block = "\nEXTRACTED FACTS:\n"
        for i, span in enumerate(spans[:5], 1):
            spans_block += f"{i}. {span.get('text', '')[:100]}...\\n"

        judge_prompt = f"""Technical quality comparison.

QUESTION: {question}
{spans_block}

VERSION A (Prepolish - Natural):
{prepolish}
Stats: {pre_wc}w, {pre_extr:.0%} extractiveness

VERSION B (Postpolish - Polished):
{postpolish}
Stats: {post_wc}w, {post_extr:.0%} extractiveness

EVALUATE:
1. Completeness (all key facts present?)
2. Length (50-150w ideal)
3. Extractiveness (15-45% ideal)
4. Factual grounding (no hallucinations?)

Return JSON:
{{
  "winner": "A|B",
  "reason": "Brief explanation",
  "score_A": 0-10,
  "score_B": 0-10
}}"""

        try:
            response = self.generate_response(judge_prompt, temperature=0.0, max_tokens=200)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                winner = result.get("winner", "B").upper()
                tracker.add_tech_judge_result(prepolish_better=(winner == "A"))
                return {
                    "winner": winner,
                    "reason": result.get("reason", ""),
                    "score_A": result.get("score_A", 5),
                    "score_B": result.get("score_B", 5)
                }
        except:
            pass

        # Default: prefer postpolish (more extractive)
        tracker.add_tech_judge_result(prepolish_better=False)
        return {"winner": "B", "reason": "Default (judge failed)", "score_A": 5, "score_B": 6}

    def user_satisfaction_judge_comparison(
        self,
        prepolish: str,
        postpolish: str,
        question: str,
        contexts: List[Dict],
        history_lines: List[str]
    ) -> Dict:
        """User Satisfaction Judge - συγκρίνει prepolish vs postpolish"""
        history_block = ""
        if history_lines:
            history_block = f"\\nCONTEXT: {' | '.join(history_lines[-2:])}\n"

        judge_prompt = f"""You are a USER who asked a question. Which answer do you prefer?

{history_block}YOUR QUESTION: {question}

ANSWER A (Natural):
{prepolish}

ANSWER B (Polished):
{postpolish}

As a user, consider:
1. Clarity
2. Completeness for your needs
3. Naturalness (does it sound human?)
4. Directness

Return JSON:
{{
  "preferred": "A|B",
  "reason": "Brief explanation",
  "confidence": "HIGH|MEDIUM|LOW"
}}"""

        try:
            response = self.generate_response(judge_prompt, temperature=0.0, max_tokens=200)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                preferred = result.get("preferred", "A").upper()
                tracker.add_usersat_judge_result(prepolish_better=(preferred == "A"))
                return {
                    "preferred": preferred,
                    "reason": result.get("reason", ""),
                    "confidence": result.get("confidence", "MEDIUM")
                }
        except:
            pass

        # Default: prefer prepolish (more natural)
        tracker.add_usersat_judge_result(prepolish_better=True)
        return {"preferred": "A", "reason": "Default (judge failed)", "confidence": "LOW"}

    # ========================================================================
    # STEP 4: PICK BEST VERSION
    # ========================================================================

    def pick_best_version(
        self,
        prepolish: str,
        postpolish: str,
        tech_judgment: Dict,
        usersat_judgment: Optional[Dict]
    ) -> str:
        """Επιλέγει καλύτερη version με βάση judges"""
        # Scoring system
        scores = {"A": 0, "B": 0}

        # Technical judge (weight: 0.4)
        tech_winner = tech_judgment.get("winner", "B")
        tech_score_A = tech_judgment.get("score_A", 5)
        tech_score_B = tech_judgment.get("score_B", 5)
        scores["A"] += tech_score_A * 0.4
        scores["B"] += tech_score_B * 0.4

        # User satisfaction judge (weight: 0.6 if available)
        if usersat_judgment:
            usersat_preferred = usersat_judgment.get("preferred", "A")
            usersat_conf = usersat_judgment.get("confidence", "MEDIUM")

            conf_weight = {"HIGH": 1.0, "MEDIUM": 0.7, "LOW": 0.4}.get(usersat_conf, 0.7)
            if usersat_preferred == "A":
                scores["A"] += 6.0 * conf_weight
            else:
                scores["B"] += 6.0 * conf_weight
        else:
            # No user satisfaction judge → slight preference for postpolish (extractiveness)
            scores["B"] += 3.0

        # Decision
        chosen = "A" if scores["A"] >= scores["B"] else "B"

        if DEBUG_MODE:
            print(f"  ├─ Scores: A={scores['A']:.1f}, B={scores['B']:.1f} → Chose {chosen}")

        tracker.add_choice(chose_prepolish=(chosen == "A"))

        return prepolish if chosen == "A" else postpolish

    # ========================================================================
    # STEP 5: MICRO-ADJUSTMENTS
    # ========================================================================

    def apply_micro_adjustments(
        self,
        answer: str,
        question: str,
        spans: List[Dict]
    ) -> str:
        """Κάνει small adjustments αν χρειάζεται"""
        if not ENABLE_FINAL_MICRO_ADJUSTMENTS or is_pure_idk(answer):
            return answer

        wc = word_count(answer)
        extr = calculate_extractiveness(answer, spans)

        # Check if adjustments needed
        needs_adjustment = False
        adjustment_reason = ""

        if wc < JUDGE_WC_MIN:
            needs_adjustment = True
            adjustment_reason = f"Too short ({wc}w) - expand slightly"
        elif wc > ADJUSTMENT_MAX_LENGTH:
            needs_adjustment = True
            adjustment_reason = f"Too long ({wc}w) - trim to {ADJUSTMENT_MAX_LENGTH}w"
        elif extr < 0.10:
            needs_adjustment = True
            adjustment_reason = f"Low extractiveness ({extr:.0%}) - add more exact phrases"

        if not needs_adjustment:
            return answer

        if DEBUG_MODE:
            print(f"  ├─ Micro-adjustment: {adjustment_reason}")

        adjustment_prompt = f"""Make SMALL adjustments to this answer.

ISSUE: {adjustment_reason}

CURRENT ANSWER:
{answer}

TASK: Fix the issue with MINIMAL changes.
- Keep all facts
- Maintain naturalness
- NO meta-phrases

ADJUSTED ANSWER:"""

        adjusted = self.generate_response(adjustment_prompt, temperature=GENERATION_TEMP_ADJUSTMENT, max_tokens=400)
        adjusted = normalize_idk_partial_safe(clean_model_output(adjusted))

        if adjusted and adjusted != IDK_PHRASE and word_count(adjusted) >= 30:
            tracker.add_micro_adjustment()
            return adjusted

        return answer

    # ========================================================================
    # MAIN PIPELINE
    # ========================================================================

    def process_task(self, task: Dict) -> Dict:
        """
        🎯 REFINED PIPELINE:
        1. Extract spans
        2. Generate 2 versions (prepolish + postpolish)
        3. Judge both with technical + user satisfaction
        4. Pick best
        5. Apply micro-adjustments
        """
        inputs = task.get("input", []) or []
        question = self.get_current_question(task)
        history_lines = self.build_history_lines(inputs, question)

        contexts_original = task.get("contexts", [])[:N_PASSAGES]

        debug_steps = {} if DEBUG_MODE else {}

        # ========== CASE 1: No gold docs ==========
        if not contexts_original or len(contexts_original) == 0:
            if self.is_conversational(question):
                # Conversational response
                prompt = f"Reply naturally (under {MAX_WORDS_CONV}w).\\n\\n"
                if history_lines:
                    prompt += "\\n".join(history_lines[-4:]) + "\\n"
                prompt += f"User: {question}"

                raw = self.generate_response(prompt, temperature=0.3, max_tokens=160)
                raw = clean_model_output(raw)

                task["contexts"] = []
                task["predictions"] = [{"text": raw}]
                task["debug_info"] = {"type": "CONVERSATIONAL"}
                tracker.add_conversational()
                return task
            else:
                # Unanswerable
                task["contexts"] = []
                task["predictions"] = [{"text": IDK_PHRASE}]
                task["debug_info"] = {"type": "UNANSWERABLE"}
                tracker.add_prediction(is_idk=True)
                return task

        # ========== CASE 2: Gold docs available - REFINED PIPELINE ==========

        # STEP 1: Extract spans
        if ENABLE_SPAN_EXTRACTION:
            spans = self.extract_relevant_spans_with_retry(question, contexts_original, history_lines)
        else:
            spans = contexts_original[:3]

        if not spans:
            spans = contexts_original[:3]

        debug_steps["spans_extracted"] = len(spans)

        # STEP 2: Dual generation
        if ENABLE_DUAL_GENERATION:
            # 2a. Generate PREPOLISH (natural)
            prepolish = self.generate_prepolish_answer(spans, question, history_lines)
            prepolish = self.calibrate_idk_response(prepolish, question, spans, history_lines)

            # 2b. Generate POSTPOLISH (extractively polished)
            postpolish = self.generate_postpolish_answer(spans, prepolish)

            tracker.add_dual_generation()
            debug_steps["dual_generation"] = True

            if DEBUG_MODE:
                print(f"  ├─ Prepolish: {word_count(prepolish)}w")
                print(f"  ├─ Postpolish: {word_count(postpolish)}w")

            # If both are IDK, return IDK
            if is_pure_idk(prepolish) and is_pure_idk(postpolish):
                task["contexts"] = contexts_original
                task["predictions"] = [{"text": IDK_PHRASE}]
                task["debug_info"] = {"type": "IDK", "steps": debug_steps}
                tracker.add_prediction(is_idk=True)
                return task

            # STEP 3: Judge both versions
            tech_judgment = None
            usersat_judgment = None

            if ENABLE_TECHNICAL_JUDGE:
                tech_judgment = self.technical_judge_comparison(
                    prepolish, postpolish, question, spans
                )
                debug_steps["tech_judge_winner"] = tech_judgment.get("winner", "?")

            if ENABLE_USER_SATISFACTION_JUDGE and random.random() < USER_SAT_JUDGE_SAMPLE_RATE:
                usersat_judgment = self.user_satisfaction_judge_comparison(
                    prepolish, postpolish, question, contexts_original, history_lines
                )
                debug_steps["usersat_judge_preferred"] = usersat_judgment.get("preferred", "?")

            # STEP 4: Pick best version
            final_answer = self.pick_best_version(
                prepolish, postpolish, tech_judgment or {}, usersat_judgment
            )
            debug_steps["final_choice"] = "prepolish" if final_answer == prepolish else "postpolish"
        else:
            # Fallback: single generation
            final_answer = self.generate_prepolish_answer(spans, question, history_lines)
            final_answer = self.calibrate_idk_response(final_answer, question, spans, history_lines)

        # STEP 5: Micro-adjustments
        if ENABLE_FINAL_MICRO_ADJUSTMENTS:
            adjusted = self.apply_micro_adjustments(final_answer, question, spans)
            if adjusted != final_answer:
                debug_steps["micro_adjusted"] = True
                final_answer = adjusted

        # Finalize
        final_answer = final_answer.strip()
        debug_steps["final_wc"] = word_count(final_answer)

        is_idk = is_pure_idk(final_answer)

        task["contexts"] = contexts_original
        task["contexts_used"] = [s.get("text", "")[:100] for s in spans[:5]]
        task["predictions"] = [{"text": final_answer}]
        task["debug_info"] = {
            "type": "IDK" if is_idk else "FULL_ANSWER",
            "answer_preview": final_answer[:80],
            "steps": debug_steps
        }

        tracker.add_prediction(is_idk=is_idk)
        return task

    # ========================================================================
    # EXECUTION
    # ========================================================================

    def run_generation(self, input_file: str, output_file: str, max_tasks: int = None):
        print("=" * 70)
        print("🎯 Task B REFINED - Starting...")
        print("=" * 70)

        tasks = self.load_tasks(input_file, max_tasks)
        processed = [None] * len(tasks)

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_idx = {executor.submit(self.process_task, task): idx for idx, task in enumerate(tasks)}

            for future in tqdm(as_completed(future_to_idx), total=len(tasks), desc="Refined Pipeline"):
                idx = future_to_idx[future]
                try:
                    out_task = future.result()
                    processed[idx] = out_task

                    pred = out_task["predictions"][0]
                    wc = word_count(pred["text"])
                    debug = out_task.get("debug_info", {})
                    ans_type = debug.get("type", "?")

                    tqdm.write(f"#{idx+1}: {wc}w [{ans_type}]")
                except Exception as e:
                    tqdm.write(f"⚠ Task {idx+1} error: {e}")
                    processed[idx] = tasks[idx]

        print(tracker.get_report())
        self.save_predictions(processed, output_file)
        print("=" * 70)
        print("✅ Task B REFINED COMPLETE!")
        print("=" * 70)
        return output_file

    def generate_response_with_system(
        self,
        system: str,
        user: str,
        temperature: float = 0.0,
        max_tokens: int = 360
    ) -> str:
        headers = {
            "Content-Type": "application/json",
            "api-key": AZURE_DS_KEY
        }

        payload = {
            "model": "DeepSeek-V3.2",
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user}
            ],
            "max_tokens": max_tokens,
            "temperature": temperature
        }

        for attempt in range(7):
            try:
                response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
                if response.status_code == 200:
                    res_json = response.json()
                    usage = res_json.get("usage", {})
                    tracker.add_usage(usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0))
                    return res_json["choices"][0]["message"]["content"].strip()
                elif response.status_code == 429:
                    time.sleep(5 * (2 ** attempt))
                    continue
            except Exception as e:
                time.sleep(5)

        return ""

    def generate_response(self, prompt: str, temperature: float = 0.0, max_tokens: int = 360) -> str:
        return self.generate_response_with_system(
            "You are a helpful assistant.",
            prompt,
            temperature,
            max_tokens
        )

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    INPUT_FILE = "/content/drive/MyDrive/Project/generation/reference.jsonl"
    PREDICTIONS_OUTPUT = "/content/drive/MyDrive/Project/outputs/task_b_refined_pipeline.jsonl"

    generator = TaskBGenerationRefined()
    generator.run_generation(INPUT_FILE, PREDICTIONS_OUTPUT, max_tasks=None)

    print("\n" + "=" * 70)
    print("🎯 REFINED PIPELINE SUMMARY")
    print("=" * 70)
    print("Pipeline Flow:")
    print("  1️⃣  Extract spans (with retry)")
    print("  2️⃣  Generate 2 versions (prepolish + postpolish)")
    print("  3️⃣  Judge both (technical + user satisfaction)")
    print("  4️⃣  Pick best version (weighted scoring)")
    print("  5️⃣  Micro-adjustments (if needed)")
    print("=" * 70)
